# NB-05_sintesis_final

Process Flow SAS: **Síntesis_Final** — `PFD-FzVr69cJ7GUTkjGn`

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text
import pyreadstat

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-05_sintesis_final.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## S2_13_Inicio_2

Carga la tabla principal de posiciones desde presíntesis, corrige códigos de contragente/instrumento por reglas de negocio históricas y acumula ajustes de reajustes, SIFMI, dividendos de hogares, DCV y patrimonios separados en BD_CTSI

*confianza: low · verificador: revise · SAS: PROC SQL CREATE/UPDATE/DELETE masivos + PROC DATASETS APPEND encadenados sobre tabla principal, con datasets WORK temporales de sesión (#tmp)*

In [ ]:
# ========= S2_13_Inicio_2 =========
# TRAE TABLA PRINCIPAL DESDE PRESÍNTESIS PARA COMENZAR EL PROCESO
# M-001: ruta absoluta reemplazada por ruta relativa al workspace
ruta_presintesis = Path("sasdata") / "BCCH" / "GEM_DCNI" / "02_CNSI" / "02_PRE_SINTESIS" / "BD_CTSI_CIERRE.sas7bdat"
bd_ctsi_presintesis, _ = pyreadstat.read_sas7bdat(str(ruta_presintesis))
# reemplazo estilo SAS: la tabla se recrea por completo -> DELETE FROM sin WHERE + append
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.BD_CTSI"))
    _log("DELETE TABLAS.dbo.BD_CTSI", res.rowcount)
bd_ctsi_presintesis.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("BD_CTSI cargada desde presintesis", bd_ctsi_presintesis)


In [ ]:
# COMPRIME TABLA PRINCIPAL (COMPRESS=YES es un atributo de almacenamiento SAS sin equivalente en SQL Server; no aplica)
# AGREGA COLUMNA DE PROC A BASE BD_CTSI + ajusta FUENTE a char(12)
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE TABLAS.dbo.BD_CTSI ADD PROC varchar(255) NULL"))
    conn.execute(text("ALTER TABLE TABLAS.dbo.BD_CTSI ALTER COLUMN FUENTE varchar(255) NULL"))


In [ ]:
# ELIMINA AÑO 2002 DE LA BASE
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.BD_CTSI WHERE AÑO = :anio"), {"anio": 2002})
    _log("DELETE BD_CTSI año 2002", res.rowcount)


In [ ]:
# ELIMINA STOCKS QUE NO SON DE LA CUENTA CORRESPONDIENTE
sql_delete_stocks = """
DELETE FROM TABLAS.dbo.BD_CTSI
WHERE C_CUENTA IN ('Bce Final','Bce Inicio','Financiera','Rec Precio','Rec Precio Reaj','Rec Volumen')
  AND C_SCN IN ('D.41','K.1','P.2','P.51','P.11','D.42','D.5','B.9','B.90','B.10.2','B.10.3','P.52','D.62','D.75','D.1')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_delete_stocks))
    _log("DELETE BD_CTSI stocks no correspondientes", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES 36912 a 36 en todos los sectores porque es lo mismo y la síntesis está con 36. ESTO AFECTA EL PATRIMONIO Y PAGO DE DIVIDENDOS
# cierre 2021: se deja contragente 36912 tal cual ya que ahora existe ese sector. Se cambia sector 363 a 36912, ya que no tenemos sector 363
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE IN (:c1)"), {"nuevo": "36912", "c1": "363"})
    _log("UPDATE C_CAGENTE 363->36912", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES 7 a 53 en todos los sectores porque es lo mismo. ESTO AFECTA EL PATRIMONIO Y PAGO DE DIVIDENDOS
sql_up_7_53 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE IN ('7','7.1','7.2')"
with engine.begin() as conn:
    res = conn.execute(text(sql_up_7_53), {"nuevo": "53"})
    _log("UPDATE C_CAGENTE 7->53", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES 512 a 511 en todos los sectores porque es lo mismo. ESTO AFECTA EL PATRIMONIO Y PAGO DE DIVIDENDOS
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE IN ('512')"), {"nuevo": "511"})
    _log("UPDATE C_CAGENTE 512->511", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES NACIONALES EN EMISIONES DE TITULO DE HOLDING Y CASAS MATRICES A 51
sql_up_holding = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_CAGENTE NOT IN ('6') AND SECTOR IN (37,36907) AND C_SCN IN ('AF.31','AF.32') AND C_ENTRADA='H'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_holding), {"nuevo": "53"})
    _log("UPDATE C_CAGENTE holding/casas matrices ->53", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES NACIONALES EN EMISIONES DE TITULO DE EMPRESAS A 53
sql_up_empresas = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_CAGENTE NOT IN ('6','54') AND SECTOR IN (51021,5101) AND C_SCN IN ('AF.31','AF.32') AND C_ENTRADA='H'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_empresas), {"nuevo": "53"})
    _log("UPDATE C_CAGENTE empresas ->53", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES en inversiones en af31 con CA 6 EN FONDOS MUTUOS A AF32
sql_up_af31_af32 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.32', N_SCN='Valores distintos de acciones a largo plazo'
WHERE C_CAGENTE='6' AND SECTOR IN (3390101,3390102) AND C_SCN IN ('AF.31') AND C_ENTRADA='D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af31_af32))
    _log("UPDATE AF.31->AF.32 fondos mutuos", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES EN AF41 ACTIVO DEL RM DESDE VACIO A 53
# valor faltante SAS ('') equivale a blanco/NULL en SQL Server
sql_up_af41_vacio = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE (C_CAGENTE IS NULL OR C_CAGENTE = '') AND SECTOR=6 AND C_SCN='AF.41' AND C_ENTRADA='D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af41_vacio), {"nuevo": "53"})
    _log("UPDATE AF.41 RM vacio->53", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES EN AF7 ACTIVO DE RESTO DE EMPRESAS CON CONTRAGENTE HOGARES
sql_up_af7 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_CAGENTE='511' AND SECTOR=51022 AND C_SCN='AF.7' AND C_ENTRADA='D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af7), {"nuevo": "53"})
    _log("UPDATE AF.7 hogares->53", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES EN AF5 ACTIVO DE FONDOS DE INVERSIÓN DESDE 0 A 53 (CIERRE 2021)
# valor faltante SAS ('') equivale a blanco/NULL en SQL Server
sql_up_af5_0 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE (C_CAGENTE IN ('0') OR C_CAGENTE IS NULL OR C_CAGENTE = '') AND SECTOR=339011 AND C_SCN='AF.5' AND C_ENTRADA='D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af5_0), {"nuevo": "53"})
    _log("UPDATE AF.5 fondos inversion 0->53", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTES EN AF5 ACTIVO DE FONDOS DE INVERSIÓN DESDE 339012 CORRESPONDIENTE A FONDOS DE INVERSIÓN PRIVADOS A CA 36912 (CIERRE 2021)
sql_up_af5_339012 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_CAGENTE IN ('339012') AND SECTOR=339011 AND C_SCN='AF.5' AND C_ENTRADA='D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af5_339012), {"nuevo": "36912"})
    _log("UPDATE AF.5 339012->36912", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (41,42,412 Y 413)
sql_up_gob_s1s19 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_CAGENTE IN ('S1','S19') AND SECTOR IN (41,42,412,413)
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_gob_s1s19), {"nuevo": "53"})
    _log("UPDATE gobierno S1/S19->53", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S11->51)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S11' AND SECTOR IN (41,42,412,413)"), {"nuevo": "51"})
    _log("UPDATE gobierno S11->51", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S12->3)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S12' AND SECTOR IN (41,42,412,413)"), {"nuevo": "3"})
    _log("UPDATE gobierno S12->3", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S121->31)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S121' AND SECTOR IN (41,42,412,413)"), {"nuevo": "31"})
    _log("UPDATE gobierno S121->31", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S122->321)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S122' AND SECTOR IN (41,42,412,413)"), {"nuevo": "321"})
    _log("UPDATE gobierno S122->321", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S123->331)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S123' AND SECTOR IN (41,42,412,413)"), {"nuevo": "331"})
    _log("UPDATE gobierno S123->331", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S13->4)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S13' AND SECTOR IN (41,42,412,413)"), {"nuevo": "4"})
    _log("UPDATE gobierno S13->4", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S131->41)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S131' AND SECTOR IN (41,42,412,413)"), {"nuevo": "41"})
    _log("UPDATE gobierno S131->41", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S14->511)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S14' AND SECTOR IN (41,42,412,413)"), {"nuevo": "511"})
    _log("UPDATE gobierno S14->511", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE CONTRAGENTE EN SECTOR GOBIERNO (S2->6)
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='S2' AND SECTOR IN (41,42,412,413)"), {"nuevo": "6"})
    _log("UPDATE gobierno S2->6", res.rowcount)


In [ ]:
# CAMBIA CÓDIGOS DE INSTRUMENTOS EN DEPÓSITOS AF29 ENTRE AFP Y FONDOS DE PENSIONES
sql_up_af29_1 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.612', N_SCN='Reserva de fondos de pensiones'
WHERE C_CAGENTE='34' AND SECTOR=361 AND C_SCN='AF.29' AND C_ENTRADA='D'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af29_1))
    _log("UPDATE AF.29->AF.612 D", res.rowcount)


In [ ]:
sql_up_af29_2 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.612', N_SCN='Reserva de fondos de pensiones'
WHERE C_CAGENTE='361' AND SECTOR=34 AND C_SCN='AF.29' AND C_ENTRADA='H'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af29_2))
    _log("UPDATE AF.29->AF.612 H", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTE 339011P
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_CAGENTE='339011_p'"), {"nuevo": "339011"})
    _log("UPDATE 339011_p->339011", res.rowcount)


In [ ]:
sql_up_36909 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE SECTOR=36909 AND C_SCN='AF.41' AND C_ENTRADA='D' AND C_CAGENTE IN ('321','36904')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_36909), {"nuevo": "53"})
    _log("UPDATE sector 36909 ->53", res.rowcount)


In [ ]:
sql_up_af5_512 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo WHERE C_SCN='AF.5' AND C_ENTRADA='H' AND C_CAGENTE='512'"
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af5_512), {"nuevo": "511"})
    _log("UPDATE AF.5 H 512->511", res.rowcount)


In [ ]:
# valor faltante SAS ('') equivale a blanco/NULL en SQL Server
sql_up_af5_352 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.5' AND C_ENTRADA='H' AND (C_CAGENTE IS NULL OR C_CAGENTE='') AND SECTOR=352
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af5_352), {"nuevo": "53"})
    _log("UPDATE AF.5 H vacio SECTOR=352 ->53", res.rowcount)


In [ ]:
sql_up_af5_339011_ = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.5' AND C_ENTRADA='H' AND C_CAGENTE='339011_' AND SECTOR=352
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af5_339011_), {"nuevo": "339011"})
    _log("UPDATE AF.5 H 339011_->339011", res.rowcount)


In [ ]:
# CIERRE 2021: PARA DEJAR BIEN CONTRAGENTE EN CUOTAS DE FONDOS
sql_up_af522 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.522' AND C_ENTRADA='D' AND C_CAGENTE='339012' AND SECTOR=339011
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af522), {"nuevo": "3390102"})
    _log("UPDATE AF.522 cuotas fondos", res.rowcount)


In [ ]:
# CIERRE 2022. CAMBIA C_CAGENTE EN AF7 ACTIVO DE BANCOS CON HOGARES A RESTO, YA QUE NO DEBE PASAR A SER PASIVO DE LOS HOGARES
sql_up_af7_bancos = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.7' AND C_ENTRADA='D' AND C_CAGENTE='511' AND SECTOR=321
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af7_bancos), {"nuevo": "53"})
    _log("UPDATE AF.7 bancos-hogares->resto", res.rowcount)


In [ ]:
# CIERRE 2023q2. CAMBIA C_CAGENTE EN AF41 ACTIVO DE FONDOS DE INVERSIÓN CON SEGUROS A RESTO
sql_up_af41_seguros = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.41' AND C_ENTRADA='D' AND C_CAGENTE IN ('352','339011') AND SECTOR=339011
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af41_seguros), {"nuevo": "53"})
    _log("UPDATE AF.41 seguros->resto", res.rowcount)


In [ ]:
# CIERRE 2023q2. CAMBIA C_CAGENTE EN AF29 ACTIVO DE FONDOS DE INVERSIÓN CON RM A BANCOS
sql_up_af29_rm = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.29' AND C_ENTRADA='D' AND C_CAGENTE IN ('6') AND SECTOR=339011 AND AÑO >= :anio
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af29_rm), {"nuevo": "321", "anio": 2023})
    _log("UPDATE AF.29 RM->bancos", res.rowcount)


In [ ]:
# CIERRE 2023q3. CAMBIA C_CAGENTE EN af32 PASIVO DE FONDOS DE INVERSIÓN CON RM A RESTO DE LA ECONOMÍA
sql_up_af32_rm = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.32' AND C_ENTRADA='H' AND C_CAGENTE IN ('6') AND SECTOR=339011 AND AÑO >= :anio
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af32_rm), {"nuevo": "53", "anio": 2023})
    _log("UPDATE AF.32 RM->resto economia", res.rowcount)


In [ ]:
# CIERRE 2023q4. CAMBIA C_CAGENTE EN af41 PASIVO DE SOCIEDADES CAUTIVAS CON GOBIERNO A BANCOS
sql_up_af41_cautivas = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = :nuevo
WHERE C_SCN='AF.41' AND C_ENTRADA='H' AND C_CAGENTE IN ('41') AND SECTOR=37
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af41_cautivas), {"nuevo": "321"})
    _log("UPDATE AF.41 cautivas->bancos", res.rowcount)


In [ ]:
# CIERRE 2025q4. CAMBIA af42 ACTIVO EN SECTOR 37 A af5
sql_up_af42_af5 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.5', C_CAGENTE='2', N_SCN='Acciones y otras participaciones de capital'
WHERE C_SCN='AF.42' AND C_ENTRADA='D' AND C_CAGENTE IN ('321') AND SECTOR=37
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af42_af5))
    _log("UPDATE AF.42->AF.5 sector 37", res.rowcount)


In [ ]:
# IMPUTA AJUSTES INICIALES DE SIFMI, DIVIDENDOS DE HOGARES Y REAJUSTES
# PROC APPEND: acumula tal cual; re-ejecutar duplica, igual que el SAS original
cols_append = ["RP_HH", "SIFMI", "REAJUSTES", "AJ_VARIOS", "FBCF_SF"]
for tabla in cols_append:
    df_tmp = pd.read_sql(text(f"SELECT * FROM TABLAS.dbo.{tabla}"), engine)
    df_tmp.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
    _log(f"APPEND BD_CTSI <- {tabla}", len(df_tmp))


In [ ]:
sql_up_proc0071 = """
UPDATE TABLAS.dbo.BD_CTSI SET PROC = :proc
WHERE FUENTE IN ('DI_RP_H','DI_Aj_SIFMI','DI_Aj_REAJ','AJ_DEP','AJ_K_RESTO')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_proc0071), {"proc": "0071"})
    _log("UPDATE PROC=0071", res.rowcount)


In [ ]:
# TRASPASA REAJUSTES DESDE CTA REC PRECIO REAJ A CUENTA FINANCIERA EN INST AF.29 Y AF.42 TODOS LOS SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_fin"))
sql_reaj_fin = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
       'Financiera' AS C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0001a' AS PROC
INTO #reaj_fin
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN IN ('AF.29','AF.42') AND T1.C_CUENTA='Rec Precio Reaj' AND T1.FUENTE='CI'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_reaj_fin))

work_conn.execute(text("DROP TABLE IF EXISTS #reaj_fin_2"))
sql_reaj_fin_2 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0001b' AS PROC
INTO #reaj_fin_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN IN ('AF.29','AF.42') AND T1.C_CUENTA='Rec Precio Reaj' AND T1.FUENTE='CI'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_reaj_fin_2))


In [ ]:
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_fin"))
_log("APPEND BD_CTSI <- REAJ_FIN", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_fin_2"))
_log("APPEND BD_CTSI <- REAJ_FIN_2", res.rowcount)
for t in ["#reaj_fin", "#reaj_fin_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA REAJUSTES DE INST AF.29 Y AF.42 DE SECTORES 34 Y 341 EN LA CTA FINANCIERA(H) CON CA HOGARES EN INST AF.612
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af612"))
sql_reaj_af612 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       '511' AS C_CAGENTE, 'Financiera' AS C_CUENTA, 'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, 'AF.612' AS C_SCN, 'Reserva de fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE, '0001c' AS PROC
INTO #reaj_af612
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR IN (34,341,342) AND T1.C_SCN IN ('AF.29','AF.42') AND T1.C_CUENTA='Rec Precio Reaj' AND T1.FUENTE='CI'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR
"""
work_conn.execute(text(sql_reaj_af612))

work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af612_2"))
sql_reaj_af612_2 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       '511' AS C_CAGENTE, T1.C_CUENTA, 'H' AS C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO, 'AF.612' AS C_SCN, 'Reserva de fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE, '0001b' AS PROC
INTO #reaj_af612_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR IN (34,341,342) AND T1.C_SCN IN ('AF.29','AF.42') AND T1.C_CUENTA='Rec Precio Reaj' AND T1.FUENTE='CI'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA
"""
work_conn.execute(text(sql_reaj_af612_2))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_af612"))
_log("APPEND BD_CTSI <- REAJ_AF612", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_af612_2"))
_log("APPEND BD_CTSI <- REAJ_AF612_2", res.rowcount)
for t in ["#reaj_af612", "#reaj_af612_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA REAJUSTES DE INST AF.29 DE SECTOR 3390101/3390102 EN LA CTA FINANCIERA(H) CON CA EMPRESAS (53) EN INST AF.521...debe crearse estructura de contragente porque todo el reajuste se va a empresas
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af52"))
sql_reaj_af52 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       '53' AS C_CAGENTE, 'Financiera' AS C_CUENTA, 'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       (CASE WHEN T1.SECTOR=3390101 THEN 'AF.521' ELSE 'AF.522' END) AS C_SCN,
       'Participaciones emitidas por fondos de inversión' AS N_SCN,
       'PS' AS FUENTE, '0001c' AS PROC
INTO #reaj_af52
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR IN (3390101,3390102) AND T1.C_SCN IN ('AF.29') AND T1.C_CUENTA='Rec Precio Reaj' AND T1.FUENTE='CI'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR
"""
work_conn.execute(text(sql_reaj_af52))


In [ ]:
# CALCULA REAJUSTES A LLEVAR A CTA FINANCIERA CORRESPONDIENTE A SECTOR HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af52_h"))
sql_reaj_af52_h = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       '511' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       (T1.DATO * T2.DATO) AS DATO, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
INTO #reaj_af52_h
FROM #reaj_af52 T1
INNER JOIN TABLAS.dbo.DEP_HH_FM T2 ON T1.AÑO=T2.AÑO AND T1.TRIM=T2.TRIM AND T1.SECTOR=T2.SECTOR
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_reaj_af52_h))

# CALCULA REAJUSTES A LLEVAR A CTA FINANCIERA CORRESPONDIENTE A SECTOR RESTO
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af52_r"))
sql_reaj_af52_r = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       (T1.DATO * (1 - T2.DATO)) AS DATO, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
INTO #reaj_af52_r
FROM #reaj_af52 T1
INNER JOIN TABLAS.dbo.DEP_HH_FM T2 ON T1.AÑO=T2.AÑO AND T1.TRIM=T2.TRIM AND T1.SECTOR=T2.SECTOR
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_reaj_af52_r))


In [ ]:
# junta tablas para ca 511 y 53 (append server-side de una #tmp a otra #tmp)
cols_reaj_af52 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
work_conn.execute(text(f"INSERT INTO #reaj_af52_h ({cols_reaj_af52}) SELECT {cols_reaj_af52} FROM #reaj_af52_r"))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_af52_h"))
_log("APPEND BD_CTSI <- REAJ_AF52_H", res.rowcount)


In [ ]:
# elimina reajustes sectorizados en ffmm
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af52_2"))
sql_reaj_af52_2 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
       'Rec Precio Reaj' AS C_CUENTA, T1.C_ENTRADA,
       T1.DATO*-1 AS DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0001b' AS PROC
INTO #reaj_af52_2
FROM #reaj_af52_h T1
"""
work_conn.execute(text(sql_reaj_af52_2))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_af52_2"))
_log("APPEND BD_CTSI <- REAJ_AF52_2", res.rowcount)

for t in ["#reaj_af52", "#reaj_af52_h", "#reaj_af52_r", "#reaj_af52_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA REAJUSTES QUE PROVIENEN DE AF29 EN FONDOS MUTUOS EN OP FINANCIERA DE LA CARTERA DE LOS INVERSIONISTAS. RESTA IMPUTACIÓN DE REC PRECIO REAJUSTE
# imputa op financiera en activo af521 y af522 de los sectores excepto, hogares y resto que ya están recogidos en el pasivo de ffmm. tampoco considera el resto del mundo para no alterar la ci del sector ni su operación financiera
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af52_3"))
sql_reaj_af52_3 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(8)))) AS C_CAGENTE,
       'Financiera' AS C_CUENTA, T1.C_ENTRADA,
       T1.DATO AS DATO,
       (CASE WHEN T1.SECTOR=3390101 THEN 'AF.521' ELSE 'AF.522' END) AS C_SCN,
       'Participaciones emitidas por fondos de inversión' AS N_SCN,
       'PS' AS FUENTE, '0001h' AS PROC
INTO #reaj_af52_3
FROM TABLAS.dbo.REAJUSTES T1
WHERE T1.SECTOR IN (3390101, 3390102) AND T1.C_ENTRADA='D' AND T1.C_CAGENTE NOT IN ('511','53','6','3390101','3390102')
"""
work_conn.execute(text(sql_reaj_af52_3))

# ajusta anterior en rec precio reaj en activo af521 y af522
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_af52_4"))
sql_reaj_af52_4 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
       'Rec Precio Reaj' AS C_CUENTA, T1.C_ENTRADA,
       T1.DATO*-1 AS DATO, T1.C_SCN, T1.N_SCN, T1.FUENTE,
       '0001i' AS PROC
INTO #reaj_af52_4
FROM #reaj_af52_3 T1
"""
work_conn.execute(text(sql_reaj_af52_4))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_af52_3"))
_log("APPEND BD_CTSI <- REAJ_AF52_3", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_af52_4"))
_log("APPEND BD_CTSI <- REAJ_AF52_4", res.rowcount)
for t in ["#reaj_af52_3", "#reaj_af52_4"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA IMPUTACIÓN DE OP FINANCIERA EN FONDOS DE PENSIONES PROVENIENTE DE AJUSTE DE REAJUSTES EN AF522 EN EL PATRIMONIO DE LOS FONDOS PARA EQUILIBRAR LA CUENTA FINANCIERA
# imputa op financiera en pasivo af612 ca hogares
work_conn.execute(text("DROP TABLE IF EXISTS #of_fp"))
sql_of_fp = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       '511' AS C_CAGENTE, 'Financiera' AS C_CUENTA, 'H' AS C_ENTRADA,
       T1.DATO AS DATO, 'AF.612' AS C_SCN, 'Reserva de fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE, '0001j' AS PROC
INTO #of_fp
FROM TABLAS.dbo.REAJUSTES T1
WHERE T1.SECTOR IN (3390101, 3390102) AND T1.C_ENTRADA='D' AND T1.C_CAGENTE IN ('34','341','342')
"""
work_conn.execute(text(sql_of_fp))

# imputa (elimina) rec precio reaj en pasivo af612 ca hogares
work_conn.execute(text("DROP TABLE IF EXISTS #reaj_fp"))
sql_reaj_fp = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       '511' AS C_CAGENTE, 'Rec Precio Reaj' AS C_CUENTA, 'H' AS C_ENTRADA,
       T1.DATO*-1 AS DATO, 'AF.612' AS C_SCN, 'Reserva de fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE, '0001k' AS PROC
INTO #reaj_fp
FROM TABLAS.dbo.REAJUSTES T1
WHERE T1.SECTOR IN (3390101, 3390102) AND T1.C_ENTRADA='D' AND T1.C_CAGENTE IN ('34','341','342')
"""
work_conn.execute(text(sql_reaj_fp))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #of_fp"))
_log("APPEND BD_CTSI <- OF_FP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #reaj_fp"))
_log("APPEND BD_CTSI <- REAJ_FP", res.rowcount)
for t in ["#of_fp", "#reaj_fp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PTMOS DE CORTO PLAZO EN HOGARES POR USO DE TARJETAS COMERCIALES
df_ajptmos_cp = pd.read_sql(text("SELECT * FROM TABLAS.dbo.BD_CTSI_AJPTMOS_CP"), engine)
df_ajptmos_cp.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND BD_CTSI <- BD_CTSI_AJPTMOS_CP", len(df_ajptmos_cp))


In [ ]:
# RECLASIFICA PARTICIPACIONES DE LOS FONDOS DE INVERSIÓN COMO INSTRUMENTO AF.522 PARA DIFERENCIARLO DE LO CORRESPONDIENTE A FONDOS MUTUOS
# EN TEORÍA SOLO EL MERCADO MONETARIO DEBE QUEDAR EN AF.521. LOS FFMM DE LP DEBEN TAMBIÉN QUEDAR COMO AF.522
sql_up_af521_522 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.522', N_SCN='Participaciones emitidas por fondos de inversión'
WHERE C_SCN='AF.521' AND C_ENTRADA='H' AND SECTOR IN (339011,3390102)
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af521_522))
    _log("UPDATE AF.521->AF.522", res.rowcount)


In [ ]:
# ACTUALIZACIONES DE REGISTROS
sql_up_af41_42_7 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.7', N_SCN='Créditos comerciales', PROC='0002'
WHERE C_SCN IN ('AF.41','AF.42') AND C_ENTRADA='D' AND SECTOR IN (334,369011)
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af41_42_7))
    _log("UPDATE AF.41/42->AF.7", res.rowcount)


In [ ]:
sql_up_d29_d21 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='D.21'
WHERE C_SCN='D.29' AND C_CUENTA='Producción' AND SECTOR=41
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_d29_d21))
    _log("UPDATE D.29->D.21", res.rowcount)


In [ ]:
sql_up_cagente_511 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='511' WHERE C_SCN IN ('D.1','D.61','D.62','D.8')"
with engine.begin() as conn:
    res = conn.execute(text(sql_up_cagente_511))
    _log("UPDATE C_CAGENTE->511 D.1/61/62/8", res.rowcount)


In [ ]:
sql_up_cagente_41 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='41' WHERE C_SCN IN ('D.21','D.29','D.5')"
with engine.begin() as conn:
    res = conn.execute(text(sql_up_cagente_41))
    _log("UPDATE C_CAGENTE->41", res.rowcount)


In [ ]:
sql_up_cagente_53_sector41 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='53' WHERE C_SCN IN ('D.21','D.29','D.5') AND SECTOR=41"
with engine.begin() as conn:
    res = conn.execute(text(sql_up_cagente_53_sector41))
    _log("UPDATE C_CAGENTE->53 sector 41", res.rowcount)


In [ ]:
sql_up_331_41 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='51', PROC='0007Z'
WHERE C_CAGENTE='331' AND SECTOR=41 AND C_SCN='AF.42' AND FUENTE='CI'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_331_41))
    _log("UPDATE 331/41 AF.42->51", res.rowcount)


In [ ]:
# IMPUTA AF42 EN ACTIVO DE GOB CENTRAL CON CA 321 USANDO INFO DEL PASIVO DE BCOS COMERCIALES CON CA CORFO(331). AJUSTA EN GOB CONTRA SNF
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_331"))
sql_af42_41_331 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR, '321' AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0007x' AS PROC
INTO #af42_41_331
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR=321 AND T1.C_CAGENTE='331' AND T1.C_ENTRADA='H'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_41_331))

work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_snf"))
sql_af42_41_snf = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       '51' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       T1.DATO*-1 AS DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0007y' AS PROC
INTO #af42_41_snf
FROM #af42_41_331 T1
"""
work_conn.execute(text(sql_af42_41_snf))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_41_331"))
_log("APPEND BD_CTSI <- AF42_41_331", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_41_snf"))
_log("APPEND BD_CTSI <- AF42_41_SNF", res.rowcount)
for t in ["#af42_41_331", "#af42_41_snf"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZACIONES DE REGISTROS
with engine.begin() as conn:
    res = conn.execute(text("DELETE FROM TABLAS.dbo.BD_CTSI WHERE C_CUENTA='Var Balance'"))
    _log("DELETE Var Balance", res.rowcount)


In [ ]:
sql_up_k1_p51 = "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '' WHERE C_SCN IN ('K.1','P.51')"
with engine.begin() as conn:
    res = conn.execute(text(sql_up_k1_p51))
    _log("UPDATE C_CAGENTE vacio K.1/P.51", res.rowcount)


In [ ]:
sql_up_af2_af22 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.22', N_SCN='Depósitos', PROC='0030'
WHERE C_SCN='AF.2'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af2_af22))
    _log("UPDATE AF.2->AF.22", res.rowcount)


In [ ]:
sql_up_af1_af42 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_SCN='AF.42', N_SCN='Préstamos a largo plazo', PROC='0032', FUENTE='PS'
WHERE C_SCN='AF.1' AND C_ENTRADA='D' AND SECTOR=6
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af1_af42))
    _log("UPDATE AF.1->AF.42", res.rowcount)


In [ ]:
sql_up_af21_31 = """
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='31'
WHERE SECTOR IN (41,42) AND C_SCN='AF.21' AND C_CAGENTE IN ('3','321')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_up_af21_31))
    _log("UPDATE AF.21 ->31", res.rowcount)


In [ ]:
# AJUSTA BONOS DE LARGO PLAZO DE BANCOS CON PATRIMONIO SEPARADO USANDO INFO DCV. AJUSTA EN BONOS DE LP DE BANCOS CON EMPRESAS
# SELECCIONA ACTIVO AF32 INFORMADA POR BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_bcos"))
sql_af32_bcos = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0015' AS PROC
INTO #af32_bcos
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.C_CAGENTE='332' AND T1.SECTOR=321
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_bcos))


In [ ]:
# DCV AF32 TENENCIA DE AUXILIARES SIN EMISOR RM
# M-001: ruta absoluta reemplazada por ruta relativa al workspace
ruta_dcv = Path("sasdata") / "BCCH" / "GEM_DCNI" / "02_CNSI" / "05_DCV" / "Data" / "DCVRES" / "base_af3_total_v2.sas7bdat"
df_dcv, _ = pyreadstat.read_sas7bdat(str(ruta_dcv))
df_dcv_filtrado = df_dcv[
    (df_dcv["C_SI_Emisor"] == "3324")
    & (df_dcv["C_INSTRUMENTO_SCN"] == "AF.32")
    & (df_dcv["C_SI_Tenedor"].isin(["321"]))
    & (df_dcv["Año"] > 2002)
    & (~df_dcv["Variable"].isin(["Valor Par Indice", "Valor Par MM$"]))
].copy()

mapa_cuenta = {
    "Cuenta Financiera": "Financiera",
    "Cuenta de Revalorización": "Rec Precio",
    "Saldo Final": "Bce Final",
    "Saldo Inicial": "Bce Inicio",
    "Cuenta Volumen": "Rec Volumen",
}
df_dcv_filtrado["C_CUENTA"] = df_dcv_filtrado["C_Cuenta"].map(mapa_cuenta)
df_dcv_filtrado["SECTOR"] = pd.to_numeric(df_dcv_filtrado["C_SI_Tenedor"], errors="coerce")
df_dcv_filtrado["C_CAGENTE"] = "332"
df_dcv_filtrado["C_ENTRADA"] = "D"
df_dcv_filtrado["N_SCN"] = "Valores distintos de acciones a largo plazo"
df_dcv_filtrado["FUENTE"] = "PS"
df_dcv_filtrado["PROC"] = "0015"
df_dcv_filtrado["MONEDA"] = "P"

af32_d_dcv = (
    df_dcv_filtrado.groupby(["Año", "Trimestre", "C_SI_Tenedor", "C_Cuenta", "C_ENTRADA", "C_INSTRUMENTO_SCN"], as_index=False)
    .agg(DATO=("Dato", "sum"))
)
af32_d_dcv = af32_d_dcv.merge(
    df_dcv_filtrado[["Año", "Trimestre", "C_SI_Tenedor", "C_Cuenta", "C_ENTRADA", "C_INSTRUMENTO_SCN", "MONEDA", "SECTOR", "C_CAGENTE", "C_CUENTA", "N_SCN", "FUENTE", "PROC"]].drop_duplicates(),
    on=["Año", "Trimestre", "C_SI_Tenedor", "C_Cuenta", "C_ENTRADA", "C_INSTRUMENTO_SCN"],
    how="left",
)
af32_d_dcv = af32_d_dcv.rename(columns={"Año": "AÑO", "Trimestre": "TRIM", "C_INSTRUMENTO_SCN": "C_SCN"})
af32_d_dcv = af32_d_dcv[["MONEDA", "AÑO", "TRIM", "SECTOR", "C_CAGENTE", "C_CUENTA", "C_ENTRADA", "DATO", "C_SCN", "N_SCN", "FUENTE", "PROC"]]
# se sube a una tabla TEMPORAL de sesion para juntar server-side con #af32_bcos (replace permitido en # porque muere con la sesion)
af32_d_dcv.to_sql("#af32_d_dcv", work_conn, if_exists="replace", index=False)


In [ ]:
# JUNTA TABLAS PARA HACER RESTA
work_conn.execute(text(f"INSERT INTO #af32_bcos ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_d_dcv"))


In [ ]:
# AGRUPA DIF DCV-CI
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dif"))
sql_af32_dif = f"""
SELECT {cols_bd_ctsi.replace('DATO', 'SUM(DATO) AS DATO')}
INTO #af32_dif
FROM #af32_bcos T1
GROUP BY MONEDA, AÑO, TRIM, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, C_CAGENTE, FUENTE, SECTOR, PROC
"""
work_conn.execute(text(sql_af32_dif))


In [ ]:
# AJUSTE POR IMPUT ANTERIOR EN CRED DE BANCOS CON RESTO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_emp"))
sql_af32_emp = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       (DATO*-1) AS DATO, 'AF.7' AS C_SCN, 'Créditos comerciales' AS N_SCN,
       FUENTE, '0015b' AS PROC
INTO #af32_emp
FROM #af32_dif
"""
work_conn.execute(text(sql_af32_emp))


In [ ]:
# ANEXA AJUSTE BONOS LP EN AUX
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_dif"))
_log("APPEND BD_CTSI <- AF32_DIF", res.rowcount)
# ANEXA AJUSTE cred com EN AUX
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_emp"))
_log("APPEND BD_CTSI <- AF32_EMP", res.rowcount)
for t in ["#af32_bcos", "#af32_d_dcv", "#af32_dif", "#af32_emp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTACIONES EN PATRIMONIO SEPARADO
# 1. IMPUTA BONOS DE LARGO PLAZO EN PASIVO DE PAT SEPARADO USANDO INFO DEL PASIVO DE CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af32_3324"))
sql_af32_3324 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       3324 AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       T1.C_CUENTA, 'H' AS C_ENTRADA,
       T1.DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0017' AS PROC
INTO #af32_3324
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CAGENTE IN ('332','3323','3324') AND T1.C_ENTRADA='D' AND T1.C_SCN='AF.32'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.DATO
"""
work_conn.execute(text(sql_af32_3324))


In [ ]:
# 2. IMPUTA PTMOS DE LARGO PLAZO EN ACTIVO DE PAT SEPARADO USANDO 2/3 DE LO IMPUTADO EN BONOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_3324"))
sql_af42_3324 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       '51022' AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO)*2/3 AS DATO, 'AF.42' AS C_SCN, 'Préstamos de largo plazo' AS N_SCN,
       T1.FUENTE, '0018' AS PROC
INTO #af42_3324
FROM #af32_3324 T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.FUENTE
"""
work_conn.execute(text(sql_af42_3324))

# 3. IMPUTA CRED COMERCIALES EN ACTIVO DE PAT SEPARADO USANDO 1/3 DE LO IMPUTADO EN BONOS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_3324"))
sql_af7_3324 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR,
       '51022' AS C_CAGENTE, T1.C_CUENTA, 'D' AS C_ENTRADA,
       SUM(T1.DATO)/3 AS DATO, 'AF.7' AS C_SCN, 'Créditos comerciales' AS N_SCN,
       T1.FUENTE, '0020' AS PROC
INTO #af7_3324
FROM #af32_3324 T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.FUENTE
"""
work_conn.execute(text(sql_af7_3324))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_3324"))
_log("APPEND BD_CTSI <- AF32_3324", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af42_3324"))
_log("APPEND BD_CTSI <- AF42_3324", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_3324"))
_log("APPEND BD_CTSI <- AF7_3324", res.rowcount)
for t in ["#af32_3324", "#af42_3324", "#af7_3324"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA BONOS DE RECONOCIMIENTO EN PASIVO DEL GOB CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af321_41"))
sql_af321_41 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       T1.C_CUENTA, 'H' AS C_ENTRADA,
       T1.DATO, T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0021' AS PROC
INTO #af321_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN='AF.321'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.DATO
"""
work_conn.execute(text(sql_af321_41))

res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af321_41"))
_log("APPEND BD_CTSI <- AF321_41", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af321_41"))


## S2_02_Din&Dep

Reconcilia activo y pasivo de instrumentos de dinero y depósitos (AF1/21/22/29/31/32) entre sectores institucionales y resto del mundo, respetando el dato de la contraparte designada y registrando el ajuste de contrapartida (AF29/AF32/AF71) en el sector correspondiente, incluyendo actualizaciones puntuales de contragente e instrumento (cierres 2020/2021) / Reconcilia depósitos y activos financieros (AF.29, AF.5/AF.32, AF.71) entre resto de la economía, RM, Banco Central, FP, AFP y otros sectores, imputando contrapartidas y actualizando contragentes en la base consolidada

*confianza: medium · verificador: revise · SAS: PROC SQL UPDATE + secuencia CREATE TABLE/APPEND/DROP contra tabla temporal de sesión (#tmp) por instrumento financiero (AF1, AF21, AF22, AF29, AF31, AF32, CD) + PROC SQL CREATE TABLE con GROUP BY/CASE + PROC DATASETS APPEND + PROC SQL UPDATE, sobre TABLAS.BD_CTSI vía temporales de sesión*

In [ ]:
# ========= S2_02_Din&Dep =========
# COMPRIME TABLA PRINCIPAL
# (COMPRESS=YES es una opción de almacenamiento SAS sin equivalente en SQL Server; no se traduce, la tabla ya existe)

# CAMBIA COD DE INSTRUMENTO DESDE AF.22 A AF.29 EN EL PASIVO DEL RESTO DEL MUNDO CON CONTRAENTE HOGARES
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.29', N_SCN = 'Otros depósitos' WHERE SECTOR = 6 AND C_SCN = 'AF.29' AND C_ENTRADA = 'H' AND C_CAGENTE = '511'"))
_log("UPDATE TABLAS.BD_CTSI (AF29->AF29 hogares RM)", res.rowcount)


In [ ]:
# CIERRE 2021: EN FONDOS DE INVERSIÓN ACTIVO AF41 CON CA BANCOS, CAMBIA INSTRUMENTO A AF.29
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.29', N_SCN = 'Otros depósitos' WHERE SECTOR = 339011 AND C_SCN = 'AF.41' AND C_ENTRADA = 'D' AND C_CAGENTE = '321'"))
_log("UPDATE TABLAS.BD_CTSI (AF41->AF29 FI bancos)", res.rowcount)


In [ ]:
# CIERRE 2021: EN HOLDINGS Y CASAS MATRICES CAMBIA CONTRAGENTE DE AF21 DESDE 321 A 31
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '31' WHERE SECTOR IN (37, 36907) AND C_SCN = 'AF.21' AND C_ENTRADA = 'D' AND C_CAGENTE = '321'"))
_log("UPDATE TABLAS.BD_CTSI (holdings AF21 CA 321->31)", res.rowcount)


In [ ]:
# ORO MONETARIO ENTRE BCENTRAL-ACTIVO Y RM-PASIVO, MANDA DATO DE BCENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af1_31_6"))
sql_af1_31_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR = 6 AND C_CAGENTE = '31' AND C_ENTRADA = 'H' AND C_SCN = 'AF.1' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0037' AS PROC
INTO #af1_31_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.1' AND FUENTE = 'CI')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.1' AND FUENTE = 'CI')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af1_31_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af1_31_6_v1"))
sql_af1_31_6_v1 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       '0038' AS PROC
INTO #af1_31_6_v1
FROM #af1_31_6 T1
"""
work_conn.execute(text(sql_af1_31_6_v1))

cols_af1 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af1}) SELECT {cols_af1} FROM #af1_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF1_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af1}) SELECT {cols_af1} FROM #af1_31_6_v1"))
_log("APPEND TABLAS.BD_CTSI (AF1_31_6_V1)", res.rowcount)
for t in ["#af1_31_6", "#af1_31_6_v1"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# TÍTULOS AF31 ENTRE RESTO DEL MUNDO-PASIVO Y FONDOS DE INVERSIÓN-ACTIVO. RESPETA DATO DE LOS FONDOS Y AJUSTA IMPUTACIÓN CONTRA AF.32 EN PASIVO DEL RM CON CA 33901 (CIERRE 2021)
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_fi"))
sql_af31_6_fi = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '33901' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d1' AS PROC
INTO #af31_6_fi
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (339011, 3390101, 3390102) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE IN ('339011', '33901') AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.31')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_fi))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_fi_aj"))
sql_af31_6_fi_aj = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_6_fi_aj
FROM #af31_6_fi T1
"""
work_conn.execute(text(sql_af31_6_fi_aj))

cols_af31fi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31fi}) SELECT {cols_af31fi} FROM #af31_6_fi"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_FI)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31fi}) SELECT {cols_af31fi} FROM #af31_6_fi_aj"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_FI_AJ)", res.rowcount)
for t in ["#af31_6_fi", "#af31_6_fi_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN ACTIVO AF29 DE GOBIERNO DESDE 3 A 321
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321' WHERE C_CAGENTE = '3' AND SECTOR = 41 AND C_SCN = 'AF.29' AND C_ENTRADA = 'D'"))
_log("UPDATE TABLAS.BD_CTSI (AF29 gob CA 3->321)", res.rowcount)


In [ ]:
# TÍTULOS AF31 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL GOBIERNO Y AJUSTA IMPUTACIÓN CONTRA AF.32 EN PASIVO DEL RM CON CA 41
# CIERRE 2021: SE CAMBIA REGLA PARA RESPETAR EL DATO DEL RM, POR ENDE SE AJUSTE EN ACTIVO AF32 DEL GOB CON CA RM
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_41"))
sql_af31_6_41 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d1' AS PROC
INTO #af31_6_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (41) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE IN ('41') AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.31')
GROUP BY T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_41_aj"))
sql_af31_6_41_aj = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_6_41_aj
FROM #af31_6_41 T1
"""
work_conn.execute(text(sql_af31_6_41_aj))

cols_af316_41 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af316_41}) SELECT {cols_af316_41} FROM #af31_6_41"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af316_41}) SELECT {cols_af316_41} FROM #af31_6_41_aj"))
_log("APPEND TABLAS.BD_CTSI (AF31_6_41_AJ)", res.rowcount)
for t in ["#af31_6_41", "#af31_6_41_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: TÍTULOS AF31 ENTRE RESTO DEL MUNDO-PASIVO Y BANCOS-SEGUROS-EMPRESAS-ACTIVO. RESPETA DATO DEL RM Y AJUSTA IMPUTACIÓN CONTRA AF.32 EN ACTIVO DE LOS SECTORES CON CA RM
# selecciona pasivo del rm con contragentes distintos a gobierno y fondos de inversión
# SECTOR se arma con INPUT(C_CAGENTE, BEST5.) salvo CA='53' que mapea a 51022 -- TRY_CAST retorna NULL si el valor no es numérico (supuesto ya declarado)
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_sect"))
sql_af31_6_sect = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       (CASE WHEN T1.C_CAGENTE = '53' THEN 51022 ELSE TRY_CAST(T1.C_CAGENTE AS int) END) AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_6_sect
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CAGENTE NOT IN ('41', '33901') AND T1.SECTOR = 6 AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.31')
GROUP BY T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CAGENTE = '53' THEN 51022 ELSE TRY_CAST(T1.C_CAGENTE AS int) END),
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_sect))


In [ ]:
# selecciona activo af31 de los sectores (excepto gobierno y fondos de inversiones) con resto del mundo
work_conn.execute(text("DROP TABLE IF EXISTS #af31_sect_6"))
sql_af31_sect_6 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d2' AS PROC
INTO #af31_sect_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR NOT IN (41, 4, 42, 412, 339011) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31')
GROUP BY T1.AÑO, T1.TRIM, T1.SECTOR,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_sect_6))

# data AF31_6_sect; set AF31_6_sect AF31_SECT_6; run; -- concatena ambas tablas en la misma #af31_6_sect
work_conn.execute(text("INSERT INTO #af31_6_sect SELECT * FROM #af31_sect_6"))


In [ ]:
# calcula delta a imputar en sectores
work_conn.execute(text("DROP TABLE IF EXISTS #af31_sect_6_imp"))
sql_af31_sect_6_imp = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af31_sect_6_imp
FROM #af31_6_sect T1
GROUP BY T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_af31_sect_6_imp))


In [ ]:
# ajusta en activo af32 de los sectores con contragente RM
work_conn.execute(text("DROP TABLE IF EXISTS #af32_sect_6_aj"))
sql_af32_sect_6_aj = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af32_sect_6_aj
FROM #af31_sect_6_imp T1
"""
work_conn.execute(text(sql_af32_sect_6_aj))

cols_af31sect = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31sect}) SELECT {cols_af31sect} FROM #af31_sect_6_imp"))
_log("APPEND TABLAS.BD_CTSI (AF31_SECT_6_IMP)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af31sect}) SELECT {cols_af31sect} FROM #af32_sect_6_aj"))
_log("APPEND TABLAS.BD_CTSI (AF32_SECT_6_AJ)", res.rowcount)
for t in ["#af31_sect_6_imp", "#af32_sect_6_aj", "#af31_6_sect", "#af31_sect_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# TÍTULOS AF32 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL RM Y LUEGO SE AJUSTA IMPUTACIÓN CONTRA AF.29 EN GOB CON CA 6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_41_2"))
sql_af32_6_41_2 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR = 41 AND C_CAGENTE = '6' AND C_ENTRADA = 'D' AND C_SCN = 'AF.32' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051d' AS PROC
INTO #af32_6_41_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32')
GROUP BY T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_6_41_2))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_41_6"))
sql_af29_41_6 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0051d3' AS PROC
INTO #af29_41_6
FROM #af32_6_41_2 T1
"""
work_conn.execute(text(sql_af29_41_6))

cols_af32_6_41 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_6_41}) SELECT {cols_af32_6_41} FROM #af32_6_41_2"))
_log("APPEND TABLAS.BD_CTSI (AF32_6_41_2)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32_6_41}) SELECT {cols_af32_6_41} FROM #af29_41_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_41_6 v1)", res.rowcount)
for t in ["#af32_6_41_2", "#af29_41_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPÓSITOS AF22 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL GOB Y LUEGO SE AJUSTA IMPUTACIÓN CONTRA AF.29 EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af22_41_6"))
sql_af22_41_6 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '41' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR = 6 AND C_CAGENTE = '41' AND C_ENTRADA = 'H' AND C_SCN = 'AF.22' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0042_04' AS PROC
INTO #af22_41_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22' AND FUENTE = 'CI')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22' AND FUENTE = 'CI')
GROUP BY T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_41_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_41_6_v2"))
sql_af29_41_6_v2 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0042_05' AS PROC
INTO #af29_41_6_v2
FROM #af22_41_6 T1
"""
work_conn.execute(text(sql_af29_41_6_v2))

cols_af22_41 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_41}) SELECT {cols_af22_41} FROM #af22_41_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_41_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_41}) SELECT {cols_af22_41} FROM #af29_41_6_v2"))
_log("APPEND TABLAS.BD_CTSI (AF29_41_6 v2)", res.rowcount)
for t in ["#af22_41_6", "#af29_41_6_v2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPÓSITOS AF29 ENTRE RESTO DEL MUNDO-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL RM Y LUEGO SE AJUSTA IMPUTACIÓN CONTRA AF.29 EN GOB CON CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_41"))
sql_af29_6_41 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN SECTOR = 41 AND C_CAGENTE = '6' AND C_ENTRADA = 'D' AND C_SCN = 'AF.29' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0051' AS PROC
INTO #af29_6_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 41 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_41_321"))
sql_af29_41_321 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0052' AS PROC
INTO #af29_41_321
FROM #af29_6_41 T1
"""
work_conn.execute(text(sql_af29_41_321))

cols_af29_6_41 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_41}) SELECT {cols_af29_6_41} FROM #af29_6_41"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_41}) SELECT {cols_af29_6_41} FROM #af29_41_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_41_321)", res.rowcount)
for t in ["#af29_6_41", "#af29_41_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DINERO AF21 ENTRE RESTO DEL MUNDO-PASIVO Y BCOS/BCO CENTRAL-ACTIVO. RESPETA DATO DEL BCOS Y BCO CENTRAL Y LUEGO SE AJUSTA IMPUTACIÓN CONTRA AF.22 EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af21_31_321_6"))
sql_af21_31_321_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0055' AS PROC
INTO #af21_31_321_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (321, 31, 336, 36912) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.21')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, TRY_CAST(T1.C_CAGENTE AS int), LTRIM(CAST(T1.SECTOR AS varchar(11))), T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.DATO
"""
work_conn.execute(text(sql_af21_31_321_6))


In [ ]:
# cierre 2021: para no dejar saldo negativo en pasivo del rm con contragentes 336 y 36912 en af22, el ajuste en estos sectores se resta de ca 53
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_321_6"))
sql_af22_31_321_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       (CASE WHEN T1.C_CAGENTE IN ('336', '36912') THEN '53' ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.22' AS C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0056' AS PROC
INTO #af22_31_321_6
FROM #af21_31_321_6 T1
"""
work_conn.execute(text(sql_af22_31_321_6))

cols_af21_31_321 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_31_321}) SELECT {cols_af21_31_321} FROM #af21_31_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF21_31_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_31_321}) SELECT {cols_af21_31_321} FROM #af22_31_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_321_6)", res.rowcount)
for t in ["#af21_31_321_6", "#af22_31_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ANULA AF21 EN EL ACTIVO DE SECTORES 5111 CON BCO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af21_5111"))
sql_af21_5111 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0053' AS PROC
INTO #af21_5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (5111) AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.21')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af21_5111))

cols_af21_5111 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_5111}) SELECT {cols_af21_5111} FROM #af21_5111"))
_log("APPEND TABLAS.BD_CTSI (AF21_5111)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af21_5111"))


In [ ]:
# IMPUTA AF21 EN HOGARES COMO LA DIF ENTRE ACTIVO Y PASIVO DEL INSTRUMENTO
work_conn.execute(text("DROP TABLE IF EXISTS #af21_511"))
sql_af21_511 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       511 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0057' AS PROC
INTO #af21_511
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_SCN = 'AF.21')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af21_511))

cols_af21_511 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af21_511}) SELECT {cols_af21_511} FROM #af21_511"))
_log("APPEND TABLAS.BD_CTSI (AF21_511)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af21_511"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN AF22 DEL GOBIERNO
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321' WHERE C_CAGENTE IN ('3', '41') AND SECTOR IN (41, 42) AND C_SCN = 'AF.22'"))
_log("UPDATE TABLAS.BD_CTSI (AF22 gob CA->321)", res.rowcount)


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-PASIVO Y BCOS COMERCIALES-ACTIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_321"))
sql_af22_31_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       321 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 321 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0062' AS PROC
INTO #af22_31_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '321' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 321 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_31_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_321"))
sql_af29_31_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0063' AS PROC
INTO #af29_31_321
FROM #af22_31_321 T1
"""
work_conn.execute(text(sql_af29_31_321))

cols_af22_31_321 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_321}) SELECT {cols_af22_31_321} FROM #af22_31_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_321}) SELECT {cols_af22_31_321} FROM #af29_31_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_31_321)", res.rowcount)
for t in ["#af22_31_321", "#af29_31_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN AF22 BCO CENTRAL-PASIVO
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '41' WHERE SECTOR = 31 AND C_ENTRADA = 'H' AND C_SCN = 'AF.22' AND C_CAGENTE IN ('5101', '53')"))
_log("UPDATE TABLAS.BD_CTSI (AF22 BC pasivo CA->41)", res.rowcount)


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-PASIVO Y GOBIERNO-ACTIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF22 CONTRAGENTE BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_41"))
sql_af22_31_41 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 41 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0067' AS PROC
INTO #af22_31_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '41' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 41 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_31_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af22_321_41"))
sql_af22_321_41 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0068' AS PROC
INTO #af22_321_41
FROM #af22_31_41 T1
"""
work_conn.execute(text(sql_af22_321_41))

cols_af22_31_41 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_41}) SELECT {cols_af22_31_41} FROM #af22_31_41"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_41}) SELECT {cols_af22_31_41} FROM #af22_321_41"))
_log("APPEND TABLAS.BD_CTSI (AF22_321_41)", res.rowcount)
for t in ["#af22_31_41", "#af22_321_41"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-PASIVO Y RM-ACTIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF71 CONTRAGENTE BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af22_31_6"))
sql_af22_31_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0072' AS PROC
INTO #af22_31_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_31_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_31_6"))
sql_af71_31_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0073' AS PROC
INTO #af71_31_6
FROM #af22_31_6 T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af71_31_6))

cols_af22_31_6 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_6}) SELECT {cols_af22_31_6} FROM #af22_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_31_6}) SELECT {cols_af22_31_6} FROM #af71_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF71_31_6)", res.rowcount)
for t in ["#af22_31_6", "#af71_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF22 ENTRE BCOS COMERCIALES-ACTIVO Y RM-PASIVO. RESPETA DATO DE BCOS Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_321_6"))
sql_af22_321_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '321' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0082' AS PROC
INTO #af22_321_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 321 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '321' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_321_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_321_6"))
sql_af29_321_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0083' AS PROC
INTO #af29_321_6
FROM #af22_321_6 T1
"""
work_conn.execute(text(sql_af29_321_6))

cols_af22_321_6 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_321_6}) SELECT {cols_af22_321_6} FROM #af22_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_321_6}) SELECT {cols_af22_321_6} FROM #af29_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_321_6)", res.rowcount)
for t in ["#af22_321_6", "#af29_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF22 ENTRE FONDO DE PENSIONES-ACTIVO Y RM-PASIVO. RESPETA DATO DE RM Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_34_6"))
sql_af22_34_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       34 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR IN (34, 341) THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0087' AS PROC
INTO #af22_34_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR IN (341, 34) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '34' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_34_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_34_6"))
sql_af29_34_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       T1.FUENTE,
       '0088' AS PROC
INTO #af29_34_6
FROM #af22_34_6 T1
"""
work_conn.execute(text(sql_af29_34_6))

cols_af22_34_6 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_34_6}) SELECT {cols_af22_34_6} FROM #af22_34_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_34_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_34_6}) SELECT {cols_af22_34_6} FROM #af29_34_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_34_6)", res.rowcount)
for t in ["#af22_34_6", "#af29_34_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AF2 Y AF22 EN RESTO DEL MUNDO CON CA 33901/351 PARA CAMBIAR INST A AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_6_33901"))
sql_af22_6_33901 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0089' AS PROC
INTO #af22_6_33901
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE IN ('33901/351', '351', '33901') AND T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.22', 'AF.2'))
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_6_33901))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_33901"))
sql_af29_6_33901 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0089' AS PROC
INTO #af29_6_33901
FROM #af22_6_33901 T1
"""
work_conn.execute(text(sql_af29_6_33901))

cols_af22_6_33901 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_33901}) SELECT {cols_af22_6_33901} FROM #af22_6_33901"))
_log("APPEND TABLAS.BD_CTSI (AF22_6_33901)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_33901}) SELECT {cols_af22_6_33901} FROM #af29_6_33901"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_33901)", res.rowcount)
for t in ["#af22_6_33901", "#af29_6_33901"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AF22 EN 5101 (SOC PUB)-ACTIVO CON RM-PASIVO. RESPETA DATO DE RM Y AJUSTA CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5101_6"))
sql_af22_5101_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       5101 AS SECTOR,
       51 AS C_SI_publ,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0091' AS PROC
INTO #af22_5101_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE = '5101' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_5101_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5101_321"))
sql_af22_5101_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_SI_publ,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0090' AS PROC
INTO #af22_5101_321
FROM #af22_5101_6 T1
"""
work_conn.execute(text(sql_af22_5101_321))

cols_af22_5101 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_5101}) SELECT {cols_af22_5101} FROM #af22_5101_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_5101_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_5101}) SELECT {cols_af22_5101} FROM #af22_5101_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_5101_321)", res.rowcount)
for t in ["#af22_5101_6", "#af22_5101_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN EMPRESAS (51021) QUE REPORTAN EN DÓLARES
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '6', FUENTE = 'PS', PROC = '0092' WHERE MONEDA = 'D' AND SECTOR = 51021 AND C_CAGENTE = '321'"))
_log("UPDATE TABLAS.BD_CTSI (51021 USD CA->6)", res.rowcount)


In [ ]:
# ACTUALIZA INSTRUMENTO EN EMPRESAS MINERAS (51022 MONEDA DOLARES) DESDE AF22 A AF29
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_SCN = 'AF.29', N_SCN = 'Otros depósitos', PROC = '0092b' WHERE MONEDA = 'D' AND SECTOR = 51022 AND C_CAGENTE = '6' AND C_SCN = 'AF.22' AND C_ENTRADA = 'D'"))
_log("UPDATE TABLAS.BD_CTSI (51022 USD AF22->AF29)", res.rowcount)


In [ ]:
# DEPOSITOS AF22 ENTRE 51021-ACTIVO Y RM-PASIVO. RESPETA DATO DE 51021 Y AJUSTA IMPUTACIÓN CONTRA AF29 CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af22_53_6"))
sql_af22_53_6 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '53' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0096' AS PROC
INTO #af22_53_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 51021 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '53' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_53_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_53_6"))
sql_af29_53_6 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0097' AS PROC
INTO #af29_53_6
FROM #af22_53_6 T1
"""
work_conn.execute(text(sql_af29_53_6))

cols_af22_53_6 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_53_6}) SELECT {cols_af22_53_6} FROM #af22_53_6"))
_log("APPEND TABLAS.BD_CTSI (AF22_53_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_53_6}) SELECT {cols_af22_53_6} FROM #af29_53_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_53_6)", res.rowcount)
for t in ["#af22_53_6", "#af29_53_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AF22 DE SECTOR 42-ACTIVO CON 321-PASIVO. RESPETA DATO DEL SECTOR 321, AJUSTA CONTRA AF71
work_conn.execute(text("DROP TABLE IF EXISTS #af22_42_321"))
sql_af22_42_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       42 AS SECTOR,
       '321' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio' THEN 'Rec Precio Reaj' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 42 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0113' AS PROC
INTO #af22_42_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 42 AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.C_CAGENTE = '42' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio' THEN 'Rec Precio Reaj' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_42_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_42_321"))
sql_af71_42_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0114' AS PROC
INTO #af71_42_321
FROM #af22_42_321 T1
"""
work_conn.execute(text(sql_af71_42_321))

cols_af22_42_321 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_42_321}) SELECT {cols_af22_42_321} FROM #af22_42_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_42_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_42_321}) SELECT {cols_af22_42_321} FROM #af71_42_321"))
_log("APPEND TABLAS.BD_CTSI (AF71_42_321)", res.rowcount)
for t in ["#af22_42_321", "#af71_42_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA EN BCO COMERCIALES EL INST AF.1 A AF.22 DEL ACTIVO Y CONTRAGENTE DESDE 53 A 321
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '321', C_SCN = 'AF.22', N_SCN = 'Depósitos', PROC = '0114b', FUENTE = 'PS' WHERE C_CAGENTE = '53' AND SECTOR = 321 AND C_SCN = 'AF.1' AND C_ENTRADA = 'D'"))
_log("UPDATE TABLAS.BD_CTSI (bancos AF1->AF22 CA 53->321)", res.rowcount)


In [ ]:
# ANULA AF22 DEL ACTIVO DE SECTOR 5111 CON CA 321. POR CIERRE 2020 ANULA TMB AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5111_321"))
sql_af22_5111_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0115' AS PROC
INTO #af22_5111_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.22', 'AF.29'))
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af22_5111_321))

cols_af22_5111 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_5111}) SELECT {cols_af22_5111} FROM #af22_5111_321"))
_log("APPEND TABLAS.BD_CTSI (AF22_5111_321)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af22_5111_321"))


In [ ]:
# CIERRE 2020. ANULA AF31,AF32,AF34, AF521, AF522 DEL ACTIVO Y PASIVO DE SECTOR 5111 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af_5111_321"))
sql_af_5111_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0115' AS PROC
INTO #af_5111_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_SCN IN ('AF.31', 'AF.32', 'AF.34', 'AF.521', 'AF.522'))
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af_5111_321))

cols_af_5111 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af_5111}) SELECT {cols_af_5111} FROM #af_5111_321"))
_log("APPEND TABLAS.BD_CTSI (AF_5111_321)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af_5111_321"))


In [ ]:
# DEPOSITOS AF22 ENTRE BCO CENTRAL-ACTIVO Y RM-PASIVO. RESPETA DATO DEL SECTOR 31 Y AJUSTA IMPUTACIÓN CONTRA AF29
work_conn.execute(text("DROP TABLE IF EXISTS #af22_6_31"))
sql_af22_6_31 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 6 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0115d' AS PROC
INTO #af22_6_31
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN
"""
work_conn.execute(text(sql_af22_6_31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_31"))
sql_af29_6_31 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       T1.FUENTE,
       '0115e' AS PROC
INTO #af29_6_31
FROM #af22_6_31 T1
"""
work_conn.execute(text(sql_af29_6_31))

cols_af22_6_31 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_31}) SELECT {cols_af22_6_31} FROM #af22_6_31"))
_log("APPEND TABLAS.BD_CTSI (AF22_6_31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af22_6_31}) SELECT {cols_af22_6_31} FROM #af29_6_31"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_31 v1)", res.rowcount)
for t in ["#af22_6_31", "#af29_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA CERTIFICADOS DE DEPÓSITOS ENTRE BCOS COMERCIALES Y RESTO DEL MUNDO. LOS RECLASIFICA COMO TÍTULOS DE LARGO PLAZO
# NOTA: fuente original es un archivo SAS externo (t_aj_cd_321_6_af32.sas7bdat) con ruta absoluta hardcodeada (M-001);
# no hay equivalente disponible en BD ni en el proyecto Python -- se declara el hueco explícitamente
raise NotImplementedError("WORK.CD_321_6 se origina de un archivo externo '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_cd_321_6_af32.sas7bdat' (ruta absoluta, M-001): no hay dataset ni tabla de BD equivalente declarada en input_datasets/file_imports para esta fuente; se requiere definir su reemplazo (ruta relativa o tabla) antes de traducir CD_321_6 / CD_321_6_2 y su APPEND a TABLAS.BD_CTSI")


In [ ]:
# DEPOSITOS AF29 ENTRE BCO CENTRAL-ACTIVO Y RM-PASIVO. RESPETA DATO DEL SECTOR 6 Y AJUSTA IMPUTACIÓN CONTRA AF71
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_31_v2"))
sql_af29_6_31_v2 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       31 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 31 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0122' AS PROC
INTO #af29_6_31_v2
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 31 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '31' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_31_v2))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_6_31"))
sql_af71_6_31 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       '0123' AS PROC
INTO #af71_6_31
FROM #af29_6_31_v2 T1
"""
work_conn.execute(text(sql_af71_6_31))

cols_af29_6_31_v2 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_31_v2}) SELECT {cols_af29_6_31_v2} FROM #af29_6_31_v2"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_31 v2)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_31_v2}) SELECT {cols_af29_6_31_v2} FROM #af71_6_31"))
_log("APPEND TABLAS.BD_CTSI (AF71_6_31)", res.rowcount)
for t in ["#af29_6_31_v2", "#af71_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE BCO COMERCIAL-ACTIVO Y RM-PASIVO. RESPETA DATO DEL SECTOR 6 Y AJUSTA IMPUTACIÓN CONTRA AF32
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_321"))
sql_af29_6_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       321 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.SECTOR = 321 THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0127' AS PROC
INTO #af29_6_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 321 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 6 AND T1.C_CAGENTE = '321' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_6_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_321"))
sql_af32_6_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       T1.FUENTE,
       '0128' AS PROC
INTO #af32_6_321
FROM #af29_6_321 T1
"""
work_conn.execute(text(sql_af32_6_321))

cols_af29_6_321 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_321}) SELECT {cols_af29_6_321} FROM #af29_6_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_6_321}) SELECT {cols_af29_6_321} FROM #af32_6_321"))
_log("APPEND TABLAS.BD_CTSI (AF32_6_321)", res.rowcount)
for t in ["#af29_6_321", "#af32_6_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# OTROS DEPÓSITOS AF29 ENTRE FFMM Y FONDOS DE INVERSIÓN (3390102+339011+3390102)-ACTIVO Y RM-PASIVO. RESPETA DATO DEL RM Y AJUSTA IMPUTACIÓN EN FFMM MM CON CONTRAGENTE 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_ffmm_6"))
sql_af29_ffmm_6 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       3390101 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0129' AS PROC
INTO #af29_ffmm_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE = '33901' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR IN (3390102, 339011, 3390101) AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_ffmm_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_ffmm_321"))
sql_af29_ffmm_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0130' AS PROC
INTO #af29_ffmm_321
FROM #af29_ffmm_6 T1
"""
work_conn.execute(text(sql_af29_ffmm_321))

cols_af29_ffmm = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_ffmm}) SELECT {cols_af29_ffmm} FROM #af29_ffmm_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_FFMM_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_ffmm}) SELECT {cols_af29_ffmm} FROM #af29_ffmm_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_FFMM_321)", res.rowcount)
for t in ["#af29_ffmm_6", "#af29_ffmm_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# OTROS DEPÓSITOS AF29 ENTRE CIAS DE SEGURO DE VIDA-ACTIVO Y RM-PASIVO. RESPETA DATO DEL RM Y AJUSTA IMPUTACIÓN EN CIAS DE SEGURO EN BONOS DE LP CON CONTRAGENTE 6
work_conn.execute(text("DROP TABLE IF EXISTS #af29_351_6"))
sql_af29_351_6 = """
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       351 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0129a' AS PROC
INTO #af29_351_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 6 AND T1.C_CAGENTE = '351' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
   OR (T1.SECTOR = 351 AND T1.C_CAGENTE = '6' AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29')
GROUP BY T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_351_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_351_321"))
sql_af29_351_321 = """
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0130b' AS PROC
INTO #af29_351_321
FROM #af29_351_6 T1
"""
work_conn.execute(text(sql_af29_351_321))

cols_af29_351 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_351}) SELECT {cols_af29_351} FROM #af29_351_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_351_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af29_351}) SELECT {cols_af29_351} FROM #af29_351_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_351_321)", res.rowcount)
for t in ["#af29_351_6", "#af29_351_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE RESTO DE LA ECONOMÍA (activo) Y RM (pasivo, CA6)
work_conn.execute(text("DROP TABLE IF EXISTS #af29_resto_6"))
sql_af29_resto_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       36912 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA='D' THEN t1.DATO*-1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       'Otros depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0129c' AS PROC
INTO #af29_resto_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=6 AND t1.C_CAGENTE='33901/351' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.29')
   OR ((SUBSTRING(LTRIM(STR(t1.SECTOR,8)),1,2) IN ('33','36','37') OR t1.SECTOR IN (411))
       AND t1.C_ENTRADA='D' AND t1.C_SCN='AF.29' AND t1.C_CAGENTE='6'
       AND t1.SECTOR NOT IN (3390101,3390102,33901,339011,339))
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_af29_resto_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_resto_321"))
sql_af29_resto_321 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       (t1.DATO*-1) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       t1.FUENTE,
       '0130c' AS PROC
INTO #af29_resto_321
FROM #af29_resto_6 t1
"""
work_conn.execute(text(sql_af29_resto_321))


In [ ]:
# APPEND server-side de ambas tablas a TABLAS.BD_CTSI (columnas explícitas, alinea por nombre como FORCE)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_resto_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_RESTO_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_resto_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_RESTO_321)", res.rowcount)


In [ ]:
for t in ["#af29_resto_6", "#af29_resto_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE RESTO DE LA ECONOMÍA-ACTIVO Y RM-PASIVO. En RM resta lo que debe quedar con sector 51021
# y el resto lo asigna a Hogares(10%) y Otras Sociedades(90%)
# CIERRE 2021: CAMBIA REC PRECIO REAJ A REC PRECIO
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_511"))
sql_af29_6_511 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       511 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.SECTOR=51021 THEN t1.DATO*-1 ELSE t1.DATO END)*0.1 AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0132' AS PROC
INTO #af29_6_511
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=51021 AND t1.C_CAGENTE='6' AND t1.C_ENTRADA='D' AND t1.C_SCN='AF.29')
   OR (t1.SECTOR=6 AND t1.C_CAGENTE='53' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.29')
GROUP BY t1.[AÑO], t1.TRIM,
         CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END,
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af29_6_511))


In [ ]:
# ESTIMADO MINERAS
work_conn.execute(text("DROP TABLE IF EXISTS #af29_6_51022"))
sql_af29_6_51022 = """
SELECT 'D' AS MONEDA, t1.[AÑO], t1.TRIM,
       51022 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.SECTOR=51021 THEN t1.DATO*-1 ELSE t1.DATO END)*0.9 AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0133' AS PROC
INTO #af29_6_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=51021 AND t1.C_CAGENTE='6' AND t1.C_ENTRADA='D' AND t1.C_SCN='AF.29')
   OR (t1.SECTOR=6 AND t1.C_CAGENTE='53' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.29')
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af29_6_51022))


In [ ]:
# DATO EFECTIVO DE MINERAS PARA COMPARAR CON ESTIMACIÓN ANTERIOR
work_conn.execute(text("DROP TABLE IF EXISTS #af29_mineras"))
sql_af29_mineras = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       SUM(t1.DATO*-1) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0133' AS PROC
INTO #af29_mineras
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=51022 AND t1.C_CAGENTE='6' AND t1.C_ENTRADA='D' AND t1.C_SCN='AF.29' AND t1.MONEDA='D')
GROUP BY t1.MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af29_mineras))


In [ ]:
# JUNTA AMBAS TABLAS DE MINERAS PARA CALCULAR DELTA A IMPUTAR EN AF29 (append server-side)
cols_mineras = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
work_conn.execute(text(f"INSERT INTO #af29_6_51022 ({cols_mineras}) SELECT {cols_mineras} FROM #af29_mineras"))


In [ ]:
# CALCULA DELTA MINERAS A IMPUTAR EN AF29 CON CA6
work_conn.execute(text("DROP TABLE IF EXISTS #delta_mineras"))
sql_delta_mineras = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       t1.FUENTE,
       t1.PROC
INTO #delta_mineras
FROM #af29_6_51022 t1
GROUP BY t1.MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE, t1.PROC
"""
work_conn.execute(text(sql_delta_mineras))


In [ ]:
# AJUSTE EN AF29 DE SECTOR MINERAS CON CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #delta_321"))
sql_delta_321 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       (t1.DATO*-1) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       t1.FUENTE,
       '0133b' AS PROC
INTO #delta_321
FROM #delta_mineras t1
"""
work_conn.execute(text(sql_delta_321))


In [ ]:
# APPEND server-side de AF29_6_511, DELTA_MINERAS y DELTA_321 a TABLAS.BD_CTSI
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_6_511"))
_log("APPEND TABLAS.BD_CTSI (AF29_6_511)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #delta_mineras"))
_log("APPEND TABLAS.BD_CTSI (DELTA_MINERAS)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #delta_321"))
_log("APPEND TABLAS.BD_CTSI (DELTA_321)", res.rowcount)


In [ ]:
for t in ["#af29_6_51022", "#af29_6_511", "#delta_321", "#delta_mineras", "#af29_mineras"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA AF5 DEL RM CON CA 351 CON AF5+AF521 REPORTADO POR EL SECTOR 351 CON CA 6.
# MANDA EL DATO DEL RM, DIFERENCIA LA IMPUTA EN AF.5 DEL SECTOR 351 CON CA 6.
# CONTRA AJUSTE LO HACE EN ACTIVO AF32 DEL SECTOR 351 CON CONTRAGENTE 6
work_conn.execute(text("DROP TABLE IF EXISTS #af5_351_6"))
sql_af5_351_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       351 AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA='D' THEN t1.DATO*-1 ELSE t1.DATO END) AS DATO,
       'AF.5' AS C_SCN,
       'Acciones y otras participaciones de capital' AS N_SCN,
       'PS' AS FUENTE,
       '0129d' AS PROC
INTO #af5_351_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=6 AND t1.C_CAGENTE='351' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.5')
   OR (t1.SECTOR=351 AND t1.C_CAGENTE='6' AND t1.C_ENTRADA='D' AND t1.C_SCN IN ('AF.5','AF.522'))
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA
"""
work_conn.execute(text(sql_af5_351_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_351_6"))
sql_af32_351_6 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       (t1.DATO*-1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       t1.FUENTE,
       '0130d' AS PROC
INTO #af32_351_6
FROM #af5_351_6 t1
"""
work_conn.execute(text(sql_af32_351_6))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_351_6"))
_log("APPEND TABLAS.BD_CTSI (AF5_351_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_351_6"))
_log("APPEND TABLAS.BD_CTSI (AF32_351_6)", res.rowcount)


In [ ]:
for t in ["#af32_351_6", "#af5_351_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA EN FP EL INST AF.29 DEL ACTIVO CONTRAGENTE DESDE 6 A 321
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321' WHERE C_CAGENTE='6' AND SECTOR=:sector AND C_SCN=:c_scn"), {"sector": 34, "c_scn": "AF.29"})
    _log("UPDATE TABLAS.BD_CTSI (FP AF.29 6->321)", res.rowcount)


In [ ]:
# ACTUALIZA EN CCAF EL INST AF.29 DEL ACTIVO CONTRAGENTE DESDE 9 A 511
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='511' WHERE C_CAGENTE='9' AND SECTOR=:sector AND C_SCN=:c_scn"), {"sector": 411, "c_scn": "AF.29"})
    _log("UPDATE TABLAS.BD_CTSI (CCAF AF.29 9->511)", res.rowcount)


In [ ]:
# ACTUALIZA EN COOP EL INST AF.29 DEL PASIVO CONTRAGENTE DESDE 321 A 511
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='511' WHERE C_CAGENTE='321' AND SECTOR=:sector AND C_SCN=:c_scn AND C_ENTRADA=:c_entrada"), {"sector": 322, "c_scn": "AF.29", "c_entrada": "H"})
    _log("UPDATE TABLAS.BD_CTSI (COOP AF.29 321->511)", res.rowcount)


In [ ]:
# ACTUALIZA EN GOB GRAL EL INST AF.29 CONTRAGENTE DESDE 3 A 321
with engine.begin() as conn:
    res = conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='321', PROC=:proc WHERE C_CAGENTE='3' AND SECTOR=:sector AND C_SCN=:c_scn"), {"proc": "0137", "sector": 41, "c_scn": "AF.29"})
    _log("UPDATE TABLAS.BD_CTSI (GOB GRAL AF.29 3->321)", res.rowcount)


In [ ]:
# DEPOSITOS AF29 ENTRE BANCO CENTRAL-PASIVO Y BCOS COM-ACTIVO. Respeta dato de sector 31 y ajusta contra AF.29 contragente 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_321"))
sql_af29_31_321 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       321 AS SECTOR,
       '31' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.SECTOR=321 THEN t1.DATO*-1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0146' AS PROC
INTO #af29_31_321
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=321 AND t1.C_CAGENTE='31' AND t1.C_ENTRADA='D' AND t1.C_SCN='AF.29')
   OR (t1.SECTOR=31 AND t1.C_CAGENTE='321' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.29')
GROUP BY t1.[AÑO], t1.TRIM,
         CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END,
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af29_31_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_321_321"))
sql_af29_321_321 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       t1.DATO*-1 AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       t1.FUENTE,
       '0147' AS PROC
INTO #af29_321_321
FROM #af29_31_321 t1
"""
work_conn.execute(text(sql_af29_321_321))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_31_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_31_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_321_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_321_321)", res.rowcount)


In [ ]:
for t in ["#af29_31_321", "#af29_321_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE BANCO CENTRAL-PASIVO Y GOB-ACTIVO. Respeta dato de sector 31 y ajusta contra AF.29 contragente 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_41"))
sql_af29_31_41 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0150' AS PROC
INTO #af29_31_41
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=31 AND t1.C_CAGENTE='41' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.29')
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af29_31_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_321_41"))
sql_af29_321_41 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       t1.DATO*-1 AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0151' AS PROC
INTO #af29_321_41
FROM #af29_31_41 t1
"""
work_conn.execute(text(sql_af29_321_41))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_31_41"))
_log("APPEND TABLAS.BD_CTSI (AF29_31_41)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_321_41"))
_log("APPEND TABLAS.BD_CTSI (AF29_321_41)", res.rowcount)


In [ ]:
for t in ["#af29_31_41", "#af29_321_41"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE BANCO CENTRAL-PASIVO Y RM-ACTIVO. Respeta dato de sector 31 y ajusta contra AF.71
work_conn.execute(text("DROP TABLE IF EXISTS #af29_31_6"))
sql_af29_31_6 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.SECTOR=6 THEN t1.DATO*-1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0155' AS PROC
INTO #af29_31_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=6 AND t1.C_CAGENTE='31' AND t1.C_ENTRADA='D' AND t1.C_SCN='AF.29')
   OR (t1.SECTOR=31 AND t1.C_CAGENTE='6' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.29')
GROUP BY t1.[AÑO], t1.TRIM,
         CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END,
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af29_31_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_31_6"))
sql_af71_31_6 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       t1.DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       t1.FUENTE,
       '0156' AS PROC
INTO #af71_31_6
FROM #af29_31_6 t1
"""
work_conn.execute(text(sql_af71_31_6))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF29_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF71_31_6)", res.rowcount)


In [ ]:
for t in ["#af29_31_6", "#af71_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# DEPOSITOS AF29 ENTRE FP-PASIVO Y AFP-ACTIVO. Respeta dato de sector 34 y ajusta contra AF.29 contragente 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_34_361"))
sql_af29_34_361 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       361 AS SECTOR,
       '34' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.SECTOR=361 THEN t1.DATO*-1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0156i' AS PROC
INTO #af29_34_361
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.SECTOR=361 AND t1.C_CAGENTE='34' AND t1.C_ENTRADA='D' AND t1.C_SCN='AF.612')
   OR (t1.SECTOR=34 AND t1.C_CAGENTE='361' AND t1.C_ENTRADA='H' AND t1.C_SCN='AF.612')
GROUP BY t1.[AÑO], t1.TRIM,
         CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE t1.C_CUENTA END,
         t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af29_34_361))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af29_361_321"))
sql_af29_361_321 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       t1.DATO*-1 AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       t1.FUENTE,
       '0156j' AS PROC
INTO #af29_361_321
FROM #af29_34_361 t1
"""
work_conn.execute(text(sql_af29_361_321))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_34_361"))
_log("APPEND TABLAS.BD_CTSI (AF29_34_361)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_361_321"))
_log("APPEND TABLAS.BD_CTSI (AF29_361_321)", res.rowcount)


In [ ]:
for t in ["#af29_34_361", "#af29_361_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S2_03_Reservas&Capital

Reasigna la reserva de fondos de pensiones del sector 412 al 511, redistribuye el pasivo AF62 del sector 352 hacia los activos 321/51022/41/511 según ratios fijos (20/45/2/33%), y ajusta a valor de mercado e imputa inversiones en AF5 de holdings/casas matrices contra bancos, entre holdings, y contra empresas supervisadas usando ratios de patrimonio/inversión a valor libro

*confianza: low · verificador: revise · SAS: PROC SQL UPDATE/CREATE TABLE + PROC DATASETS APPEND sobre TABLAS.BD_CTSI, con joins contra archivos .sas7bdat externos de ratios*

In [ ]:
# ========= S2_03_Reservas&Capital =========
# COMPRIME TABLA PRINCIPAL
# SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;
# COMPRESS=YES es una opción de almacenamiento del dataset SAS: reescribe la
# tabla con exactamente las mismas filas para guardarlas comprimidas. En SQL
# Server la compresión es DDL (ALTER TABLE ... REBUILD WITH DATA_COMPRESSION),
# se decide una vez con el DBA y no desde el pipeline: el paso no tiene efecto
# de datos que traducir. Queda registrado y sin ejecutar — replicarlo con un
# borrado + reinserción deja TABLAS.BD_CTSI vacía si la celda se corta a la mitad.


In [ ]:
# ACTUALIZA CONTRAGENTE EN RESERVA DE FONDO DE PENSIONES DEL SECTOR 412 DE CA 9 A 511
res = work_conn.execute(text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE='511', PROC='0160' WHERE C_CAGENTE='9' AND SECTOR=412 AND C_SCN='AF.612'"))
_log("UPDATE TABLAS.dbo.BD_CTSI (contragente 412->511)", res.rowcount)


In [ ]:
# AF62 DEL PASIVO DEL SECTOR 352 LO ASIGNA AL ACTIVO DE LOS SECTORES 321(20%),51022(45%),41(2%) Y 511(33%)
for tmp in ["#af62_352_321", "#af62_352_51022", "#af62_352_41", "#af62_352_511"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp}"))

sql_af62_352_321 = """
SELECT MONEDA, [AÑO], TRIM,
       321 AS SECTOR,
       '352' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO*0.2) AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0162' AS PROC
INTO #af62_352_321
FROM TABLAS.dbo.BD_CTSI
WHERE C_SCN='AF.62'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af62_352_321))

sql_af62_352_51022 = """
SELECT MONEDA, [AÑO], TRIM,
       51022 AS SECTOR,
       '352' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO*0.45) AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0163' AS PROC
INTO #af62_352_51022
FROM TABLAS.dbo.BD_CTSI
WHERE C_SCN='AF.62'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af62_352_51022))

sql_af62_352_41 = """
SELECT MONEDA, [AÑO], TRIM,
       41 AS SECTOR,
       '352' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO*0.02) AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0164' AS PROC
INTO #af62_352_41
FROM TABLAS.dbo.BD_CTSI
WHERE C_SCN='AF.62'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af62_352_41))

sql_af62_352_511 = """
SELECT MONEDA, [AÑO], TRIM,
       511 AS SECTOR,
       '352' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO*0.33) AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0165' AS PROC
INTO #af62_352_511
FROM TABLAS.dbo.BD_CTSI
WHERE C_SCN='AF.62'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af62_352_511))


In [ ]:
# APPEND server-side de las 4 particiones AF62_352_* hacia TABLAS.BD_CTSI
cols_af62 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp in ["#af62_352_321", "#af62_352_51022", "#af62_352_41", "#af62_352_511"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af62}) SELECT {cols_af62} FROM {tmp}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp}", res.rowcount)
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp}"))


In [ ]:
# AF62 CONTRA ACTIVO DE SECTORES 41 Y 321 EN INST AF71 CON CA 352 DEBIDO A IMPUTACIONES ANTERIORES
work_conn.execute(text("DROP TABLE IF EXISTS #af62_41_321"))
sql_af62_41_321 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO*-1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0166' AS PROC
INTO #af62_41_321
FROM TABLAS.dbo.BD_CTSI
WHERE C_SCN='AF.62' AND SECTOR IN (41,321) AND C_ENTRADA='D' AND FUENTE='PS'
GROUP BY MONEDA, [AÑO], TRIM, C_CUENTA, C_ENTRADA, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af62_41_321))

cols_af62_41_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_41_321 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af62_41_321}) SELECT {cols_af62_41_321} FROM #af62_41_321"
res = work_conn.execute(text(sql_append_41_321))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af62_41_321", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af62_41_321"))


In [ ]:
# CALCULA VALOR DE MERCADO DEL ACTIVO DE LOS HOLDINGS Y CASAS MATRICES CON BANCOS
# ratio_inv_hc_c.sas7bdat: archivo externo fuera del alcance de rutas relativas conocidas del proyecto (ver M-001).
# Se levanta el hueco explícito: no se inventa una ruta ni se omite la tabla contractual.
raise NotImplementedError(
    "WORK.VM_AF5_HC_D y WORK.VM_AF5_HC_RP requieren el archivo externo "
    "ratio_inv_hc_c.sas7bdat (ratio de valor de mercado por AÑO/TRIM/SECTOR/C_CAGENTE/C_ENTRADA/C_CUENTA/C_SCN). "
    "El SAS original lo referencia con ruta absoluta "
    "'/sasdata/BCCH/GEM_DCNI/02_CNSI/08_SI_EMP/36907_37_HOLDING/ratio_inv_hc_c.sas7bdat'; "
    "no hay ruta relativa ni tabla de BD equivalente declarada para este archivo en el contexto del proyecto. "
    "Falta: (1) migrar el archivo a una ruta relativa del workspace o a una tabla de BD, "
    "(2) leerlo con pyreadstat.read_sas7bdat, (3) hacer el join descrito por SECTOR/C_CAGENTE/C_ENTRADA/C_CUENTA/C_SCN, "
    "(4) calcular DATO=(DATO*RATIO)-DATO agrupado, y (5) el bloque REC PRECIO derivado (VM_AF5_HC_RP) y el APPEND de ambas tablas a TABLAS.BD_CTSI con PROC 0300/0303."
)


In [ ]:
# IMPUTA INVERSIONES EN AF5 DE HOLDINGS Y CASAS MATRICES ENTRE EL SECTOR USANDO INFO DEL PATRIMONIO A VALOR LIBRO. AJUSTA IMPUTACIÓN EN CA 2.
# ratio_pat_hc_c.sas7bdat: archivo externo fuera del alcance de rutas relativas conocidas del proyecto (ver M-001).
raise NotImplementedError(
    "La cadena WORK.VM_AF5_HC_P / WORK.VL_AF5_HC_P / WORK.RP_AF5_HC_P / WORK.AG_AF5_HC_P / "
    "WORK.INV_AF5_HC_HC / WORK.IMP_AF5_HC_HC / WORK.IMP_AF5_HC_2 (PROC 0304/0304b) requiere el archivo externo "
    "ratio_pat_hc_c.sas7bdat (ratio de patrimonio a valor libro por AÑO/TRIM/SECTOR/C_CAGENTE/C_ENTRADA/C_CUENTA/C_SCN). "
    "Ruta absoluta original: '/sasdata/BCCH/GEM_DCNI/02_CNSI/08_SI_EMP/36907_37_HOLDING/ratio_pat_hc_c.sas7bdat'; "
    "sin ruta relativa ni tabla de BD equivalente declarada. Falta migrar el archivo y replicar: "
    "VM_AF5_HC_P (patrimonio VM sector 37/36907 con C_CAGENTE 37/36907, C_ENTRADA=H), "
    "VL_AF5_HC_P (DATO/RATIO - DATO vía join con el archivo), "
    "RP_AF5_HC_P (rec precio: -DATO si C_CUENTA='Bce Inicio', DATO si no), "
    "UNION de las 3 tablas, AG_AF5_HC_P (invierte SECTOR<->C_CAGENTE vía INPUT/PUT), "
    "INV_AF5_HC_HC (-1 * inversión con C_ENTRADA='D'), UNION con AG_AF5_HC_P, "
    "IMP_AF5_HC_HC (agregado, descartando DATO=0) e IMP_AF5_HC_2 (mismo monto invertido en C_CAGENTE='2'), "
    "y el APPEND final de IMP_AF5_HC_HC e IMP_AF5_HC_2 a TABLAS.BD_CTSI con PROC 0304/0304b."
)


In [ ]:
# CALCULA VALOR DE MERCADO DEL ACTIVO DE LOS HOLDINGS Y CASAS MATRICES CON HOLDINGS Y CASAS MATRICES (INTRA)
# Reusa ratio_pat_hc_c.sas7bdat: mismo archivo externo sin ruta relativa conocida.
raise NotImplementedError(
    "WORK.VM_AF5_HC_INTRA y WORK.RP_AF5_HC_INTRA (PROC 0304c) requieren el mismo archivo externo "
    "ratio_pat_hc_c.sas7bdat que el bloque anterior, con join cruzado invirtiendo SECTOR/C_CAGENTE "
    "(T1.SECTOR=INPUT(T2.C_CAGENTE,BEST5.) AND T1.C_CAGENTE=LEFT(PUT(T2.SECTOR,CHAR6.))), filtro "
    "C_ENTRADA='D' AND SECTOR IN (37,36907) AND C_CAGENTE IN ('37','36907'). Falta migrar el archivo "
    "a ruta relativa o tabla de BD para calcular VM_AF5_HC_INTRA (DATO*T2.DATO - DATO), derivar "
    "RP_AF5_HC_INTRA (rec precio) y hacer el APPEND de ambas a TABLAS.BD_CTSI con PROC 0304c."
)


In [ ]:
# IMPUTA INVERSIONES EN AF5 DE HOLDINGS/CASAS MATRICES CON EMPRESAS SUPERVISADAS, USANDO PATRIMONIO A VALOR LIBRO. AJUSTA IMPUTACIÓN EN CA 2.
# ratio_AF5_emp_hold_n.sas7bdat: archivo externo fuera del alcance de rutas relativas conocidas del proyecto.
raise NotImplementedError(
    "La cadena WORK.VM_AF5_HC_EMP / WORK.VL_AF5_HC_EMP / WORK.RP_AF5_HC_EMP / WORK.AG_AF5_HC_EMP / "
    "WORK.INV_AF5_HC_EMP / WORK.IMP_AF5_HC_EMP / WORK.IMP_AF5_HC_2 (PROC 0305/0305b) requiere el archivo externo "
    "ratio_AF5_emp_hold_n.sas7bdat (ratio patrimonio/inversión empresas-holding por AÑO/TRIM/SECTOR/C_CAGENTE/C_CUENTA). "
    "Ruta absoluta original: '/sasdata/BCCH/GEM_DCNI/02_CNSI/08_SI_EMP/S11_EMPRESAS/ratio_AF5_emp_hold_n.sas7bdat'; "
    "sin ruta relativa ni tabla de BD equivalente declarada. Falta migrar el archivo y replicar: "
    "VM_AF5_HC_EMP (patrimonio VM sector 51021/5101 con C_CAGENTE 37/36907, C_ENTRADA=H), "
    "VL_AF5_HC_EMP (DATO/RATIO - DATO), RP_AF5_HC_EMP (rec precio), UNION de las 3, "
    "AG_AF5_HC_EMP (invierte SECTOR<->C_CAGENTE), INV_AF5_HC_EMP (-1 * inversión sector 37/36907 con "
    "C_CAGENTE 51021/5101, C_ENTRADA=D), UNION con AG_AF5_HC_EMP, IMP_AF5_HC_EMP (agregado, descarta DATO=0), "
    "IMP_AF5_HC_2 (mismo monto en C_CAGENTE='2'), y el APPEND final de IMP_AF5_HC_EMP e IMP_AF5_HC_2 a "
    "TABLAS.BD_CTSI con PROC 0305/0305b."
)


In [ ]:
# CALCULA VALOR DE MERCADO DEL ACTIVO DE LOS HOLDINGS Y CASAS MATRICES CON EMPRESAS SUPERVISADAS
# Reusa ratio_AF5_emp_hold_n.sas7bdat: mismo archivo externo sin ruta relativa conocida.
# CIERRE 2021: FALTABA INGRESAR COMO FILTRO EL C_SCN (comentario original del SAS sobre este bloque)
raise NotImplementedError(
    "WORK.VM_AF5_HC_ES y WORK.RP_AF5_HC_ES (PROC 0305c) requieren el mismo archivo externo "
    "ratio_AF5_emp_hold_n.sas7bdat, join cruzado invirtiendo SECTOR/C_CAGENTE, filtro "
    "C_ENTRADA='D' AND SECTOR IN (37,36907) AND C_CAGENTE IN ('51021','5101') AND C_SCN='AF.5' "
    "(nota original: 'CIERRE 2021: FALTABA INGRESAR COMO FILTRO EL C_SCN'). Falta migrar el archivo "
    "a ruta relativa o tabla de BD para calcular VM_AF5_HC_ES (DATO*RATIO - DATO), derivar RP_AF5_HC_ES "
    "y hacer el APPEND de ambas a TABLAS.BD_CTSI con PROC 0305c."
)


## S2_04_HH&Dep

Reclasifica contragentes de patrimonio de empresas a hogares (incl. prorrateo 1/3 del sector 36912), e imputa activo/pasivo de hogares y ajustes de depósitos (AF.22/AF.29/AF.7) por contrapartida, acumulando los ajustes en la tabla maestra BD_CTSI

*confianza: medium · verificador: approve · SAS: PROC SQL UPDATE/DELETE + CREATE TABLE con agregaciones sobre tabla base + PROC APPEND FORCE reiterado*

In [ ]:
# ========= S2_04_HH&Dep =========
# COMPRIME TABLA PRINCIPAL: en SQL Server no existe COMPRESS=YES de SAS; no hay equivalente de compresión de tabla a nivel de fila que replicar aquí. La tabla ya existe en la BD, no se recrea.


In [ ]:
# CAMBIA CONTRAGENTE 511 EN EL PATRIMONIO DE EMPRESAS SUPERVISADAS A 53 PARA RECALCULAR LA PARTE QUE VA A HOGARES EN BASE AL % DEL TOTAL DEL PAT DE EMPRESAS
with engine.begin() as conn:
    res = conn.execute(
        text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0014' WHERE C_CAGENTE = '511' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 51021")
    )
    _log("UPDATE TABLAS.dbo.BD_CTSI (0014)", res.rowcount)


In [ ]:
# CIERRE 2020. ELIMINA PASIVO Y ACTIVO AF7 DE EMPRESAS HOGARES CON CONTRAGENTE HOGARES
with engine.begin() as conn:
    res = conn.execute(
        text("DELETE FROM TABLAS.dbo.BD_CTSI WHERE SECTOR = 5111 AND C_SCN = 'AF.7' AND C_CAGENTE = '511'")
    )
    _log("DELETE TABLAS.dbo.BD_CTSI (AF.7 sector 5111)", res.rowcount)


In [ ]:
# CIERRE 2021: ELIMINA CONTRAGENTE HOGARES EN PATRIMONIO DE RESTO DE EMPRESAS SECTOR 51022, PORQUE LE GENERA MUCHA VOLATILIDAD AL SECTOR
with engine.begin() as conn:
    res = conn.execute(
        text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0014a' WHERE C_CAGENTE = '511' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 51022")
    )
    _log("UPDATE TABLAS.dbo.BD_CTSI (0014a)", res.rowcount)


In [ ]:
# CIERRE 2021: EN EMPRESAS HOGARES (SECTOR 5111) DEJA TODO EL PATRIMONIO PARA HOGARES (LO DEJA EN CA 53 PARA QUE LO IMPUTE PORQUE ASÍ ESTABA ANTES
with engine.begin() as conn:
    res = conn.execute(
        text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0014b' WHERE C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 5111")
    )
    _log("UPDATE TABLAS.dbo.BD_CTSI (0014b)", res.rowcount)


In [ ]:
# CIERRE 2021: AJUSTA PATRIMONIO DE SECTOR 36912 ASIGNADO A HOGARES EN 1/3 YA QUE ES MUY VOLÁTIL Y DISTORSIONA LA CTA FINANCIERA. LA DIFERENCIA SE LLEVA A CONTRAGENTE 36912 y 53
# calcula patrimonio ajustado a hogares
work_conn.execute(text("DROP TABLE IF EXISTS #pat_36912_hh"))
sql_pat_36912_hh = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) / 3.0 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0014d' AS PROC
INTO #pat_36912_hh
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CAGENTE = '511' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 36912
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
         T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_pat_36912_hh))


In [ ]:
# deja 1/3 en ca 36912
work_conn.execute(text("DROP TABLE IF EXISTS #pat_36912_36"))
sql_pat_36912_36 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '36912' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * 1.0 / 3.0 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0014e' AS PROC
INTO #pat_36912_36
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CAGENTE = '511' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 36912
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
         T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_pat_36912_36))


In [ ]:
# deja 1/3 en ca 53
work_conn.execute(text("DROP TABLE IF EXISTS #pat_36912_53"))
sql_pat_36912_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * 1.0 / 3.0 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0014e' AS PROC
INTO #pat_36912_53
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CAGENTE = '511' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 36912
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
         T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_pat_36912_53))


In [ ]:
# elimina patrimonio inicial asignado a hogares
work_conn.execute(text("DROP TABLE IF EXISTS #pat_36912_hh_elimina"))
sql_pat_36912_hh_elimina = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0014f' AS PROC
INTO #pat_36912_hh_elimina
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_CAGENTE = '511' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND T1.SECTOR = 36912
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE,
         T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_pat_36912_hh_elimina))
pat_36912_hh = pd.read_sql(text("SELECT * FROM #pat_36912_hh"), work_conn)
pat_36912_36 = pd.read_sql(text("SELECT * FROM #pat_36912_36"), work_conn)
pat_36912_53 = pd.read_sql(text("SELECT * FROM #pat_36912_53"), work_conn)
pat_36912_hh_elimina = pd.read_sql(text("SELECT * FROM #pat_36912_hh_elimina"), work_conn)
_log("pat_36912_hh_elimina", pat_36912_hh_elimina)


In [ ]:
# APPEND (proc datasets ... force) de las 4 tablas de ajuste 36912 a la tabla principal, server-side
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_name in ["#pat_36912_hh", "#pat_36912_36", "#pat_36912_53", "#pat_36912_hh_elimina"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_name}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)


In [ ]:
# limpia temporales de sesión (equivalente a DROP TABLE pat_36912_hh, pat_36912_36, pat_36912_hh_elimina de SAS)
for tmp_name in ["#pat_36912_hh", "#pat_36912_36", "#pat_36912_53", "#pat_36912_hh_elimina"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_name}"))


In [ ]:
# IMPUTA ACTIVO DE HOGARES CON INFO DE CONTRAPARTIDA
work_conn.execute(text("DROP TABLE IF EXISTS #activo_hh"))
sql_activo_hh = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0015' AS PROC
INTO #activo_hh
FROM TABLAS.dbo.BD_CTSI T1
WHERE C_CAGENTE IN ('5', '511', 'S14') AND C_ENTRADA = 'H'
  AND C_CUENTA IN ('Bce Final', 'Financiera', 'Bce Inicio', 'Rec Precio', 'Rec Precio Reaj', 'Rec Volumen')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR
"""
work_conn.execute(text(sql_activo_hh))


In [ ]:
# UPDATE sobre la temporal: cambia contragente 5111 a 51022 para AF.5
work_conn.execute(text("UPDATE #activo_hh SET C_CAGENTE = '51022' WHERE C_CAGENTE = '5111' AND C_SCN = 'AF.5'"))
activo_hh = pd.read_sql(text("SELECT * FROM #activo_hh"), work_conn)
_log("activo_hh", activo_hh)


In [ ]:
# append server-side de activo_hh a la tabla principal
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #activo_hh"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #activo_hh", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #activo_hh"))


In [ ]:
# IMPUTA PASIVO DE HOGARES CON INFO DE CONTRAPARTIDA
work_conn.execute(text("DROP TABLE IF EXISTS #pasivo_hh"))
sql_pasivo_hh = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       LTRIM(RTRIM(CAST(T1.SECTOR AS varchar(11)))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0016' AS PROC
INTO #pasivo_hh
FROM TABLAS.dbo.BD_CTSI T1
WHERE C_CAGENTE IN ('5', '511', 'S14') AND C_ENTRADA = 'D'
  AND C_CUENTA IN ('Bce Final', 'Financiera', 'Bce Inicio', 'Rec Precio', 'Rec Precio Reaj', 'Rec Volumen')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR
"""
work_conn.execute(text(sql_pasivo_hh))
pasivo_hh = pd.read_sql(text("SELECT * FROM #pasivo_hh"), work_conn)
_log("pasivo_hh", pasivo_hh)


In [ ]:
# append server-side de pasivo_hh a la tabla principal
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #pasivo_hh"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #pasivo_hh", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #pasivo_hh"))


In [ ]:
# DEPOSITOS AF29 ENTRE BCOS-PASIVO Y MUNICIPALIDADES-ACTIVO. RESPETA DATO DE SECTOR 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_42_321"))
sql_af29_42_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       42 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0142' AS PROC
INTO #af29_42_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR = 321 AND T1.C_CAGENTE = '42' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_42_321))
af29_42_321 = pd.read_sql(text("SELECT * FROM #af29_42_321"), work_conn)
_log("af29_42_321", af29_42_321)


In [ ]:
# append server-side de af29_42_321 a la tabla principal
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_42_321"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af29_42_321", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af29_42_321"))


In [ ]:
# AJUSTA TOTAL DE DEPÓSITOS BANCARIOS IMPUTANDO DIF ENTRE ACTIVO Y PASIVO EN EL ACTIVO DEL SECTOR 51022 CON CONTRAGENTE 321
work_conn.execute(text("DROP TABLE IF EXISTS #af22_total"))
sql_af22_total = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       CASE WHEN T1.SECTOR = 321 AND T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio'
            WHEN T1.SECTOR <> 321 AND T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio'
            ELSE T1.C_CUENTA END AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Depósitos' AS N_SCN,
       'PS' AS FUENTE,
       '0118' AS PROC
INTO #af22_total
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.22' AND T1.C_CAGENTE = '321')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.22' AND T1.SECTOR = 321)
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.SECTOR = 321 AND T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio'
              WHEN T1.SECTOR <> 321 AND T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio'
              ELSE T1.C_CUENTA END,
         T1.C_SCN
"""
work_conn.execute(text(sql_af22_total))
af22_total = pd.read_sql(text("SELECT * FROM #af22_total"), work_conn)
_log("af22_total", af22_total)


In [ ]:
# CIERRE 2020. EL AJUSTE ANTERIOR SE LLEVA A ACTIVO AF29 SECTOR 51022 CA 321 PARA EQUILIBRAR EL BALANCE
work_conn.execute(text("DROP TABLE IF EXISTS #af29_aj"))
sql_af29_aj = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.29' AS C_SCN,
       'Otros depósitos' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af29_aj
FROM #af22_total T1
"""
work_conn.execute(text(sql_af29_aj))
af29_aj = pd.read_sql(text("SELECT * FROM #af29_aj"), work_conn)
_log("af29_aj", af29_aj)


In [ ]:
# append server-side de af22_total y af29_aj a la tabla principal
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af22_total"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af22_total", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af29_aj", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af22_total"))
work_conn.execute(text("DROP TABLE IF EXISTS #af29_aj"))


In [ ]:
# AJUSTA TOTAL DE OTROS DEPÓSITOS IMPUTANDO DIF ENTRE ACTIVO Y PASIVO EN EL ACTIVO DEL SECTOR 51022 CON CONTRAGENTE 321
work_conn.execute(text("DROP TABLE IF EXISTS #af29_total"))
sql_af29_total = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0159' AS PROC
INTO #af29_total
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.29') OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.29')
GROUP BY T1.[AÑO], T1.TRIM,
         T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af29_total))
af29_total = pd.read_sql(text("SELECT * FROM #af29_total"), work_conn)
_log("af29_total", af29_total)


In [ ]:
# CIERRE 2020. AJUSTE ANTERIOR SE IMPUTA EN ACTIVO AF7 SECTOR 51022 CA 53, PARA EQUILIBRAR BALANCE
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste"))
sql_af7_ajuste = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af7_ajuste
FROM #af29_total T1
"""
work_conn.execute(text(sql_af7_ajuste))
af7_ajuste = pd.read_sql(text("SELECT * FROM #af7_ajuste"), work_conn)
_log("af7_ajuste", af7_ajuste)


In [ ]:
# append server-side de af29_total y af7_ajuste a la tabla principal, y limpieza final de temporales
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af29_total"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af29_total", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_ajuste"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_ajuste", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af29_total"))
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste"))


## S2_05_Ptmos

Reclasifica contrapartidas de préstamos de corto/largo plazo entre sectores por cierres 2021/2022 y concilia diferenciales activo-pasivo (bancos, RM, banco central, fondos de pensión, fondos de inversión, auxiliares, empresas, gobierno) imputando el ajuste en el sector contrario según la regla de negocio de cada tramo / Concilia y ajusta préstamos de corto y largo plazo entre múltiples pares de sectores (RM, gobierno, empresas, bancos, hogares) respetando el dato de un lado e imputando/ajustando el contragente correspondiente, incluyendo reclasificaciones de AF.41→AF.42 y derivaciones a AF.71/AF.7 como partida de ajuste / Imputa préstamos de largo plazo entre sectores (51021/51022/34/5111), reclasifica CA en el pasivo del sector 5111, reconcilia el activo de fondos de pensiones contra el pasivo de empresas privadas ajustando AF.7, y concilia el total de préstamos LP activo vs pasivo imputando la diferencia en el sector 51022 CA 321, con su correspondiente ajuste en AF.7

*confianza: medium · verificador: unverified · SAS: PROC SQL: UPDATE de reclasificación de contrapartidas + secuencia de CREATE TABLE/APPEND de conciliación de préstamos (AF.41/AF.42/AF.71) por sector + PROC SQL CREATE TABLE + PROC DATASETS APPEND + UPDATE sobre TABLAS.BD_CTSI, con tablas de sesión # (session_sql) + PROC SQL CREATE TABLE con SUM/GROUP BY + PROC DATASETS APPEND + PROC SQL UPDATE, sobre tablas temporales de sesión (#tmp)*

In [ ]:
# ========= S2_05_Ptmos =========
# COMPRIME TABLA PRINCIPAL
# SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;
# COMPRESS=YES es una opción de almacenamiento del dataset SAS: reescribe la
# tabla con exactamente las mismas filas para guardarlas comprimidas. En SQL
# Server la compresión es DDL (ALTER TABLE ... REBUILD WITH DATA_COMPRESSION),
# se decide una vez con el DBA y no desde el pipeline: el paso no tiene efecto
# de datos que traducir. Queda registrado y sin ejecutar — replicarlo con
# DELETE FROM + INSERT deja TABLAS.BD_CTSI vacía si la celda se corta a la mitad.


In [ ]:
# CIERRE 2021: CAMBIOS EN ALGUNOS CONTRAGENTE DE PRESTAMO PASIVOS DE EMPRESAS
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca "
    "WHERE SECTOR IN (51021,5101) AND C_SCN=:scn AND C_ENTRADA=:entrada "
    "AND C_CAGENTE IN (:ca1,:ca2,:ca3)"
), {"nueva_ca": "321", "scn": "AF.41", "entrada": "H", "ca1": "511", "ca2": "53", "ca3": "31"})
_log("UPDATE BD_CTSI C_CAGENTE->321 (51021,5101/AF.41/H)", res.rowcount)

res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_SCN=:nuevo_scn, N_SCN=:nuevo_nscn "
    "WHERE SECTOR IN (51021,5101) AND C_SCN=:scn AND C_ENTRADA=:entrada "
    "AND C_CAGENTE IN (:ca1,:ca2,:ca3)"
), {"nuevo_scn": "AF.42", "nuevo_nscn": "Préstamos a largo plazo", "scn": "AF.41", "entrada": "H",
    "ca1": "351", "ca2": "352", "ca3": "41"})
_log("UPDATE BD_CTSI AF.41->AF.42 (51021,5101/H/351,352,41)", res.rowcount)

res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca "
    "WHERE SECTOR IN (51021,5101) AND C_SCN=:scn AND C_ENTRADA=:entrada AND C_CAGENTE IN (:ca1)"
), {"nueva_ca": "321", "scn": "AF.42", "entrada": "H", "ca1": "511"})
_log("UPDATE BD_CTSI C_CAGENTE->321 (51021,5101/AF.42/H/511)", res.rowcount)

res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca "
    "WHERE SECTOR IN (37) AND C_SCN=:scn AND C_ENTRADA=:entrada AND C_CAGENTE IN (:ca1)"
), {"nueva_ca": "321", "scn": "AF.42", "entrada": "H", "ca1": "33222"})
_log("UPDATE BD_CTSI C_CAGENTE->321 (37/AF.42/H/33222)", res.rowcount)

res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_SCN=:nuevo_scn, N_SCN=:nuevo_nscn "
    "WHERE SECTOR IN (37) AND C_SCN=:scn AND C_ENTRADA=:entrada AND C_CAGENTE IN (:ca1,:ca2,:ca3)"
), {"nuevo_scn": "AF.42", "nuevo_nscn": "Préstamos a largo plazo", "scn": "AF.41", "entrada": "H",
    "ca1": "351", "ca2": "352", "ca3": "35"})
_log("UPDATE BD_CTSI AF.41->AF.42 (37/H/351,352,35)", res.rowcount)

res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca "
    "WHERE SECTOR IN (51021,5101) AND C_SCN=:scn AND C_ENTRADA=:entrada AND C_CAGENTE IN (:ca1)"
), {"nueva_ca": "321", "scn": "AF.42", "entrada": "H", "ca1": "31"})
_log("UPDATE BD_CTSI C_CAGENTE->321 (51021,5101/AF.42/H/31)", res.rowcount)


In [ ]:
# CIERRE 2022: CAMBIOS EN ALGUNOS CONTRAGENTE DE PRESTAMOS PASIVOS DE AUXILIARES
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca "
    "WHERE SECTOR IN (37) AND C_SCN=:scn AND C_ENTRADA=:entrada AND C_CAGENTE IN (:ca1)"
), {"nueva_ca": "321", "scn": "AF.41", "entrada": "H", "ca1": "2"})
_log("UPDATE BD_CTSI C_CAGENTE->321 (37/AF.41/H/2)", res.rowcount)


In [ ]:
# AJUSTA PTMO DE CP ENTRE FONDOS DE PENSIONES-ACTIVO Y RM-PASIVO. RESPETA DATO DE RM Y AJUSTA EN AF.5
# ACTIVO DE FP CON CONTRAGENTE RM. cierre 2022q3: incorpora ajuste en FP por cambios en datos de balanza
work_conn.execute(text("DROP TABLE IF EXISTS #af41_34_6"))
sql_af41_34_6 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        34 AS SECTOR,
        '6' AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.SECTOR=34 THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0179a' AS PROC
INTO #af41_34_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=34 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=6 AND T1.C_CAGENTE='34')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_34_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_34_6"))
sql_af5_34_6 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO*-1 AS DATO,
        'AF.5' AS C_SCN,
        'Acciones y otras participaciones de capital' AS N_SCN,
        'PS' AS FUENTE,
        '0180a' AS PROC
INTO #af5_34_6
FROM #af41_34_6 T1
"""
work_conn.execute(text(sql_af5_34_6))

cols_af41_34_6 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_34_6}) SELECT {cols_af41_34_6} FROM #af41_34_6"))
_log("APPEND BD_CTSI desde #af41_34_6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_34_6}) SELECT {cols_af41_34_6} FROM #af5_34_6"))
_log("APPEND BD_CTSI desde #af5_34_6", res.rowcount)
for t in ["#af41_34_6", "#af5_34_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO DE CP ENTRE BCO CENTRAL-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE BCO CENTRAL Y AJUSTA EN
# PTMO DE LP DE SECTOR 321 CON CA 31. cierre 2021: se concilia con cuentas originales
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_31"))
sql_af41_321_31 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        321 AS SECTOR,
        '31' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.SECTOR=321 THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0179' AS PROC
INTO #af41_321_31
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=31 AND T1.C_CAGENTE='321')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=321 AND T1.C_CAGENTE='31')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_321_31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_31"))
sql_af42_321_31 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        T1.DATO*-1 AS DATO,
        'AF.42' AS C_SCN,
        'Préstamos a largo plazo' AS N_SCN,
        'PS' AS FUENTE,
        '0180' AS PROC
INTO #af42_321_31
FROM #af41_321_31 T1
"""
work_conn.execute(text(sql_af42_321_31))

cols_af41_321_31 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321_31}) SELECT {cols_af41_321_31} FROM #af41_321_31"))
_log("APPEND BD_CTSI desde #af41_321_31", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321_31}) SELECT {cols_af41_321_31} FROM #af42_321_31"))
_log("APPEND BD_CTSI desde #af42_321_31", res.rowcount)
for t in ["#af41_321_31", "#af42_321_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO DE CP ENTRE BCO CENTRAL-ACTIVO Y RM-PASIVO. RESPETA DATO DE BCO CENTRAL Y AJUSTA EN
# PTMO DE LP DE SECTOR 6 CON CA 321. cierre 2021: no se cambia Rec Precio Reaj sino que se concilia con cuentas originales
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_31"))
sql_af41_6_31 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        6 AS SECTOR,
        '31' AS C_CAGENTE,
        CASE WHEN T1.SECTOR=31 AND T1.C_CUENTA IN ('Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.SECTOR=6 THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0180d' AS PROC
INTO #af41_6_31
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=31 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=6 AND T1.C_CAGENTE='31')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM,
         CASE WHEN T1.SECTOR=31 AND T1.C_CUENTA IN ('Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END,
         T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_6_31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_31"))
sql_af42_6_31 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '321' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO*-1 AS DATO,
        'AF.42' AS C_SCN,
        'Préstamos a largo plazo' AS N_SCN,
        'PS' AS FUENTE,
        '0180e' AS PROC
INTO #af42_6_31
FROM #af41_6_31 T1
"""
work_conn.execute(text(sql_af42_6_31))

cols_af41_6_31 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_31}) SELECT {cols_af41_6_31} FROM #af41_6_31"))
_log("APPEND BD_CTSI desde #af41_6_31", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_31}) SELECT {cols_af41_6_31} FROM #af42_6_31"))
_log("APPEND BD_CTSI desde #af42_6_31", res.rowcount)
for t in ["#af41_6_31", "#af42_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN PTMOS DE CP DEL SECTOR 412 DE CA 9 A 321
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca "
    "WHERE C_CAGENTE=:vieja_ca AND SECTOR=412 AND C_SCN=:scn"
), {"nueva_ca": "321", "vieja_ca": "9", "scn": "AF.41"})
_log("UPDATE BD_CTSI C_CAGENTE->321 (412/AF.41/9)", res.rowcount)

# CIERRE 2021: CAMBIA EN FONDOS DE INVERSIÓN PTMOS DE LARGO PLAZO OTORGADO A RESTO DEL MUNDO A PTMOS DE CORTO PLAZO
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_SCN=:nuevo_scn, N_SCN=:nuevo_nscn, PROC=:proc "
    "WHERE C_CAGENTE=:ca AND SECTOR=339011 AND C_SCN=:scn_viejo AND C_ENTRADA=:entrada"
), {"nuevo_scn": "AF.41", "nuevo_nscn": "Préstamos a corto plazo", "proc": "0210a",
    "ca": "6", "scn_viejo": "AF.42", "entrada": "D"})
_log("UPDATE BD_CTSI AF.42->AF.41 (339011/D/CA 6)", res.rowcount)


In [ ]:
# CIERRE 2021: IMPUTA PTMOS DE CP DE FONDOS DE INVERSIÓN ACTIVO CON CA RM EN EL PASIVO DEL RM.
# AJUSTA EN PASIVO DEL RM CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_fi"))
sql_af41_6_fi = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        6 AS SECTOR,
        '33901' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.SECTOR=6 THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0210b' AS PROC
INTO #af41_6_fi
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR IN (339011,3390102) AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('33901','33901/351'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_6_fi))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_53"))
sql_af41_6_53 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO*-1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0210c' AS PROC
INTO #af41_6_53
FROM #af41_6_fi T1
"""
work_conn.execute(text(sql_af41_6_53))

cols_af41_6_fi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_fi}) SELECT {cols_af41_6_fi} FROM #af41_6_fi"))
_log("APPEND BD_CTSI desde #af41_6_fi", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_fi}) SELECT {cols_af41_6_fi} FROM #af41_6_53"))
_log("APPEND BD_CTSI desde #af41_6_53", res.rowcount)
for t in ["#af41_6_fi", "#af41_6_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA PTMOS DE CP DE FONDOS DE INVERSIÓN-OFIS-AUX ACTIVO CON CA OFIS Y AUXILIARES EN EL PASIVO
# DE ESOS SECTORES. AJUSTA EN PASIVO DE OFIS Y AUX CON CA 321
# SELECCIONA ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi"))
sql_af41_oa_fi = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS float) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(8))) AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0210d' AS PROC
INTO #af41_oa_fi
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.41'
  AND (T1.SECTOR IN (339011,3390102,411,37) OR SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='36' OR SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='33')
  AND (SUBSTRING(T1.C_CAGENTE,1,2)='36' OR SUBSTRING(T1.C_CAGENTE,1,2)='33' OR T1.C_CAGENTE IN ('411','37'))
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         CAST(T1.C_CAGENTE AS float), LTRIM(CAST(T1.SECTOR AS varchar(8)))
"""
work_conn.execute(text(sql_af41_oa_fi))


In [ ]:
# SELECCIONA PASIVO: si alguno de estos sectores informa ptmos con fondos de inversión, se imputa solo la diferencia
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi_2"))
sql_af41_oa_fi_2 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO)*-1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0210d' AS PROC
INTO #af41_oa_fi_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.41'
  AND (SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='36' OR SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='33' OR T1.SECTOR IN (411,37))
  AND (SUBSTRING(T1.C_CAGENTE,1,2)='36' OR SUBSTRING(T1.C_CAGENTE,1,2)='33' OR T1.C_CAGENTE IN ('411','37'))
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af41_oa_fi_2))

# concatena activo + pasivo (equivalente a DATA AF41_OA_FI;SET AF41_OA_FI AF41_OA_FI_2;)
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi_all"))
work_conn.execute(text(
    "SELECT * INTO #af41_oa_fi_all FROM #af41_oa_fi "
    "UNION ALL SELECT * FROM #af41_oa_fi_2"
))


In [ ]:
# IMPUTA DIFERENCIAL DE PTMOS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_fi_def"))
sql_af41_oa_fi_def = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        SUM(DATO) AS DATO,
        C_SCN,
        N_SCN,
        FUENTE,
        PROC
INTO #af41_oa_fi_def
FROM #af41_oa_fi_all
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_af41_oa_fi_def))


In [ ]:
# AJUSTA EN CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af41_oa_321"))
sql_af41_oa_321 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        C_SCN,
        N_SCN,
        FUENTE,
        PROC
INTO #af41_oa_321
FROM #af41_oa_fi_def
"""
work_conn.execute(text(sql_af41_oa_321))

cols_af41_oa = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_oa}) SELECT {cols_af41_oa} FROM #af41_oa_fi_def"))
_log("APPEND BD_CTSI desde #af41_oa_fi_def", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_oa}) SELECT {cols_af41_oa} FROM #af41_oa_321"))
_log("APPEND BD_CTSI desde #af41_oa_321", res.rowcount)
for t in ["#af41_oa_321", "#af41_oa_fi_def", "#af41_oa_fi", "#af41_oa_fi_2", "#af41_oa_fi_all"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN PTMOS DE CP DEL SECTOR RM DE CA 53 A 321
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca, PROC=:proc "
    "WHERE C_CAGENTE=:vieja_ca AND SECTOR=6 AND C_SCN=:scn AND C_ENTRADA=:entrada"
), {"nueva_ca": "321", "proc": "0210", "vieja_ca": "53", "scn": "AF.41", "entrada": "H"})
_log("UPDATE BD_CTSI C_CAGENTE 53->321 (6/AF.41/H)", res.rowcount)


In [ ]:
# AJUSTA TOTAL DE PTMOS DE CP IMPUTANDO DIF ENTRE ACTIVO Y PASIVO EN EL PASIVO DEL SECTOR 51022
# CON CONTRAGENTE 321 Y AJUSTANDO CONTRA PTMO DE LP EN EL MISMO SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af41_total"))
sql_af41_total = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        '321' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0170' AS PROC
INTO #af41_total
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41') OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_total))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af41_aj"))
sql_af41_aj = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        'AF.42' AS C_SCN,
        'Préstamos a largo plazo' AS N_SCN,
        'PS' AS FUENTE,
        '0171' AS PROC
INTO #af41_aj
FROM #af41_total
"""
work_conn.execute(text(sql_af41_aj))

cols_af41_total = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_total}) SELECT {cols_af41_total} FROM #af41_total"))
_log("APPEND BD_CTSI desde #af41_total", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_total}) SELECT {cols_af41_total} FROM #af41_aj"))
_log("APPEND BD_CTSI desde #af41_aj", res.rowcount)
for t in ["#af41_total", "#af41_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE EN PTMOS DE CP DEL SECTOR RM DE CA 41 A 53
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca "
    "WHERE C_CAGENTE=:vieja_ca AND SECTOR=6 AND C_SCN=:scn"
), {"nueva_ca": "53", "vieja_ca": "41", "scn": "AF.41"})
_log("UPDATE BD_CTSI C_CAGENTE 41->53 (6/AF.41)", res.rowcount)


In [ ]:
# AJUSTA PTMOS DE CP ENTRE RM-ACTIVO Y 5101-PASIVO. RESPETA DATO DEL RM Y AJUSTA EN 5101 CON CONTRAGENTE 321
# CIERRE 2021: concilia cuentas con sus nombres originales
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_5101"))
sql_af41_6_5101 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        5101 AS SECTOR,
        '6' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0197' AS PROC
INTO #af41_6_5101
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=6 AND T1.C_CAGENTE='5101')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=5101 AND T1.C_CAGENTE='6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_6_5101))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_5101"))
sql_af41_321_5101 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        C_SCN,
        N_SCN,
        'PS' AS FUENTE,
        '0198' AS PROC
INTO #af41_321_5101
FROM #af41_6_5101
"""
work_conn.execute(text(sql_af41_321_5101))

cols_af41_6_5101 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_5101}) SELECT {cols_af41_6_5101} FROM #af41_6_5101"))
_log("APPEND BD_CTSI desde #af41_6_5101", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_5101}) SELECT {cols_af41_6_5101} FROM #af41_321_5101"))
_log("APPEND BD_CTSI desde #af41_321_5101", res.rowcount)
for t in ["#af41_6_5101", "#af41_321_5101"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA DE PTMOS DE CP ENTRE RM-ACTIVO Y 51021/51022 MINERAS-PASIVO. RESPETA DATO DEL RM Y AJUSTA EN 51021
# CON CONTRAGENTE 321. CIERRE 2020: agrega contragente 51021 (nuevo en cta del RM). CIERRE 2021: concilia con nombres originales
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_51021"))
sql_af41_6_51021 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51021 AS SECTOR,
        '6' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0202' AS PROC
INTO #af41_6_51021
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('53','51021'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=51021 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=51022 AND T1.MONEDA='D' AND T1.C_CAGENTE='6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_6_51021))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_51021"))
sql_af41_321_51021 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        C_SCN,
        N_SCN,
        'PS' AS FUENTE,
        '0203' AS PROC
INTO #af41_321_51021
FROM #af41_6_51021
"""
work_conn.execute(text(sql_af41_321_51021))

cols_af41_6_51021 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_51021}) SELECT {cols_af41_6_51021} FROM #af41_6_51021"))
_log("APPEND BD_CTSI desde #af41_6_51021", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_51021}) SELECT {cols_af41_6_51021} FROM #af41_321_51021"))
_log("APPEND BD_CTSI desde #af41_321_51021", res.rowcount)
for t in ["#af41_6_51021", "#af41_321_51021"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DE PTMOS DE CP RM-ACTIVO CON CA 31 USANDO INFO DE LO REPORTADO EN PASIVO DEL SECTOR 31 CON CA 6. AJUSTA CONTRA AF7
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_31_v2"))
sql_af41_6_31_v2 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        6 AS SECTOR,
        '31' AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0205' AS PROC
INTO #af41_6_31_v2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=31 AND T1.C_CAGENTE='6'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_6_31_v2))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_6_31"))
sql_af71_6_31 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        'AF.71' AS C_SCN,
        'Ajuste conciliación' AS N_SCN,
        'PS' AS FUENTE,
        '0206' AS PROC
INTO #af71_6_31
FROM #af41_6_31_v2
"""
work_conn.execute(text(sql_af71_6_31))

cols_af41_6_31_v2 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_31_v2}) SELECT {cols_af41_6_31_v2} FROM #af41_6_31_v2"))
_log("APPEND BD_CTSI desde #af41_6_31_v2", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_31_v2}) SELECT {cols_af41_6_31_v2} FROM #af71_6_31"))
_log("APPEND BD_CTSI desde #af71_6_31", res.rowcount)
for t in ["#af41_6_31_v2", "#af71_6_31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DE PTMOS DE CP EN PASIVO DE 51022 CON CA FMNM Y OFIS USANDO INFO DE LO REPORTADO EN ACTIVO DE ESOS SECTORES
# CIERRE 2021: solo considera contragentes de empresas y bancos (por cambio en FI también hay ptmos a OFIS y AUX; sin CA 334 porque se cambia a auxiliares)
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022"))
sql_af41_51022 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0208' AS PROC
INTO #af41_51022
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.41'
  AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='33'
  AND T1.SECTOR NOT IN (334,3390101)
  AND T1.C_CAGENTE IN ('53','51021','5101','51022','5102','9')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_af41_51022))


In [ ]:
# CIERRE 2021: incorpora el pasivo de empresas con OFIS para restar a la consulta anterior e imputar la diferencia
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022_2"))
sql_af41_51022_2 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO)*-1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0208' AS PROC
INTO #af41_51022_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR IN (51021,51022,5101)
  AND SUBSTRING(T1.C_CAGENTE,1,2)='33' AND T1.C_CAGENTE NOT IN ('3390101','334')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_51022_2))

# equivalente a DATA AF41_51022;SET AF41_51022 AF41_51022_2;
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022_all"))
work_conn.execute(text(
    "SELECT * INTO #af41_51022_all FROM #af41_51022 "
    "UNION ALL SELECT * FROM #af41_51022_2"
))


In [ ]:
# AGRUPA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51022_def"))
sql_af41_51022_def = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        SUM(DATO) AS DATO,
        C_SCN,
        N_SCN,
        FUENTE,
        PROC
INTO #af41_51022_def
FROM #af41_51022_all
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_af41_51022_def))


In [ ]:
# AJUSTA EN PTMOS DE CP DE SECTOR 51022 CONTRAGENTE BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_321"))
sql_af41_51_321 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        SUM(DATO)*-1 AS DATO,
        C_SCN,
        N_SCN,
        FUENTE,
        PROC
INTO #af41_51_321
FROM #af41_51022_def
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_af41_51_321))

cols_af41_51 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_51}) SELECT {cols_af41_51} FROM #af41_51_321"))
_log("APPEND BD_CTSI desde #af41_51_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_51}) SELECT {cols_af41_51} FROM #af41_51022_def"))
_log("APPEND BD_CTSI desde #af41_51022_def", res.rowcount)
for t in ["#af41_51022", "#af41_51022_2", "#af41_51022_all", "#af41_51022_def", "#af41_51_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA DE PTMOS DE CP EN PASIVO DE EMPRESAS CONTRAGENTE AUXILIARES USANDO LO REPORTADO EN EL
# ACTIVO DE AUXILIARES CON CONTRAGENTE EMPRESAS. AJUSTA EN PTMOS DE EMPRESAS CON BANCOS
# ACTIVO DE AUXILIARES
work_conn.execute(text("DROP TABLE IF EXISTS #af41_36_51"))
sql_af41_36_51 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(5))) AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0208a' AS PROC
INTO #af41_36_51
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.41'
  AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='36'
  AND T1.C_CAGENTE IN ('53','51021','5101','51022','5102','9','51')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, LTRIM(CAST(T1.SECTOR AS varchar(5)))
"""
work_conn.execute(text(sql_af41_36_51))


In [ ]:
# PASIVO DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_36"))
sql_af41_51_36 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO)*-1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0208a' AS PROC
INTO #af41_51_36
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR IN (51021,51022,5101)
  AND SUBSTRING(T1.C_CAGENTE,1,2)='36'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_51_36))

# equivalente a DATA AF41_36_51;SET AF41_36_51 AF41_51_36;
work_conn.execute(text("DROP TABLE IF EXISTS #af41_36_51_all"))
work_conn.execute(text(
    "SELECT * INTO #af41_36_51_all FROM #af41_36_51 "
    "UNION ALL SELECT * FROM #af41_51_36"
))


In [ ]:
# AGRUPA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_36_51_ag"))
sql_af41_36_51_ag = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        SUM(DATO) AS DATO,
        C_SCN,
        N_SCN,
        FUENTE,
        PROC
INTO #af41_36_51_ag
FROM #af41_36_51_all
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_af41_36_51_ag))


In [ ]:
# GENERA IMPUTACIÓN EN PASIVO DE EMPRESAS CON CA BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_321"))
sql_af41_51_321_b = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        SUM(DATO)*-1 AS DATO,
        C_SCN,
        N_SCN,
        FUENTE,
        PROC
INTO #af41_51_321
FROM #af41_36_51_ag
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_af41_51_321_b))

cols_af41_36_51 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_36_51}) SELECT {cols_af41_36_51} FROM #af41_36_51_ag"))
_log("APPEND BD_CTSI desde #af41_36_51_ag", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_36_51}) SELECT {cols_af41_36_51} FROM #af41_51_321"))
_log("APPEND BD_CTSI desde #af41_51_321 (empresas-bancos)", res.rowcount)
for t in ["#af41_36_51_ag", "#af41_51_321", "#af41_36_51", "#af41_51_36", "#af41_36_51_all"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA DE PTMOS DE CP EN ACTIVO DE EMPRESAS USANDO LO REPORTADO EN EL PASIVO DE EMPRESAS CON
# CONTRAGENTE EMPRESAS. AJUSTA EN CREDITOS COMERCIALES DEL SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51"))
sql_af41_51 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '51022' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO)*-1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0208' AS PROC
INTO #af41_51
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.41'
  AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='51'
  AND T1.SECTOR NOT IN (511,5111)
  AND T1.C_CAGENTE IN ('53','51021','5101','51022','5102','9','51')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_51))


In [ ]:
# PASIVO DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_2"))
sql_af41_51_2 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS float) AS SECTOR,
        '51022' AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0208' AS PROC
INTO #af41_51_2
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR IN (51021,51022,5101)
  AND SUBSTRING(T1.C_CAGENTE,1,2) IN ('51','53','9','2')
  AND T1.C_CAGENTE NOT IN ('511')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS float)
"""
work_conn.execute(text(sql_af41_51_2))

res = work_conn.execute(text(
    "UPDATE #af41_51_2 SET SECTOR=51022 WHERE SECTOR IN (5102,2)"
))
_log("UPDATE #af41_51_2 SECTOR->51022", res.rowcount)

# equivalente a DATA AF41_51;SET AF41_51 AF41_51_2;
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_all"))
work_conn.execute(text(
    "SELECT * INTO #af41_51_all FROM #af41_51 "
    "UNION ALL SELECT * FROM #af41_51_2"
))


In [ ]:
# AGRUPA
work_conn.execute(text("DROP TABLE IF EXISTS #af41_51_def"))
sql_af41_51_def = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        SUM(DATO) AS DATO,
        C_SCN,
        N_SCN,
        FUENTE,
        PROC
INTO #af41_51_def
FROM #af41_51_all
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_af41_51_def))


In [ ]:
# AJUSTA EN AF.7 DEL SECTOR RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51"))
sql_af7_51 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        SUM(DATO)*-1 AS DATO,
        'AF.7' AS C_SCN,
        'Créditos comerciales' AS N_SCN,
        FUENTE,
        PROC
INTO #af7_51
FROM #af41_51_def
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, FUENTE, PROC
"""
work_conn.execute(text(sql_af7_51))

cols_af41_51_def = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_51_def}) SELECT {cols_af41_51_def} FROM #af41_51_def"))
_log("APPEND BD_CTSI desde #af41_51_def", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_51_def}) SELECT {cols_af41_51_def} FROM #af7_51"))
_log("APPEND BD_CTSI desde #af7_51", res.rowcount)
for t in ["#af41_51_def", "#af7_51", "#af41_51", "#af41_51_2", "#af41_51_all"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA DE PTMOS DE CP ENTRE BCOS-ACTIVO Y SECTORES-PASIVO. RESPETA DATO DE BCOS E IMPUTA DIF EN 51022
# CON CONTRAGENTE 321. CIERRE 2021: concilia con nombres originales y agrega contragente 322 (empresas tiene ese CA)
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321"))
sql_af41_321 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        '321' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0184' AS PROC
INTO #af41_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=321)
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.C_CAGENTE IN ('321','322'))
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_51022"))
sql_af42_321_51022 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        51022 AS SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        'AF.42' AS C_SCN,
        'Préstamos a largo plazo' AS N_SCN,
        'PS' AS FUENTE,
        '0185' AS PROC
INTO #af42_321_51022
FROM #af41_321
"""
work_conn.execute(text(sql_af42_321_51022))

cols_af41_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321}) SELECT {cols_af41_321} FROM #af41_321"))
_log("APPEND BD_CTSI desde #af41_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321}) SELECT {cols_af41_321} FROM #af42_321_51022"))
_log("APPEND BD_CTSI desde #af42_321_51022", res.rowcount)
for t in ["#af41_321", "#af42_321_51022"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PTMOS DE CP DEL ACTIVO DE GOB CON CA BCOS Y PASIVO DE BCOS CON GOB. RESPETA DATO DE GOBIERNO
# E IMPUTA EN BCOS. AJUSTA IMPUTACIÓN EN PASIVO DE BCOS CON CA 53. CIERRE 2021: concilia con nombres originales
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_41"))
sql_af41_321_41 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        321 AS SECTOR,
        '41' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0187a' AS PROC
INTO #af41_321_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=41 AND T1.C_CAGENTE='321')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=321 AND T1.C_CAGENTE='41')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_321_41))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321_53"))
sql_af41_321_53 = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        321 AS SECTOR,
        '53' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        C_SCN,
        N_SCN,
        'PS' AS FUENTE,
        '0187b' AS PROC
INTO #af41_321_53
FROM #af41_321_41
"""
work_conn.execute(text(sql_af41_321_53))

cols_af41_321_41 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321_41}) SELECT {cols_af41_321_41} FROM #af41_321_41"))
_log("APPEND BD_CTSI desde #af41_321_41", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321_41}) SELECT {cols_af41_321_41} FROM #af41_321_53"))
_log("APPEND BD_CTSI desde #af41_321_53", res.rowcount)
for t in ["#af41_321_41", "#af41_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DE PTMOS DE CP ENTRE BCOS-PASIVO CON CONTRAGENTE RM, USANDO INFO DEL PASIVO DE BCOS CON CONTRAGENTES 53 Y 36901
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321p"))
sql_af41_321p = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO)*-1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0187' AS PROC
INTO #af41_321p
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=321 AND T1.C_CAGENTE IN ('53','36901')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_321p))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af41_321p_6"))
sql_af41_321p_6 = """
SELECT  'P' AS MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '6' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        C_SCN,
        N_SCN,
        'PS' AS FUENTE,
        '0188' AS PROC
INTO #af41_321p_6
FROM #af41_321p
"""
work_conn.execute(text(sql_af41_321p_6))

cols_af41_321p = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321p}) SELECT {cols_af41_321p} FROM #af41_321p"))
_log("APPEND BD_CTSI desde #af41_321p", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_321p}) SELECT {cols_af41_321p} FROM #af41_321p_6"))
_log("APPEND BD_CTSI desde #af41_321p_6", res.rowcount)
for t in ["#af41_321p", "#af41_321p_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE CP ENTRE RM-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE RM E AJUSTA CONTRA PTMOS DE LP DEL
# SECTOR 321 CON CA RM. CIERRE 2021: no se cambia cuenta Rec Precio a Rec Precio Reaj
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_321"))
sql_af41_6_321 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        321 AS SECTOR,
        '6' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0192' AS PROC
INTO #af41_6_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND T1.SECTOR=321 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND T1.SECTOR=6 AND T1.C_CAGENTE='321')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_6_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_321"))
sql_af42_6_321 = """
SELECT  'P' AS MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        'AF.42' AS C_SCN,
        'Préstamos a largo plazo' AS N_SCN,
        'PS' AS FUENTE,
        '0193' AS PROC
INTO #af42_6_321
FROM #af41_6_321
"""
work_conn.execute(text(sql_af42_6_321))

cols_af41_6_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_321}) SELECT {cols_af41_6_321} FROM #af41_6_321"))
_log("APPEND BD_CTSI desde #af41_6_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_321}) SELECT {cols_af41_6_321} FROM #af42_6_321"))
_log("APPEND BD_CTSI desde #af42_6_321 (RM-Bancos)", res.rowcount)
for t in ["#af41_6_321", "#af42_6_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIACIÓN PTMOS DE LP, RESPETANDO DATO DEL TOTAL ACTIVO E IMPUTANDO DIFERENCIA EN PASIVO DEL SECTOR
# 51022 CON CONTRAGENTE BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_all"))
sql_af42_all = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        '321' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        'Préstamos a largo plazo' AS N_SCN,
        'PS' AS FUENTE,
        '0175' AS PROC
INTO #af42_all
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42') OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN
"""
work_conn.execute(text(sql_af42_all))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN PASIVO AF7 DE SECTOR 51022 CON CA 53. REGLA NUEVA CIERRE 2020
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste"))
sql_af7_ajuste = """
SELECT  MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '53' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        'AF.7' AS C_SCN,
        'Créditos comerciales' AS N_SCN,
        FUENTE,
        PROC
INTO #af7_ajuste
FROM #af42_all
"""
work_conn.execute(text(sql_af7_ajuste))

cols_af42_all = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_all}) SELECT {cols_af42_all} FROM #af42_all"))
_log("APPEND BD_CTSI desde #af42_all", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_all}) SELECT {cols_af42_all} FROM #af7_ajuste"))
_log("APPEND BD_CTSI desde #af7_ajuste", res.rowcount)
for t in ["#af42_all", "#af7_ajuste"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CAMBIA CONTRAGENTE DE DEUDA SUBORDINADA ACTIVO DEL BANCO CENTRAL
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nueva_ca, PROC=:proc "
    "WHERE C_SCN=:scn AND SECTOR=31 AND C_CAGENTE=:vieja_ca AND C_ENTRADA=:entrada"
), {"nueva_ca": "37", "proc": "0214a", "scn": "AF.42", "vieja_ca": "321", "entrada": "D"})
_log("UPDATE BD_CTSI C_CAGENTE 321->37 (31/AF.42/D)", res.rowcount)


In [ ]:
# AJUSTA PTMOS DE LP ENTRE BCO CENTRAL-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE BCENTRAL Y AJUSTA CONTRA
# PTMOS DE LP DEL SECTOR 321 CON CA RM. CIERRE 2021: no se cambia nombre de cuenta, queda igual a Banco Central
work_conn.execute(text("DROP TABLE IF EXISTS #af42_31_321"))
sql_af42_31_321 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        321 AS SECTOR,
        '31' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0214' AS PROC
INTO #af42_31_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=321 AND T1.C_CAGENTE='31')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=31 AND T1.C_CAGENTE='321')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_31_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_321_6"))
sql_af42_321_6 = """
SELECT  'P' AS MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '6' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        C_SCN,
        N_SCN,
        'PS' AS FUENTE,
        '0215' AS PROC
INTO #af42_321_6
FROM #af42_31_321
"""
work_conn.execute(text(sql_af42_321_6))

cols_af42_31_321 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_31_321}) SELECT {cols_af42_31_321} FROM #af42_31_321"))
_log("APPEND BD_CTSI desde #af42_31_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_31_321}) SELECT {cols_af42_31_321} FROM #af42_321_6"))
_log("APPEND BD_CTSI desde #af42_321_6 (BCentral-Bancos)", res.rowcount)
for t in ["#af42_31_321", "#af42_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE BCO CENTRAL-ACTIVO Y HOLDING-PASIVO. RESPETA DATO DE BCENTRAL Y AJUSTA CONTRA
# PTMOS DE LP DEL SECTOR 37 CON CA 321. CIERRE 2021: no se cambia nombre de cuenta, queda igual a Banco Central
work_conn.execute(text("DROP TABLE IF EXISTS #af42_31_37"))
sql_af42_31_37 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        37 AS SECTOR,
        '31' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0214b' AS PROC
INTO #af42_31_37
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=37 AND T1.C_CAGENTE='31')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=31 AND T1.C_CAGENTE='37')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_31_37))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_37_321"))
sql_af42_37_321 = """
SELECT  'P' AS MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        '321' AS C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        C_SCN,
        N_SCN,
        'PS' AS FUENTE,
        '0215b' AS PROC
INTO #af42_37_321
FROM #af42_31_37
"""
work_conn.execute(text(sql_af42_37_321))

cols_af42_31_37 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_31_37}) SELECT {cols_af42_31_37} FROM #af42_31_37"))
_log("APPEND BD_CTSI desde #af42_31_37", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_31_37}) SELECT {cols_af42_31_37} FROM #af42_37_321"))
_log("APPEND BD_CTSI desde #af42_37_321 (BCentral-Holding)", res.rowcount)
for t in ["#af42_31_37", "#af42_37_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y BCOS-PASIVO. RESPETA DATO DE RM Y AJUSTA CONTRA AJ DE CONCILIACIÓN
# DEL SECTOR 321 CON CA RM
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_321"))
sql_af42_6_321_v2 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        321 AS SECTOR,
        '6' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0220' AS PROC
INTO #af42_6_321
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=321 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=6 AND T1.C_CAGENTE='321')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_6_321_v2))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_321_6"))
sql_af71_321_6 = """
SELECT  'P' AS MONEDA,
        [AÑO],
        TRIM,
        SECTOR,
        C_CAGENTE,
        C_CUENTA,
        C_ENTRADA,
        DATO*-1 AS DATO,
        'AF.71' AS C_SCN,
        'Ajuste conciliación' AS N_SCN,
        'PS' AS FUENTE,
        '0221' AS PROC
INTO #af71_321_6
FROM #af42_6_321
"""
work_conn.execute(text(sql_af71_321_6))

cols_af42_6_321_v2 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_321_v2}) SELECT {cols_af42_6_321_v2} FROM #af42_6_321"))
_log("APPEND BD_CTSI desde #af42_6_321 (RM-Bancos LP)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_321_v2}) SELECT {cols_af42_6_321_v2} FROM #af71_321_6"))
_log("APPEND BD_CTSI desde #af71_321_6", res.rowcount)
for t in ["#af42_6_321", "#af71_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y GOBIERNO-PASIVO. RESPETA DATO DE RM Y AJUSTA CONTRA AJ DE CONCILIACIÓN
# DEL SECTOR 41 CON CA RM. CIERRE 2021: no cambia Rec Precio a Rec Precio Reaj (RM solo tiene Rec Precio y este sector manda)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_41"))
sql_af42_6_41 = """
SELECT  'P' AS MONEDA,
        T1.[AÑO],
        T1.TRIM,
        41 AS SECTOR,
        '6' AS C_CAGENTE,
        T1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0225' AS PROC
INTO #af42_6_41
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=41 AND T1.C_CAGENTE='6')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=6 AND T1.C_CAGENTE='41')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""

In [ ]:
# AJUSTA PTMOS DE CP entre RM-ACTIVO Y 51021/51022-PASIVO. RESPETA DATO DEL RM Y AJUSTA EN 51021 CON CONTRAGENTE 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_51021"))
sql_af41_6_51021 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51021 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0202a' AS PROC
INTO #af41_6_51021
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.41' AND SECTOR=6 AND C_CAGENTE IN ('53','51021'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND SECTOR=51021 AND C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND SECTOR=51022 AND MONEDA='D' AND C_CAGENTE='6')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.41' AND SECTOR=51022 AND MONEDA='P' AND C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af41_6_51021))


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF41_6_51021 force (server-side, columnas explícitas)
cols_af41_6_51021 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af41_6_51021 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af41_6_51021})
SELECT {cols_af41_6_51021}
FROM #af41_6_51021
"""
res = work_conn.execute(text(sql_append_af41_6_51021))
_log("APPEND TABLAS.dbo.BD_CTSI (AF41_6_51021)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af41_6_51021"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y EMPRESAS PRIVADAS-PASIVO. RESPETA DATO DE RM Y AJUSTA CONTRA PTMOS DE LP DEL SECTOR 51022 CON CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES. ELIMINA 334 PORQUE AHORA ES DE AUXILIARES. INCORPORA CONTRAGENTE 51021 EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_51022"))
sql_af42_6_51022 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0235' AS PROC
INTO #af42_6_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=6 AND C_CAGENTE IN ('53','51021'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR IN (51021,51022) AND C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_6_51022))
af42_6_51022 = pd.read_sql(text("SELECT * FROM #af42_6_51022"), work_conn)
_log("af42_6_51022", af42_6_51022)


In [ ]:
sql_af42_321_51022 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       'H' AS C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0236' AS PROC
FROM #af42_6_51022
"""
af42_321_51022 = pd.read_sql(text(sql_af42_321_51022), work_conn)
_log("af42_321_51022", af42_321_51022)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_6_51022 force
cols_af42_6_51022 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_6_51022 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_51022})
SELECT {cols_af42_6_51022}
FROM #af42_6_51022
"""
res = work_conn.execute(text(sql_append_af42_6_51022))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_6_51022)", res.rowcount)
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_321_51022 force
af42_321_51022.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_321_51022)", len(af42_321_51022))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_51022"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE RM-ACTIVO Y BCCH-PASIVO. RESPETA DATO DE BCHH Y AJUSTA CONTRA AF71 DEL SECTOR 6 CON CA 31
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_31"))
sql_af42_6_31 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0238' AS PROC
INTO #af42_6_31
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=6 AND C_CAGENTE='31')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_6_31))


In [ ]:
sql_af71_6_31 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA,
       'D' AS C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0239' AS PROC
FROM #af42_6_31
"""
af71_6_31 = pd.read_sql(text(sql_af71_6_31), work_conn)
_log("af71_6_31", af71_6_31)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_6_31 force
cols_af42_6_31 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_6_31 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_6_31})
SELECT {cols_af42_6_31}
FROM #af42_6_31
"""
res = work_conn.execute(text(sql_append_af42_6_31))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_6_31)", res.rowcount)
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF71_6_31 force
af71_6_31.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF71_6_31)", len(af71_6_31))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_6_31"))


In [ ]:
# AJUSTA PTMOS DE LP ENTRE GOBIERNO CENTRAL-ACTIVO Y BCCH-PASIVO. RESPETA DATO DE BCHH Y AJUSTA CONTRA AF71 DEL SECTOR 41 CON CA 31
# CIERRE 2021: NO HACE CAMBIO EN CUENTA DE REC PRECIO A REC PRECIO REAJ
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_31"))
sql_af42_41_31 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0243' AS PROC
INTO #af42_41_31
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=41 AND C_CAGENTE='31')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE='41')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_41_31))


In [ ]:
sql_af71_41_31 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       FUENTE,
       '0244' AS PROC
FROM #af42_41_31
"""
af71_41_31 = pd.read_sql(text(sql_af71_41_31), work_conn)
_log("af71_41_31", af71_41_31)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF71_41_31 force / WORK.AF42_41_31 force
af71_41_31.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF71_41_31)", len(af71_41_31))
cols_af42_41_31 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_41_31 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_41_31})
SELECT {cols_af42_41_31}
FROM #af42_41_31
"""
res = work_conn.execute(text(sql_append_af42_41_31))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_41_31)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_31"))


In [ ]:
# CAMBIA CONTRAGENTE DE 3 A 31 EN PTMOS DE LP-PASIVO DEL SECTOR 41
with engine.begin() as conn:
    res = conn.execute(
        text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nuevo_ca, PROC=:proc WHERE C_CAGENTE=:ca_actual AND SECTOR=:sector AND C_SCN=:scn AND C_ENTRADA=:entrada"),
        {"nuevo_ca": "31", "proc": "0245", "ca_actual": "3", "sector": 41, "scn": "AF.42", "entrada": "H"},
    )
    _log("UPDATE TABLAS.dbo.BD_CTSI (0245)", res.rowcount)


In [ ]:
# ACTUALIZA CONTRAGENTES A 321 EN EL PASIVO DE PTMOS DE LP DE TODOS LOS SECTORES, EXCEPTO ALGUNOS CA
# EXCEPTO PARA EL SECTOR HOGARES (511) POR LOS PTMOS DE SECURITIZADORAS QUE APARECEN EN 2020
excluidos_ca = ['6','321','31','322','331','334','33211','33212','33222','339011','3324','351','41','411','412','42','51021','413','51022']
placeholders_ca = ", ".join(f":ca{i}" for i in range(len(excluidos_ca)))
params_update_321 = {"nuevo_ca": "321", "proc": "0245b", "scn": "AF.42", "entrada": "H", "sector_excl": 511}
params_update_321.update({f"ca{i}": v for i, v in enumerate(excluidos_ca)})
with engine.begin() as conn:
    res = conn.execute(
        text(f"UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nuevo_ca, PROC=:proc WHERE C_SCN=:scn AND C_ENTRADA=:entrada AND C_CAGENTE NOT IN ({placeholders_ca}) AND SECTOR <> :sector_excl"),
        params_update_321,
    )
    _log("UPDATE TABLAS.dbo.BD_CTSI (0245b)", res.rowcount)


In [ ]:
# CAMBIA CONTRAGENTE DE 3 A 31 EN PTMOS DE LP-PASIVO DEL SECTOR 42
with engine.begin() as conn:
    res = conn.execute(
        text("UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:nuevo_ca, PROC=:proc WHERE C_CAGENTE=:ca_actual AND SECTOR=:sector AND C_SCN=:scn AND C_ENTRADA=:entrada"),
        {"nuevo_ca": "31", "proc": "0246", "ca_actual": "3", "sector": 42, "scn": "AF.42", "entrada": "H"},
    )
    _log("UPDATE TABLAS.dbo.BD_CTSI (0246)", res.rowcount)


In [ ]:
# AJUSTA PTMOS DE LP ENTRE GOBIERNO CENTRAL-PASIVO Y BCCH-ACTIVO. RESPETA DATO DE BCHH Y AJUSTA CONTRA AF71 DEL SECTOR 41 CON CA 31
# CIERRE 2021. NO SE HACE CAMBIO DE REC PRECIO A REC PRECIO REAJ
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_31_2"))
sql_af42_41_31_2 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0250' AS PROC
INTO #af42_41_31_2
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE='41')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR=41 AND C_CAGENTE='31')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_41_31_2))


In [ ]:
sql_af71_41_31_2 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO*-1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE,
       '0251' AS PROC
FROM #af42_41_31_2
"""
af71_41_31_2 = pd.read_sql(text(sql_af71_41_31_2), work_conn)
_log("af71_41_31_2", af71_41_31_2)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_41_31_2 force / WORK.AF71_41_31_2 force
cols_af42_41_31_2 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_41_31_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_41_31_2})
SELECT {cols_af42_41_31_2}
FROM #af42_41_31_2
"""
res = work_conn.execute(text(sql_append_af42_41_31_2))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_41_31_2)", res.rowcount)
af71_41_31_2.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF71_41_31_2)", len(af71_41_31_2))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_31_2"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 333 CON CA 31, USANDO LA INFO DEL ACTIVO DEL SECTOR 31 CON CA 333
sql_af42_333_31 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0252' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE='333')
GROUP BY T1.SECTOR, T1.C_CAGENTE, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
af42_333_31 = pd.read_sql(text(sql_af42_333_31), engine)
_log("af42_333_31", af42_333_31)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_333_31 force
af42_333_31.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_333_31)", len(af42_333_31))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 322, USANDO LA INFO DEL ACTIVO DEL SECTOR 322 CON CA 51
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_332_51"))
sql_af42_332_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0255' AS PROC
INTO #af42_332_51
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=322 AND C_CAGENTE='51')
GROUP BY T1.SECTOR, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_332_51))


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_332_51 force
cols_af42_332_51 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_332_51 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_332_51})
SELECT {cols_af42_332_51}
FROM #af42_332_51
"""
res = work_conn.execute(text(sql_append_af42_332_51))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_332_51)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af42_332_51"))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR CONTRA PASIVO DE PTMOS DE LP DEL SECTOR 51022 CON CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
sql_af42_51022_321 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0256' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=322 AND C_CAGENTE='51')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
af42_51022_321 = pd.read_sql(text(sql_af42_51022_321), engine)
_log("af42_51022_321", af42_51022_321)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_51022_321 force
af42_51022_321.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_51022_321)", len(af42_51022_321))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 3324, USANDO LA INFO DEL ACTIVO DEL SECTOR 3324 CON CA 51022. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_3324_51022"))
sql_af42_3324_51022 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0257' AS PROC
INTO #af42_3324_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=3324 AND C_CAGENTE='51022')
GROUP BY T1.C_CAGENTE, T1.SECTOR, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_3324_51022))


In [ ]:
sql_af42_51022_321_ = """
SELECT MONEDA, [AÑO], TRIM, SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN, N_SCN, FUENTE,
       '0258' AS PROC
FROM #af42_3324_51022
"""
af42_51022_321_ = pd.read_sql(text(sql_af42_51022_321_), work_conn)
_log("af42_51022_321_", af42_51022_321_)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_3324_51022 force / WORK.AF42_51022_321_ force
cols_af42_3324_51022 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_3324_51022 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_3324_51022})
SELECT {cols_af42_3324_51022}
FROM #af42_3324_51022
"""
res = work_conn.execute(text(sql_append_af42_3324_51022))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_3324_51022)", res.rowcount)
af42_51022_321_.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_51022_321_)", len(af42_51022_321_))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_3324_51022"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 33211/33212/33222/411, USANDO LA INFO DEL ACTIVO DE LOS SECTORES 33211/33212/33222/411 CON CA 9. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES. INCORPORA PASIVO DE EMPRESAS CON ESOS CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_v_51022"))
sql_af42_v_51022 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '33' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0260' AS PROC
INTO #af42_v_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (33211,33212,33222,411,3324,335) AND T1.C_CAGENTE IN ('9','53','51021','51022','5102','51'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (51022,51021,5101) AND (SUBSTRING(T1.C_CAGENTE,1,2)='33' OR T1.C_CAGENTE='411'))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_v_51022))


In [ ]:
sql_af42_51022_v = """
SELECT MONEDA, [AÑO], TRIM, SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN, N_SCN, FUENTE,
       '0260B' AS PROC
FROM #af42_v_51022
"""
af42_51022_v = pd.read_sql(text(sql_af42_51022_v), work_conn)
_log("af42_51022_v", af42_51022_v)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_V_51022 force / WORK.AF42_51022_V force
cols_af42_v_51022 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_v_51022 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_v_51022})
SELECT {cols_af42_v_51022}
FROM #af42_v_51022
"""
res = work_conn.execute(text(sql_append_af42_v_51022))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_V_51022)", res.rowcount)
af42_51022_v.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_51022_V)", len(af42_51022_v))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_v_51022"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 33212 CON CA 339011, USANDO LA INFO DEL ACTIVO DEL SECTOR 339011 CON CA 3/3321. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 33212 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af42_339011"))
sql_af42_339011 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       33212 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0263' AS PROC
INTO #af42_339011
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=339011 AND T1.C_CAGENTE IN ('3','3321'))
GROUP BY T1.SECTOR, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_339011))


In [ ]:
sql_af42_33212 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0264' AS PROC
FROM #af42_339011
"""
af42_33212 = pd.read_sql(text(sql_af42_33212), work_conn)
_log("af42_33212", af42_33212)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_339011 force / WORK.AF42_33212 force
cols_af42_339011 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_339011 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_339011})
SELECT {cols_af42_339011}
FROM #af42_339011
"""
res = work_conn.execute(text(sql_append_af42_339011))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_339011)", res.rowcount)
af42_33212.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_33212)", len(af42_33212))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_339011"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 351, USANDO LA INFO DEL ACTIVO DEL SECTOR 351/352 CON CA <>511. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. CONCILIA CUENTAS CON SUS NOMBRE ORIGINALES. INCORPORA CALCULO NETO CONSIDERANDO LO QUE EMPRESAS DICE HABER RECIBIDO DE SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_351_352"))
sql_af42_351_352 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '351' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0266' AS PROC
INTO #af42_351_352
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR IN (351,352,353) AND C_CAGENTE <> '511')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR IN (51021,51022,5101,5102) AND C_CAGENTE IN ('351','352','353','35'))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_351_352))


In [ ]:
sql_af42_51022_351 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0267' AS PROC
FROM #af42_351_352
"""
af42_51022_351 = pd.read_sql(text(sql_af42_51022_351), work_conn)
_log("af42_51022_351", af42_51022_351)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_351_352 force / WORK.AF42_51022_351 force
cols_af42_351_352 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_351_352 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_351_352})
SELECT {cols_af42_351_352}
FROM #af42_351_352
"""
res = work_conn.execute(text(sql_append_af42_351_352))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_351_352)", res.rowcount)
af42_51022_351.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_51022_351)", len(af42_51022_351))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_351_352"))


In [ ]:
# CIERRE 2021. IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 37 CON CA 351, USANDO LA INFO DEL ACTIVO DEL SECTOR 351/352 CON CA AUXILIARES. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR AUX CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af42_35_36"))
sql_af42_35_36 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       36 AS SECTOR,
       '35' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0266' AS PROC
INTO #af42_35_36
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (351,352,353) AND SUBSTRING(T1.C_CAGENTE,1,2)='36')
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR IN (351,352,353) AND T1.C_CAGENTE='37')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(6))),1,2)='33' AND T1.C_CAGENTE IN ('351','352','353','35'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND T1.SECTOR=37 AND T1.C_CAGENTE IN ('351','352','353','35'))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_35_36))


In [ ]:
sql_af42_51022_351_b = """
SELECT MONEDA, [AÑO], TRIM, SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0267' AS PROC
FROM #af42_35_36
"""
af42_51022_351_b = pd.read_sql(text(sql_af42_51022_351_b), work_conn)
_log("af42_51022_351_b", af42_51022_351_b)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_35_36 force / WORK.AF42_51022_351 force (segunda vez, PROC=0267)
cols_af42_35_36 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_35_36 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_35_36})
SELECT {cols_af42_35_36}
FROM #af42_35_36
"""
res = work_conn.execute(text(sql_append_af42_35_36))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_35_36)", res.rowcount)
af42_51022_351_b.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_51022_351_b)", len(af42_51022_351_b))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_35_36"))


In [ ]:
# CIERRE 2021: IMPUTA ACTIVO DE AF42 EN GOBIERNO CON CONTRAGENTE 37, USANDO PASIVO DE SECTOR 37 CA GOBIERNO, AJUSTA EN AF42 ACTIVO GOB CON CA 53
sql_af42_41_37 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '37' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0270' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR=37 AND C_CAGENTE='41'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
af42_41_37 = pd.read_sql(text(sql_af42_41_37), engine)
_log("af42_41_37", af42_41_37)


In [ ]:
af42_41_53 = af42_41_37.copy()
af42_41_53["C_CAGENTE"] = "53"
af42_41_53["DATO"] = af42_41_53["DATO"] * -1
af42_41_53["C_SCN"] = af42_41_37["C_SCN"]
af42_41_53["N_SCN"] = af42_41_37["N_SCN"]
af42_41_53["PROC"] = "0271"
af42_41_53 = af42_41_53[["MONEDA","AÑO","TRIM","SECTOR","C_CAGENTE","C_CUENTA","C_ENTRADA","DATO","C_SCN","N_SCN","FUENTE","PROC"]]
_log("af42_41_53", af42_41_53)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=AF42_41_37 force / AF42_41_53 force
af42_41_37.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_41_37)", len(af42_41_37))
af42_41_53.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_41_53)", len(af42_41_53))


In [ ]:
# CIERRE 2021: IMPUTA ACTIVO DE AF42 EN EMPRESAS CON CONTRAGENTE EMPRESAS, USANDO PASIVO DE SECTOR 51021 CA 51021, AJUSTA EN AF7 ACTIVO 51021 CON CA 51021
sql_af42_51_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       CAST(T1.C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(5))) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0270' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR=51021 AND C_CAGENTE='51021'
GROUP BY T1.C_CAGENTE, T1.SECTOR, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
af42_51_51 = pd.read_sql(text(sql_af42_51_51), engine)
_log("af42_51_51", af42_51_51)


In [ ]:
af7_51_51 = af42_51_51.copy()
af7_51_51["DATO"] = af7_51_51["DATO"] * -1
af7_51_51["C_SCN"] = "AF.7"
af7_51_51["N_SCN"] = "Créditos comerciales"
af7_51_51["PROC"] = "0271"
af7_51_51 = af7_51_51[["MONEDA","AÑO","TRIM","SECTOR","C_CAGENTE","C_CUENTA","C_ENTRADA","DATO","C_SCN","N_SCN","FUENTE","PROC"]]
_log("af7_51_51", af7_51_51)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=AF42_51_51 force / AF7_51_51 force
af42_51_51.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_51_51)", len(af42_51_51))
af7_51_51.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF7_51_51)", len(af7_51_51))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 41, USANDO LA INFO DEL ACTIVO DEL SECTOR 41 CON CA 53/51. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. NO SE CAMBIA REC PRECIO A REC PRECIO REAJ EN EMPRESAS PORQUE SE GENERAN REAJUSTES QUE NO CORRESPONDEN
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_51"))
sql_af42_41_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '41' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0270' AS PROC
INTO #af42_41_51
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=41 AND C_CAGENTE IN ('51','53'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR IN (51021,51022) AND C_CAGENTE='41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af42_41_51))


In [ ]:
sql_af42_51022_41 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR,
       '321' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO*-1 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0271' AS PROC
FROM #af42_41_51
"""
af42_51022_41 = pd.read_sql(text(sql_af42_51022_41), work_conn)
_log("af42_51022_41", af42_51022_41)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_41_51 force / WORK.AF42_51022_41 force
cols_af42_41_51 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_41_51 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_41_51})
SELECT {cols_af42_41_51}
FROM #af42_41_51
"""
res = work_conn.execute(text(sql_append_af42_41_51))
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_41_51)", res.rowcount)
af42_51022_41.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_51022_41)", len(af42_51022_41))
work_conn.execute(text("DROP TABLE IF EXISTS #af42_41_51"))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 51022 CON CA 42, USANDO LA INFO DEL ACTIVO DEL SECTOR 42 CON CA 53/51. AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 51022 CA 321
# CIERRE 2021. NO SE CAMBIA REC PRECIO A REC PRECIO REAJ EN EMPRESAS PORQUE SE GENERAN REAJUSTES QUE NO CORRESPONDEN
sql_af42_42_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0273' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=42 AND C_CAGENTE IN ('51','53'))
GROUP BY T1.SECTOR, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
af42_42_51 = pd.read_sql(text(sql_af42_42_51), engine)
_log("af42_42_51", af42_42_51)


In [ ]:
af42_42_321 = af42_42_51.copy()
af42_42_321["C_CAGENTE"] = "321"
af42_42_321["DATO"] = af42_42_321["DATO"] * -1
af42_42_321["FUENTE"] = "PS"
af42_42_321["PROC"] = "0274"
af42_42_321 = af42_42_321[["MONEDA","AÑO","TRIM","SECTOR","C_CAGENTE","C_CUENTA","C_ENTRADA","DATO","C_SCN","N_SCN","FUENTE","PROC"]]
_log("af42_42_321", af42_42_321)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_42_51 force / WORK.AF42_42_321 force
af42_42_51.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_42_51)", len(af42_42_51))
af42_42_321.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_42_321)", len(af42_42_321))


In [ ]:
# IMPUTA PASIVO DE PTMOS DE LP AL SECTOR 41 CON CA 31 (EN SECTOR 31 CON CA 53/331/41). RESPETA DATO DEL SECTOR 31 Y AJUSTA CONTRA PASIVO DE PTMOS LP DE SECTOR 41 CA 321
# CIERRE 2021. NO SE HACE CAMBIO EN CUENTA DESDE REC PRECIO REAJ Y REC VOLUMEN A REC PRECIO. SE CONCILIA TAL CUAL PARA AJUSTARSE BIEN A DATO DE CENTRAL
sql_af42_31_41 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '31' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0282' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=31 AND C_CAGENTE IN ('331','53','41'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR=41 AND C_CAGENTE='31')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
af42_31_41 = pd.read_sql(text(sql_af42_31_41), engine)
_log("af42_31_41", af42_31_41)


In [ ]:
af42_321_41 = af42_31_41.copy()
af42_321_41["SECTOR"] = 41
af42_321_41["C_CAGENTE"] = "321"
af42_321_41["DATO"] = af42_321_41["DATO"] * -1
af42_321_41["FUENTE"] = "PS"
af42_321_41["PROC"] = "0282B"
af42_321_41 = af42_321_41[["MONEDA","AÑO","TRIM","SECTOR","C_CAGENTE","C_CUENTA","C_ENTRADA","DATO","C_SCN","N_SCN","FUENTE","PROC"]]
_log("af42_321_41", af42_321_41)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=WORK.AF42_321_41 force / WORK.AF42_31_41 force
af42_321_41.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_321_41)", len(af42_321_41))
af42_31_41.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_31_41)", len(af42_31_41))


In [ ]:
# CIERRE 2025Q4: AJUSTA AF42 ENTRE GOBIERNO POR PTMO AL FAPP
sql_af42_41_342 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '342' AS C_CAGENTE,
       'D' AS C_ENTRADA,
       T1.C_CUENTA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0282c' AS PROC
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND SECTOR=41 AND C_CAGENTE='342')
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42' AND SECTOR=342 AND C_CAGENTE='41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
af42_41_342 = pd.read_sql(text(sql_af42_41_342), engine)
_log("af42_41_342", af42_41_342)


In [ ]:
af7_41_53 = af42_41_342.copy()
af7_41_53["C_CAGENTE"] = "53"
af7_41_53["DATO"] = af7_41_53["DATO"] * -1
af7_41_53["C_SCN"] = "AF.7"
af7_41_53["N_SCN"] = "Créditos comerciales"
af7_41_53["FUENTE"] = "PS"
af7_41_53["PROC"] = "0282c"
af7_41_53 = af7_41_53[["MONEDA","AÑO","TRIM","SECTOR","C_CAGENTE","C_CUENTA","C_ENTRADA","DATO","C_SCN","N_SCN","FUENTE","PROC"]]
_log("af7_41_53", af7_41_53)


In [ ]:
# PROC APPEND base=tablas.bd_ctsi data=AF42_41_342 force / AF7_41_53 force
af42_41_342.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF42_41_342)", len(af42_41_342))
af7_41_53.to_sql("BD_CTSI", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND TABLAS.dbo.BD_CTSI (AF7_41_53)", len(af7_41_53))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51021_9"))
sql_af42_51021_9 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       LEFT(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       -- (CASE WHEN T1.C_CUENTA='Rec Precio' THEN 'Rec Precio Reaj' ELSE T1.C_CUENTA END) AS
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0276' AS PROC
INTO #af42_51021_9
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=51021 AND LEFT(CAST(T1.SECTOR AS varchar(11)))='9')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_51021_9))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51022_51021"))
sql_af42_51022_51021 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0277' AS PROC
INTO #af42_51022_51021
FROM #af42_51021_9 t1
"""
work_conn.execute(text(sql_af42_51022_51021))


In [ ]:
cols_af42_51021_9 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_51021_9 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_51021_9})
SELECT {cols_af42_51021_9}
FROM #af42_51021_9
"""
res = work_conn.execute(text(sql_append_af42_51021_9))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51021_9)", res.rowcount)


In [ ]:
cols_af42_51022_51021 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_51022_51021 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_51022_51021})
SELECT {cols_af42_51022_51021}
FROM #af42_51022_51021
"""
res = work_conn.execute(text(sql_append_af42_51022_51021))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_51022_51021)", res.rowcount)


In [ ]:
for t in ["#af42_51021_9", "#af42_51022_51021"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CAMBIA CA EN PTMOS DE LP DEL PASIVO DEL SECTOR 5111. POR CIERRE 2020 TMB INCORPORA AF41
sql_update_sector51022 = text(
    "UPDATE TABLAS.dbo.BD_CTSI SET SECTOR=:sector_nuevo, PROC=:proc "
    "WHERE SECTOR=:sector_viejo AND C_SCN IN ('AF.42','AF.41') AND C_ENTRADA=:entrada"
)
res = work_conn.execute(sql_update_sector51022, {"sector_nuevo": 51022, "proc": "0278", "sector_viejo": 5111, "entrada": "H"})
_log("UPDATE TABLAS.dbo.BD_CTSI (sector 5111->51022)", res.rowcount)


In [ ]:
# CIERRE 2022: IMPUTA PTMOS DE LP ACTIVO DE FONDOS DE PENSIONES EN PASIVO DE EMPRESAS PRIVADAS. RECONCILIA CONTRA AF.7 EMPRESAS CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af42_emp_fp"))
sql_af42_emp_fp = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51021 AS SECTOR,
       '34' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0279' AS PROC
INTO #af42_emp_fp
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42' AND T1.SECTOR=34 AND T1.C_CAGENTE='51021')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af42_emp_fp))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN PASIVO AF7 DE SECTOR 51021 CON CA 53. REGLA NUEVA CIERRE 2022
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste_emp_fp"))
sql_af7_ajuste_emp_fp = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       T1.FUENTE,
       '0279a' AS PROC
INTO #af7_ajuste_emp_fp
FROM #af42_emp_fp t1
"""
work_conn.execute(text(sql_af7_ajuste_emp_fp))


In [ ]:
cols_af42_emp_fp = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_emp_fp = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_emp_fp})
SELECT {cols_af42_emp_fp}
FROM #af42_emp_fp
"""
res = work_conn.execute(text(sql_append_af42_emp_fp))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_emp_fp)", res.rowcount)


In [ ]:
cols_af7_ajuste_emp_fp = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af7_ajuste_emp_fp = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af7_ajuste_emp_fp})
SELECT {cols_af7_ajuste_emp_fp}
FROM #af7_ajuste_emp_fp
"""
res = work_conn.execute(text(sql_append_af7_ajuste_emp_fp))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_ajuste_emp_fp)", res.rowcount)


In [ ]:
for t in ["#af42_emp_fp", "#af7_ajuste_emp_fp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIACIÓN TOTAL PTMOS DE LP - COMPARA ACTIVO VS PASIVO, MANDA DATO DEL ACTIVO, IMPUTA DIF EN PASIVO DE SECTOR 51022 CON CA 321
work_conn.execute(text("DROP TABLE IF EXISTS #af42_total"))
sql_af42_total = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE,
       '0175' AS PROC
INTO #af42_total
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.42') OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.42')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_af42_total))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN PASIVO AF7 DE SECTOR 51022 CON CA 53. REGLA NUEVA CIERRE 2020
work_conn.execute(text("DROP TABLE IF EXISTS #af7_ajuste"))
sql_af7_ajuste = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af7_ajuste
FROM #af42_total t1
"""
work_conn.execute(text(sql_af7_ajuste))


In [ ]:
cols_af42_total = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af42_total = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af42_total})
SELECT {cols_af42_total}
FROM #af42_total
"""
res = work_conn.execute(text(sql_append_af42_total))
_log("APPEND TABLAS.dbo.BD_CTSI (af42_total)", res.rowcount)


In [ ]:
cols_af7_ajuste = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af7_ajuste = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af7_ajuste})
SELECT {cols_af7_ajuste}
FROM #af7_ajuste
"""
res = work_conn.execute(text(sql_append_af7_ajuste))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_ajuste)", res.rowcount)


In [ ]:
for t in ["#af42_total", "#af7_ajuste"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S2_06_Bonos

Reclasifica títulos AF.31/AF.32 al pasivo con contraparte 53, ajusta bonos de corto y largo plazo a valor de mercado y por reciprocidad de deuda entre sectores usando información DCV, e imputa la cartera de bonos que los sectores no informan directamente / Imputa y reclasifica tenencias de bonos (AF.32/AF.34/AF.71) entre sectores contraparte (41, 5101, 51021, 321, 351, 33901, 339011) conciliando activo/pasivo y ajustando contra bonos de largo plazo y derivados de resto del mundo; recompras a valor de mercado pasan su contragente a 9 / Imputa y ajusta bonos de largo plazo (AF.32) en pasivo de OFIS/auxiliares/patrimonio separado y sus contrapartidas en préstamos y créditos comerciales, comparando la contabilidad interna (CI) contra el DCV, y propaga los ajustes acumulativos a la tabla histórica BD_CTSI / Imputa y reclasifica activo/pasivo de bonos de largo plazo (créditos comerciales, valor de mercado, tenencias por sector emisor/tenedor vía DCV) y acumula los ajustes en la tabla histórica de cuentas nacionales TABLAS.BD_CTSI / Ajusta bonos de corto y largo plazo a valor de mercado con DCV, reclasifica tenencias entre AF.31/AF.32 por sector y contragente, e imputa diferencias activo-pasivo entre sectores emisores (31/32/33/36/4/51) y hogares, dejando el resultado acumulado en TABLAS.BD_CTSI

*confianza: low · SAS: PROC SQL (CREATE TABLE / UPDATE / DELETE) + PROC DATASETS APPEND FORCE encadenados sobre TABLAS.BD_CTSI, con fuentes DCV externas (.sas7bdat) + PROC SQL CREATE TABLE / UPDATE / PROC DATASETS APPEND sobre TABLAS.BD_CTSI, con tablas #tmp de sesión + PROC SQL / PROC DATASETS APPEND encadenados sobre TABLAS.BD_CTSI (tabla temporal de sesión), con imputaciones cruzadas contra fuentes DCV externas + PROC SQL (CREATE TABLE/UPDATE/DELETE con JOINs y agregaciones), PROC DATASETS APPEND, DATA step de concatenación — todo sobre tablas temporales de sesión (#) y TABLAS.BD_CTSI + PROC SQL (CREATE TABLE/DELETE/DROP) + PROC DATASETS APPEND, encadenados sobre tablas temporales de sesión y TABLAS.BD_CTSI*

In [ ]:
# ========= S2_06_Bonos =========
# COMPRIME TABLA PRINCIPAL
# SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;
# COMPRESS=YES es una opción de almacenamiento del dataset SAS: reescribe la
# tabla con exactamente las mismas filas para guardarlas comprimidas. En SQL
# Server la compresión es DDL (ALTER TABLE ... REBUILD WITH DATA_COMPRESSION),
# se decide una vez con el DBA y no desde el pipeline: el paso no tiene efecto
# de datos que traducir. Queda registrado y sin ejecutar — replicarlo con un
# borrado + reinserción deja TABLAS.BD_CTSI vacía si la celda se corta a la mitad.


In [ ]:
# ELIMINA TÍTULOS DE CP Y LP CON CA DISTINTOS DE 53 O 6 PARA DEJARLOS TODOS CON CA 53 EN EL PASIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af3_aj_ca"))
sql_af3_aj_ca = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO * -1) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0424' AS PROC
INTO #af3_aj_ca
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'H' AND t1.C_SCN IN ('AF.31','AF.32') AND t1.SECTOR <> 6 AND t1.C_CAGENTE NOT IN ('53','6','9','54'))
GROUP BY t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af3_aj_ca))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af3_im_ca"))
sql_af3_im_ca = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, '53' AS C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       t1.DATO * -1 AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0425' AS PROC
INTO #af3_im_ca
FROM #af3_aj_ca t1
"""
work_conn.execute(text(sql_af3_im_ca))


In [ ]:
# PROC APPEND FORCE: alinea por nombre de columna, server-side desde la #tmp
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af3_aj_ca"))
_log("APPEND TABLAS.BD_CTSI (AF3_AJ_CA)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af3_im_ca"))
_log("APPEND TABLAS.BD_CTSI (AF3_IM_CA)", res.rowcount)


In [ ]:
for t in ["#af3_aj_ca", "#af3_im_ca"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTE BONOS EMITIDOS DE CORTO PLAZO USANDO INFO DCV. para luego llevar a mercado los bonos de cp usando los precios (cambio por cierre 2019)
# INFO AF31 EN CI, EXCEPTO EL DEL RM. CIERRE 2021 EXCEPTO SI CONTRAGENTE EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af31_emisor"))
sql_af31_emisor = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, '53' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE t1.C_CUENTA END) AS C_CUENTA,
       t1.C_ENTRADA,
       SUM(t1.DATO * -1) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0430' AS PROC
INTO #af31_emisor
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'H' AND t1.C_SCN IN ('AF.31') AND t1.SECTOR <> 6 AND t1.C_CAGENTE NOT IN ('6'))
GROUP BY t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE,
         (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE t1.C_CUENTA END),
         t1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af31_emisor))


In [ ]:
# INFO DESDE DCV A VALOR PAR — fuente es un archivo SAS externo (.sas7bdat) que ya no existe en Python:
# el DCV vive en la base de datos (deja de ser un archivo). No hay ruta de reemplazo declarada por el plan.
raise NotImplementedError("WORK.AF31_DCV_PAS depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/base_af3_total_v2.sas7bdat': no hay tabla de BD ni ruta relativa equivalente declarada para esta fuente DCV")


In [ ]:
# UPDATEs de reclasificación de sector sobre AF31_DCV_PAS — dependen de la celda anterior no resuelta
raise NotImplementedError("UPDATE WORK.AF31_DCV_PAS (reclasificación de SECTOR 3321/3322/36901) depende de AF31_DCV_PAS, no resuelta por falta de fuente DCV")


In [ ]:
# PROC APPEND BASE=WORK.AF31_EMISOR DATA=WORK.AF31_DCV_PAS — depende de AF31_DCV_PAS no resuelta
raise NotImplementedError("APPEND WORK.AF31_EMISOR += WORK.AF31_DCV_PAS depende de AF31_DCV_PAS, no resuelta por falta de fuente DCV")


In [ ]:
# CREA DIF ENTRE DCV Y CI DEL AF31 — depende de AF31_EMISOR ya con el append de AF31_DCV_PAS aplicado
work_conn.execute(text("DROP TABLE IF EXISTS #af31_ajuste"))
sql_af31_ajuste = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE, t1.PROC
INTO #af31_ajuste
FROM #af31_emisor t1
GROUP BY t1.MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE, t1.PROC
"""
work_conn.execute(text(sql_af31_ajuste))


In [ ]:
# AJUSTA DIFERENCIA EN AF32 PARA TODOS LOS SECTORES (ANTES SE HACÍA EN AF7 PARA SECTORES 33 Y 36. SE CAMBIA EN CIERRE 2019)
work_conn.execute(text("DROP TABLE IF EXISTS #af31_ajuste_2"))
sql_af31_ajuste_2 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       (t1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       t1.FUENTE, '0431' AS PROC
INTO #af31_ajuste_2
FROM #af31_ajuste t1
"""
work_conn.execute(text(sql_af31_ajuste_2))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_ajuste"))
_log("APPEND TABLAS.BD_CTSI (AF31_AJUSTE)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_ajuste_2"))
_log("APPEND TABLAS.BD_CTSI (AF31_AJUSTE_2)", res.rowcount)


In [ ]:
for t in ["#af31_ajuste", "#af31_ajuste_2", "#af31_emisor", "#af31_dcv_pas"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE CORTO PLAZO EN TODOS LOS SECTORES, EXCEPTO RM
# Fuente es un archivo SAS externo (.sas7bdat) sin tabla de BD ni ruta equivalente declarada
raise NotImplementedError("WORK.T_AJ_MDO_AF31 depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_mdo_af31.sas7bdat': no hay tabla de BD ni ruta relativa equivalente declarada para esta fuente DCV")


In [ ]:
raise NotImplementedError("UPDATE T_AJ_MDO_AF31 (reclasificación de SECTOR 3321/3322/36901) depende de T_AJ_MDO_AF31, no resuelta por falta de fuente DCV")


In [ ]:
# VM EN BCE FINAL — depende de t_aj_mdo_af31 no resuelto
raise NotImplementedError("WORK.AF31_VM_BF depende de T_AJ_MDO_AF31 (join con TABLAS.BD_CTSI), no resuelta por falta de fuente DCV")


In [ ]:
# VM EN BCE INICIO — depende de t_aj_mdo_af31 no resuelto
raise NotImplementedError("WORK.AF31_VM_BI depende de T_AJ_MDO_AF31 (join con TABLAS.BD_CTSI), no resuelta por falta de fuente DCV")


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
raise NotImplementedError("APPEND TABLAS.BD_CTSI += WORK.AF31_VM_BF / WORK.AF31_VM_BI depende de tablas no resueltas por falta de fuente DCV")


In [ ]:
# DROP de WORK.AF31_VM_BF, WORK.AF31_VM_BI y t_aj_mdo_af31 — no aplica: ninguna de las tres se llegó a crear en este tramo


In [ ]:
# CALCULA REC PRECIO EN BONOS POR EFECTO DE CALCULO DE SALDOS A VALOR DE MERCADO-PASIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af31_rp"))
sql_af31_rp = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, '53' AS C_CAGENTE,
       'Rec Precio' AS C_CUENTA, t1.C_ENTRADA,
       (CASE WHEN t1.C_CUENTA <> 'Bce Final' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0426a' AS PROC
INTO #af31_rp
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN IN ('AF.31')
GROUP BY t1.[AÑO], t1.TRIM, t1.C_ENTRADA,
         (CASE WHEN t1.C_CUENTA <> 'Bce Final' THEN t1.DATO * -1 ELSE t1.DATO END),
         t1.C_SCN, t1.N_SCN, t1.SECTOR
"""
work_conn.execute(text(sql_af31_rp))


In [ ]:
# AGRUPA ANTERIOR
work_conn.execute(text("DROP TABLE IF EXISTS #af31_rp_2"))
sql_af31_rp_2 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE, t1.PROC
INTO #af31_rp_2
FROM #af31_rp t1
WHERE t1.DATO <> 0
GROUP BY t1.MONEDA, t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.C_CAGENTE, t1.FUENTE, t1.SECTOR, t1.PROC
"""
work_conn.execute(text(sql_af31_rp_2))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_rp_2"))
_log("APPEND TABLAS.BD_CTSI (AF31_RP_2)", res.rowcount)


In [ ]:
for t in ["#af31_rp_2", "#af31_rp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA CARTERA DE BONOS DE CORTO PLAZO EN TODOS LOS SECTORES DE LA ECON NACIONAL USANDO INFO DCV
# ELIMINA CARTERA AF31 D INFORMADA POR SECTORES, INCLUYENDO SECTOR 3390101. TAMPOCO CONSIDERA CONTRAGENTE 6.
# cierre 2021: tampoco elimina cartera de resto del mundo con contragente 321 porque no aparece en el DCV
work_conn.execute(text("DROP TABLE IF EXISTS #af31_d_ci"))
sql_af31_d_ci = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO * -1) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0433' AS PROC
INTO #af31_d_ci
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.31' AND t1.C_CAGENTE <> '6'
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.C_CAGENTE, t1.SECTOR
"""
work_conn.execute(text(sql_af31_d_ci))
res = work_conn.execute(text("DELETE FROM #af31_d_ci WHERE SECTOR = 6 AND C_CAGENTE = '321'"))
_log("DELETE #af31_d_ci (SECTOR=6, C_CAGENTE=321)", res.rowcount)


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN AF32 D
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ajuste"))
sql_af32_ajuste = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       (t1.DATO * -1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       t1.FUENTE, '0434' AS PROC
INTO #af32_ajuste
FROM #af31_d_ci t1
"""
work_conn.execute(text(sql_af32_ajuste))


In [ ]:
# DCV AF31 TENENCIA. AGREGA TENEDOR 6 PARA IMPUTAR LO DE CORTO DADO QUE NO ESTÁ EN LA CTA DEL RM
# Fuente es un archivo SAS externo (.sas7bdat) sin tabla de BD ni ruta equivalente declarada
raise NotImplementedError("WORK.AF31_D_DCV depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_dcv_af31_act_c21.sas7bdat': no hay tabla de BD ni ruta relativa equivalente declarada para esta fuente DCV")


In [ ]:
# AGRUPA IMPUTACIÓN DEFINITIVA DESDE DCV — depende de AF31_D_DCV no resuelta
raise NotImplementedError("WORK.AF31_DCV depende de WORK.AF31_D_DCV, no resuelta por falta de fuente DCV")


In [ ]:
raise NotImplementedError("UPDATE AF31_DCV SET SECTOR=33901 WHERE SECTOR=33 depende de AF31_DCV, no resuelta por falta de fuente DCV")


In [ ]:
# AJUSTA IMPUTACION DCV DE AF31 CONTRA AF32 EXCEPTO EN TENEDOR 511 — depende de AF31_DCV no resuelta
raise NotImplementedError("WORK.AF32_D_DCV depende de WORK.AF31_DCV, no resuelta por falta de fuente DCV")


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_d_ci"))
_log("APPEND TABLAS.BD_CTSI (AF31_D_CI)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_ajuste"))
_log("APPEND TABLAS.BD_CTSI (AF32_AJUSTE)", res.rowcount)
raise NotImplementedError("APPEND TABLAS.BD_CTSI += WORK.AF31_DCV / WORK.AF32_D_DCV depende de tablas no resueltas por falta de fuente DCV")


In [ ]:
for t in ["#af31_d_ci", "#af32_ajuste"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
# WORK.AF31_D_DCV, WORK.AF32_D_DCV, WORK.AF31_DCV no se llegaron a crear en este tramo (fuente DCV no resuelta)


In [ ]:
# CONCILIACION BONOS DE LP DEL SECTOR 6 Y 31. IMPUTA BONOS DE LP EN EL PASIVO DEL SECTOR 31 CON CA 6, USANDO INFO DEL ACTIVO DEL SECTOR 6 CON CA 31.
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 31 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_31_6"))
sql_af32_31_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       CAST(t1.C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(t1.SECTOR AS varchar(11))) AS C_CAGENTE,
       t1.C_CUENTA, 'H' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0451' AS PROC
INTO #af32_31_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.32' AND t1.SECTOR = 6 AND t1.C_CAGENTE = '31'
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.SECTOR, t1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_31_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_31_53"))
sql_af32_31_53 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, '53' AS C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       (t1.DATO * -1) AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE, '0451b' AS PROC
INTO #af32_31_53
FROM #af32_31_6 t1
"""
work_conn.execute(text(sql_af32_31_53))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_31_6"))
_log("APPEND TABLAS.BD_CTSI (AF32_31_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_31_53"))
_log("APPEND TABLAS.BD_CTSI (AF32_31_53)", res.rowcount)


In [ ]:
for t in ["#af32_31_53", "#af32_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL PASIVO DEL SECTOR 321 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU ACTIVO EL SECTOR 6 CON CA 321. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 321 CON CA 53.
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_6"))
sql_af32_321_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       321 AS SECTOR, '6' AS C_CAGENTE,
       t1.C_CUENTA, 'H' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'H' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0455' AS PROC
INTO #af32_321_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.32' AND t1.SECTOR = 6 AND t1.C_CAGENTE = '321')
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.32' AND t1.SECTOR = 321 AND t1.C_CAGENTE = '6')
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_321_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_53"))
sql_af32_321_53 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, '53' AS C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       t1.DATO * -1 AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE, '0456' AS PROC
INTO #af32_321_53
FROM #af32_321_6 t1
"""
work_conn.execute(text(sql_af32_321_53))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_321_6"))
_log("APPEND TABLAS.BD_CTSI (AF32_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_321_53"))
_log("APPEND TABLAS.BD_CTSI (AF32_321_53)", res.rowcount)


In [ ]:
for t in ["#af32_321_6", "#af32_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE LP EN SECTOR GOBIERNO, EMPRESAS Y HOLDINGS SIN CA 6. CIERRE 2019, CIERRE 2020
# TABLAS.BONOS_EXT_PRECIO existe en el catálogo de conexiones, pero el archivo fuente aquí es t_aj_mdo_af3.sas7bdat (DCV), sin equivalente en BD
work_conn.execute(text("DROP TABLE IF EXISTS #p_mcdo_af32"))
raise NotImplementedError("WORK.P_MCDO_AF32 depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_mdo_af3.sas7bdat' filtrado por SECTOR IN ('41','37','5101','51021'): no hay tabla de BD ni ruta relativa equivalente declarada para esta fuente DCV")


In [ ]:
# VM EN BCE FINAL SECTOR 41 SIN CA 6 NI 54 (RECOMPRAS, PORQUE LLEVAN SU PROPIO PRECIO) — depende de p_mcdo_af32 no resuelta
raise NotImplementedError("WORK.AF32_VM_BF depende de WORK.P_MCDO_AF32, no resuelta por falta de fuente DCV")


In [ ]:
# VM EN BCE INICIO — depende de p_mcdo_af32 no resuelta
raise NotImplementedError("WORK.AF32_VM_BI depende de WORK.P_MCDO_AF32, no resuelta por falta de fuente DCV")


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
raise NotImplementedError("APPEND TABLAS.BD_CTSI += WORK.AF32_VM_BF / WORK.AF32_VM_BI depende de tablas no resueltas por falta de fuente DCV")


In [ ]:
# DROP de WORK.AF32_VM_BF y p_mcdo_af32 — no aplica: no se llegaron a crear en este tramo


In [ ]:
# CIERRE 2020. CALCULA REC PRECIO EN BONOS POR EFECTO DE CALCULO DE SALDOS A VALOR DE MERCADO-PASIVO EN GOBIERNO, EMPRESAS (LO QUE NO ES RECOMPRA) Y HOLDINGS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp"))
sql_af32_rp = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE,
       'Rec Precio' AS C_CUENTA, t1.C_ENTRADA,
       (CASE WHEN t1.C_CUENTA <> 'Bce Final' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN, t1.N_SCN,
       'PS' AS FUENTE, '0460c' AS PROC
INTO #af32_rp
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN IN ('AF.32') AND t1.SECTOR IN (41,37,5101,51021) AND t1.C_CAGENTE NOT IN ('6','54')
GROUP BY t1.[AÑO], t1.TRIM, t1.C_ENTRADA,
         (CASE WHEN t1.C_CUENTA <> 'Bce Final' THEN t1.DATO * -1 ELSE t1.DATO END),
         t1.C_SCN, t1.N_SCN, t1.C_CAGENTE, t1.SECTOR
"""
work_conn.execute(text(sql_af32_rp))


In [ ]:
# AGRUPA ANTERIOR
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp_2"))
sql_af32_rp_2 = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN, t1.N_SCN, t1.FUENTE, t1.PROC
INTO #af32_rp_2
FROM #af32_rp t1
WHERE t1.DATO <> 0
GROUP BY t1.MONEDA, t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.C_CAGENTE, t1.FUENTE, t1.SECTOR, t1.PROC
"""
work_conn.execute(text(sql_af32_rp_2))


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_rp_2"))
_log("APPEND TABLAS.BD_CTSI (AF32_RP_2)", res.rowcount)


In [ ]:
for t in ["#af32_rp_2", "#af32_rp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE LP EN SECTOR GOBIERNO, EMPRESAS Y HOLDINGS CON CA 6 USANDO PRECIO DEL RM. VALORA RECOMPRAS A VALOR DE MERCADO (CA 54). CIERRE 2020
# VM EN SECTOR 41 CA 6 — usa TABLAS.BONOS_EXT_PRECIO, tabla real del catálogo de conexiones
work_conn.execute(text("DROP TABLE IF EXISTS #af32_vm_6"))
sql_af32_vm_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO * t2.Precio - t1.DATO) AS DATO,
       t1.C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'VM' AS FUENTE, '0460a' AS PROC
INTO #af32_vm_6
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.BONOS_EXT_PRECIO t2
    ON t1.[AÑO] = t2.[Año] AND t1.TRIM = t2.Trim AND t1.SECTOR = t2.SECTOR
   AND t1.C_SCN = t2.C_SCN AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_CAGENTE = t2.C_CAGENTE
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.32' AND t1.SECTOR IN (41,37,51021,5101,36912)
  AND t1.C_CAGENTE IN ('6','54') AND t1.FUENTE = 'CI'
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.C_CAGENTE, t1.SECTOR
"""
work_conn.execute(text(sql_af32_vm_6))
_af32_vm_6 = pd.read_sql(text("SELECT * FROM #af32_vm_6"), work_conn)
_log("af32_vm_6", _af32_vm_6)


In [ ]:
# REC PRECIO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp_6"))
sql_af32_rp_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Rec Precio' AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(CASE WHEN T1.C_CUENTA = 'Bce Inicio' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0460c' AS PROC
INTO #af32_rp_6
FROM #af32_vm_6 T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_rp_6))


In [ ]:
# PROC DATASETS: APPEND de AF32_VM_6 y AF32_RP_6 (creado en tramo anterior) a TABLAS.BD_CTSI, luego DROP
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_vm_6 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32})
SELECT {cols_af32}
FROM #af32_vm_6
"""
res = work_conn.execute(text(sql_append_vm_6))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_vm_6", res.rowcount)

sql_append_rp_6 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32})
SELECT {cols_af32}
FROM #af32_rp_6
"""
res = work_conn.execute(text(sql_append_rp_6))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_rp_6", res.rowcount)

for t in ["#af32_vm_6", "#af32_rp_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CAMBIA CONTRAGENTE DE RECOMPRAS A 9, DESPUES QUE SON LLEVADAS A VALOR DE MERCADO
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '9' "
    "WHERE C_CAGENTE = '54' AND C_SCN = 'AF.32' AND C_ENTRADA = 'H' AND SECTOR IN (5101, 51021)"
))
_log("UPDATE TABLAS.dbo.BD_CTSI C_CAGENTE=9 recompras", res.rowcount)


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL PASIVO DEL SECTOR 41 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU ACTIVO EL SECTOR 6 CON CA 41. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 41 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_41_6"))
sql_af32_41_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0460' AS PROC
INTO #af32_41_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '41')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 41 AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_41_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_41_53"))
sql_af32_41_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0461' AS PROC
INTO #af32_41_53
FROM #af32_41_6 T1
"""
work_conn.execute(text(sql_af32_41_53))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_41_6 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_41_6"
res = work_conn.execute(text(sql_append_41_6))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_41_6", res.rowcount)

sql_append_41_53 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_41_53"
res = work_conn.execute(text(sql_append_41_53))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_41_53", res.rowcount)

for t in ["#af32_41_6", "#af32_41_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA BONOS DE LP EN EL PASIVO DEL SECTOR 5101 CON CA 6, USANDO INFO DE ACTIVO EL SECTOR 6 CON CA 5101
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 5101 CON CA 53
# CIERRE 2020. AJUSTA DIFERENCIAL PORQUE AHORA EMPRESAS VIENE CON CONTRAGENTE EN LOS BONOS
work_conn.execute(text("DROP TABLE IF EXISTS #af32_5101_6"))
sql_af32_5101_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       5101 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0463' AS PROC
INTO #af32_5101_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '5101')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 5101 AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_5101_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_5101_53"))
sql_af32_5101_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0464' AS PROC
INTO #af32_5101_53
FROM #af32_5101_6 T1
"""
work_conn.execute(text(sql_af32_5101_53))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_5101_6 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_5101_6"
res = work_conn.execute(text(sql_append_5101_6))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_5101_6", res.rowcount)

sql_append_5101_53 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_5101_53"
res = work_conn.execute(text(sql_append_5101_53))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_5101_53", res.rowcount)

for t in ["#af32_5101_6", "#af32_5101_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN ACTIVO AF32 DEL SECTOR 6 CON CA 53 CAMBIA CA A SECTOR 51021
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51021', PROC = '0465' "
    "WHERE SECTOR = 6 AND C_SCN = 'AF.32' AND C_ENTRADA = 'D' AND C_CAGENTE = '53'"
))
_log("UPDATE TABLAS.dbo.BD_CTSI sector6 CA 53->51021", res.rowcount)


In [ ]:
# IMPUTA BONOS DE LP EN EL PASIVO DEL SECTOR 51021 CON CA 6, USANDO INFO DE ACTIVO EL SECTOR 6 CON CA 51021
# AJUSTA IMPUTACIÓN CONTRA BONOS DE LP DEL SECTOR 5101 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51021_6"))
sql_af32_51021_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51021 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0467' AS PROC
INTO #af32_51021_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '51021')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 51021 AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_51021_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51021_53"))
sql_af32_51021_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0468' AS PROC
INTO #af32_51021_53
FROM #af32_51021_6 T1
"""
work_conn.execute(text(sql_af32_51021_53))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_51021_6 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_51021_6"
res = work_conn.execute(text(sql_append_51021_6))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_51021_6", res.rowcount)

sql_append_51021_53 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_51021_53"
res = work_conn.execute(text(sql_append_51021_53))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_51021_53", res.rowcount)

for t in ["#af32_51021_6", "#af32_51021_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# RESTA BONOS DE LP EMITIDOS POR LOS HOLDINGS EN EL MERCADO EXTERNO DE LAS EMISIONES DE EMPRESAS EN EL MERCADO EXTERNO.
# SE RESTA LO QUE DICE EL ACTIVO DEL RM QUE LOS SECTORES AUXILIARES DICEN EMITIR FUERA (CIERRE 2020)
# AJUSTA ESTA IMPUTACIÓN EN LOS BONOS DE LP DE EMPRESAS EN EL MERCADO LOCAL (CONTRAGENTE 53).
# CIERRE 2022Q2: INCORPORA AJUSTE TMB POR EMISIONES EXTERNAS DE SECTOR 36
work_conn.execute(text("DROP TABLE IF EXISTS #af32_37_6"))
sql_af32_37_6 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51021 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0468a' AS PROC
INTO #af32_37_6
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR IN (37, 36912) AND T1.C_CAGENTE = '6' AND T1.DATO IS NOT NULL)
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE IN ('37', '36912') AND T1.DATO IS NOT NULL)
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_37_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51021_53_v2"))
sql_af32_51021_53_v2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0468b' AS PROC
INTO #af32_51021_53_v2
FROM #af32_37_6 T1
"""
work_conn.execute(text(sql_af32_51021_53_v2))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_37_6 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_37_6"
res = work_conn.execute(text(sql_append_37_6))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_37_6", res.rowcount)

sql_append_51021_53_v2 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_51021_53_v2"
res = work_conn.execute(text(sql_append_51021_53_v2))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_51021_53_v2", res.rowcount)

for t in ["#af32_51021_53_v2", "#af32_37_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN ACTIVO AF32 DEL SECTOR 36904 CON CA 6 CAMBIA CA A SECTOR 53
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0469' "
    "WHERE SECTOR = 36904 AND C_SCN = 'AF.32' AND C_ENTRADA = 'H' AND C_CAGENTE = '6'"
))
_log("UPDATE TABLAS.dbo.BD_CTSI sector36904 CA 6->53", res.rowcount)


In [ ]:
# CONCILIACION BONOS DE LP DEL SECTOR 6 Y 31. IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 31 CON CA 6,
# COMPARANDO CON INFO DEL PASIVO DEL SECTOR 6 CON CA 31.
# RESPETA DATO DEL PASIVO DEL SECTOR 31. AJUSTA IMPUTACIÓN CONTRA AJUSTE DE CONCILIACIÓN EN ACTIVO DE SECTOR 31 CON CA 6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_31_6_a"))
sql_af32_31_6_a = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       31 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.SECTOR = 6 AND T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio'
             WHEN T1.SECTOR = 31 AND T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio'
             ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0473' AS PROC
INTO #af32_31_6_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 31 AND T1.C_CAGENTE = '6')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '31')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         (CASE WHEN T1.SECTOR = 6 AND T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio'
               WHEN T1.SECTOR = 31 AND T1.C_CUENTA IN ('Rec Precio Reaj', 'Rec Volumen') THEN 'Rec Precio'
               ELSE T1.C_CUENTA END)
"""
work_conn.execute(text(sql_af32_31_6_a))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_31_6"))
sql_af71_31_6 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       T1.FUENTE,
       '0474' AS PROC
INTO #af71_31_6
FROM #af32_31_6_a T1
"""
work_conn.execute(text(sql_af71_31_6))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_31_6_a = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_31_6_a"
res = work_conn.execute(text(sql_append_31_6_a))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_31_6_a", res.rowcount)

sql_append_71_31_6 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af71_31_6"
res = work_conn.execute(text(sql_append_71_31_6))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af71_31_6", res.rowcount)

for t in ["#af32_31_6_a", "#af71_31_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
raise NotImplementedError(
    "WORK.AF32_BCOS_DCV lee un archivo SAS externo '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/base_af3_total_v2.sas7bdat' "
    "(comentario: DCV AF32 tenencia de bonos de bancos con CA gob, bcos y central). No hay ruta relativa ni file_imports "
    "provistos por el parser para este .sas7bdat; se requiere definir la ubicación del archivo en el workspace (p.ej. vía pyreadstat) "
    "antes de poder traducir este bloque."
)


In [ ]:
raise NotImplementedError(
    "WORK.AF32_BANCOS, WORK.AF32_BANCOS_DELTA, WORK.AF32_BANCOS_AJ y WORK.AF7_EMP dependen de #af32_bcos_dcv, "
    "que no pudo materializarse porque su fuente es el archivo externo base_af3_total_v2.sas7bdat (ver bloque anterior). "
    "Sin esa tabla no se puede calcular el DELTA de tenencia de bonos de bancos ni los ajustes en AF32/AF7 contra resto del mundo y empresas."
)


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 321 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU PASIVO EL SECTOR 6 CON CA 321.
# RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA ACTIVO DE AJUSTE Y DISCREPANCIA DEL SECTOR 321 CON CA 6
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_6_a"))
sql_af32_321_6_a = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0478' AS PROC
INTO #af32_321_6_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE IN ('321', '322'))
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR IN (321, 322) AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         (CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END)
"""
work_conn.execute(text(sql_af32_321_6_a))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_321_6_a"))
sql_af71_321_6_a = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       '0479' AS PROC
INTO #af71_321_6_a
FROM #af32_321_6_a T1
"""
work_conn.execute(text(sql_af71_321_6_a))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_321_6_a = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_321_6_a"
res = work_conn.execute(text(sql_append_321_6_a))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_321_6_a", res.rowcount)

sql_append_71_321_6_a = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af71_321_6_a"
res = work_conn.execute(text(sql_append_71_321_6_a))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af71_321_6_a", res.rowcount)

for t in ["#af32_321_6_a", "#af71_321_6_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN ACTIVO AF32 DEL SECTOR 339011 CON CA 53 CAMBIA CA A SECTOR 6
res = work_conn.execute(text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '6', PROC = '0481' "
    "WHERE SECTOR = 339011 AND C_SCN = 'AF.32' AND C_ENTRADA = 'D' AND C_CAGENTE = '53'"
))
_log("UPDATE TABLAS.dbo.BD_CTSI sector339011 CA 53->6", res.rowcount)


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 351 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU PASIVO EL SECTOR 6 CON CA 351
# CON ACTIVO DE LOS SECTORES 351/352 CON CA 6. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA ACTIVO DE BONOS DE LP DEL SECTOR 351 CON CA 53
# CIERRE 2021: SE CAMBIA AJUSTE A AF34 CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af32_351_6_a"))
sql_af32_351_6_a = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       351 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0484' AS PROC
INTO #af32_351_6_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '351')
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR IN (351, 352) AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END)
"""
work_conn.execute(text(sql_af32_351_6_a))


In [ ]:
# imputa derivados en activo de seguros con ca resto del mundo
work_conn.execute(text("DROP TABLE IF EXISTS #af32_351_53_a"))
sql_af32_351_53_a = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.34' AS C_SCN,
       'Derivados financieros' AS N_SCN,
       T1.FUENTE,
       '0485' AS PROC
INTO #af32_351_53_a
FROM #af32_351_6_a T1
"""
work_conn.execute(text(sql_af32_351_53_a))


In [ ]:
# imputa derivados en pasivo de resto del mundo con contragente seguros
work_conn.execute(text("DROP TABLE IF EXISTS #af34_6_35"))
sql_af34_6_35 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '35' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0485' AS PROC
INTO #af34_6_35
FROM #af32_351_53_a T1
"""
work_conn.execute(text(sql_af34_6_35))


In [ ]:
# ajusta en derivados en pasivo de resto del mundo con contragente resto
work_conn.execute(text("DROP TABLE IF EXISTS #af34_6_53"))
sql_af34_6_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0485' AS PROC
INTO #af34_6_53
FROM #af34_6_35 T1
"""
work_conn.execute(text(sql_af34_6_53))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_351_6_a = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_351_6_a"
res = work_conn.execute(text(sql_append_351_6_a))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_351_6_a", res.rowcount)

sql_append_351_53_a = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_351_53_a"
res = work_conn.execute(text(sql_append_351_53_a))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_351_53_a", res.rowcount)

sql_append_34_6_35 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af34_6_35"
res = work_conn.execute(text(sql_append_34_6_35))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af34_6_35", res.rowcount)

sql_append_34_6_53 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af34_6_53"
res = work_conn.execute(text(sql_append_34_6_53))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af34_6_53", res.rowcount)

for t in ["#af32_351_6_a", "#af32_351_53_a", "#af34_6_35", "#af34_6_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DIF DE BONOS DE LP EN EL ACTIVO DEL SECTOR 3390101/3390102/339011 CON CA 6, COMPARANDO LO QUE DICE TENER EN SU PASIVO EL SECTOR 6 CON CA 33901/351 y 33901
# CON ACTIVO DE LOS SECTORES 33901/339011 CON CA 6. RESPETA DATO DEL SECTOR 6
# AJUSTA IMPUTACIÓN CONTRA ACTIVO DE BONOS DE LP DEL SECTOR 33 CON CA 53
# CIERRE 2021: PARA AJUSTAR LA IMPUTACIÓN SE CAMBIA AL SECTOR 33901 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33_6_a"))
sql_af32_33_6_a = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       33901 AS SECTOR,
       '6' AS C_CAGENTE,
       (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0484b' AS PROC
INTO #af32_33_6_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE IN ('33901/351', '33901'))
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR IN (3390101, 3390102, 339011, 33222, 33212) AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN,
         (CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Rec Precio' WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END)
"""
work_conn.execute(text(sql_af32_33_6_a))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33_53_a"))
sql_af32_33_53_a = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0485b' AS PROC
INTO #af32_33_53_a
FROM #af32_33_6_a T1
"""
work_conn.execute(text(sql_af32_33_53_a))


In [ ]:
cols_af32 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_33_6_a = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_33_6_a"
res = work_conn.execute(text(sql_append_33_6_a))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_33_6_a", res.rowcount)

sql_append_33_53_a = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af32}) SELECT {cols_af32} FROM #af32_33_53_a"
res = work_conn.execute(text(sql_append_33_53_a))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_33_53_a", res.rowcount)

for t in ["#af32_33_6_a", "#af32_33_53_a"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ANOTACION: este es el tramo 3 de 5 de un nodo dividido por el traductor;
# continua tras AF32_51022_6 (tramo anterior) y sigue con la imputacion de
# bonos LP en el pasivo del sector OFIS+AUX, comparando contra el DCV.

# IMPUTA DIF DE BONOS DE LP EN EL PASIVO DEL SECTOR OFIS+AUX, COMPARANDO LO QUE DICE EL DCV DEL PASIVO DE OFIS+AUX. RESPETA DATO DCV. CIERRE 2019 HACE AJUSTE PRIMERO DEL VALOR PAR
# AJUSTA IMPUTACION CONTRA CREDITOS COMERCIALES/PTMOS CA 53 EN EL PASIVO. cierre 2021: elimina a holdings de este calculo porque ya se valoro a mercado anteriormente y esta consulta revierte ese efecto
work_conn.execute(text("DROP TABLE IF EXISTS #af32_oa_emision"))
sql_af32_oa_emision = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0493' AS PROC
INTO #af32_oa_emision
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE T1.SECTOR = T2.C_SI AND (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND T1.FUENTE IN ('CI','VM','PS') AND T2.C_SI_SCN IN ('S.123','S.124','S.128') AND T1.C_CAGENTE <> '6' AND T1.SECTOR <> 37)
GROUP BY t1.[AÑO], T1.TRIM,
         (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_oa_emision))


In [ ]:
# INFO DESDE DCV A VALOR PAR
# NOTA: la fuente original es un archivo SAS externo
# (/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/base_af3_total_v2.sas7bdat)
# que no forma parte del catalogo de conexiones de BD ni de input_datasets del nodo;
# no hay ruta relativa ni dataset declarado para resolverlo, así que se declara el hueco.
raise NotImplementedError(
    "AF32_OA_DCV depende del archivo externo base_af3_total_v2.sas7bdat "
    "(DCVRES) que no está mapeado como conexión de BD ni como input_dataset del nodo; "
    "falta la ruta relativa/fuente autorizada para leerlo con pyreadstat"
)


In [ ]:
# NOTA: los 4 UPDATE de SECTOR sobre WORK.AF32_OA_DCV dependen de la celda anterior
# (que no se pudo resolver); se declaran igual por completitud del nodo pero no pueden
# ejecutarse hasta que AF32_OA_DCV exista como #tmp.
raise NotImplementedError(
    "Los 4 UPDATE sobre #af32_oa_dcv (SECTOR 3321->33212, 3322->33222, "
    "36901->369011, 33901->339011) dependen de que #af32_oa_dcv exista; "
    "bloqueado por la celda anterior (fuente DCV externa no resuelta)"
)


In [ ]:
# JUNTA TABLAS PARA HACER RESTA: PROC DATASETS APPEND BASE=WORK.AF32_OA_EMISION DATA=WORK.AF32_OA_DCV FORCE
# NOTA: depende de #af32_oa_dcv, no resuelto (ver celda anterior).
raise NotImplementedError(
    "APPEND de #af32_oa_dcv en #af32_oa_emision bloqueado: #af32_oa_dcv no se "
    "pudo materializar (fuente DCV externa no resuelta)"
)


In [ ]:
# CALCULA DIF DCV-CI
# elimina emisiones de patrimonio separado porque se imputan en otra parte del proceso
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dcv_ci_dif"))
sql_af32_dcv_ci_dif = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0493' AS PROC
INTO #af32_dcv_ci_dif
FROM #af32_oa_emision T1
WHERE T1.SECTOR NOT IN (3324)
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_dcv_ci_dif))


In [ ]:
# AJUSTE EN CRED COM o ptmos segun corresponda en el sector
work_conn.execute(text("DROP TABLE IF EXISTS #af71_dcv_ci_dif"))
sql_af71_dcv_ci_dif = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       (CASE WHEN T1.SECTOR IN (411,33212,33222,339011) THEN '321' WHEN T1.SECTOR=37 THEN '6' ELSE '53' END) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       (CASE WHEN T1.SECTOR IN (411,33212,33222,339011,37) THEN 'AF.42' ELSE 'AF.7' END) AS C_SCN,
       (CASE WHEN T1.SECTOR IN (411,33212,33222,339011,37) THEN 'Préstamos a largo plazo' ELSE 'Créditos comerciales' END) AS N_SCN,
       T1.FUENTE, '0494' AS PROC
INTO #af71_dcv_ci_dif
FROM #af32_dcv_ci_dif T1
"""
work_conn.execute(text(sql_af71_dcv_ci_dif))


In [ ]:
# AJUSTE EN PTMOS DE LP PASIVO DE RESTO DE EMPRESAS CON BCOS DEBIDO A AJUSTE 0494 EN LA PARTE QUE SE IMPUTA EN PTMOS BANCARIOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_bcos_dcv"))
sql_af42_bcos_dcv = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494a' AS PROC
INTO #af42_bcos_dcv
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN = 'AF.42' AND T1.C_CAGENTE = '321'
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_bcos_dcv))


In [ ]:
# AJUSTE EN PTMOS DE LP ACTIVO DE RESTO DEL MUNDO CON CONTRAGENTE 37 DEBIDO A AJUSTE 0494 EN LA PARTE QUE SE IMPUTA EN PTMOS CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af42_rm_dcv"))
sql_af42_rm_dcv = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '37' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494b' AS PROC
INTO #af42_rm_dcv
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN = 'AF.42' AND T1.C_CAGENTE = '6' AND T1.SECTOR = 37
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_rm_dcv))


In [ ]:
# AJUSTE EN PTMOS DE LP ACTIVO DE RESTO DEL MUNDO CON CONTRAGENTE 53 DEBIDO A AJUSTE 0494b
work_conn.execute(text("DROP TABLE IF EXISTS #af42_rm_resto"))
sql_af42_rm_resto = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494c' AS PROC
INTO #af42_rm_resto
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN = 'AF.42' AND T1.C_CAGENTE = '6' AND T1.SECTOR = 37
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_rm_resto))


In [ ]:
# AJUSTE EN PTMOS DE LP PASIVO DE RESTO DE EMPRESAS CON EL RESTO DEL MUNDO A AJUSTE 0494
work_conn.execute(text("DROP TABLE IF EXISTS #af42_resto_rm"))
sql_af42_resto_rm = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0494d' AS PROC
INTO #af42_resto_rm
FROM #af71_dcv_ci_dif T1
WHERE T1.C_SCN = 'AF.42' AND T1.C_CAGENTE = '6' AND T1.SECTOR = 37
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_af42_resto_rm))


In [ ]:
# APPEND server-side de los 6 ajustes DCV-CI a TABLAS.BD_CTSI (mismas columnas de #tmp)
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_name in ["#af32_dcv_ci_dif", "#af71_dcv_ci_dif", "#af42_bcos_dcv", "#af42_rm_dcv", "#af42_rm_resto", "#af42_resto_rm"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_name}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)


In [ ]:
# DROP de temporales de este bloque (re-ejecutable)
for t in ["#af32_oa_emision", "#af32_dcv_ci_dif", "#af71_dcv_ci_dif", "#af42_bcos_dcv", "#af42_rm_dcv", "#af42_rm_resto", "#af42_resto_rm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA EMISIONES DE OFIS EN PODER DE NO RESIDENTES COMPARANDO CON LO QUE DICE EL RM. RESPETA DATO DEL RM. AJUSTA EN PTMOS DE LP CA 321 EN PASIVO DE OFIS
# CALCULA DIFERENCIA A IMPUTAR EN BONOS DE LP
# CIERRE 2021: SE DEJA EN EL SECTOR 33 POR APERTURA DE LOS FONDOS NO MONEY MARKET
# selecciona activo del resto del mundo con ofis
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33_6"))
sql_af32_33_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       33 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0495' AS PROC
INTO #af32_33_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR = 6 AND T1.C_CAGENTE LIKE '33%')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.32' AND TRY_CAST(SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(9))), 1, 2) AS int) = 33 AND T1.C_CAGENTE = '6' AND T1.SECTOR NOT IN (334,331))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR
"""
work_conn.execute(text(sql_af32_33_6))


In [ ]:
# AJUSTA EN PTMOS DE LP CA BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_33_321"))
sql_af42_33_321 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       t1.SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       t1.DATO * -1 AS DATO,
       'AF.42' AS C_SCN,
       'Préstamos a largo plazo' AS N_SCN,
       T1.FUENTE,
       '0495a' AS PROC
INTO #af42_33_321
FROM #af32_33_6 t1
"""
work_conn.execute(text(sql_af42_33_321))


In [ ]:
# AJUSTA EN PTMOS DE LP DE RESTO DE EMPRESAS CA BANCOS PARA EQUILIBRAR LOS PTMOS
work_conn.execute(text("DROP TABLE IF EXISTS #af42_51_321"))
sql_af42_51_321 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '321' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       t1.DATO AS DATO,
       'AF.42' AS C_SCN,
       'Préstamos a largo plazo' AS N_SCN,
       T1.FUENTE,
       '0495b' AS PROC
INTO #af42_51_321
FROM #af32_33_6 t1
"""
work_conn.execute(text(sql_af42_51_321))


In [ ]:
# APPEND server-side de AF32_33_6, AF42_33_321, AF42_51_321 a TABLAS.BD_CTSI
for tmp_name in ["#af32_33_6", "#af42_33_321", "#af42_51_321"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_name}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)

for t in ["#af32_33_6", "#af42_33_321", "#af42_51_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: IMPUTA ACTIVO AF31 DEL RESTO CON CONTRAGENTES BANCOS EN PASIVO DE BANCOS CON RM. AJUSTA EN PASIVO AF32 DE BANCOS CON 53
# selecciona activo rm con bcos
work_conn.execute(text("DROP TABLE IF EXISTS #af31_6_321"))
sql_af31_6_321 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0435a' AS PROC
INTO #af31_6_321
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.31' AND T1.SECTOR = 6 AND T1.C_CAGENTE IN ('321','322','32')) OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.31' AND T1.SECTOR IN (321,322) AND T1.C_CAGENTE IN ('6'))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af31_6_321))


In [ ]:
# ajusta en af32 pasivo de bancos con ca 53
work_conn.execute(text("DROP TABLE IF EXISTS #af32_321_6"))
sql_af32_321_6 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_321_6
FROM #af31_6_321 t1
"""
work_conn.execute(text(sql_af32_321_6))


In [ ]:
# APPEND server-side de AF31_6_321 y AF32_321_6 a TABLAS.BD_CTSI
for tmp_name in ["#af31_6_321", "#af32_321_6"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_name}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)

for t in ["#af31_6_321", "#af32_321_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA CARTERA DE BONOS DE LARGO PLAZO EN EL SECTOR AUXILIARES FINANCIEROS USANDO INFO DCV
# SELECCIONA ACTIVO AF32 INFORMADA
work_conn.execute(text("DROP TABLE IF EXISTS #af32_aux"))
sql_af32_aux = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0436d' AS PROC
INTO #af32_aux
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE T1.SECTOR = T2.C_SI AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T2.C_SI_SCN = 'S.124'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T2.C_SI_publ
"""
work_conn.execute(text(sql_af32_aux))


In [ ]:
# DCV AF32 TENENCIA DE AUXILIARES SIN EMISOR RM
# NOTA: fuente es archivo SAS externo t_aj_dcv_af32_act_ao.sas7bdat (DCVRES),
# no mapeado como conexión de BD ni declarado en input_datasets del nodo.
raise NotImplementedError(
    "AF32_D_DCV (variante 'DCV AF32 tenencia de auxiliares sin emisor RM') depende "
    "del archivo externo t_aj_dcv_af32_act_ao.sas7bdat (DCVRES) sin ruta relativa "
    "autorizada ni conexión de BD que lo represente"
)


In [ ]:
# JUNTA TABLAS PARA HACER RESTA: APPEND BASE=WORK.AF32_AUX DATA=WORK.AF32_D_DCV FORCE
# NOTA: depende de #af32_d_dcv (celda anterior no resuelta)
raise NotImplementedError(
    "APPEND de #af32_d_dcv en #af32_aux bloqueado: #af32_d_dcv no se pudo "
    "materializar (fuente DCV externa t_aj_dcv_af32_act_ao no resuelta)"
)


In [ ]:
# AGRUPA DIF DCV-CI
# NOTA: consume #af32_aux post-APPEND del bloque anterior, que quedó incompleto
# porque #af32_d_dcv no se resolvió; se traduce la consulta igual, correrá
# solo sobre lo que #af32_aux tenga en ese momento (sin el aporte del DCV).
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dif"))
sql_af32_dif_aux = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_dif
FROM #af32_aux T1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE, T1.SECTOR, T1.PROC
"""
work_conn.execute(text(sql_af32_dif_aux))


In [ ]:
# AJUSTE EN CREDITOS COMERCIALES
work_conn.execute(text("DROP TABLE IF EXISTS #af7_aj_oa"))
sql_af7_aj_oa = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       T1.FUENTE, '0436e' AS PROC
INTO #af7_aj_oa
FROM #af32_dif T1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.FUENTE
"""
work_conn.execute(text(sql_af7_aj_oa))


In [ ]:
# ANEXA AJUSTE BONOS LP EN AUX / ANEXA AJUSTE cred com EN AUX
for tmp_name in ["#af32_dif", "#af7_aj_oa"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_name}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)

for t in ["#af32_aux", "#af32_d_dcv", "#af32_dif", "#af7_aj_oa"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA DIFERENCIAL DE BONOS SECURITIZADOS EN RESTO DE EMPRESAS USANDO DCV MENOS TODO LO QUE SE ENCUENTRA EN CARTERA DE LOS INVERSIONISTAS
# SELECCIONA ACTIVO DE SECTORES EN PATRIMONIO SEPARADO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_pat_tot"))
sql_af32_pat_tot = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '3324' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0436j' AS PROC
INTO #af32_pat_tot
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.C_CAGENTE IN ('332','3323','3324')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_pat_tot))


In [ ]:
# DCV AF32 EMISION TOTAL PAT SEPARADO
# NOTA: misma fuente externa base_af3_total_v2.sas7bdat que en AF32_OA_DCV, sin
# conexión de BD ni ruta relativa disponible.
raise NotImplementedError(
    "AF32_D_DCV (variante 'DCV AF32 emision total pat separado') depende del "
    "archivo externo base_af3_total_v2.sas7bdat (DCVRES) sin ruta relativa "
    "autorizada ni conexión de BD que lo represente"
)


In [ ]:
# JUNTA TABLAS PARA HACER RESTA: APPEND BASE=WORK.AF32_PAT_TOT DATA=WORK.AF32_D_DCV FORCE
# NOTA: depende de #af32_d_dcv (celda anterior no resuelta)
raise NotImplementedError(
    "APPEND de #af32_d_dcv en #af32_pat_tot bloqueado: #af32_d_dcv (variante "
    "pat separado) no se pudo materializar (fuente DCV externa no resuelta)"
)


In [ ]:
# AGRUPA DIF DCV-CI (sobre AF32_PAT_TOT tras el intento de APPEND)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dif"))
sql_af32_dif_pat = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_dif
FROM #af32_pat_tot T1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE, T1.SECTOR, T1.PROC
"""
work_conn.execute(text(sql_af32_dif_pat))


In [ ]:
# AJUSTE EN cred com DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_emp"))
sql_af7_emp = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       t1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       T1.FUENTE, '0436k' AS PROC
INTO #af7_emp
FROM #af32_dif T1
"""
work_conn.execute(text(sql_af7_emp))


In [ ]:
# ANEXA AJUSTE BONOS LP EN AUX / ANEXA AJUSTE cred com EN AUX (variante pat separado)
for tmp_name in ["#af32_dif", "#af7_emp"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_name}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)

for t in ["#af32_pat_tot", "#af32_d_dcv", "#af32_dif", "#af7_emp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA BONOS DE LARGO PLAZO EMITIDOS POR PATRIMONIO SEPARADO, USANDO INFO DE LA CARTERA DE LOS AUX FINANCIEROS Y EMPRESAS
# TAMBIEN GENERA ACTIVO DE PRESTAMOS DE LP (2/3) Y CRED COMERCIALES (1/3) USANDO IMPUTACION DE BONOS
# SELECCIONA AF32 DE PAT SEPARADO DESDE AUX FIN
work_conn.execute(text("DROP TABLE IF EXISTS #af32_pat_aux"))
sql_af32_pat_aux = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       3324 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0436f' AS PROC
INTO #af32_pat_aux
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T1.SECTOR IN (36,51022) AND T1.C_CAGENTE = '3324'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af32_pat_aux))


In [ ]:
# IMPUTA ACTIVO DE PRESTAMOS DE LP
work_conn.execute(text("DROP TABLE IF EXISTS #af42_pat"))
sql_af42_pat = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '51022' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       (T1.DATO * 2.0 / 3.0) AS DATO,
       'AF.42' AS C_SCN,
       'Préstamos a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0436g' AS PROC
INTO #af42_pat
FROM #af32_pat_aux t1
"""
work_conn.execute(text(sql_af42_pat))
res_check = pd.read_sql(text("SELECT COUNT(*) AS n FROM #af42_pat"), work_conn)
_log("af42_pat filas", res_check["n"].iloc[0])


In [ ]:
# PROC SQL: IMPUTA ACTIVO DE CRÉDITOS COMERCIALES (server-side, WORK.AF7_PAT nace de #af32_pat_aux ya materializada en tramos previos)
work_conn.execute(text("DROP TABLE IF EXISTS #af7_pat"))
sql_af7_pat = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '51022' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       (T1.DATO*1.0/3) AS DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       'PS' AS FUENTE, '0436h' AS PROC
INTO #af7_pat
FROM #af32_pat_aux t1
"""
work_conn.execute(text(sql_af7_pat))


In [ ]:
# PROC SQL: IMPUTA PASIVO DE PRÉSTAMOS DE LP EN SECTOR 51022 CON CA 3324
work_conn.execute(text("DROP TABLE IF EXISTS #af42_resto"))
sql_af42_resto = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '3324' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0436i' AS PROC
INTO #af42_resto
FROM #af32_pat t1
"""
work_conn.execute(text(sql_af42_resto))


In [ ]:
# PROC DATASETS: APPEND de las 4 tablas hacia TABLAS.BD_CTSI, server-side desde las #tmp
cols_bd_ctsi_pat = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_src in ["#af32_pat_aux", "#af42_pat", "#af7_pat", "#af42_resto"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_pat}) SELECT {cols_bd_ctsi_pat} FROM {tmp_src}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_src}", res.rowcount)


In [ ]:
# PROC SQL: drop de las 4 temporales del tramo (DROP TABLE de SAS -> DROP TABLE IF EXISTS sobre # de sesión)
for t in ["#af32_pat_aux", "#af42_pat", "#af7_pat", "#af42_resto"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# NOTA: bloque comentado en el SAS original (ajuste manual de precio 2019 T1/T2 sector 41, eliminado en cierre 2019) NO se traduce: quedó anulado en el fuente.
# IMPUTA VALOR DE MERCADO EN PASIVO DE BONOS DE LP, EXCEPTO EN SECTORES 6 Y 3324 Y CA 6
# El archivo t_aj_mdo_af3.sas7bdat es una fuente externa (ruta SAS absoluta) sin equivalente de conexión declarado en el contexto del proyecto: no se puede resolver con la información disponible.
raise NotImplementedError("WORK.P_MCDO_AF32 depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_mdo_af3.sas7bdat' — archivo SAS7BDAT externo sin ruta de proyecto ni conexión declarada; requiere definir cómo se lee este insumo (pyreadstat + ruta relativa del workspace) antes de traducir")

In [ ]:
# PROC SQL: UPDATE p_mcdo_af32 SET SECTOR=339011 WHERE SECTOR=33901
# PROC SQL: UPDATE p_mcdo_af32 SET SECTOR=369011 WHERE SECTOR=36901
# Dependen de p_mcdo_af32 (celda anterior, no resuelta) — mismo hueco declarado
raise NotImplementedError("UPDATEs sobre p_mcdo_af32 (SECTOR 33901->339011, 36901->369011) no se pueden traducir: p_mcdo_af32 depende de la fuente externa no resuelta arriba")

In [ ]:
# PROC SQL: VM EN BCE FINAL (WORK.AF32_VM_BF) — depende de p_mcdo_af32, no resuelto
raise NotImplementedError("WORK.AF32_VM_BF requiere p_mcdo_af32 (fuente externa t_aj_mdo_af3.sas7bdat no resuelta)")

In [ ]:
# PROC SQL: VM EN BCE INICIO (WORK.AF32_VM_BI) — depende de p_mcdo_af32, no resuelto
raise NotImplementedError("WORK.AF32_VM_BI requiere p_mcdo_af32 (fuente externa t_aj_mdo_af3.sas7bdat no resuelta)")

In [ ]:
# PROC DATASETS: APPEND AF32_VM_BF y AF32_VM_BI hacia TABLAS.BD_CTSI — no se pueden ejecutar sin las tablas fuente
raise NotImplementedError("APPEND de WORK.AF32_VM_BF / WORK.AF32_VM_BI a TABLAS.BD_CTSI bloqueado: ambas tablas dependen de p_mcdo_af32 no resuelto")

In [ ]:
# CALCULA REC PRECIO EN BONOS POR EFECTO DE CALCULO DE SALDOS A VALOR DE MERCADO-PASIVO (comentario original del SAS: VM EN BCE INICIO)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp"))
sql_af32_rp = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Rec Precio' AS C_CUENTA,
       T1.C_ENTRADA,
       CASE WHEN T1.C_CUENTA <> 'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0403' AS PROC
INTO #af32_rp
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.31', 'AF.32')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.SECTOR,
         CASE WHEN T1.C_CUENTA <> 'Bce Final' THEN T1.DATO*-1 ELSE T1.DATO END
"""
work_conn.execute(text(sql_af32_rp))


In [ ]:
# PROC SQL: AGRUPA ANTERIOR (WORK.AF32_RP_2)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_rp_2"))
sql_af32_rp_2 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_rp_2
FROM #af32_rp T1
WHERE T1.DATO <> 0
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE, T1.SECTOR, T1.PROC
"""
work_conn.execute(text(sql_af32_rp_2))


In [ ]:
# PROC DATASETS: APPEND WORK.AF32_RP_2 a TABLAS.BD_CTSI
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_rp2 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_rp_2"
res = work_conn.execute(text(sql_append_rp2))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af32_rp_2", res.rowcount)


In [ ]:
# PROC SQL: drop AF32_RP_2 y AF32_RP
for t in ["#af32_rp_2", "#af32_rp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTE TOTAL BONOS COMPARANDO TOTAL EMITIDO CON TOTAL ACTIVO (WORK.AF3_AJ_TOT)
work_conn.execute(text("DROP TABLE IF EXISTS #af3_aj_tot"))
sql_af3_aj_tot = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO*-1 ELSE T1.DATO END AS DATO,
       T1.C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN,
       'PS' AS FUENTE, '0309' AS PROC
INTO #af3_aj_tot
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN IN ('AF.32')
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN,
         CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO*-1 ELSE T1.DATO END
"""
work_conn.execute(text(sql_af3_aj_tot))


In [ ]:
# PROC SQL: WORK.AF3_AJ_TOT2 (agrupa anterior)
work_conn.execute(text("DROP TABLE IF EXISTS #af3_aj_tot2"))
sql_af3_aj_tot2 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af3_aj_tot2
FROM #af3_aj_tot T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.SECTOR, T1.C_CAGENTE, T1.PROC
"""
work_conn.execute(text(sql_af3_aj_tot2))


In [ ]:
# PROC DATASETS: APPEND WORK.AF3_AJ_TOT2 a TABLAS.BD_CTSI
sql_append_aj_tot2 = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af3_aj_tot2"
res = work_conn.execute(text(sql_append_aj_tot2))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af3_aj_tot2", res.rowcount)


In [ ]:
# PROC SQL: drop AF3_AJ_TOT y AF3_AJ_TOT2
for t in ["#af3_aj_tot", "#af3_aj_tot2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# EN BONOS DE LP DEL ACTIVO DEL SECTOR 51 CON CA DISTINTOS DE 6 Y 53 PARA CAMBIAR CA A TODOS A 53
# IMPUTA DATA DE SECTOR 51 CON CA 53 (WORK.AF32_51)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51"))
sql_af32_51 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       '53' AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_03' AS PROC, 'PS' AS FUENTE
INTO #af32_51
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T2.C_SI_publ = 51 AND (T3.C_SI_publ <> 6 AND T3.C_SI_publ <> 53)
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_SI_publ, T3.C_SI_publ,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_51))


In [ ]:
# PROC SQL: ELIMINA DATA DE SECTOR 51 (WORK.AF32_51_DEL)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_del"))
sql_af32_51_del = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       LTRIM(CAST(T3.C_SI_publ AS varchar(11))) AS C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_02' AS PROC
INTO #af32_51_del
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE = T3.C_CAGENTE
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T2.C_SI_publ = 51 AND (T3.C_SI_publ <> 6 AND T3.C_SI_publ <> 53)
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_SI_publ, T3.C_SI_publ,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_51_del))


In [ ]:
# PROC DATASETS: APPEND WORK.AF32_51_DEL y WORK.AF32_51 a TABLAS.BD_CTSI
for tmp_src in ["#af32_51_del", "#af32_51"]:
    sql_append = f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM {tmp_src}"
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_src}", res.rowcount)


In [ ]:
# PROC SQL: drop AF32_51_DEL y AF32_51
for t in ["#af32_51_del", "#af32_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA TENENCIAS DE BONOS DE LARGO PLAZO DE BONOS EMITIDOS POR HOLDINGS Y CASAS MATRICES EN TODOS LOS SECTORES, EXCEPTO AUXILIARES FINANCIEROS PORQUE YA SE IMPUTO
# ESTA IMPUTACION SE RESTA DE LAS TENENCIAS DE LOS SECTORES RESPECTIVOS DE BONOS DE EMPRESAS
# WORK.AF32_D_HC depende de un archivo SAS7BDAT externo ('t_aj_dcv_af32_act_hc.sas7bdat') sin ruta de proyecto ni conexión declarada en el contexto
raise NotImplementedError("WORK.AF32_D_HC depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_dcv_af32_act_hc.sas7bdat' — archivo externo sin ruta de proyecto ni conexión declarada; requiere definir cómo se lee este insumo antes de traducir")

In [ ]:
# PROC SQL: UPDATE AF32_D_HC SECTOR=51 WHERE SECTOR IN (8,511) — depende de AF32_D_HC no resuelto
raise NotImplementedError("UPDATE sobre WORK.AF32_D_HC bloqueado: AF32_D_HC depende de archivo externo no resuelto")

In [ ]:
# PROC SQL: SELECCIONA TENENCIAS DE BONOS DE LP DE EMISIONES DE AUX FIN (WORK.AF32_CA36)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca36"))
sql_af32_ca36 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_010' AS PROC
INTO #af32_ca36
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND (T1.C_CAGENTE LIKE '%36%' OR T1.C_CAGENTE LIKE '%37%') AND T1.SECTOR NOT IN (36, 36904, 36906, 6) AND T1.[AÑO] > 2002
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_ca36))


In [ ]:
# PROC DATASETS: JUNTA TABLAS PARA SACAR DIFERENCIA (APPEND AF32_CA36 en AF32_D_HC) — bloqueado: AF32_D_HC no se pudo materializar (fuente externa sin resolver)
raise NotImplementedError("APPEND WORK.AF32_CA36 sobre WORK.AF32_D_HC bloqueado: AF32_D_HC depende de archivo externo no resuelto")

In [ ]:
# PROC SQL: AGRUPA TABLA PARA CALCULAR DIFERENCIA A IMPUTAR. MANDA DCV (WORK.AF32_D_AUX) — depende de AF32_D_HC ya combinado con CA36
raise NotImplementedError("WORK.AF32_D_AUX bloqueado: depende de WORK.AF32_D_HC (archivo externo no resuelto)")

In [ ]:
# PROC SQL: UPDATE AF32_D_AUX SECTOR=33901 WHERE SECTOR=33 (CIERRE 2021: se deja ajuste en sector 33901 por apertura de FMNM) — depende de AF32_D_AUX no resuelto
raise NotImplementedError("UPDATE sobre WORK.AF32_D_AUX bloqueado: AF32_D_AUX no resuelto")

In [ ]:
# PROC SQL: AJUSTA EN EMISIONES DE EMPRESAS (WORK.AF32_AUX_AJUSTE) — depende de AF32_D_AUX
raise NotImplementedError("WORK.AF32_AUX_AJUSTE bloqueado: depende de WORK.AF32_D_AUX no resuelto")

In [ ]:
# PROC DATASETS: APPEND WORK.AF32_D_AUX y WORK.AF32_AUX_AJUSTE a TABLAS.BD_CTSI — bloqueado
raise NotImplementedError("APPEND de WORK.AF32_D_AUX / WORK.AF32_AUX_AJUSTE a TABLAS.BD_CTSI bloqueado: tablas fuente no resueltas")

In [ ]:
# PROC SQL: drop AF32_D_AUX, AF32_AUX_AJUSTE, AF32_CA36, AF32_D_HC (solo AF32_CA36 llegó a crearse)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca36"))


In [ ]:
# IMPUTA CARTERA DE BONOS EMITIDOS POR CORREDORES DE BOLSA EN EL SECTOR OFIS USANDO INFO DCV
# SELECCIONA ACTIVO AF32 INFORMADA (WORK.AF32_OFIS)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ofis"))
sql_af32_ofis = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_011a' AS PROC
INTO #af32_ofis
FROM TABLAS.dbo.BD_CTSI T1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND T2.C_SI_SCN = 'S.123' AND T1.C_CAGENTE = '36904'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T2.C_SI_publ
"""
work_conn.execute(text(sql_af32_ofis))


In [ ]:
# PROC SQL: DCV AF32 TENENCIA DE OFIS DE CORREDORES (WORK.AF32_D_DCV_) — fuente externa sin resolver
raise NotImplementedError("WORK.AF32_D_DCV_ depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_dcv_af32_act_ao.sas7bdat' — archivo externo sin ruta de proyecto ni conexión declarada")

In [ ]:
# PROC DATASETS: JUNTA TABLAS PARA HACER RESTA (APPEND AF32_D_DCV_ en AF32_OFIS) — bloqueado
raise NotImplementedError("APPEND WORK.AF32_D_DCV_ sobre WORK.AF32_OFIS bloqueado: AF32_D_DCV_ depende de archivo externo no resuelto")

In [ ]:
# PROC SQL: AGRUPA DIF DCV-CI (WORK.AF32_DIF) — depende de AF32_OFIS ya combinado con AF32_D_DCV_
raise NotImplementedError("WORK.AF32_DIF bloqueado: depende de WORK.AF32_OFIS combinado con WORK.AF32_D_DCV_ (archivo externo no resuelto)")

In [ ]:
# PROC SQL: AJUSTE EN AF32 CON CA 51 (WORK.AF32_AJ_OFIS) — depende de AF32_DIF
raise NotImplementedError("WORK.AF32_AJ_OFIS bloqueado: depende de WORK.AF32_DIF no resuelto")

In [ ]:
# PROC DATASETS: APPEND WORK.AF32_DIF y WORK.AF32_AJ_OFIS a TABLAS.BD_CTSI — bloqueado
raise NotImplementedError("APPEND de WORK.AF32_DIF / WORK.AF32_AJ_OFIS a TABLAS.BD_CTSI bloqueado: tablas fuente no resueltas")

In [ ]:
# PROC SQL: drop AF32_OFIS, AF32_D_DCV_, AF32_DIF, AF32_AJ_OFIS (solo AF32_OFIS llegó a crearse)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ofis"))


In [ ]:
# IMPUTA TENENCIAS DE BONOS DE LARGO PLAZO DE BONOS EMITIDOS POR OFIS EN BANCOS, SEGUROS Y PENSIONES
# ESTA IMPUTACION SE RESTA DE LAS TENENCIAS DE LOS SECTORES RESPECTIVOS DE BONOS DE EMPRESAS
# DCV AF32 TENENCIA DE BONOS DE OFIS, EN ACTIVOS DE BANCOS, SEGUROS Y PENSIONES (WORK.AF32_OFIS_DCV) — fuente externa sin resolver
raise NotImplementedError("WORK.AF32_OFIS_DCV depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_dcv_af32_act_hc.sas7bdat' — archivo externo sin ruta de proyecto ni conexión declarada")

In [ ]:
# PROC SQL: DCV AF32 TENENCIA DE BONOS DE FONDOS DE INVERSION EN ACTIVOS DE BANCOS, SEGUROS Y PENSIONES (WORK.AF32_OFIS_DCV_2) — fuente externa sin resolver
raise NotImplementedError("WORK.AF32_OFIS_DCV_2 depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/base_af3_total_v2.sas7bdat' — archivo externo sin ruta de proyecto ni conexión declarada")

In [ ]:
# DATA step: AF32_OFIS_DCV = SET AF32_OFIS_DCV AF32_OFIS_DCV_2 (concatenación) — bloqueado: ambas fuentes no resueltas
raise NotImplementedError("Concatenación DATA AF32_OFIS_DCV bloqueada: WORK.AF32_OFIS_DCV y WORK.AF32_OFIS_DCV_2 dependen de archivos externos no resueltos")

In [ ]:
# PROC SQL: UPDATE AF32_OFIS_DCV SECTOR=51 WHERE SECTOR IN (8,511) — bloqueado
raise NotImplementedError("UPDATE sobre WORK.AF32_OFIS_DCV bloqueado: tabla no resuelta")

In [ ]:
# PROC SQL: SELECCIONA TENENCIAS DE BONOS DE LP DE EMISIONES DE OFIS EN BANCOS, SEGUROS Y PENSIONES, EXCEPTO TENENCIAS DE PAT SEPARADO (WORK.AF32_CA33)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca33"))
sql_af32_ca33 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_014' AS PROC
INTO #af32_ca33
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.32' AND (T1.C_CAGENTE LIKE '%33%' OR T1.C_CAGENTE LIKE '%411%') AND T1.SECTOR IN (32, 321, 351, 352, 35, 353, 34, 341, 322) AND T1.[AÑO] > 2002
GROUP BY T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE,
         CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END
"""
work_conn.execute(text(sql_af32_ca33))


In [ ]:
# PROC SQL: DELETE FROM WORK.AF32_CA33 WHERE C_CAGENTE IN ('332','3324','3323')
res = work_conn.execute(text("DELETE FROM #af32_ca33 WHERE C_CAGENTE IN ('332','3324','3323')"))
_log("DELETE #af32_ca33", res.rowcount)


In [ ]:
# PROC DATASETS: JUNTA TABLAS PARA SACAR DIFERENCIA (APPEND AF32_CA33 en AF32_OFIS_DCV) — bloqueado: AF32_OFIS_DCV no resuelto
raise NotImplementedError("APPEND WORK.AF32_CA33 sobre WORK.AF32_OFIS_DCV bloqueado: AF32_OFIS_DCV depende de archivos externos no resueltos")

In [ ]:
# PROC SQL: AGRUPA TABLA PARA CALCULAR DIFERENCIA A IMPUTAR. MANDA DCV (WORK.AF32_D_OFI) — depende de AF32_OFIS_DCV
raise NotImplementedError("WORK.AF32_D_OFI bloqueado: depende de WORK.AF32_OFIS_DCV no resuelto")

In [ ]:
# PROC SQL: AJUSTA EN EMISIONES DE EMPRESAS (WORK.AF32_OFI_AJUSTE) — depende de AF32_D_OFI
raise NotImplementedError("WORK.AF32_OFI_AJUSTE bloqueado: depende de WORK.AF32_D_OFI no resuelto")

In [ ]:
# PROC DATASETS: APPEND WORK.AF32_D_OFI y WORK.AF32_OFI_AJUSTE a TABLAS.BD_CTSI — bloqueado
raise NotImplementedError("APPEND de WORK.AF32_D_OFI / WORK.AF32_OFI_AJUSTE a TABLAS.BD_CTSI bloqueado: tablas fuente no resueltas")

In [ ]:
# PROC SQL: drop AF32_D_OFI, AF32_OFI_AJUSTE, AF32_OFIS_DCV, AF32_CA33 (solo AF32_CA33 llegó a crearse)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca33"))


In [ ]:
# IMPUTA TENENCIAS DE BONOS DE LP DE OFIS CON CONTRAGENTE BANCOS/GOBIERNO/BCENTRAL/OFIS USANDO INFO DCV. AJUSTA IMPUTACION EN LAS TENENCIAS DE OFIS CON EMPRESAS
# CIERRE 2021: EL AJUSTE SE IMPUTA EN SECTOR FMNM (SECTOR 33901) PORQUE AHORA SE PUBLICA SEPARADO DE OFIS.
# DCV AF32 TENENCIA DE BONOS DE OFIS CON CONTRAGENTE BANCOS/GOBIERNO/BCENTRAL/OFIS...SIN GOB (WORK.AF32_OFIS_BCOS_DCV) — fuente externa sin resolver; último bloque de este tramo
raise NotImplementedError("WORK.AF32_OFIS_BCOS_DCV depende de '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/t_aj_dcv_af32_act_ofis.sas7bdat' — archivo externo sin ruta de proyecto ni conexión declarada; requiere definir cómo se lee este insumo antes de traducir")

In [ ]:
work_conn.execute(text("DELETE FROM WORK.AF32_OFIS_BCOS_DCV WHERE C_CAGENTE IN ('3324')"))
# Nota: WORK.AF32_OFIS_BCOS_DCV es un dataset WORK que este tramo asume creado en un tramo anterior del mismo nodo (no aparece en input_datasets del tramo 5); se referencia tal cual lo dejó ese bloque previo.


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_33ca321"))
# SELECCIONA TENENCIAS DE BONOS DE LP DE OFIS CON BANCOS/GOBIERNO/BCENTRAL/OFIS
# CIERRE 2021: AGREGA CONTRAGENTE 411 PORQUE ESTABA QUEDANDO FUERA DE LA REGLA
sql_af32_33ca321 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       33901 AS SECTOR,
       T1.C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_016' AS PROC
INTO #af32_33ca321
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND (T1.C_CAGENTE IN ('321','3') OR T1.C_CAGENTE LIKE '31' OR T1.C_CAGENTE LIKE '33%' OR T1.C_CAGENTE='411')
  AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(8))),1,2)='33' AND T1.SECTOR NOT IN (334,331,3324) AND t1.[AÑO]>2002
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_33ca321))
work_conn.execute(text("DELETE FROM #af32_33ca321 WHERE C_CAGENTE IN ('3324')"))


In [ ]:
# JUNTA TABLAS PARA SACAR DIFERENCIA (PROC APPEND, fuente en #tmp, server-side)
cols_af32_ofis_bcos_dcv = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_af32_ofis_bcos_dcv = f"""
INSERT INTO #af32_ofis_bcos_dcv ({cols_af32_ofis_bcos_dcv})
SELECT {cols_af32_ofis_bcos_dcv}
FROM #af32_33ca321
"""
res = work_conn.execute(text(sql_append_af32_ofis_bcos_dcv))
_log("APPEND #af32_ofis_bcos_dcv", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ofis_bcos_dif"))
# CALCULA DIF A IMPUTAN EN OFIS
sql_af32_ofis_bcos_dif = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, T1.PROC
INTO #af32_ofis_bcos_dif
FROM #af32_ofis_bcos_dcv T1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_af32_ofis_bcos_dif))
af32_ofis_bcos_dif = pd.read_sql(text("SELECT * FROM #af32_ofis_bcos_dif"), work_conn)
_log("af32_ofis_bcos_dif", af32_ofis_bcos_dif)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ofi_aj"))
# AJUSTA EN EMISIONES DE EMPRESAS
sql_af32_ofi_aj = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       ('51') AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE, '0498_016' AS PROC
INTO #af32_ofi_aj
FROM #af32_ofis_bcos_dif t1
"""
work_conn.execute(text(sql_af32_ofi_aj))
af32_ofi_aj = pd.read_sql(text("SELECT * FROM #af32_ofi_aj"), work_conn)
_log("af32_ofi_aj", af32_ofi_aj)


In [ ]:
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.AF32_OFIS_BCOS_DIF FORCE (server-side desde #tmp)
sql_append_1 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #af32_ofis_bcos_dif
"""
res = work_conn.execute(text(sql_append_1))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ofis_bcos_dif)", res.rowcount)

# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.AF32_OFI_AJ FORCE
sql_append_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #af32_ofi_aj
"""
res = work_conn.execute(text(sql_append_2))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ofi_aj)", res.rowcount)

for t in ["#af32_ofis_bcos_dif", "#af32_ofi_aj", "#af32_ofis_bcos_dcv", "#af32_33ca321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_37"))
# CALCULA DIFERENCIA ENTRE BONOS DE LP EMITIDOS POR SECTOR 37 CON CA 6 Y TENENCIA DE SECTOR 6 CON CA 37.
# RESPETA DATO DEL SECTOR 37, POR LO TANTO IMPUTA DIF EN RM CON CA 37
# CIERRE 2022Q2 INCORPORA EN AUSTE LAS EMISIONES DEL SECTOR 36912 EN EL EXTERIOR
# AJUSTA IMPUTACIÓN DE SECTOR RM CON CA 51021
sql_af32_6_37 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       6 AS SECTOR, '37' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af32_6_37
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE IN ('37','36912'))
   OR (T1.C_ENTRADA='H' AND T1.C_SCN='AF.32' AND T1.SECTOR IN (37,36912) AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_6_37))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_6_51021"))
# AJUSTA IMPUTACIÓN
sql_af32_6_51021 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR, '51021' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_13' AS PROC, T1.FUENTE
INTO #af32_6_51021
FROM #af32_6_37 t1
"""
work_conn.execute(text(sql_af32_6_51021))

cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
sql_append_3 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #af32_6_37
"""
res = work_conn.execute(text(sql_append_3))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_6_37)", res.rowcount)

sql_append_4 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #af32_6_51021
"""
res = work_conn.execute(text(sql_append_4))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_6_51021)", res.rowcount)

for t in ["#af32_6_51021", "#af32_6_37"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2020. IMPUTA ACTIVO DEL RM CON CA 36 DE BONOS DE LP EN PASIVO DEL SECTOR 36 CON CA 6.
# ESTE MONTO SE AJUSTA CREANDO UN ACTIVO AF7 EN SECTOR 36 CON CA 51022
work_conn.execute(text("DROP TABLE IF EXISTS #af32_36_6"))
# IMPUTA PASIVO AF32 36 CA RM
sql_af32_36_6 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       36 AS SECTOR, '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af32_36_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T1.SECTOR=6 AND T1.C_CAGENTE='36' AND T1.FUENTE='CI')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af32_36_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af7_36_51"))
# AJUSTA IMPUTACION ANTERIOR CREANDO ACTIVO AF7 36 CA 51022, PORQUE ES SECTOR NO MEDIDO
sql_af7_36_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       36 AS SECTOR, '51022' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'AF.7' AS C_SCN,
       'Créditos comerciales' AS N_SCN,
       '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af7_36_51
FROM #af32_36_6 t1
"""
work_conn.execute(text(sql_af7_36_51))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_2"))
# AJUSTA IMPUTACION ANTERIOR EN ACTIVO AF32 EMPRESAS CON CA EMPRESAS, PORQUE ES SECTOR NO MEDIDO
sql_af32_51_2 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR, '2' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       '0498_12' AS PROC, 'PS' AS FUENTE
INTO #af32_51_2
FROM #af32_36_6 t1
"""
work_conn.execute(text(sql_af32_51_2))

cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_36_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_36_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_36_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af7_36_51)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51_2)", res.rowcount)

for t in ["#af32_36_6", "#af7_36_51", "#af32_51_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA TOTAL EMITIDO CON LAS TENENCIAS DE LOS SECTORES EMISORES 31/32/33/36/4/51
work_conn.execute(text("DROP TABLE IF EXISTS #af32_d"))
# SELECCIONA CARTERAS CON CA 31/32/33/36/4/51 -- CIERRE 2021: INCORPORA CARTERA DE SECTOR 38
sql_af32_d = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       t3.C_SI_publ AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_17' AS PROC, 'PS' AS FUENTE
INTO #af32_d
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE=T3.C_CAGENTE
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32'
  AND (t3.C_SI_publ=31 OR t3.C_SI_publ=32 OR t3.C_SI_publ=36 OR t3.C_SI_publ=4 OR t3.C_SI_publ=51 OR t3.C_SI_publ=33 OR t3.C_SI_publ=38)
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t3.C_SI_publ
"""
work_conn.execute(text(sql_af32_d))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_h"))
# SELECCIONA PASIVO DE SECTORES 31/32/33/36/4/51
sql_af32_h = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       t2.C_SI_publ AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, '0498_17' AS PROC, 'PS' AS FUENTE
INTO #af32_h
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE=T3.C_CAGENTE
WHERE T1.C_ENTRADA='H' AND T1.C_SCN='AF.32'
  AND (t2.C_SI_publ=31 OR t2.C_SI_publ=32 OR t2.C_SI_publ=36 OR t2.C_SI_publ=4 OR t2.C_SI_publ=51 OR t2.C_SI_publ=33 OR t2.C_SI_publ=38)
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, t1.C_SCN, t2.C_SI_publ
"""
work_conn.execute(text(sql_af32_h))

cols_af32_d = "MONEDA, [AÑO], TRIM, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO #af32_d ({cols_af32_d}) SELECT {cols_af32_d} FROM #af32_h"))
_log("APPEND #af32_d (af32_h)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_dif"))
# CALCULA DIF PASIVO-ACTIVO PARA IMPUTAR EN ACTIVO DE LOS SECTORES 33/36/51 USANDO INFO DE ESTRUCTURA DE CONTRAGENTES DEL DCV...TODO SE VA A EMPRESAS
sql_af32_dif = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T2.C_SECTOR AS SECTOR,
       LTRIM(CAST(T1.C_CAGENTE AS varchar(11))) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO*T2.DATO) AS DATO,
       T1.C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, T1.PROC, T1.FUENTE
INTO #af32_dif
FROM #af32_d t1
INNER JOIN TABLAS.dbo.T_EST_DCV_AF32_PROMEDIOS T2 ON T1.C_CAGENTE=T2.C_CAGENTE AND T1.C_SCN=T2.C_SCN
WHERE T2.DATO<>0
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.C_CUENTA, T2.C_SECTOR, t1.C_SCN, LTRIM(CAST(T1.C_CAGENTE AS varchar(11))), T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_af32_dif))
af32_dif = pd.read_sql(text("SELECT * FROM #af32_dif"), work_conn)
_log("af32_dif", af32_dif)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca53"))
# AJUSTA IMPUTACION ANTERIOR EN ACTIVO AF32 CON CA 53 DEL RESPECTIVO SECTOR
sql_af32_ca53 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0498_18' AS PROC, T1.FUENTE
INTO #af32_ca53
FROM #af32_dif T1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_ca53))
af32_ca53 = pd.read_sql(text("SELECT * FROM #af32_ca53"), work_conn)
_log("af32_ca53", af32_ca53)

cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_dif"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_dif)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_ca53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ca53)", res.rowcount)

for t in ["#af32_dif", "#af32_ca53", "#af32_d", "#af32_h"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CAMBIA CONTRAGENTE DE 53 A 51 EN TÍTULOS DE LP EN EL ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51"))
# SELECCIONA DATA DE ACTIVO PARA CAMBIAR CONTRAGENTE
sql_af32_51 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       '51' AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, 'PS' AS FUENTE, '0498_20' AS PROC
INTO #af32_51
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE=T3.C_CAGENTE
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND t3.C_SI_publ=53 AND T1.DATO<>0
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T2.C_SI_publ
"""
work_conn.execute(text(sql_af32_51))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_53"))
# ELIMINA DATA
sql_af32_53 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       LTRIM(CAST(t3.C_SI_publ AS varchar(11))) AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*-1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE, '0498_21' AS PROC
INTO #af32_53
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE=T3.C_CAGENTE
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND t3.C_SI_publ=53 AND T1.DATO<>0
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T2.C_SI_publ, t3.C_SI_publ
"""
work_conn.execute(text(sql_af32_53))

cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_53)", res.rowcount)

for t in ["#af32_53", "#af32_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA EN SECTORES 51 Y 8 (HOGARES) DE ACTIVO AF.32 Y AF.31 CON TODOS LOS CA DEJANDO EL 70% EN CP Y 30% EN LP CON LOS MISMOS CA
work_conn.execute(text("DROP TABLE IF EXISTS #af31_51_8"))
# SELECCIONA DATA DE SECTOR 51 Y 8
sql_af31_51_8 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       LTRIM(CAST(t3.C_SI_publ AS varchar(11))) AS C_CAGENTE,
       (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN
INTO #af31_51_8
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS T3 ON T1.C_CAGENTE=T3.C_CAGENTE
WHERE (T1.C_ENTRADA='D' AND T1.C_SCN='AF.31' AND T2.C_SI_publ=8)
   OR (T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T2.C_SI_publ=51)
GROUP BY t1.[AÑO], T1.TRIM, (CASE WHEN t1.C_CUENTA='Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END), T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T2.C_SI_publ, t3.C_SI_publ
"""
work_conn.execute(text(sql_af31_51_8))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af31_51_70"))
# IMPUTA 70% DEL CP DE SECTOR 8 EN SECTOR 51
sql_af31_51_70 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       CASE WHEN T1.C_CAGENTE IN ('4','41') THEN SUM(T1.DATO) ELSE SUM(T1.DATO*0.7) END AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_003' AS PROC, 'PS' AS FUENTE
INTO #af31_51_70
FROM #af31_51_8 T1
WHERE T1.C_SCN='AF.31' AND T1.SECTOR=8 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af31_51_70))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af31_8_70"))
# ANULA 70% DEL CP PARA CONTRARESTAR AJUSTE ANTERIOR EN SECTOR 8
sql_af31_8_70 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-0.7) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_004' AS PROC, 'PS' AS FUENTE
INTO #af31_8_70
FROM #af31_51_8 T1
WHERE T1.C_SCN='AF.31' AND T1.SECTOR=8 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af31_8_70))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_30"))
# IMPUTA 30% DEL LP EN SECTOR 51
sql_af32_51_30 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*0.3) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_005' AS PROC, 'PS' AS FUENTE
INTO #af32_51_30
FROM #af31_51_8 T1
WHERE T1.C_SCN='AF.32' AND T1.SECTOR=51 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_51_30))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_51_30_"))
# IMPUTA 30% DEL LP EN SECTOR 51 (contrapartida negativa en el sector original)
sql_af32_51_30_ = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-0.3) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_006' AS PROC, 'PS' AS FUENTE
INTO #af32_51_30_
FROM #af31_51_8 T1
WHERE T1.C_SCN='AF.32' AND T1.SECTOR=51 AND T1.C_CAGENTE NOT IN ('4','41')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE, T1.SECTOR
"""
work_conn.execute(text(sql_af32_51_30_))

cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_51_70"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_51_70)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_8_70"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_8_70)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51_30"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51_30)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_51_30_"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_51_30_)", res.rowcount)

for t in ["#af31_51_8", "#af31_51_70", "#af31_8_70", "#af32_51_30", "#af32_51_30_"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA TITULOS DE CP DE SECTORES 8/511 CON CA 31 Y LO CAMBIA A LP. LO DE CP ADEMÁS LO IMPUTA A SECTOR 51 AJUSTANDOLO CONTRA TITULOS DE LP
work_conn.execute(text("DROP TABLE IF EXISTS #af31_ca31"))
# SELECCIONA DATA DE SECTORES 8/511 CON CA 31 PARA ELIMINARLO DEL CP
sql_af31_ca31 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_011' AS PROC, 'PS' AS FUENTE
INTO #af31_ca31
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.31' AND T2.C_SI_SCN_N1='S.14' AND T1.C_CAGENTE IN ('31','41') AND T1.DATO<>0
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af31_ca31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_ca31"))
# IMPUTA DATA DE SECTORES 8/511 CON CA 31 EN LP
sql_af32_ca31 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, '0500_014' AS PROC, T1.FUENTE
INTO #af32_ca31
FROM #af31_ca31 t1
"""
work_conn.execute(text(sql_af32_ca31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af31_41_ca31"))
# IMPUTA DATA DE SECTORES 8/511 CON CA 31 EN CP DE SECTOR 51
sql_af31_41_ca31 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_012' AS PROC, T1.FUENTE
INTO #af31_41_ca31
FROM #af31_ca31 t1
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE, T1.FUENTE
"""
work_conn.execute(text(sql_af31_41_ca31))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_41_ca31"))
# AJUSTA IMPUTACIÓN ANTERIOR EN SECTOR 51 CON CA 31 EN BONOS DE LP
sql_af32_41_ca31 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO*-1) AS DATO,
       'AF.32' AS C_SCN,
       'Valores distintos de acciones a largo plazo' AS N_SCN, '0500_013' AS PROC, T1.FUENTE
INTO #af32_41_ca31
FROM #af31_41_ca31 t1
"""
work_conn.execute(text(sql_af32_41_ca31))

cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_ca31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_ca31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af31_41_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af31_41_ca31)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_41_ca31"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_41_ca31)", res.rowcount)

for t in ["#af31_ca31", "#af32_ca31", "#af31_41_ca31", "#af32_41_ca31"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))

# DROP TABLE WORK.af32_vm_bi suelto en el SAS: tabla que este tramo no crea (nace en otro tramo/nodo)
work_conn.execute(text("DROP TABLE IF EXISTS #af32_vm_bi"))


In [ ]:
# RECLASIFICA BONOS DE LARGO PLAZO DEL ACTIVO DE HOGARES CON CA 31 A BONOS DE LP DEL SECTOR 51 CON CA 31
work_conn.execute(text("DROP TABLE IF EXISTS #af32_hh"))
# SELECCIONA DATA DE SECTORES 8/511 CON CA 31 PARA ELIMINARLO DEL LP
sql_af32_hh = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_015' AS PROC, 'PS' AS FUENTE
INTO #af32_hh
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T2.C_SI_SCN_N1='S.14' AND T1.C_CAGENTE IN ('31','41') AND T1.DATO<>0
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_hh))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af32_hh_aj"))
# SELECCIONA DATA DE SECTORES 8/511 CON CA 31 PARA ELIMINARLO DEL LP (imputación en sector 51)
sql_af32_hh_aj = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN, '0500_016' AS PROC, 'PS' AS FUENTE
INTO #af32_hh_aj
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR=T2.C_SI
WHERE T1.C_ENTRADA='D' AND T1.C_SCN='AF.32' AND T2.C_SI_SCN_N1='S.14' AND T1.C_CAGENTE IN ('31','41') AND T1.DATO<>0
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af32_hh_aj))

cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, PROC, FUENTE"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_hh"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_hh)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af32_hh_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (af32_hh_aj)", res.rowcount)

for t in ["#af32_hh", "#af32_hh_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# Lectura de archivo externo SAS7BDAT: base_af3_total_v2.sas7bdat (antes ruta absoluta /sasdata/BCCH/... — M-001: reemplazada por ruta relativa al workspace)
ruta_dcv = Path("entradas") / "DCVRES" / "base_af3_total_v2.sas7bdat"
dcv, _meta_dcv = pyreadstat.read_sas7bdat(str(ruta_dcv))
_log("dcv", dcv)

## S2_07_Cuotas_Fondos

Reclasifica cuotas de fondos mutuos e inversión (AF.521/AF.522) entre money market y no money market por sector/contraparte, concilia activo-pasivo con el resto del mundo y fondos, e imputa patrimonio del Banco Central en gobierno central

*confianza: medium · verificador: unverified · SAS: PROC SQL UPDATE múltiples + CREATE TABLE/PROC APPEND con tablas WORK temporales de sesión*

In [ ]:
# ========= S2_07_Cuotas_Fondos =========
# COMPRIME TABLA PRINCIPAL (SET tablas.BD_CTSI sobre sí misma con COMPRESS=YES).
# COMPRESS es una opción de almacenamiento de SAS sin equivalente en SQL Server: no hay statement que traducir.


In [ ]:
# ACTUALIZA COD DE INSTRUMENTOS Y CONTRAGENTES PARA SEPARAR MONEY MARKET DE NO MONEY MARKET
# Y PARA DEJAR DENTRO DE LO NO MONEY MARKET LO QUE ES CA 3390102 DE 339011
with engine.begin() as conn:
    # MUNICIPALIDADES ES TODO MONEY MARKET
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 42, "entrada": "D", "scn": "AF.521", "cagente": "51"})
    _log("UPDATE BD_CTSI municipalidades", res.rowcount)

    # BANCOS ES TODO MONEY MARKET
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 321, "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI bancos", res.rowcount)

    # TCC ES TODO MONEY MARKET
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 334, "entrada": "D", "scn": "AF.521", "cagente": "339"})
    _log("UPDATE BD_CTSI tcc", res.rowcount)

    # ISAPRES ES TODO MONEY MARKET
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE IN (:cag1, :cag2)
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 353, "entrada": "D", "scn": "AF.521", "cag1": "33901", "cag2": "53"})
    _log("UPDATE BD_CTSI isapres", res.rowcount)

    # AFP ES TODO MONEY MARKET. SE DECIDE ASÍ PORQUE SON MONTOS BAJOS Y SE ASUME QUE PARA HACER CAJA Y REQUIERE LIQUIDEZ
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, C_SCN = :nuevo_scn, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_scn": "AF.521", "nuevo_proc": "0503", "sector": 361, "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI afp", res.rowcount)

    # MUTUALIDADES ES TODO MONEY MARKET IGUAL QUE GOB CENTRAL
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 412, "entrada": "D", "scn": "AF.521", "cagente": "339"})
    _log("UPDATE BD_CTSI mutualidades", res.rowcount)

    # GOBIERNO ES TODO MONEY MARKET LO NACIONAL
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 41, "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI gobierno", res.rowcount)

    # FONDOS DE INVERSIÓN SE SEPARA LO QUE ES FI DE FM. EN FM SE ASUME QUE ES NO MONEY MARKET. CIERRE 2021 AJUSTA ESTE CAMBIO DE CONTRAGENTE
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, C_SCN = :nuevo_scn, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE <> :cagente_excl
    """), {"nueva_cagente": "339011", "nuevo_scn": "AF.522", "nuevo_proc": "0503", "sector": 339011, "entrada": "D", "scn": "AF.521", "cagente_excl": "3390101"})
    _log("UPDATE BD_CTSI FI vs FM (1)", res.rowcount)

    # FONDOS DE INVERSIÓN SE SEPARA LO QUE ES FI DE FM. EN FM SE ASUME QUE ES NO MONEY MARKET
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, C_SCN = :nuevo_scn, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390102", "nuevo_scn": "AF.522", "nuevo_proc": "0503", "sector": 339011, "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI FI vs FM (2)", res.rowcount)

    # CORREDORES DE BOLSA SE DEJA EN FM MONEY MARKET
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 36904, "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI corredores de bolsa", res.rowcount)

    # AFI, BOLSA DE VALORES, SECURITIZADORAS, LEASING, FACTORING SE DEJA EN FM MONEY MARKET. CIERRE 2021: INCORPORA SECTOR 36909
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR IN (36905,36906,33232,33231,33222,33221,33212,33211,36909)
              AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE IN (:cag1, :cag2)
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "entrada": "D", "scn": "AF.521", "cag1": "339", "cag2": "33901"})
    _log("UPDATE BD_CTSI afi/bolsa/securitizadoras/leasing/factoring", res.rowcount)

    # LEASING BANCARIO
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR IN (33211) AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI leasing bancario", res.rowcount)

    # CLASIFICADORAS, CASAS MATRICES, AFIS, HOLDINGS
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR IN (36910,36907,36905,37) AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI clasificadoras/casas matrices/afis/holdings", res.rowcount)

    # FACTORING BANCARIO SE DEJA EN FM MONEY MARKET
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR IN (33221,369011) AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI factoring bancario", res.rowcount)

    # LO NO CLASIFICADO EN FONDOS DE PENSIONES SE DEJA EN AF522 CON CA 339011
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "339011", "nuevo_proc": "0503", "sector": 34, "entrada": "D", "scn": "AF.522", "cagente": "339"})
    _log("UPDATE BD_CTSI no clasificado fondos pensiones", res.rowcount)

    # LO CLASIFICADO EN CA 339 EN EMPRESAS PUBLICAS Y PRIVADAS SE DEJA EN AF522 CON CA 339011
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_SCN = :nuevo_scn, C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR IN (51021,5101) AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nuevo_scn": "AF.522", "nueva_cagente": "339011", "nuevo_proc": "0503", "entrada": "D", "scn": "AF.521", "cagente": "339"})
    _log("UPDATE BD_CTSI empresas publicas/privadas CA339 (1)", res.rowcount)

    # LO CLASIFICADO EN CA 339 EN EMPRESAS PUBLICAS Y PRIVADAS SE DEJA EN AF522 CON CA 339011
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR IN (51021,5101) AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "entrada": "D", "scn": "AF.521", "cagente": "33901"})
    _log("UPDATE BD_CTSI empresas publicas/privadas CA339 (2)", res.rowcount)

    # CIERRE 2021: CAMBIA INSTRUMENTO DESDE AF.5 A AF.522 EN EL ACTIVO DE FONDOS DE INVERSIÓN CON CONTRAGENTE 339011
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_SCN = :nuevo_scn, N_SCN = :nuevo_n_scn, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nuevo_scn": "AF.522", "nuevo_n_scn": "Participaciones emitidas por fondos  de inversión del mercado monetario", "nuevo_proc": "0503", "sector": 339011, "entrada": "D", "scn": "AF.5", "cagente": "339011"})
    _log("UPDATE BD_CTSI AF.5 a AF.522 fondos inversion", res.rowcount)

    # CIERRE 2021: INCORPORA CAMBIO DE CONTRAGENTE EN CUOTAS DE FONDOS DE UNIVERSIDADES
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE SECTOR = :sector AND C_ENTRADA = :entrada AND C_SCN = :scn AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "3390101", "nuevo_proc": "0503", "sector": 413, "entrada": "D", "scn": "AF.521", "cagente": "S129"})
    _log("UPDATE BD_CTSI cuotas fondos universidades", res.rowcount)


In [ ]:
# IMPUTA CARTERA DE FONDOS DE SECTORES CON CA RESTO DEL MUNDO EN EL PASIVO DEL RM Y AJUSTA CONTRA EL PASIVO AF.5 DEL RM
# IMPUTA EN RM EXCEPTO LA CARTERA DE GOB CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #fondos_ca6"))
sql_fondos_ca6 = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        t1.C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0504' AS PROC
INTO    #fondos_ca6
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   (T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.C_CAGENTE = '6' AND T1.SECTOR <> 41)
GROUP BY 'P', t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, t1.N_SCN, CAST(T1.C_CAGENTE AS int), LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_fondos_ca6))
fondos_ca6 = pd.read_sql(text("SELECT * FROM #fondos_ca6"), work_conn)
_log("fondos_ca6", fondos_ca6)


In [ ]:
# AJUSTA IMPUTACIÓN EN AF5. CIERRE 2021: DADO QUE RESTO DEL MUNDO NO TIENE AF.5 CON AUXILIARES, SE RESTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #ajuste_af5_6"))
sql_ajuste_af5_6 = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        t1.C_CUENTA,
        T1.C_ENTRADA,
        (T1.DATO * -1) AS DATO,
        'AF.5' AS C_SCN,
        'Acciones y otras participaciones de capital' AS N_SCN,
        T1.FUENTE,
        '0505' AS PROC
INTO    #ajuste_af5_6
FROM    #fondos_ca6 t1
"""
work_conn.execute(text(sql_ajuste_af5_6))
# corrige contragente de sector 36xx a CA 53 (regla explícita del SAS)
work_conn.execute(text("UPDATE #ajuste_af5_6 SET C_CAGENTE = '53' WHERE SUBSTRING(C_CAGENTE, 1, 2) = '36'"))
ajuste_af5_6 = pd.read_sql(text("SELECT * FROM #ajuste_af5_6"), work_conn)
_log("ajuste_af5_6", ajuste_af5_6)


In [ ]:
# APPEND server-side a la tabla principal (equivalente a PROC APPEND FORCE)
cols_fondos = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fondos}) SELECT {cols_fondos} FROM #fondos_ca6"))
_log("APPEND BD_CTSI FONDOS_CA6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fondos}) SELECT {cols_fondos} FROM #ajuste_af5_6"))
_log("APPEND BD_CTSI AJUSTE_AF5_6", res.rowcount)
for t in ["#fondos_ca6", "#ajuste_af5_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA ACTIVO DE AF521 DE SECTOR GOBIERNO CON CA RM CON EL PASIVO DEL RM CA 41.
# RESPETA DATO DE RM, POR LO TANTO AJUSTA DATO DE GOBIERNO Y EL CONTRAAJUSTE ES AL ACTIVO AF521 DE GOBIERNO CON CA 3390101
# CALCULA DIF ENTRE RM Y GOB
work_conn.execute(text("DROP TABLE IF EXISTS #af521_gob_ca6"))
sql_af521_gob_ca6 = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        41 AS SECTOR,
        '6' AS C_CAGENTE,
        (CASE WHEN T1.SECTOR = 6 AND t1.C_CUENTA IN ('Rec Volumen','Rec Precio') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0523' AS PROC
INTO    #af521_gob_ca6
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.521' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '41')
     OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.521' AND T1.SECTOR = 41 AND T1.C_CAGENTE = '6')
GROUP BY 'P', t1.AÑO, T1.TRIM,
         (CASE WHEN T1.SECTOR = 6 AND t1.C_CUENTA IN ('Rec Volumen','Rec Precio') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af521_gob_ca6))
af521_gob_ca6 = pd.read_sql(text("SELECT * FROM #af521_gob_ca6"), work_conn)
_log("af521_gob_ca6", af521_gob_ca6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af521_gob_cafm"))
sql_af521_gob_cafm = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '3390101' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        (T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0524' AS PROC
INTO    #af521_gob_cafm
FROM    #af521_gob_ca6 AS T1
"""
work_conn.execute(text(sql_af521_gob_cafm))
af521_gob_cafm = pd.read_sql(text("SELECT * FROM #af521_gob_cafm"), work_conn)
_log("af521_gob_cafm", af521_gob_cafm)

cols_gob = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_gob}) SELECT {cols_gob} FROM #af521_gob_ca6"))
_log("APPEND BD_CTSI AF521_GOB_CA6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_gob}) SELECT {cols_gob} FROM #af521_gob_cafm"))
_log("APPEND BD_CTSI AF521_GOB_CAFM", res.rowcount)
for t in ["#af521_gob_ca6", "#af521_gob_cafm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ACTUALIZA CONTRAGENTE A 33901 EN ACTIVO AF.521 Y AF.522 DE TODOS LOS SECTORES PARA LOS CA NO ASIGNADOS
with engine.begin() as conn:
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE C_ENTRADA = :entrada AND C_SCN IN ('AF.521','AF.522') AND C_CAGENTE = :cagente
    """), {"nueva_cagente": "33901", "nuevo_proc": "0506", "entrada": "D", "cagente": "53"})
    _log("UPDATE BD_CTSI CA no asignados", res.rowcount)

    # IMPUTA PASIVO DE FONDOS MUTUOS E INVERSIÓN CON CA 6 EN ACTIVO DE RESTO DEL MUNDO DEL INST AF.521 Y AF.522.
    # DIFERENCIAL CON LO INFORMADO POR EL RM SE VA A AF.5 CON CA 33
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_SCN = :nuevo_scn
        WHERE C_ENTRADA = :entrada AND SECTOR = :sector AND C_SCN = :scn
    """), {"nuevo_scn": "AF.521", "entrada": "D", "sector": 6, "scn": "AF.522"})
    _log("UPDATE BD_CTSI pasivo FFMM/FI a AF.521", res.rowcount)


In [ ]:
# IMPUTA FFMM MONEY MARKET EN ACTIVO RM
work_conn.execute(text("DROP TABLE IF EXISTS #ffmm_ca6"))
sql_ffmm_ca6 = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        6 AS SECTOR,
        '3390101' AS C_CAGENTE,
        (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0518' AS PROC
INTO    #ffmm_ca6
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.521' AND T1.SECTOR = 3390101 AND T1.C_CAGENTE = '6'
GROUP BY 'P', t1.AÑO, T1.TRIM,
         (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_ffmm_ca6))
ffmm_ca6 = pd.read_sql(text("SELECT * FROM #ffmm_ca6"), work_conn)
_log("ffmm_ca6", ffmm_ca6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af521_6aj"))
sql_af521_6aj = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '33901' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        (T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0519' AS PROC
INTO    #af521_6aj
FROM    #ffmm_ca6 AS T1
"""
work_conn.execute(text(sql_af521_6aj))
af521_6aj = pd.read_sql(text("SELECT * FROM #af521_6aj"), work_conn)
_log("af521_6aj", af521_6aj)

cols_ffmm = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ffmm}) SELECT {cols_ffmm} FROM #ffmm_ca6"))
_log("APPEND BD_CTSI FFMM_CA6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ffmm}) SELECT {cols_ffmm} FROM #af521_6aj"))
_log("APPEND BD_CTSI AF521_6AJ", res.rowcount)
for t in ["#ffmm_ca6", "#af521_6aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA FFMM NO MONEY MARKET EN ACTIVO RM
work_conn.execute(text("DROP TABLE IF EXISTS #ffmmnm_ca6"))
sql_ffmmnm_ca6 = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        6 AS SECTOR,
        '3390102' AS C_CAGENTE,
        (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0518a' AS PROC
INTO    #ffmmnm_ca6
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.522' AND T1.SECTOR = 3390102 AND T1.C_CAGENTE = '6'
GROUP BY 'P', t1.AÑO, T1.TRIM,
         (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_ffmmnm_ca6))
ffmmnm_ca6 = pd.read_sql(text("SELECT * FROM #ffmmnm_ca6"), work_conn)
_log("ffmmnm_ca6", ffmmnm_ca6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af522_6aj"))
sql_af522_6aj = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '33901' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        (T1.DATO * -1) AS DATO,
        'AF.521' AS C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0519a' AS PROC
INTO    #af522_6aj
FROM    #ffmmnm_ca6 AS T1
"""
work_conn.execute(text(sql_af522_6aj))
af522_6aj = pd.read_sql(text("SELECT * FROM #af522_6aj"), work_conn)
_log("af522_6aj", af522_6aj)

cols_ffmmnm = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ffmmnm}) SELECT {cols_ffmmnm} FROM #ffmmnm_ca6"))
_log("APPEND BD_CTSI FFMMNM_CA6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_ffmmnm}) SELECT {cols_ffmmnm} FROM #af522_6aj"))
_log("APPEND BD_CTSI AF522_6AJ", res.rowcount)
for t in ["#ffmmnm_ca6", "#af522_6aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA FI EN ACTIVO RM
work_conn.execute(text("DROP TABLE IF EXISTS #fi_ca6"))
sql_fi_ca6 = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        6 AS SECTOR,
        '339011' AS C_CAGENTE,
        (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0518b' AS PROC
INTO    #fi_ca6
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.522' AND T1.SECTOR = 339011 AND T1.C_CAGENTE = '6'
GROUP BY 'P', t1.AÑO, T1.TRIM,
         (CASE WHEN t1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_fi_ca6))
fi_ca6 = pd.read_sql(text("SELECT * FROM #fi_ca6"), work_conn)
_log("fi_ca6", fi_ca6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af522_6aj_2"))
sql_af522_6aj_2 = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '33901' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        (T1.DATO * -1) AS DATO,
        'AF.521' AS C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0519b' AS PROC
INTO    #af522_6aj_2
FROM    #fi_ca6 AS T1
"""
work_conn.execute(text(sql_af522_6aj_2))
af522_6aj_2 = pd.read_sql(text("SELECT * FROM #af522_6aj_2"), work_conn)
_log("af522_6aj_2", af522_6aj_2)

cols_fi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fi}) SELECT {cols_fi} FROM #fi_ca6"))
_log("APPEND BD_CTSI FI_CA6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fi}) SELECT {cols_fi} FROM #af522_6aj_2"))
_log("APPEND BD_CTSI AF522_6AJ_2", res.rowcount)
for t in ["#fi_ca6", "#af522_6aj_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA LO RESIDUAL QUE QUEDA EN AF521 CON CA 33901 A AF5 CON CA 33
work_conn.execute(text("DROP TABLE IF EXISTS #fondos_s6"))
sql_fondos_s6 = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0518c' AS PROC
INTO    #fondos_s6
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.521' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '33901')
GROUP BY 'P', t1.AÑO, T1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.SECTOR, t1.C_CAGENTE
"""
work_conn.execute(text(sql_fondos_s6))
fondos_s6 = pd.read_sql(text("SELECT * FROM #fondos_s6"), work_conn)
_log("fondos_s6", fondos_s6)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_s6_ca33"))
sql_af5_s6_ca33 = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '33' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO * -1 AS DATO,
        'AF.5' AS C_SCN,
        'Acciones y otras participaciones de capital' AS N_SCN,
        'PS' AS FUENTE,
        '0519c' AS PROC
INTO    #af5_s6_ca33
FROM    #fondos_s6 t1
"""
work_conn.execute(text(sql_af5_s6_ca33))
af5_s6_ca33 = pd.read_sql(text("SELECT * FROM #af5_s6_ca33"), work_conn)
_log("af5_s6_ca33", af5_s6_ca33)

cols_s6 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_s6}) SELECT {cols_s6} FROM #fondos_s6"))
_log("APPEND BD_CTSI FONDOS_S6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_s6}) SELECT {cols_s6} FROM #af5_s6_ca33"))
_log("APPEND BD_CTSI AF5_S6_CA33", res.rowcount)
for t in ["#fondos_s6", "#af5_s6_ca33"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO DE SECTORES NACIONALES (EXCEPTO HOGARES) DE INST AF.521 CON CA 3390101 EN PASIVO DE SECTOR 3390101 CA 321.
# AJUSTA EN PASIVO AF521 SECTOR 3390101 CA 53
# IMPUTA CARTERAS DE INST FFMM MONEY MARKET EN PASIVO DE SECTOR 3390101
work_conn.execute(text("DROP TABLE IF EXISTS #af521_invnac"))
sql_af521_invnac = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0510' AS PROC
INTO    #af521_invnac
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   ((T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.521' AND T1.SECTOR <> 6 AND T1.C_CAGENTE = '3390101')
     AND (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.521' AND T1.SECTOR <> 511 AND T1.C_CAGENTE = '3390101'))
GROUP BY 'P', t1.AÑO, T1.TRIM,
         (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, CAST(T1.C_CAGENTE AS int), LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_af521_invnac))
af521_invnac = pd.read_sql(text("SELECT * FROM #af521_invnac"), work_conn)
_log("af521_invnac", af521_invnac)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af521_ca53"))
sql_af521_ca53 = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0510' AS PROC
INTO    #af521_ca53
FROM    #af521_invnac T1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.SECTOR, T1.FUENTE
"""
work_conn.execute(text(sql_af521_ca53))
af521_ca53 = pd.read_sql(text("SELECT * FROM #af521_ca53"), work_conn)
_log("af521_ca53", af521_ca53)

cols_invnac = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_invnac}) SELECT {cols_invnac} FROM #af521_invnac"))
_log("APPEND BD_CTSI AF521_INVNAC", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_invnac}) SELECT {cols_invnac} FROM #af521_ca53"))
_log("APPEND BD_CTSI AF521_CA53", res.rowcount)
for t in ["#af521_invnac", "#af521_ca53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ANULA CONTRAGENTES DISTINTOS DE HOGARES Y RM DEL PATRIMONIO DE LOS FI, Y LOS DEJA EN CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af522_fi_p"))
sql_af522_fi_p = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        t1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0510x' AS PROC
INTO    #af522_fi_p
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   ((T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.522' AND T1.SECTOR = 339011 AND T1.C_CAGENTE <> '511')
     AND (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.522' AND T1.SECTOR = 339011 AND T1.C_CAGENTE <> '6'))
GROUP BY 'P', t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_af522_fi_p))
af522_fi_p = pd.read_sql(text("SELECT * FROM #af522_fi_p"), work_conn)
_log("af522_fi_p", af522_fi_p)


In [ ]:
# IMPUTA AJUSTE ANTERIOR EN CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af522_fi_53"))
sql_af522_fi_53 = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        t1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        t1.FUENTE,
        '0510z' AS PROC
INTO    #af522_fi_53
FROM    #af522_fi_p t1
GROUP BY 'P', t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.SECTOR, T1.FUENTE
"""
work_conn.execute(text(sql_af522_fi_53))
af522_fi_53 = pd.read_sql(text("SELECT * FROM #af522_fi_53"), work_conn)
_log("af522_fi_53", af522_fi_53)

cols_fi_p = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fi_p}) SELECT {cols_fi_p} FROM #af522_fi_p"))
_log("APPEND BD_CTSI AF522_FI_P", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_fi_p}) SELECT {cols_fi_p} FROM #af522_fi_53"))
_log("APPEND BD_CTSI AF522_FI_53", res.rowcount)
for t in ["#af522_fi_p", "#af522_fi_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA CARTERAS DE INST FONDOS NO MONEY MARKET EN PASIVO DE SECTORES 3390102 Y 339011. AJUSTA IMPUTACIÓN EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af522_invnac"))
sql_af522_invnac = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        (CASE WHEN CAST(T1.C_CAGENTE AS int) = 3390102 THEN 3390102 ELSE 339011 END) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END) AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0510a' AS PROC
INTO    #af522_invnac
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   ((T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.522' AND SECTOR <> 6 AND C_CAGENTE IN ('3390102','339011','33901','339'))
     AND (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.522' AND SECTOR <> 511 AND C_CAGENTE IN ('3390102','339011','33901','339')))
GROUP BY 'P', t1.AÑO, T1.TRIM,
         (CASE WHEN t1.C_CUENTA IN ('Rec Precio Reaj','Rec Volumen') THEN 'Rec Precio' ELSE T1.C_CUENTA END),
         T1.C_ENTRADA, T1.C_SCN, T1.N_SCN,
         (CASE WHEN CAST(T1.C_CAGENTE AS int) = 3390102 THEN 3390102 ELSE 339011 END),
         LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_af522_invnac))
af522_invnac = pd.read_sql(text("SELECT * FROM #af522_invnac"), work_conn)
_log("af522_invnac", af522_invnac)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af522_ca53"))
sql_af522_ca53 = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0510a' AS PROC
INTO    #af522_ca53
FROM    #af522_invnac T1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN, T1.SECTOR, T1.FUENTE
"""
work_conn.execute(text(sql_af522_ca53))
af522_ca53 = pd.read_sql(text("SELECT * FROM #af522_ca53"), work_conn)
_log("af522_ca53", af522_ca53)

cols_invnac2 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_invnac2}) SELECT {cols_invnac2} FROM #af522_invnac"))
_log("APPEND BD_CTSI AF522_INVNAC", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_invnac2}) SELECT {cols_invnac2} FROM #af522_ca53"))
_log("APPEND BD_CTSI AF522_CA53", res.rowcount)
for t in ["#af522_invnac", "#af522_ca53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIACIÓN FINAL DE FM Y FI. COMPARA PASIVO CON ACTIVO, LO QUE FALTA EN ACTIVO LO IMPUTA AL ACTIVO DEL SECTOR 51022
work_conn.execute(text("DROP TABLE IF EXISTS #fondos_cierre"))
sql_fondos_cierre = """
SELECT  'P' AS MONEDA,
        t1.AÑO,
        T1.TRIM,
        51022 AS SECTOR,
        (CASE WHEN T1.C_SCN = 'AF.521' THEN '3390101' ELSE '339011' END) AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0534' AS PROC
INTO    #fondos_cierre
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   (T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.521','AF.522') AND SECTOR <> 6)
     OR (T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.521','AF.522') AND C_CAGENTE <> '6')
GROUP BY 'P', t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_SCN = 'AF.521' THEN '3390101' ELSE '339011' END),
         T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_fondos_cierre))
fondos_cierre = pd.read_sql(text("SELECT * FROM #fondos_cierre"), work_conn)
_log("fondos_cierre", fondos_cierre)

cols_cierre = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_cierre}) SELECT {cols_cierre} FROM #fondos_cierre"))
_log("APPEND BD_CTSI FONDOS_CIERRE", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #fondos_cierre"))


In [ ]:
# IMPUTA PATRIMONIO DEL BANCO CENTRAL EN GOBIERNO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #pat_bc"))
sql_pat_bc = """
SELECT  T1.MONEDA,
        t1.AÑO,
        T1.TRIM,
        41 AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        T1.DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0536' AS PROC
INTO    #pat_bc
FROM    TABLAS.dbo.BD_CTSI t1
WHERE   (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5' AND C_CAGENTE = '53' AND SECTOR = 31)
"""
work_conn.execute(text(sql_pat_bc))
pat_bc = pd.read_sql(text("SELECT * FROM #pat_bc"), work_conn)
_log("pat_bc", pat_bc)

cols_pat_bc = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_pat_bc}) SELECT {cols_pat_bc} FROM #pat_bc"))
_log("APPEND BD_CTSI PAT_BC", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #pat_bc"))


In [ ]:
# ACTUALIZA EN ACTIVO AF5 DEL SECTOR 41 CONTRAGENTES RM Y EMPRESAS A CONTRAGENTE 5101
with engine.begin() as conn:
    res = conn.execute(text("""
        UPDATE TABLAS.dbo.BD_CTSI
        SET C_CAGENTE = :nueva_cagente, PROC = :nuevo_proc
        WHERE C_CAGENTE IN (:cag1, :cag2) AND SECTOR = :sector AND C_SCN = :scn
    """), {"nueva_cagente": "5101", "nuevo_proc": "0538", "cag1": "6", "cag2": "51", "sector": 41, "scn": "AF.5"})
    _log("UPDATE BD_CTSI activo AF5 sector 41 a CA 5101", res.rowcount)


## S2_08_Derivados

Imputa contrapartidas de derivados (AF.34/AF.71) entre sectores institucionales y contragente cuando falta el registro simétrico activo/pasivo, ajusta con signo invertido en el sector 53/3 según corresponda, y finalmente neutraliza el pasivo de derivados generando el activo neto por sector, todo acumulado en TABLAS.BD_CTSI

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE (agregaciones/imputaciones) + PROC DATASETS APPEND FORCE, encadenado sobre tablas WORK #tmp de sesión*

In [ ]:
# ========= S2_08_Derivados =========
# COMPRIME TABLA PRINCIPAL (no aplica en SQL Server; el DATA step con COMPRESS=YES es solo optimización de storage SAS, sin efecto de negocio)


In [ ]:
# IMPUTA PASIVO DE DERIVADOS EN SECTOR 321 CON CA 6, USANDO INFO DEL ACTIVO DEL RM CON CA 321. AJUSTA IMPUTACIÓN EN PASIVO DE SECTOR 321 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af34_6_321"))
sql_af34_6_321 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0408' AS PROC
INTO #af34_6_321
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321' AND T1.SECTOR = 6
GROUP BY 'P', t1.[AÑO], T1.TRIM,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
        T1.C_SCN, T1.N_SCN,
        CAST(T1.C_CAGENTE AS int),
        LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_af34_6_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af34_321_53"))
sql_af34_321_53 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO * -1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        T1.PROC
INTO #af34_321_53
FROM #af34_6_321 t1
"""
work_conn.execute(text(sql_af34_321_53))


In [ ]:
# APPEND server-side de ambas temporales a la tabla base (PROC APPEND FORCE alinea por nombre de columna)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_6_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_6_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_321_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_321_53)", res.rowcount)
for t in ["#af34_6_321", "#af34_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO DE DERIVADOS EN SECTOR 321 CON CA 31, USANDO INFO DEL ACTIVO DEL 31 CON CA 321. AJUSTA IMPUTACIÓN EN PASIVO DE SECTOR 321 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af34_31_321"))
sql_af34_31_321 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0409' AS PROC
INTO #af34_31_321
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321' AND T1.SECTOR = 31
GROUP BY 'P', t1.[AÑO], T1.TRIM,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
        T1.C_SCN, T1.N_SCN,
        CAST(T1.C_CAGENTE AS int),
        LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_af34_31_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af34_321_53_2"))
sql_af34_321_53_2 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO * -1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0410' AS PROC
INTO #af34_321_53_2
FROM #af34_31_321 t1
"""
work_conn.execute(text(sql_af34_321_53_2))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_31_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_31_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_321_53_2"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_321_53 v2)", res.rowcount)
for t in ["#af34_31_321", "#af34_321_53_2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO DE DERIVADOS EN SECTOR 321 CON CA 6, USANDO INFO DEL PASIVO DEL 6 CON CA 321. AJUSTA IMPUTACIÓN EN ACTIVO DE SECTOR 321 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af34_321_6"))
sql_af34_321_6 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0411' AS PROC
INTO #af34_321_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321' AND T1.SECTOR = 6
GROUP BY 'P', t1.[AÑO], T1.TRIM,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
        T1.C_SCN, T1.N_SCN,
        CAST(T1.C_CAGENTE AS int),
        LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_af34_321_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af34_321_53d"))
sql_af34_321_53d = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO * -1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0412' AS PROC
INTO #af34_321_53d
FROM #af34_321_6 t1
"""
work_conn.execute(text(sql_af34_321_53d))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_321_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_321_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_321_53d"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_321_53D)", res.rowcount)
for t in ["#af34_321_6", "#af34_321_53d"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO DE DERIVADOS EN SECTOR 339011 CON CA 6, USANDO INFO DEL PASIVO DEL 6 CON CA 33901/351. AJUSTA IMPUTACIÓN EN ACTIVO DE AF71 DE SECTOR 339011 CON CA 6
# AJUSTE CIERRE 2024: INCORPORA AJUSTE EN TÉRMINOS DE DELTA, YA QUE POR DESCUADRE EN AF.32 CON RM EN 2024Q4, SE DECIDE GENERAR AF.34 ACTIVO EN FI
work_conn.execute(text("DROP TABLE IF EXISTS #af34_339011_6"))
sql_af34_339011_6 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        339011 AS SECTOR,
        '6' AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0413' AS PROC
INTO #af34_339011_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '33901/351' AND T1.SECTOR = 6)
   OR (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '6' AND T1.SECTOR = 339011)
GROUP BY 'P', t1.[AÑO], T1.TRIM,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
        T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af34_339011_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_339011_6"))
sql_af71_339011_6 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO * -1 AS DATO,
        'AF.71' AS C_SCN,
        'Ajuste conciliación' AS N_SCN,
        T1.FUENTE,
        '0414' AS PROC
INTO #af71_339011_6
FROM #af34_339011_6 t1
"""
work_conn.execute(text(sql_af71_339011_6))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_339011_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF71_339011_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_339011_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_339011_6)", res.rowcount)
for t in ["#af71_339011_6", "#af34_339011_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# COMPARA ACTIVO DE FP CON RM CON PASIVO DE RM CON FP. LA DIFERENCIA LA IMPUTA EN ACTIVO DE FP CON RM, Y AJUSTA ACTIVO DE AF34 DE FP CON CA 3
work_conn.execute(text("DROP TABLE IF EXISTS #af34_34_6"))
sql_af34_34_6 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        34 AS SECTOR,
        '6' AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0418' AS PROC
INTO #af34_34_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.SECTOR IN (34, 341) AND T1.C_CAGENTE = '6')
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34' AND T1.SECTOR = 6 AND T1.C_CAGENTE = '34')
GROUP BY 'P', t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af34_34_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af34_34_3"))
sql_af34_34_3 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '3' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO * -1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0419' AS PROC
INTO #af34_34_3
FROM #af34_34_6 t1
"""
work_conn.execute(text(sql_af34_34_3))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_34_6"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_34_6)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_34_3"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_34_3)", res.rowcount)
for t in ["#af34_34_6", "#af34_34_3"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: COMPARA ACTIVO DE DERIVADOS DE SECTORES OFIS-AUX-PENSIONES CON CONTRAGENTE BANCOS. IMPUTA EN PASIVO DE BANCOS Y AJUSTA EN BANCOS CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af34_sect_321"))
sql_af34_sect_321 = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        CAST(T1.C_CAGENTE AS int) AS SECTOR,
        LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END AS C_CUENTA,
        'H' AS C_ENTRADA,
        SUM(T1.DATO) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0419a' AS PROC
INTO #af34_sect_321
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34' AND T1.C_CAGENTE = '321'
  AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(6))), 1, 2) IN ('34', '33', '36', '37')
  AND T1.SECTOR NOT IN (339011, 3390101, 3390102, 33901, 331)
GROUP BY 'P', t1.[AÑO], T1.TRIM,
        CASE WHEN T1.C_CUENTA = 'Rec Precio Reaj' THEN 'Rec Precio' ELSE T1.C_CUENTA END,
        T1.C_SCN, T1.N_SCN,
        CAST(T1.C_CAGENTE AS int),
        LTRIM(CAST(T1.SECTOR AS varchar(11)))
"""
work_conn.execute(text(sql_af34_sect_321))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af34_321_53_3"))
sql_af34_321_53_3 = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        T1.DATO * -1 AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0419b' AS PROC
INTO #af34_321_53_3
FROM #af34_sect_321 t1
"""
work_conn.execute(text(sql_af34_321_53_3))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_sect_321"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_SECT_321)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_321_53_3"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_321_53 v3)", res.rowcount)
for t in ["#af34_sect_321", "#af34_321_53_3"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRA AJUSTE EN DERIVADOS COMPARANDO ACTIVO CON PASIVO, E IMPUTANDO DIFERENCIA EN ACTIVO DE SECTOR 51022 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af34_cierre"))
sql_af34_cierre = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        T1.TRIM,
        51022 AS SECTOR,
        '53' AS C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0422' AS PROC
INTO #af34_cierre
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.34') OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34')
GROUP BY 'P', t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af34_cierre))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_cierre"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_CIERRE)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af34_cierre"))


In [ ]:
# GENERA ACTIVOS NETOS DE DERIVADOS ELIMINANDO EL PASIVO E IMPUTANDO ESE PASIVO NEGATIVO EN EL ACTIVO DE LOS SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af34_elimina"))
sql_af34_elimina = """
SELECT  T1.MONEDA,
        t1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        T1.C_ENTRADA,
        SUM(T1.DATO * -1) AS DATO,
        T1.C_SCN,
        T1.N_SCN,
        'PS' AS FUENTE,
        '0651' AS PROC
INTO #af34_elimina
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.34'
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_af34_elimina))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af34_netea"))
sql_af34_netea = """
SELECT  T1.MONEDA,
        T1.[AÑO],
        T1.TRIM,
        T1.SECTOR,
        T1.C_CAGENTE,
        T1.C_CUENTA,
        'D' AS C_ENTRADA,
        T1.DATO,
        T1.C_SCN,
        T1.N_SCN,
        T1.FUENTE,
        '0652' AS PROC
INTO #af34_netea
FROM #af34_elimina t1
"""
work_conn.execute(text(sql_af34_netea))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_elimina"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_ELIMINA)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af34_netea"))
_log("APPEND TABLAS.dbo.BD_CTSI (AF34_NETEA)", res.rowcount)
for t in ["#af34_elimina", "#af34_netea"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S2_09_Capital_Cierre

Concilia el patrimonio (AF.5) entre sectores institucionales (bancos, seguros e isapres, auxiliares financieros, OFIS, resto del mundo y empresas) imputando activo/pasivo faltante por contraparte y ajustando el residuo en un contragente de cierre, dejando la base histórica balanceada por sector / Cierra la conciliación de AF.5 excluyendo sectores no compensables de la imputación, acumula las imputaciones de empresas en la base histórica y calcula/imputa la diferencia final entre pasivo y activo por sector, agregándola también a la base histórica

*confianza: medium · verificador: unverified · SAS: PROC SQL: UPDATE de contragentes + secuencia de CREATE TABLE/SELECT INTO con SUM/GROUP BY para conciliar patrimonio (bancos, seguros, auxiliares, OFIS, RM, empresas) e INSERT/APPEND acumulativo hacia la tabla histórica + PROC SQL DELETE + PROC DATASETS APPEND/DROP + PROC SQL CREATE TABLE con JOIN/GROUP BY + DATA step SET (concatenación) para cierre de AF5*

In [ ]:
# ========= S2_09_Capital_Cierre =========
# COMPRIME TABLA PRINCIPAL (COMPRESS=YES no tiene equivalente en SQL Server: se omite, es solo almacenamiento)
# El SET tablas.BD_CTSI sobre sí misma en SAS es un no-op funcional salvo la compresión; no se traduce ninguna operación de datos.


In [ ]:
# CIERRE 2021: EN SECTOR 361 INST AF5 PASIVO CAMBIA CONTRAGENTE DESDE 3 A 53
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI
    SET C_CAGENTE = '53'
    WHERE C_CAGENTE = :cagente_origen AND C_SCN = :c_scn AND C_ENTRADA = :c_entrada AND SECTOR = :sector
"""), {"cagente_origen": "3", "c_scn": "AF.5", "c_entrada": "H", "sector": 361})
_log("UPDATE BD_CTSI sector 361 CA 3->53", res.rowcount)


In [ ]:
# CIERRE 2021: EN SECTOR 339011 INST AF5 ACTIVO CAMBIA CONTRAGENTE DESDE 3 A 53
# Nota: SAS compara C_CAGENTE='' (blanco); en SQL Server eso es NULL o cadena vacía real, no ambos por defecto:
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI
    SET C_CAGENTE = '53'
    WHERE (C_CAGENTE IS NULL OR C_CAGENTE = '') AND C_SCN = :c_scn AND C_ENTRADA = :c_entrada AND SECTOR = :sector
"""), {"c_scn": "AF.5", "c_entrada": "D", "sector": 339011})
_log("UPDATE BD_CTSI sector 339011 CA ''->53", res.rowcount)


In [ ]:
# ELIMINA ACTIVO AF5 DEL SECTOR 5111 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111"))
sql_af5_5111 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO * -1) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0539' AS PROC
INTO #af5_5111
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE = '53' AND SECTOR = 5111
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_5111))


In [ ]:
# APPEND server-side: append tal cual, columnas explícitas (FORCE alinea por nombre)
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_5111 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi} FROM #af5_5111
"""
res = work_conn.execute(text(sql_append_5111))
_log("APPEND BD_CTSI desde AF5_5111", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111"))


In [ ]:
# IMPUTA ACTIVO DE AF5 EN SECTOR 511 CON CA 53, USANDO INFO DEL PASIVO AF5 DE SECTOR 5111 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511"))
sql_af5_511 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, 511 AS SECTOR, C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0543' AS PROC
INTO #af5_511
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND C_CAGENTE = '53' AND SECTOR = 5111 AND FUENTE = 'CI'
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE
"""
work_conn.execute(text(sql_af5_511))


In [ ]:
sql_append_511 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi} FROM #af5_511
"""
res = work_conn.execute(text(sql_append_511))
_log("APPEND BD_CTSI desde AF5_511 (0543)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511"))


In [ ]:
# ELIMINA ACTIVO Y PASIVO AF7 DEL SECTOR 5111 CON CA 53. CIERRE 2020: ELIMINA TMB CA 511,6,4 DE AF7. CIERRE 2021: ELIMINA TODO EL AF5 DEL SECTOR 5111 CON TODOS LOS CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_7_5111"))
sql_af5_7_5111 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO * -1) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0544' AS PROC
INTO #af5_7_5111
FROM TABLAS.dbo.BD_CTSI
WHERE C_SCN IN ('AF.7') AND C_CAGENTE IN ('511','53','6','4') AND SECTOR = 5111 AND DATO <> 0
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_7_5111))


In [ ]:
sql_append_7_5111 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi} FROM #af5_7_5111
"""
res = work_conn.execute(text(sql_append_7_5111))
_log("APPEND BD_CTSI desde AF5_7_5111 (0544 AF.7)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_7_5111"))


In [ ]:
# ELIMINA ACTIVO Y PASIVO AF5 DEL SECTOR 5111 CON CA 53. CIERRE 2021: ELIMINA TODO EL AF5 DEL SECTOR 5111 CON TODOS LOS CONTRAGENTES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111_2"))
sql_af5_5111_2 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO * -1) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0544' AS PROC
INTO #af5_5111_2
FROM TABLAS.dbo.BD_CTSI
WHERE C_SCN IN ('AF.5') AND SECTOR = 5111 AND DATO <> 0
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_5111_2))


In [ ]:
sql_append_5111_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi} FROM #af5_5111_2
"""
res = work_conn.execute(text(sql_append_5111_2))
_log("APPEND BD_CTSI desde AF5_5111 (0544 AF.5)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_5111_2"))


In [ ]:
# IMPUTA PASIVO DE AF5 EN SECTOR 51022 CON CA 53, USANDO INFO DEL PASIVO AF5 DE SECTOR 5111 CON CA 53 FUENTE CI
# CIERRE 2021: ELIMINA CONTRAGENTE PORQUE AHORA VIENEN SECTORIZADO EL PATRIMONIO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022"))
sql_af5_51022 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0550' AS PROC
INTO #af5_51022
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 5111 AND FUENTE = 'CI'
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, C_CAGENTE
"""
work_conn.execute(text(sql_af5_51022))


In [ ]:
sql_append_51022 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi} FROM #af5_51022
"""
res = work_conn.execute(text(sql_append_51022))
_log("APPEND BD_CTSI desde AF5_51022 (0550)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022"))


In [ ]:
# IMPUTA ACTIVO DE AF5 EN SECTOR 511 CON CA 322, USANDO INFO DEL PASIVO AF5 DE SECTOR 322 FUENTE CI
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_322"))
sql_af5_511_322 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, 511 AS SECTOR, '322' AS C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0551' AS PROC
INTO #af5_511_322
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 322 AND FUENTE = 'CI'
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE
"""
work_conn.execute(text(sql_af5_511_322))


In [ ]:
sql_append_511_322 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi} FROM #af5_511_322
"""
res = work_conn.execute(text(sql_append_511_322))
_log("APPEND BD_CTSI desde AF5_511 (0551)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_322"))


In [ ]:
# IMPUTA ACTIVO AF5 EN SECTOR 511 CON CA 51021, USANDO INFO DE PASIVO DE SECTOR 51021 (SOLO EL 0.036280969257607 DE ESE TOTAL)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_saa"))
sql_af5_511_saa = """
SELECT 'P' AS MONEDA, AÑO, TRIM, 511 AS SECTOR, '51021' AS C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO * 0.036280969257607) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0559' AS PROC
INTO #af5_511_saa
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 51021
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af5_511_saa))


In [ ]:
sql_append_511_saa = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi} FROM #af5_511_saa
"""
res = work_conn.execute(text(sql_append_511_saa))
_log("APPEND BD_CTSI desde AF5_511_SAA (0559)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_511_saa"))


In [ ]:
# CONCILIA PATRIMONIO BANCARIO: reasignaciones puntuales de contragente
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '34', PROC = '0560a'
    WHERE C_CAGENTE = '361' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 321
"""))
_log("UPDATE BD_CTSI CA 361->34 sector 321", res.rowcount)

res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51021', PROC = '0560a'
    WHERE C_CAGENTE = '321' AND C_SCN = 'AF.5' AND C_ENTRADA = 'D' AND SECTOR = 353
"""))
_log("UPDATE BD_CTSI CA 321->51021 sector 353", res.rowcount)

res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51021', PROC = '0560a'
    WHERE C_CAGENTE = '321' AND C_SCN = 'AF.5' AND C_ENTRADA = 'D' AND SECTOR = 361
"""))
_log("UPDATE BD_CTSI CA 321->51021 sector 361", res.rowcount)

res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51022', PROC = '0560a'
    WHERE C_CAGENTE = '511' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 411
"""))
_log("UPDATE BD_CTSI CA 511->51022 sector 411", res.rowcount)


In [ ]:
# SELECCIONA PAT BANCOS SECTORIZADO PARA GOB, EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS
# CIERRE 2021: EN OFIS NO SE DEBE SELECCIONAR LOS FONDOS MUTUOS NI DE INVERSIÓN, PARA NO ELIMINAR SU TENENCIA DE ACCIONES DE BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321"))
sql_af5_321 = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(3))) AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0560b' AS PROC
INTO #af5_321
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 321
  AND C_CAGENTE NOT IN ('511','351','352','353','34','341','53','36904','3390101','3390102','339011','33901')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, SECTOR
"""
work_conn.execute(text(sql_af5_321))


In [ ]:
# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE BANCOS EN SECTORES GOB, EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321_act"))
sql_af5_321_act = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0560b' AS PROC
INTO #af5_321_act
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE = '321'
  AND SECTOR NOT IN (511,351,352,353,34,341,53,36904,339001,3390102,339011,33901)
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_321_act))


In [ ]:
# concatenación de ambos subconjuntos (DATA AF5_321; SET AF5_321 AF5_321_ACT;)
cols_af5_321 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321_full"))
sql_af5_321_full = f"""
SELECT {cols_af5_321} INTO #af5_321_full FROM #af5_321
UNION ALL
SELECT {cols_af5_321} FROM #af5_321_act
"""
work_conn.execute(text(sql_af5_321_full))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321_imp"))
sql_af5_321_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_321_imp
FROM #af5_321_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_321_imp))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN GOBIERNO QUE AJUSTA EN 5101. CIERRE 2021: SI ES SECTOR 36 SE RESTA DEL MISMO SECTOR CON CA 53 Y NO EN 51022
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp"))
sql_af5_53_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 41 THEN '5101' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp
FROM #af5_321_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_321_imp"))
_log("APPEND BD_CTSI desde AF5_321_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp"))
_log("APPEND BD_CTSI desde AF5_53_IMP (bancario)", res.rowcount)
for t in ["#af5_53_imp", "#af5_321_imp", "#af5_321", "#af5_321_act", "#af5_321_full"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO DE AF5 DE RESTO DE EMPRESAS CON BANCOS (RM). SELECCIONA PAT BANCOS SECTORIZADO CON RM Y SOC DE INVERSIONES. CIERRE 2021: INCORPORA CONTRAGENTE 36912
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321_6"))
sql_af5_321_6 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, 6 AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(3))) AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0560c' AS PROC
INTO #af5_321_6
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 321 AND C_CAGENTE IN ('6','36','36912')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR
"""
work_conn.execute(text(sql_af5_321_6))


In [ ]:
# SELECCIONA ACTIVO DE AF.5 DEL RM CON CONTRAGENTE BANCOS PARA CREAR INV DE SECTOR 53 EN 321
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_act"))
sql_af5_rm_act = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0560c' AS PROC
INTO #af5_rm_act
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE = '321' AND SECTOR = 6 AND FUENTE = 'CI'
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_rm_act))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_321_6_full"))
sql_af5_321_6_full = f"""
SELECT {cols_af5_321} INTO #af5_321_6_full FROM #af5_321_6
UNION ALL
SELECT {cols_af5_321} FROM #af5_rm_act
"""
work_conn.execute(text(sql_af5_321_6_full))


In [ ]:
# CALCULA DIFERENCIAL A IMPUTAR EN SECTOR 51022 CON BANCOS...MISMO QUE SE IMPUTARÁ EN PASIVO DE EMPRESAS CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_dif_53"))
sql_af5_dif_53 = """
SELECT MONEDA, AÑO, TRIM, 51022 AS SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_dif_53
FROM #af5_321_6_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_dif_53))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN ACTIVO AF5 DE RESTO DE EMPRESAS CON CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_dif_aj"))
sql_af5_dif_aj = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_dif_aj
FROM #af5_dif_53
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_dif_aj))


In [ ]:
# IMPUTA AF PASIVO EN RESTO DE EMPRESAS CON CA RESTO DEL MUNDO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pas_r_rm"))
sql_af5_pas_r_rm = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_pas_r_rm
FROM #af5_dif_53
"""
work_conn.execute(text(sql_af5_pas_r_rm))


In [ ]:
# IMPUTA AF PASIVO EN RESTO DE EMPRESAS CON CA RESTO DEL MUNDO (ajuste, signo invertido, CA 53)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pas_r_r"))
sql_af5_pas_r_r = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_pas_r_r
FROM #af5_pas_r_rm
"""
work_conn.execute(text(sql_af5_pas_r_r))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_dif_53"))
_log("APPEND BD_CTSI desde AF5_DIF_53", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_dif_aj"))
_log("APPEND BD_CTSI desde AF5_DIF_AJ", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_pas_r_rm"))
_log("APPEND BD_CTSI desde AF5_PAS_R_RM", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_pas_r_r"))
_log("APPEND BD_CTSI desde AF5_PAS_R_R", res.rowcount)
for t in ["#af5_dif_aj", "#af5_321_6", "#af5_rm_act", "#af5_321_6_full", "#af5_dif_53", "#af5_pas_r_rm", "#af5_pas_r_r"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO AF5 DEL RM CON CONTRAGENTE SOC DE INVERSIONES (36). SELECCIONA INVERSION AF5 DE SECTOR 36 CON CA BANCOS PARA IMPUTARLO EN SECTOR 6 CON CA 36
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_321"))
sql_af5_36_321 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, 6 AS SECTOR, '36' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0560d' AS PROC
INTO #af5_36_321
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND SECTOR IN (36,36912) AND C_CAGENTE = '321'
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af5_36_321))


In [ ]:
# AJUSTA ANTERIOR EN SECTOR RM CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_53"))
sql_af5_6_53 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_53
FROM #af5_36_321
"""
work_conn.execute(text(sql_af5_6_53))


In [ ]:
# AJUSTA IMPUTACION DE PASIVO EN AF5 DEL SECTOR 51022 CON CONTRAGENTE 53. CIERRE 2021: SE MANTIENE ESTE AJUSTE EN PATRIMONIO DE EMPRESAS PORQUE SINO AUMENTA MUCHO SU VALOR
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_pas"))
sql_af5_51022_pas = """
SELECT MONEDA, AÑO, TRIM, 51022 AS SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_51022_pas
FROM #af5_36_321
"""
work_conn.execute(text(sql_af5_51022_pas))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_321"))
_log("APPEND BD_CTSI desde AF5_36_321", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_53"))
_log("APPEND BD_CTSI desde AF5_6_53 (bancario)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_51022_pas"))
_log("APPEND BD_CTSI desde AF5_51022_PAS (bancario)", res.rowcount)
for t in ["#af5_6_53", "#af5_36_321", "#af5_51022_pas"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO SEGUROS E ISAPRES
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '51022', PROC = '0561a'
    WHERE C_CAGENTE = '53' AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND SECTOR = 353
"""))
_log("UPDATE BD_CTSI CA 53->51022 sector 353", res.rowcount)


In [ ]:
# SELECCIONA PAT SEGUROS E ISAPRES SECTORIZADO PARA EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS. CIERRE 2021: EN OFIS NO SE INCLUYEN FONDOS MUTUOS NI DE INVERSIÓN
work_conn.execute(text("DROP TABLE IF EXISTS #af5_35"))
sql_af5_35 = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(3))) AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0561b' AS PROC
INTO #af5_35
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR IN (351,352,353)
  AND C_CAGENTE NOT IN ('511','412','41','53','36904','3390101','3390102','339011','33901')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, SECTOR
"""
work_conn.execute(text(sql_af5_35))


In [ ]:
# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE seguros e isapres EN SECTORES EMP SUPERVISADAS, AUX (SIN CORREDORAS DE BOLSAS), OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #af5_35_act"))
sql_af5_35_act = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0561b' AS PROC
INTO #af5_35_act
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE IN ('35','351','352','353','351/352')
  AND SECTOR NOT IN (511,412,41,53,36904,3390101,3390102,339011,33901)
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_35_act))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_35_full"))
sql_af5_35_full = f"""
SELECT {cols_af5_321} INTO #af5_35_full FROM #af5_35
UNION ALL
SELECT {cols_af5_321} FROM #af5_35_act
"""
work_conn.execute(text(sql_af5_35_full))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_35_imp"))
sql_af5_35_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_35_imp
FROM #af5_35_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_35_imp))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN SECTOR 36 QUE LO CAMBIA A 51022 CON CA 53 Y EN SECTOR 6 QUE LO RESTA DE CA 33 (contenido en OFIS no en empresas)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp2"))
sql_af5_53_imp2 = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 6 THEN '33' WHEN SECTOR = 361 THEN '321' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp2
FROM #af5_35_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp2))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_35_imp"))
_log("APPEND BD_CTSI desde AF5_35_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp2"))
_log("APPEND BD_CTSI desde AF5_53_IMP (seguros)", res.rowcount)
for t in ["#af5_53_imp2", "#af5_35_imp", "#af5_35", "#af5_35_act", "#af5_35_full"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO AF5 DEL RM CON CONTRAGENTE SOC DE INVERSIONES (36) POR IMPUTACIÓN DE SEGUROS E ISAPRES. SELECCIONA INVERSION AF5 DE SECTOR 36 CON CA SEGUROS PARA IMPUTARLO EN SECTOR 6 CON CA 36
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_35"))
sql_af5_36_35 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, 6 AS SECTOR, '36' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0561d' AS PROC
INTO #af5_36_35
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND SECTOR IN (36,36912) AND C_CAGENTE IN ('351','352')
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af5_36_35))


In [ ]:
# AJUSTA ANTERIOR EN SECTOR RM CA 33
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_53_2"))
sql_af5_6_53_2 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '33' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_53_2
FROM #af5_36_35
"""
work_conn.execute(text(sql_af5_6_53_2))


In [ ]:
# AJUSTA IMPUTACION DE PASIVO EN AF5 DEL SECTOR 51022 CON CONTRAGENTE 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_pas2"))
sql_af5_51022_pas2 = """
SELECT MONEDA, AÑO, TRIM, 51022 AS SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       DATO * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_51022_pas2
FROM #af5_36_35
"""
work_conn.execute(text(sql_af5_51022_pas2))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_35"))
_log("APPEND BD_CTSI desde AF5_36_35", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_53_2"))
_log("APPEND BD_CTSI desde AF5_6_53 (seguros)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_51022_pas2"))
_log("APPEND BD_CTSI desde AF5_51022_PAS (seguros)", res.rowcount)
for t in ["#af5_6_53_2", "#af5_36_35", "#af5_51022_pas2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO DE AUXILIARES FINANCIEROS. SELECCIONA PAT AUXILIARES SECTORIZADO EN TODOS MENOS HOGARES, CORREDORAS Y RESTO. CIERRE 2021: TAMPOCO SE CONSIDERA FONDOS MUTUOS E INVERSIÓN, incorpora sector 334
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36"))
sql_af5_36 = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(6))) AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0562b' AS PROC
INTO #af5_36
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND CAST(SECTOR AS varchar(6)) LIKE '%36%'
       AND C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','339011','3390102','33901'))
   OR (C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 37
       AND C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','339011','3390102','33901'))
   OR (C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 334
       AND C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','339011','3390102','33901'))
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, SECTOR
"""
work_conn.execute(text(sql_af5_36))


In [ ]:
# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE auxiliares EN SECTORES EXCEPTO HOGARES, CORREDORES DE BOLSA Y RESTO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_act"))
sql_af5_36_act = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0562b' AS PROC
INTO #af5_36_act
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE LIKE '%36%' AND SECTOR NOT IN (511,34,53,36904,3390101,339011,3390102,33901))
   OR (C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE LIKE '%37%' AND SECTOR NOT IN (511,34,53,36904,3390101,339011,3390102,33901))
   OR (C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE LIKE '%334%' AND SECTOR NOT IN (511,34,53,36904,3390101,339011,3390102,33901))
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_36_act))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_full"))
sql_af5_36_full = f"""
SELECT {cols_af5_321} INTO #af5_36_full FROM #af5_36
UNION ALL
SELECT {cols_af5_321} FROM #af5_36_act
"""
work_conn.execute(text(sql_af5_36_full))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_imp"))
sql_af5_36_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_36_imp
FROM #af5_36_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_36_imp))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN SECTOR 36 QUE LO CAMBIA A 51022 CON CA 53 Y EN SECTOR 6 QUE LO RESTA DE CA 33
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp3"))
sql_af5_53_imp3 = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 6 THEN '33' WHEN SECTOR = 361 THEN '321' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp3
FROM #af5_36_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp3))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_imp"))
_log("APPEND BD_CTSI desde AF5_36_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp3"))
_log("APPEND BD_CTSI desde AF5_53_IMP (auxiliares)", res.rowcount)
for t in ["#af5_53_imp3", "#af5_36_imp", "#af5_36", "#af5_36_act", "#af5_36_full"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA ANTERIOR EN SECTOR 51022 CA 33 (impuestas inversiones en 36/37 por auxiliares se restan de 51022 CA 53)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_53"))
sql_af5_51022_53 = """
SELECT 'p' AS MONEDA, AÑO, TRIM, 51022 AS SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0562c' AS PROC
INTO #af5_51022_53
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND SECTOR IN (36,36912) AND C_CAGENTE LIKE '%36%')
   OR (C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND SECTOR IN (36,36912) AND C_CAGENTE = '37')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af5_51022_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_51022_53"))
_log("APPEND BD_CTSI desde AF5_51022_53", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51022_53"))


In [ ]:
# CONCILIA PATRIMONIO DE OFIS. SELECCIONA PAT OFIS SECTORIZADO EN TODOS MENOS HOGARES, CORREDORAS Y RESTO. CIERRE 2021: TAMPOCO SE INCLUYEN FONDOS MUTUOS E INVERSIÓN, PARA RESPETAR SUS DATOS ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_33"))
sql_af5_33 = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(6))) AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0563b' AS PROC
INTO #af5_33
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND CAST(SECTOR AS varchar(6)) LIKE '%33%'
       AND C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','3390102','339011','33901'))
   OR (C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 411
       AND C_CAGENTE NOT IN ('511','34','36904','53','9','3390101','3390102','339011','33901'))
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, SECTOR
"""
work_conn.execute(text(sql_af5_33))


In [ ]:
# SELECCIONA ACTIVO DE AF.5 CON CONTRAGENTE ofis EN TODOS MENOS HOGARES, CORREDORAS Y RESTO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_33_act"))
sql_af5_33_act = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0563b' AS PROC
INTO #af5_33_act
FROM TABLAS.dbo.BD_CTSI
WHERE (C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE LIKE '%33%' AND SECTOR NOT IN (511,34,53,36904,3390101,3390102,339011,33901))
   OR (C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND C_CAGENTE LIKE '%411%' AND SECTOR NOT IN (511,34,53,36904,3390101,3390102,339011,33901))
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_33_act))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_33_full"))
sql_af5_33_full = f"""
SELECT {cols_af5_321} INTO #af5_33_full FROM #af5_33
UNION ALL
SELECT {cols_af5_321} FROM #af5_33_act
"""
work_conn.execute(text(sql_af5_33_full))


In [ ]:
# CALCULA DIF A IMPUTAR EN CARTERA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_33_imp"))
sql_af5_33_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_33_imp
FROM #af5_33_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_33_imp))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53, EXCEPTO EN SECTOR RM QUE LO CAMBIA A 36
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp4"))
sql_af5_53_imp4 = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 6 THEN '36' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp4
FROM #af5_33_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp4))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_33_imp"))
_log("APPEND BD_CTSI desde AF5_33_IMP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp4"))
_log("APPEND BD_CTSI desde AF5_53_IMP (OFIS)", res.rowcount)
for t in ["#af5_33", "#af5_33_act", "#af5_33_imp", "#af5_53_imp4", "#af5_33_full"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA PASIVO EN AUXILIARES (SOC DE INV 36) CON CA RM, USANDO INFO DE IMPUTACIÓN REALIZADA EN PROC 0563b
# porque se asume que el RM invierte en soc de inv para a través de estas inv indirectamente en auxiliares
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_6"))
sql_af5_36_6 = """
SELECT MONEDA, AÑO, TRIM, 36 AS SECTOR, '6' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0563e' AS PROC
INTO #af5_36_6
FROM TABLAS.dbo.BD_CTSI
WHERE PROC = '0563b' AND SECTOR = 6 AND C_CAGENTE = '36'
GROUP BY MONEDA, AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af5_36_6))


In [ ]:
# REBAJA DE SECTOR 36 CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_36_53"))
sql_af5_36_53 = """
SELECT MONEDA, AÑO, TRIM, 36 AS SECTOR, '53' AS C_CAGENTE, C_CUENTA, 'H' AS C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0563e' AS PROC
INTO #af5_36_53
FROM TABLAS.dbo.BD_CTSI
WHERE PROC = '0563b' AND SECTOR = 6 AND C_CAGENTE = '36'
GROUP BY MONEDA, AÑO, TRIM, C_CUENTA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af5_36_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_6"))
_log("APPEND BD_CTSI desde AF5_36_6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_36_53"))
_log("APPEND BD_CTSI desde AF5_36_53", res.rowcount)
for t in ["#af5_36_6", "#af5_36_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO RESTO DEL MUNDO. 1.CONCILIA RM CON BCO CENTRAL: RESPETA DATO DEL BCENTRAL Y AJUSTA EN RM CON CA 53. SELECCIONA PAT RM CON BCENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_31"))
sql_af5_6_31 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0564a' AS PROC
INTO #af5_6_31
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 6 AND C_CAGENTE = '31'
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_6_31))


In [ ]:
# SELECCIONA ACTIVO AF5 DEL BC CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_31_6"))
sql_af5_31_6 = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(6))) AS C_CAGENTE,
       C_CUENTA, 'H' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0564a' AS PROC
INTO #af5_31_6
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND SECTOR = 31 AND C_CAGENTE = '6'
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR
"""
work_conn.execute(text(sql_af5_31_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_31_full"))
sql_af5_6_31_full = f"""
SELECT {cols_af5_321} INTO #af5_6_31_full FROM #af5_6_31
UNION ALL
SELECT {cols_af5_321} FROM #af5_31_6
"""
work_conn.execute(text(sql_af5_6_31_full))


In [ ]:
# CALCULA DIF A IMPUTAR EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_imp"))
sql_af5_6_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_imp
FROM #af5_6_31_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_6_imp))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp5"))
sql_af5_53_imp5 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp5
FROM #af5_6_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp5))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_imp"))
_log("APPEND BD_CTSI desde AF5_6_IMP (BC)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp5"))
_log("APPEND BD_CTSI desde AF5_53_IMP (BC)", res.rowcount)
for t in ["#af5_6_31", "#af5_31_6", "#af5_6_31_full", "#af5_6_imp", "#af5_53_imp5"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021: CAMBIA CONTRAGENTE EN PASIVO AF5 DEL RM DESDE 3390101 A 33901
res = work_conn.execute(text("""
    UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '33901'
    WHERE SECTOR = 6 AND C_SCN = 'AF.5' AND C_ENTRADA = 'H' AND C_CAGENTE = '3390101'
"""))
_log("UPDATE BD_CTSI CA 3390101->33901 RM", res.rowcount)


In [ ]:
# 2.CONCILIA RM CON BCOS, SEGUROS, OFIS Y PENSIONES. RESPETA DATO DEL RM IMPUTANDO DIFERENCIAL EN SECTORES CON CA RM Y AJUSTA EN SECTORES CON CA 53
# SELECCIONA PAT RM CON SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_sect"))
sql_af5_6_sect = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(6))) AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0564b' AS PROC
INTO #af5_6_sect
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR = 6 AND C_CAGENTE NOT IN ('31','511','53','9')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, SECTOR
"""
work_conn.execute(text(sql_af5_6_sect))


In [ ]:
# SELECCIONA ACTIVO AF5 DE SECTORES CON RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_sect_6"))
sql_af5_sect_6 = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(SUBSTRING(LTRIM(CAST(SECTOR AS varchar(8))), 1, 5) AS int) AS SECTOR,
       C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0564b' AS PROC
INTO #af5_sect_6
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND SECTOR NOT IN (31,511,53,9) AND C_CAGENTE = '6'
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_sect_6))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_sect_full"))
sql_af5_6_sect_full = f"""
SELECT {cols_af5_321} INTO #af5_6_sect_full FROM #af5_6_sect
UNION ALL
SELECT {cols_af5_321} FROM #af5_sect_6
"""
work_conn.execute(text(sql_af5_6_sect_full))


In [ ]:
# CALCULA DIF A IMPUTAR EN RM
work_conn.execute(text("DROP TABLE IF EXISTS #af5_6_imp2"))
sql_af5_6_imp2 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_6_imp2
FROM #af5_6_sect_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_6_imp2))


In [ ]:
# CIERRE 2021: NO SE HACE EL CAMBIO DE SECTOR 33->36 PARA QUE QUEDE EN EL SECTOR CORRECTO QUE ES FMNM Y NO AUXILIARES (comentado en el SAS original)
# AJUSTA IMPUTACION ANTERIOR EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp6"))
sql_af5_53_imp6 = """
SELECT MONEDA, AÑO, TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp6
FROM #af5_6_imp2
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp6))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_6_imp2"))
_log("APPEND BD_CTSI desde AF5_6_IMP (sectores)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp6"))
_log("APPEND BD_CTSI desde AF5_53_IMP (sectores)", res.rowcount)
for t in ["#af5_6_sect", "#af5_sect_6", "#af5_6_sect_full", "#af5_6_imp2", "#af5_53_imp6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA PATRIMONIO DE EMPRESAS. SELECCIONA PAT EMPRESAS CON SECTORES A CONCILIAR. CIERRE 2021: EXCEPTO EN SECTORES FONDOS MUTUOS E INVERSIÓN, PARA RESPETAR SUS DATOS ORIGINALES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51_pas"))
sql_af5_51_pas = """
SELECT 'P' AS MONEDA, AÑO, TRIM,
       CAST(C_CAGENTE AS int) AS SECTOR,
       LTRIM(CAST(SECTOR AS varchar(6))) AS C_CAGENTE,
       C_CUENTA, 'D' AS C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0565a' AS PROC
INTO #af5_51_pas
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_SCN = 'AF.5' AND SECTOR IN (51021,5101)
  AND C_CAGENTE NOT IN ('511','35','351','352','353','34','341','36904','53','3390101','3390102','339011','33901')
GROUP BY AÑO, TRIM, C_CUENTA, C_SCN, N_SCN, C_CAGENTE, SECTOR
"""
work_conn.execute(text(sql_af5_51_pas))


In [ ]:
# SELECCIONA ACTIVO AF5 DE SECTORES CON EMPRESAS SUPERVISADAS
work_conn.execute(text("DROP TABLE IF EXISTS #af5_sect_51"))
sql_af5_sect_51 = """
SELECT 'P' AS MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, 'PS' AS FUENTE, '0565a' AS PROC
INTO #af5_sect_51
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_SCN = 'AF.5' AND SECTOR NOT IN (511,35,351,352,353,34,341,36904,3390101,3390102,339011,33901)
  AND C_CAGENTE IN ('51021','5101','2','51')
GROUP BY AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE
"""
work_conn.execute(text(sql_af5_sect_51))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_51_pas_full"))
sql_af5_51_pas_full = f"""
SELECT {cols_af5_321} INTO #af5_51_pas_full FROM #af5_51_pas
UNION ALL
SELECT {cols_af5_321} FROM #af5_sect_51
"""
work_conn.execute(text(sql_af5_51_pas_full))


In [ ]:
# CALCULA DIF A IMPUTAR EN SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #af5_sect_imp"))
sql_af5_sect_imp = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_sect_imp
FROM #af5_51_pas_full
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_sect_imp))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53 (sector 361 ajusta en 321)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_53_imp7"))
sql_af5_53_imp7 = """
SELECT MONEDA, AÑO, TRIM, SECTOR,
       CASE WHEN SECTOR = 361 THEN '321' ELSE '53' END AS C_CAGENTE,
       C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_53_imp7
FROM #af5_sect_imp
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, FUENTE, PROC
"""
work_conn.execute(text(sql_af5_53_imp7))


In [ ]:
# APPEND server-side de la conciliación de patrimonio de empresas (paridad con el resto del nodo: crea + inserta + limpia)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_sect_imp"))
_log("APPEND BD_CTSI desde AF5_SECT_IMP (empresas)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af5_53_imp7"))
_log("APPEND BD_CTSI desde AF5_53_IMP (empresas)", res.rowcount)
for t in ["#af5_51_pas", "#af5_sect_51", "#af5_51_pas_full", "#af5_sect_imp", "#af5_53_imp7"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# PORQ EN ESTOS SECTORES NO SE DEBE COMPENSAR LA IMPUTACIÓN
res = work_conn.execute(
    text("DELETE FROM #af5_53_imp WHERE SECTOR IN (4,41,42,412,413,331)")
)
_log("DELETE #af5_53_imp", res.rowcount)


In [ ]:
# APPEND server-side de las tablas de imputación hacia TABLAS.BD_CTSI (acumula, igual que el SAS original)
cols_bd_ctsi = (
    "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
)

sql_append_sect_imp = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #af5_sect_imp
"""
res = work_conn.execute(text(sql_append_sect_imp))
_log("APPEND TABLAS.dbo.BD_CTSI (AF5_SECT_IMP)", res.rowcount)

sql_append_53_imp = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #af5_53_imp
"""
res = work_conn.execute(text(sql_append_53_imp))
_log("APPEND TABLAS.dbo.BD_CTSI (AF5_53_IMP)", res.rowcount)


In [ ]:
# Limpieza de temporales de sesión ya volcadas a la base
for t in ["#af5_51_pas", "#af5_sect_51", "#af5_sect_imp", "#af5_53_imp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO AF5 DEL RM CON CONTRAGENTE SOC DE INVERSIONES (36) POR IMPUTACIÓN DE INV EN AF5 DE SOC DE INV EN EMPRESAS.
# REBAJA DE SECTOR RM CON CA 53 EN AF5 ACTIVO.
# ADEMÁS POR ESTE MISMO MONTO IMPUTA PASIVO AF5 DE SECTOR 36 CON CA RM, AJUSTANDO EN PAS AF5 SECTOR 51022 CON CA 53
# SELECCIONA INVERSION AF5 DE SECTOR 36 CON CA EMPRESAS PARA IMPUTARLO EN SECTOR 6 CON CA 36
# CIERRE 2021: ELIMINA ESTE DATO PORQUE AHORA EXISTE EL SECTOR SOC DE NV EN LA SÍNTESIS Y SE ESTABA ASIGNANDO MUCHO A RM
# (todo el bloque AF5_36_51 / AF5_6_53 / AF5_36_PAS / AF5_51022_PAS y sus APPEND/DROP quedó comentado en el SAS original: no se traduce)


In [ ]:
# CIERRE DE AF5. CALCULA DIFERENCIA DE PASIVO MENOS ACTIVO (RESPETA PASIVO).
# IMPUTA DIFERENCIA EN ACTIVO DE SECTOR 51022 CON CA 53.
# CALCULA TOTAL PASIVO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pasivo"))
sql_af5_pasivo = """
SELECT 'P' AS MONEDA,
       T1.AÑO,
       T1.TRIM,
       51022 AS SECTOR,
       CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LEFT(CONVERT(char(4), T2.C_SI_publ), 4) END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0560' AS PROC
INTO #af5_pasivo
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'AF.5') AND T1.SECTOR = T2.C_SI
GROUP BY 'P', T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LEFT(CONVERT(char(4), T2.C_SI_publ), 4) END
"""
work_conn.execute(text(sql_af5_pasivo))


In [ ]:
# CALCULA TOTAL ACTIVO POR CADA SECTOR (mismo comentario del SAS: CALCULA TOTAL PASIVO POR CADA SECTOR)
work_conn.execute(text("DROP TABLE IF EXISTS #af5_activo"))
sql_af5_activo = """
SELECT 'P' AS MONEDA,
       T1.AÑO,
       T1.TRIM,
       51022 AS SECTOR,
       LEFT(CONVERT(char(4), T2.C_SI_publ), 4) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0560' AS PROC
INTO #af5_activo
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_CONTRAPARTIDAS T2
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'AF.5') AND T1.C_CAGENTE = T2.C_CAGENTE
GROUP BY T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         LEFT(CONVERT(char(4), T2.C_SI_publ), 4)
"""
work_conn.execute(text(sql_af5_activo))


In [ ]:
# DATA AF5_PASIVO; SET AF5_PASIVO AF5_ACTIVO; -> concatenación server-side
work_conn.execute(text("DROP TABLE IF EXISTS #af5_pasivo_full"))
sql_af5_pasivo_full = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #af5_pasivo_full
FROM #af5_pasivo
UNION ALL
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC
FROM #af5_activo
"""
work_conn.execute(text(sql_af5_pasivo_full))


In [ ]:
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_delta"))
sql_af5_delta = """
SELECT T1.MONEDA,
       T1.AÑO,
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #af5_delta
FROM #af5_pasivo_full T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN,
         T1.SECTOR, T1.C_CAGENTE, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_af5_delta))
af5_delta = pd.read_sql(text("SELECT * FROM #af5_delta"), work_conn)
_log("af5_delta", af5_delta)


In [ ]:
# APPEND server-side de AF5_DELTA hacia TABLAS.BD_CTSI (acumula, igual que el SAS original)
cols_af5_delta = (
    "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
)
sql_append_delta = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_af5_delta})
SELECT {cols_af5_delta}
FROM #af5_delta
"""
res = work_conn.execute(text(sql_append_delta))
_log("APPEND TABLAS.dbo.BD_CTSI (AF5_DELTA)", res.rowcount)

for t in ["#af5_pasivo", "#af5_activo", "#af5_pasivo_full", "#af5_delta"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## S2_10_CCom

Cierre e imputación de créditos comerciales (AF.7/AF.71/AF.9): reasigna contrapartes, imputa activos/pasivos faltantes entre sectores usando reglas de espejo y porcentajes de conciliación, y acumula saldos de cuenta financiera en la base consolidada

*confianza: medium · verificador: approve · SAS: PROC SQL: UPDATE condicional + 11 bloques CREATE TABLE con CASE/agregación e INPUT/PUT, seguidos de PROC APPEND FORCE hacia TABLAS.BD_CTSI*

In [ ]:
# ========= S2_10_CCom =========
# COMPRIME TABLA PRINCIPAL (COMPRESS=YES es una opción de almacenamiento SAS sin equivalente en SQL Server; no aplica)
# No hay operación T-SQL equivalente a recompactar la tabla: se omite, no cambia los datos.


In [ ]:
# CIERRE 2021: AJUSTES DE CONTRAGENTE EN EMPRESAS Y HOLDINGS
sql_update_cagente = """
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '53', PROC = '0600a'
WHERE SECTOR IN (51021, 5101, 37) AND C_SCN = 'AF.7' AND (C_CAGENTE = '' OR C_CAGENTE = '71')
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_update_cagente))
    _log("UPDATE TABLAS.dbo.BD_CTSI (cierre 2021 contragente)", res.rowcount)


In [ ]:
# IMPUTA ACTIVO DE CRÉDITOS COMERCIALES EN SECTOR HOGARES CON CA 51, USANDO INFO DEL PASIVO DEL SECTOR 51 CON CA 511 (NEGATIVO DE ESTO)
work_conn.execute(text("DROP TABLE IF EXISTS #af7_511"))
sql_af7_511 = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       511 AS SECTOR,
       '51' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(t1.DATO * -1) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0602' AS PROC
INTO #af7_511
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION t2 ON t1.SECTOR = t2.C_SI
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS t3 ON t1.C_CAGENTE = t3.C_CAGENTE
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND t2.C_SI_SCN_N1 = 'S.11' AND t3.C_SI_SCN_N0 = 'S.14'
GROUP BY t1.AÑO, t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.C_ENTRADA
"""
work_conn.execute(text(sql_af7_511))
af7_511 = pd.read_sql(text("SELECT * FROM #af7_511"), work_conn)
_log("af7_511", af7_511)


In [ ]:
# ELIMINA PASIVO DE CREDITOS COMERCIALES EN SECTOR 51 CON CA 511 (USA CONSULTA ANTERIOR)
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51"))
sql_af7_51 = """
SELECT MONEDA, AÑO, TRIM,
       51 AS SECTOR,
       '511' AS C_CAGENTE,
       C_CUENTA,
       'H' AS C_ENTRADA,
       DATO,
       C_SCN, N_SCN, FUENTE,
       '0603' AS PROC
INTO #af7_51
FROM #af7_511
"""
work_conn.execute(text(sql_af7_51))
af7_51 = pd.read_sql(text("SELECT * FROM #af7_51"), work_conn)
_log("af7_51", af7_51)


In [ ]:
# IMPUTA INFO DE ACTIVO DE CREDITOS COMERCIALES EN SECTOR 51 CON CA 53, USANDO EL NEGATIVO DEL PASIVO DEL SECTOR 51 CON CA 511
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51_53"))
sql_af7_51_53 = """
SELECT MONEDA, AÑO, TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       DATO,
       C_SCN, N_SCN, FUENTE,
       '0604' AS PROC
INTO #af7_51_53
FROM #af7_511
"""
work_conn.execute(text(sql_af7_51_53))
af7_51_53 = pd.read_sql(text("SELECT * FROM #af7_51_53"), work_conn)
_log("af7_51_53", af7_51_53)


In [ ]:
# PROC DATASETS APPEND FORCE (x3) hacia TABLAS.BD_CTSI, server-side desde las #tmp
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp in ["#af7_511", "#af7_51", "#af7_51_53"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi} FROM {tmp}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp}", res.rowcount)


In [ ]:
# DROP TABLE AF7_511, AF7_51, AF7_51_53 (limpieza de temporales de sesión)
for t in ["#af7_511", "#af7_51", "#af7_51_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA CRED COMERCIALES EN ACTIVO DEL SECTOR 352 CON CA 6, USANDO INFO DEL PASIVO DEL SECTOR 6 CON CA 352
# Corrección: C_CAGENTE='352' es constante fijada por el WHERE; se castea a INT (no float) para que SECTOR quede como 352 entero y no se propague 352.0 a las tablas derivadas
work_conn.execute(text("DROP TABLE IF EXISTS #af7_352_6"))
sql_af7_352_6 = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       CAST(t1.C_CAGENTE AS int) AS SECTOR,
       CAST(t1.SECTOR AS varchar(11)) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0614' AS PROC
INTO #af7_352_6
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND t1.C_CAGENTE = '352' AND t1.SECTOR = 6
GROUP BY t1.AÑO, t1.TRIM, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.C_ENTRADA, t1.C_CAGENTE, t1.SECTOR
"""
work_conn.execute(text(sql_af7_352_6))
af7_352_6 = pd.read_sql(text("SELECT * FROM #af7_352_6"), work_conn)
_log("af7_352_6", af7_352_6)


In [ ]:
# IMPUTA CRED COMERCIALES EN PASIVO DE SECTOR 352 CON CA 511 USANDO 55% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_352_511"))
sql_af7_352_511 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '511' AS C_CAGENTE,
       C_CUENTA,
       'H' AS C_ENTRADA,
       DATO * 0.55 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0615' AS PROC
INTO #af7_352_511
FROM #af7_352_6
"""
work_conn.execute(text(sql_af7_352_511))
af7_352_511 = pd.read_sql(text("SELECT * FROM #af7_352_511"), work_conn)
_log("af7_352_511", af7_352_511)


In [ ]:
# IMPUTA CRED COMERCIALES EN PASIVO DE SECTOR 352 CON CA 51 USANDO 45% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_352_51"))
sql_af7_352_51 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '51' AS C_CAGENTE,
       C_CUENTA,
       'H' AS C_ENTRADA,
       DATO * 0.45 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0616' AS PROC
INTO #af7_352_51
FROM #af7_352_6
"""
work_conn.execute(text(sql_af7_352_51))
af7_352_51 = pd.read_sql(text("SELECT * FROM #af7_352_51"), work_conn)
_log("af7_352_51", af7_352_51)


In [ ]:
# IMPUTA CRED COMERCIALES EN ACTIVO DE SECTOR 511 CON CA 352, USANDO 55% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_511_352"))
sql_af7_511_352 = """
SELECT MONEDA, AÑO, TRIM,
       511 AS SECTOR,
       '352' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       DATO,
       C_SCN, N_SCN, FUENTE,
       '0616' AS PROC
INTO #af7_511_352
FROM #af7_352_511
"""
work_conn.execute(text(sql_af7_511_352))
af7_511_352 = pd.read_sql(text("SELECT * FROM #af7_511_352"), work_conn)
_log("af7_511_352", af7_511_352)


In [ ]:
# IMPUTA CRED COMERCIALES EN ACTIVO DE SECTOR 51 CON CA 352, USANDO 45% DEL PASIVO DE SECTOR 6 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #af7_51_352"))
sql_af7_51_352 = """
SELECT MONEDA, AÑO, TRIM,
       51 AS SECTOR,
       '352' AS C_CAGENTE,
       C_CUENTA,
       'D' AS C_ENTRADA,
       DATO,
       C_SCN, N_SCN, FUENTE,
       '0617' AS PROC
INTO #af7_51_352
FROM #af7_352_51
"""
work_conn.execute(text(sql_af7_51_352))
af7_51_352 = pd.read_sql(text("SELECT * FROM #af7_51_352"), work_conn)
_log("af7_51_352", af7_51_352)


In [ ]:
# PROC DATASETS APPEND FORCE (x5) hacia TABLAS.BD_CTSI
for tmp in ["#af7_352_6", "#af7_352_511", "#af7_352_51", "#af7_511_352", "#af7_51_352"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi} FROM {tmp}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp}", res.rowcount)


In [ ]:
# DROP TABLE de temporales AF7_352_6, AF7_352_511, AF7_352_51, AF7_511_352, AF7_51_352
for t in ["#af7_352_6", "#af7_352_511", "#af7_352_51", "#af7_511_352", "#af7_51_352"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DEL RESTO DEL MUNDO CON CA BCO CENTRAL Y GOBIERNO EN EL ACTIVO DE LOS SECTORES.
# REALIZA CONTRA AJUSTE EN CA 53, EXCEPTO EN BCENTRAL QUE AJUSTA EN AF.71 CA RM
work_conn.execute(text("DROP TABLE IF EXISTS #af7_rm_p"))
sql_af7_rm_p = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       CASE WHEN t1.SECTOR = 6 THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END AS SECTOR,
       '6' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0618' AS PROC
INTO #af7_rm_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.7' AND t1.SECTOR IN (31, 41) AND t1.C_CAGENTE = '6')
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND t1.SECTOR = 6 AND t1.C_CAGENTE IN ('31', '41'))
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.SECTOR = 6 THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af7_rm_p))
af7_rm_p = pd.read_sql(text("SELECT * FROM #af7_rm_p"), work_conn)
_log("af7_rm_p", af7_rm_p)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 EN GOBIERNO Y EN AF71 CA RM EN BCENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #af7_53_6"))
sql_af7_53_6 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       CASE WHEN SECTOR = 31 THEN '6' ELSE '53' END AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       DATO * -1 AS DATO,
       CASE WHEN SECTOR = 31 THEN 'AF.71' ELSE C_SCN END AS C_SCN,
       CASE WHEN SECTOR = 31 THEN 'Ajuste conciliación' ELSE N_SCN END AS N_SCN,
       FUENTE,
       '0618b' AS PROC
INTO #af7_53_6
FROM #af7_rm_p
"""
work_conn.execute(text(sql_af7_53_6))
af7_53_6 = pd.read_sql(text("SELECT * FROM #af7_53_6"), work_conn)
_log("af7_53_6", af7_53_6)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_rm_p"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_rm_p", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_53_6"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_53_6", res.rowcount)
for t in ["#af7_rm_p", "#af7_53_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES EN ACTIVO DE LOS BANCOS, EXCEPTO CRUCE BANCOS CON BANCOS, CA 41 Y 53.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_bcos_a"))
sql_af7_bcos_a = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' AND t1.C_CAGENTE <> '3' THEN CAST(t1.C_CAGENTE AS int)
            WHEN t1.C_ENTRADA = 'H' AND t1.C_CAGENTE = '3' THEN 32
            ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0619' AS PROC
INTO #af7_bcos_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.7' AND t1.SECTOR IN (321, 322, 32) AND t1.C_CAGENTE NOT IN ('321', '322', '32', '41', '53', '9'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND t1.SECTOR NOT IN (321, 41) AND t1.C_CAGENTE IN ('321', '322', '32', '3'))
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' AND t1.C_CAGENTE <> '3' THEN CAST(t1.C_CAGENTE AS int)
              WHEN t1.C_ENTRADA = 'H' AND t1.C_CAGENTE = '3' THEN 32
              ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af7_bcos_a))
af7_bcos_a = pd.read_sql(text("SELECT * FROM #af7_bcos_a"), work_conn)
_log("af7_bcos_a", af7_bcos_a)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_321_53"))
sql_af7_321_53 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * -1 AS DATO,
       C_SCN, N_SCN, FUENTE,
       '0619b' AS PROC
INTO #af7_321_53
FROM #af7_bcos_a
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_af7_321_53))
af7_321_53 = pd.read_sql(text("SELECT * FROM #af7_321_53"), work_conn)
_log("af7_321_53", af7_321_53)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_bcos_a"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_bcos_a", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_321_53"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_321_53", res.rowcount)
for t in ["#af7_bcos_a", "#af7_321_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES AUXILIARES Y OFIS EN ACTIVO DE LOS AUXILIARES FINANCIEROS.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_aux_a"))
sql_af7_aux_a = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0620' AS PROC
INTO #af7_aux_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.7' AND SUBSTRING(CAST(t1.SECTOR AS varchar(7)), 1, 2) = '36' AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('36', '33', '37'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND SUBSTRING(CAST(t1.SECTOR AS varchar(7)), 1, 2) IN ('36', '33', '37') AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('36', '37'))
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af7_aux_a))
af7_aux_a = pd.read_sql(text("SELECT * FROM #af7_aux_a"), work_conn)
_log("af7_aux_a", af7_aux_a)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR AUXILIARES
work_conn.execute(text("DROP TABLE IF EXISTS #af7_36_53"))
sql_af7_36_53 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * -1 AS DATO,
       C_SCN, N_SCN, FUENTE,
       '0620b' AS PROC
INTO #af7_36_53
FROM #af7_aux_a
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_af7_36_53))
af7_36_53 = pd.read_sql(text("SELECT * FROM #af7_36_53"), work_conn)
_log("af7_36_53", af7_36_53)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_aux_a"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_aux_a", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_36_53"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_36_53", res.rowcount)
for t in ["#af7_aux_a", "#af7_36_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES AUXILIARES, OFIS Y SEGUROS EN ACTIVO DE LOS SEGUROS.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_seg_a"))
sql_af7_seg_a = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0621' AS PROC
INTO #af7_seg_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.7' AND SUBSTRING(CAST(t1.SECTOR AS varchar(7)), 1, 2) = '35' AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('36', '33', '37', '35'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND SUBSTRING(CAST(t1.SECTOR AS varchar(7)), 1, 2) IN ('36', '33', '37', '35') AND SUBSTRING(t1.C_CAGENTE, 1, 2) = '35' AND t1.C_CAGENTE <> '351_34')
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af7_seg_a))
af7_seg_a = pd.read_sql(text("SELECT * FROM #af7_seg_a"), work_conn)
_log("af7_seg_a", af7_seg_a)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #af7_35_53"))
sql_af7_35_53 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * -1 AS DATO,
       C_SCN, N_SCN, FUENTE,
       '0621b' AS PROC
INTO #af7_35_53
FROM #af7_seg_a
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_af7_35_53))
af7_35_53 = pd.read_sql(text("SELECT * FROM #af7_35_53"), work_conn)
_log("af7_35_53", af7_35_53)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_seg_a"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_seg_a", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_35_53"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_35_53", res.rowcount)
for t in ["#af7_seg_a", "#af7_35_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES BCENTRAL, AUXILIARES, OFIS Y SEGUROS EN ACTIVO DE GOBIERNO.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_gob_a"))
sql_af7_gob_a = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0622' AS PROC
INTO #af7_gob_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.7' AND t1.SECTOR IN (4, 41, 42, 412) AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('36', '33', '37', '35', '31'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND SUBSTRING(CAST(t1.SECTOR AS varchar(7)), 1, 2) IN ('36', '33', '37', '35', '31') AND SUBSTRING(t1.C_CAGENTE, 1, 1) = '4' AND t1.C_CAGENTE <> '411')
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af7_gob_a))
af7_gob_a = pd.read_sql(text("SELECT * FROM #af7_gob_a"), work_conn)
_log("af7_gob_a", af7_gob_a)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR GOBIERNO
work_conn.execute(text("DROP TABLE IF EXISTS #af7_4_53"))
sql_af7_4_53 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * -1 AS DATO,
       C_SCN, N_SCN, FUENTE,
       '0622b' AS PROC
INTO #af7_4_53
FROM #af7_gob_a
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_af7_4_53))
af7_4_53 = pd.read_sql(text("SELECT * FROM #af7_4_53"), work_conn)
_log("af7_4_53", af7_4_53)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_gob_a"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_gob_a", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_4_53"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_4_53", res.rowcount)
for t in ["#af7_gob_a", "#af7_4_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA PASIVO DE LOS SECTORES AUXILIARES Y SEGUROS EN ACTIVO DE RESTO DEL MUNDO.
# REALIZA CONTRA AJUSTE EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #af7_rm_a"))
sql_af7_rm_a = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0623' AS PROC
INTO #af7_rm_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.7' AND t1.SECTOR = 6 AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('36', '37', '35'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7' AND SUBSTRING(CAST(t1.SECTOR AS varchar(7)), 1, 2) IN ('36', '37', '35') AND t1.C_CAGENTE = '6')
GROUP BY t1.AÑO, t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.C_CAGENTE AS int) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN CAST(t1.SECTOR AS varchar(7)) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_af7_rm_a))
af7_rm_a = pd.read_sql(text("SELECT * FROM #af7_rm_a"), work_conn)
_log("af7_rm_a", af7_rm_a)


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN AF7 CA 53 DEL SECTOR RM
work_conn.execute(text("DROP TABLE IF EXISTS #af7_6_53"))
sql_af7_6_53 = """
SELECT MONEDA, AÑO, TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) * -1 AS DATO,
       C_SCN, N_SCN, FUENTE,
       '0623b' AS PROC
INTO #af7_6_53
FROM #af7_rm_a
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_af7_6_53))
af7_6_53 = pd.read_sql(text("SELECT * FROM #af7_6_53"), work_conn)
_log("af7_6_53", af7_6_53)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_rm_a"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_rm_a", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_6_53"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_6_53", res.rowcount)
for t in ["#af7_rm_a", "#af7_6_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRA CREDITOS COMERCIALES IMPUTANDO DIFERENCIA ENTRE PASIVO Y ACTIVO AL SECTOR 51022 CON CA CORRESPONDIENTE
# CALCULA TOTAL PASIVO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af7_pasivo"))
sql_af7_pasivo = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       51022 AS SECTOR,
       CASE WHEN t1.SECTOR = 51022 THEN '53' ELSE CAST(t2.C_SI_publ AS varchar(4)) END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0648' AS PROC
INTO #af7_pasivo
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION t2 ON t1.SECTOR = t2.C_SI
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'AF.7'
GROUP BY t1.AÑO, t1.TRIM, t1.C_ENTRADA, t1.C_CUENTA, t1.C_SCN, t1.N_SCN,
         CASE WHEN t1.SECTOR = 51022 THEN '53' ELSE CAST(t2.C_SI_publ AS varchar(4)) END
"""
work_conn.execute(text(sql_af7_pasivo))
af7_pasivo = pd.read_sql(text("SELECT * FROM #af7_pasivo"), work_conn)
_log("af7_pasivo", af7_pasivo)


In [ ]:
# CALCULA TOTAL ACTIVO POR CADA SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #af7_activo"))
sql_af7_activo = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       51022 AS SECTOR,
       CAST(t2.C_SI_publ AS varchar(4)) AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       SUM(t1.DATO) * -1 AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0648' AS PROC
INTO #af7_activo
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_CONTRAPARTIDAS t2 ON t1.C_CAGENTE = t2.C_CAGENTE
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'AF.7'
GROUP BY t1.AÑO, t1.TRIM, t1.C_ENTRADA, t1.C_CUENTA, t1.C_SCN, t1.N_SCN,
         CAST(t2.C_SI_publ AS varchar(4))
"""
work_conn.execute(text(sql_af7_activo))
af7_activo = pd.read_sql(text("SELECT * FROM #af7_activo"), work_conn)
_log("af7_activo", af7_activo)


In [ ]:
# DATA AF7_PASIVO; SET AF7_PASIVO AF7_ACTIVO; -- concatena pasivo y activo en la misma #tmp de sesión
work_conn.execute(text(f"INSERT INTO #af7_pasivo ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_activo"))


In [ ]:
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af7_delta"))
sql_af7_delta = f"""
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO,
       C_SCN, N_SCN, FUENTE, PROC
INTO #af7_delta
FROM #af7_pasivo
GROUP BY MONEDA, AÑO, TRIM, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, SECTOR, C_CAGENTE, FUENTE, PROC
"""
work_conn.execute(text(sql_af7_delta))
af7_delta = pd.read_sql(text("SELECT * FROM #af7_delta"), work_conn)
_log("af7_delta", af7_delta)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af7_delta"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af7_delta", res.rowcount)
for t in ["#af7_delta", "#af7_pasivo", "#af7_activo"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# GENERA ACTIVO NETO DE ERRORES Y OMISIONES Y DISCREPANCIA ESTADÍSTICAS, ELIMINANDO EL PASIVO E IMPUTANDOLO CON SIGNO CONTRARIO EN EL ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #af71_9_elimina"))
sql_af71_9_elimina = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO * -1) AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE,
       '0654' AS PROC
INTO #af71_9_elimina
FROM TABLAS.dbo.BD_CTSI t1
WHERE C_ENTRADA = 'H' AND C_SCN IN ('AF.71', 'AF.9')
GROUP BY MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_af71_9_elimina))
af71_9_elimina = pd.read_sql(text("SELECT * FROM #af71_9_elimina"), work_conn)
_log("af71_9_elimina", af71_9_elimina)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_9_netea"))
sql_af71_9_netea = """
SELECT MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA,
       'D' AS C_ENTRADA,
       DATO, C_SCN, N_SCN, FUENTE,
       '0655' AS PROC
INTO #af71_9_netea
FROM #af71_9_elimina
"""
work_conn.execute(text(sql_af71_9_netea))
af71_9_netea = pd.read_sql(text("SELECT * FROM #af71_9_netea"), work_conn)
_log("af71_9_netea", af71_9_netea)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_9_elimina"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af71_9_elimina", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_9_netea"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af71_9_netea", res.rowcount)
for t in ["#af71_9_elimina", "#af71_9_netea"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA ACTIVO DE AF71 EN SECTOR 51 CON CA 53, USANDO TOTAL ACTIVO DE AF71 MULTIPLICADO POR AJUSTE DE CONCILIACIÓN
work_conn.execute(text("DROP TABLE IF EXISTS #af71_51"))
sql_af71_51 = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       SUM(t1.DATO * t2.Porcentaje * -1) AS DATO,
       t1.C_SCN,
       CASE WHEN t1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE t1.N_SCN END AS N_SCN,
       'PS' AS FUENTE,
       '0656' AS PROC
INTO #af71_51
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_AJUSTE_CONCILIACION t2 ON t1.AÑO = t2.Año AND t1.TRIM = t2.Trim
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN IN ('AF.71', 'AF.9') AND t1.DATO <> 0
GROUP BY t1.AÑO, t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN,
         CASE WHEN t1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE t1.N_SCN END
"""
work_conn.execute(text(sql_af71_51))
af71_51 = pd.read_sql(text("SELECT * FROM #af71_51"), work_conn)
_log("af71_51", af71_51)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af71_511"))
sql_af71_511 = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       511 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       t1.C_ENTRADA,
       SUM(t1.DATO * (1 - t2.Porcentaje) * -1) AS DATO,
       t1.C_SCN,
       CASE WHEN t1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE t1.N_SCN END AS N_SCN,
       'PS' AS FUENTE,
       '0657' AS PROC
INTO #af71_511
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_AJUSTE_CONCILIACION t2 ON t1.AÑO = t2.Año AND t1.TRIM = t2.Trim
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN IN ('AF.71', 'AF.9') AND t1.DATO <> 0
GROUP BY t1.AÑO, t1.TRIM, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN,
         CASE WHEN t1.C_SCN = 'AF.71' THEN 'Ajuste conciliación' ELSE t1.N_SCN END
"""
work_conn.execute(text(sql_af71_511))
af71_511 = pd.read_sql(text("SELECT * FROM #af71_511"), work_conn)
_log("af71_511", af71_511)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_51"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af71_51", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #af71_511"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #af71_511", res.rowcount)
for t in ["#af71_51", "#af71_511"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA SALDOS DE CTA FINANCIERA
work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))
sql_saldos = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.SECTOR,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'H' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       CASE WHEN t1.C_CUENTA = 'Financiera' THEN 'B.9'
            WHEN t1.C_CUENTA LIKE '%Bce%' THEN 'B.90'
            WHEN t1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3'
            WHEN t1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2' END AS C_SCN,
       CASE WHEN t1.C_CUENTA = 'Financiera' THEN 'Capacidad/Necesidad de financiamiento'
            WHEN t1.C_CUENTA LIKE '%Bce%' THEN 'Valor neto'
            WHEN t1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales'
            WHEN t1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen' END AS N_SCN,
       'PS' AS FUENTE,
       '0009' AS PROC
INTO #saldos
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_CUENTA IN ('Financiera', 'Bce Final', 'Bce Inicio', 'Rec Volumen', 'Rec Precio', 'Rec Precio Reaj')
GROUP BY t1.AÑO, t1.TRIM, t1.SECTOR, t1.C_CUENTA
"""
work_conn.execute(text(sql_saldos))
saldos = pd.read_sql(text("SELECT * FROM #saldos"), work_conn)
_log("saldos", saldos)


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #saldos"))
_log("APPEND TABLAS.dbo.BD_CTSI desde #saldos", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))


## S2_11_CNF

Reclasifica y reimputa transacciones de renta de la propiedad (intereses D.41, dividendos D.42, renta de fondos D.443/D.44) entre sectores institucionales según reglas de cierre por año (2019/2021), incluyendo apertura de dividendos con el resto del mundo por estructura del AF.5 y ajuste de cuotas de fondos en el exterior / Imputa apertura de dividendos pagados/recibidos (D.42) y utilidades reinvertidas (D.43, D.45) entre bancos, resto del mundo, ofis, seguros y auxiliares, ajustando siempre la contrapartida 53 para conciliar debe y haber, y acumula cada imputación en TABLAS.BD_CTSI / Genera y acumula ajustes de imputacion sobre la base de cuentas nacionales (D.5, D.61, D.62, D.8, D.71, D.72, D.75, D.41) por sector y contraparte, y actualiza contrapartidas de intereses en cierres de anios especificos, incluyendo el prorrateo de intereses del Banco Central segun estructura de activos que devengan interes / Ajusta imputaciones de SIFMI e intereses (D.41) entre bancos, resto del mundo, OFIS y auxiliares (sector 37), abriendo contrapartidas por estructura de balances y tasas implícitas, y acumula los ajustes en TABLAS.BD_CTSI / Imputa y cierra las cuentas de producción, valor agregado, excedente de explotación y ahorro por sector institucional, incluyendo transferencias de capital y ajustes SIFMI, acumulando cada ajuste en la tabla consolidada BD_CTSI / Reclasifica y ajusta cuentas de capital, financiera y resto del mundo (B.8/B.9/AF.71/D.42/D.75) por sector institucional para conciliar ahorro, préstamo neto y CCF, imputando diferencias entre sectores según tablas de sectorización / Ajusta y reclasifica en BD_CTSI las cuentas de recursos de volumen, financiera y capital para bancos/hogares/SNF, elimina saldos de balance y ceros redundantes, calcula variaciones del valor neto por volumen y precio, e imputa ajustes de conciliación cruzados entre sectores (empresas, hogares, bancos)

*confianza: low · SAS: Secuencia de UPDATE/CREATE TABLE/APPEND (PROC SQL + PROC DATASETS) sobre TABLAS.BD_CTSI vía tablas temporales de sesión + PROC SQL (CREATE TABLE con JOIN/GROUP BY sobre #tmp y TABLAS.BD_CTSI) + PROC DATASETS APPEND + DROP TABLE, todo server-side + PROC SQL CREATE TABLE (agregaciones sobre TABLAS.BD_CTSI) + PROC DATASETS APPEND + PROC SQL UPDATE, encadenados en tablas temporales de sesion + PROC SQL CREATE TABLE / UPDATE / PROC DATASETS APPEND encadenados sobre tablas temporales de sesión, con dos lecturas de archivos SAS externos (tasas y DCV) cruzadas contra TABLAS.BD_CTSI + PROC SQL CREATE TABLE + PROC DATASETS APPEND FORCE + UPDATE, cadena de imputaciones sobre TABLAS.BD_CTSI (tramo 6/8) + PROC SQL CREATE TABLE (SELECT INTO #tmp) + PROC DATASETS APPEND FORCE + PROC SQL DROP TABLE, encadenados sobre TABLAS.BD_CTSI + PROC SQL CREATE TABLE + PROC DATASETS APPEND FORCE + DELETE, encadenados sobre tablas temporales de sesión contra TABLAS.BD_CTSI*

In [ ]:
# ========= S2_11_CNF =========
# COMPRIME TABLA PRINCIPAL
# SAS: DATA tablas.BD_CTSI (COMPRESS=YES); SET tablas.BD_CTSI; RUN;
# COMPRESS=YES es una opción de almacenamiento del dataset SAS: reescribe la
# tabla con exactamente las mismas filas para guardarlas comprimidas. En SQL
# Server la compresión es DDL (ALTER TABLE ... REBUILD WITH DATA_COMPRESSION),
# se decide una vez con el DBA y no desde el pipeline: el paso no tiene efecto
# de datos que traducir. Queda registrado y sin ejecutar — replicarlo con un
# borrado + reinserción deja TABLAS.BD_CTSI vacía si la celda se corta a la mitad.


In [ ]:
# EN SECTOR SEG DE VIDA CAMBIA PRODUCCION A INTERESES RECIBIDOS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.41', N_SCN = 'Intereses', PROC = '0680'
WHERE C_SCN = 'P.11' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 351
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0680", res.rowcount)


In [ ]:
# EN SECTORES 34,341,351 CAMBIA INTERESES Y DIVIDENDOS PAGADOS A D.44
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.44', N_SCN = 'Renta de la propiedad atribuida a los titulares de pólizas de seguros', PROC = '0681'
WHERE C_SCN IN ('D.41','D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR = 351 AND (C_CAGENTE IS NULL OR C_CAGENTE = '')
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0681 (sector 351)", res.rowcount)


In [ ]:
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.44', N_SCN = 'Renta de la propiedad atribuida a los titulares de pólizas de seguros', PROC = '0681'
WHERE C_SCN IN ('D.41','D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR IN (34,341) AND C_CAGENTE = '511'
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0681 (sector 34,341)", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA CONTRAGENTE EN FONDOS DE INVERSION EN INTERESES RECIBIDOS SIN CONTRAGENTE
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '53', PROC = '0681b'
WHERE C_SCN IN ('D.41','D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 339011 AND (C_CAGENTE IS NULL OR C_CAGENTE = '')
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0681b", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA INSTRUMENTO EN FONDOS DE INVERSION EN DIVIDENDOS RECIBIDOS DESDE CA 339011 PORQUE DEBE SER D.443
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', PROC = '0681c'
WHERE C_SCN = 'D.42' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 339011 AND C_CAGENTE = '339011'
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0681c", res.rowcount)


In [ ]:
# CIERRE 2021: CAMBIA CONTRAGENTE EN MUTUALIDADES DEL INST D443
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '3390101', PROC = '0681b'
WHERE C_SCN IN ('D.443') AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND SECTOR = 412 AND (C_CAGENTE IS NULL OR C_CAGENTE = '')
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0681b (sector 412)", res.rowcount)


In [ ]:
# CIERRE 2021: IMPUTA D443 EN GASTO DEL RESTO DEL MUNDO USANDO DATO INGRESO DE FONDOS DE PENSIONES CON RM. AJUSTE EN DIVIDENDOS DEL RESTO DEL MUNDO
work_conn.execute(text("DROP TABLE IF EXISTS #d443_rm_fp"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       6 AS SECTOR,
       '34' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0783a' AS PROC
INTO #d443_rm_fp
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_CUENTA IN ('YG') AND T1.SECTOR IN (34,341) AND T1.C_ENTRADA='H' AND T1.C_SCN='D.443' AND T1.C_CAGENTE='6'
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_53"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'D.42' AS C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #d42_rm_53
FROM #d443_rm_fp t1
"""))


In [ ]:
# PROC DATASETS APPEND FORCE: alinea por nombre de columna
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_rm_fp"))
_log("APPEND TABLAS.BD_CTSI desde D443_RM_FP", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_rm_53"))
_log("APPEND TABLAS.BD_CTSI desde D42_RM_53", res.rowcount)
for t in ["#d443_rm_fp", "#d42_rm_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ELIMINA TRANSACCIONES DE LA CTA DE PRODUCCION, YG Y CAPITAL DE SECTORES 412 Y YG/CAPITAL EN 411.
# SOLO LO HACE EN SECTOR 412...NO EN 411 PORQUE ESO TENIA SENTIDO SOLO CUANDO LAS CAJAS SE INCORPORABAN EN GOBIERNO
work_conn.execute(text("DROP TABLE IF EXISTS #drop_412_411"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0783' AS PROC
INTO #drop_412_411
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA IN ('YG','Capital','Producción') AND T1.SECTOR = 412)
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #drop_412_411"))
_log("APPEND TABLAS.BD_CTSI desde DROP_412_411", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #drop_412_411"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cambia_d443_412"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       51022 AS SECTOR, 'Otras sociedades' AS N_SI, '51' AS C_SI_publ, 'Sociedades no financieras' AS N_SI_publ, '3390101' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0783b' AS PROC
INTO #cambia_d443_412
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA IN ('YG') AND T1.SECTOR = 412 AND T1.C_SCN = 'D.443' AND T1.PROC = '0783')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #cambia_d443_412"))
_log("APPEND TABLAS.BD_CTSI desde CAMBIA_D443_412", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #cambia_d443_412"))


In [ ]:
# CONCILIA TOTAL DEBE Y HABER DEL D44. MANDA EL HABER, DIF LA IMPUTA AL DEBE DEL SECTOR 35 CON CA 511.
# POR CIERRE 2019 SE AJUSTA EL DATO AL HABER, YA QUE FALTA INCORPORAR EN HOGARES LOS REAJUSTES DEL D44 DE LOS FONDOS DE PENSIONES
# PARA QUE HOGARES SE IGUALE A ESE DATO FINAL DE LOS FP
work_conn.execute(text("DROP TABLE IF EXISTS #d44_cierre"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       511 AS SECTOR,
       (CASE WHEN T1.SECTOR = 511 THEN T1.C_CAGENTE ELSE LTRIM(CAST(T1.SECTOR AS varchar(11))) END) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta de la propiedad atribuida a los titulares de pólizas de seguros' AS N_SCN,
       'PS' AS FUENTE,
       '0684' AS PROC
INTO #d44_cierre
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.44') OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.44')
GROUP BY T1.AÑO, T1.TRIM,
         (CASE WHEN T1.SECTOR = 511 THEN T1.C_CAGENTE ELSE LTRIM(CAST(T1.SECTOR AS varchar(11))) END),
         T1.C_CUENTA, T1.C_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d44_cierre"))
_log("APPEND TABLAS.BD_CTSI desde D44_CIERRE", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d44_cierre"))


In [ ]:
# ELIMINA INTERESES RECIBIDOS Y PAGADOS EN EL SECTOR 5111. POR CIERRE ELIMINA TMB D75,D5
work_conn.execute(text("DROP TABLE IF EXISTS #d41_5111"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0700' AS PROC
INTO #d41_5111
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.SECTOR = 5111 AND T1.C_SCN IN ('D.41','D.75','D.5'))
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d41_5111"))
_log("APPEND TABLAS.BD_CTSI desde D41_5111", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_5111"))


In [ ]:
# EN SECTORES 339011,3390101 y 3390102 CAMBIA INTERESES PAGADOS A D.443.
# CIERRE 2021: EN FONDOS DE INVERSIÓN NO SE HACE EL CAMBIO EN LOS INTERESES PORQ DEBEN PAGAR POR PTMOS CON BANCOS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', PROC = '0702'
WHERE C_SCN IN ('D.41') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR IN (3390101,3390102)
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0702 (D.41)", res.rowcount)


In [ ]:
# EN SECTORES 339011 CAMBIA DIVIDENDOS PAGADOS A D.443. CIERRE 2019
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', C_CAGENTE = '53', PROC = '0702'
WHERE C_SCN IN ('D.42') AND C_CUENTA = 'YG' AND C_ENTRADA = 'D' AND SECTOR IN (339011)
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0702 (D.42)", res.rowcount)


In [ ]:
# EN SECTORES QUE RECIBEN INTERESES DE FONDOS DE INV Y FONDOS MUTUOS CAMBIA INTERESES A D.443
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'D.443', N_SCN = 'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva', PROC = '0702'
WHERE C_SCN IN ('D.41') AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('339011','3390101','3390102','33901','339')
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0702 (D.41 recibido)", res.rowcount)


In [ ]:
# ACTUALIZA CONTRAGENTES EN SECTORES CON RENTAS RECIBIDAS DESDE LOS FONDOS PARA DEJARLO CONSISTENTE CON SU ACTIVO. CIERRE 2019
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '3390101', PROC = '0702b'
WHERE C_SCN = 'D.443' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('33901','339') AND SECTOR IN (321,3390101,33221,33222)
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0702b (3390101)", res.rowcount)
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI
SET C_CAGENTE = '3390102', PROC = '0702b'
WHERE C_SCN = 'D.443' AND C_CUENTA = 'YG' AND C_ENTRADA = 'H' AND C_CAGENTE IN ('33901','339') AND SECTOR IN (341,351,3390102)
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0702b (3390102)", res.rowcount)


In [ ]:
# CIERRE 2021: CALCULA D443 DE LOS SECTORES CON CONTRAGENTE RM, PARA REBAJARLO DE D443 CON EL MERCADO NACIONAL,
# EN BASE A % DE CUOTAS DE FONDOS MANTENIDAS EN EL EXTERIOR SOBRE EL TOTAL DE CUOTAS APLICADO A D.443
# CUOTAS EN EL EXTERIOR CA RM DE LOS SECTORES, EXCEPTO PENSIONES QUE YA TIENE DATO Y GOBIERNO YA QUE TIENE POCO D443 RECIBIDO EN SU TOTAL
work_conn.execute(text("DROP TABLE IF EXISTS #cf_exterior"))
work_conn.execute(text("""
SELECT T1.AÑO, T1.TRIM,
       T1.SECTOR, T1.C_CAGENTE,
       SUM(T1.DATO) AS DATO
INTO #cf_exterior
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.C_CUENTA = 'Bce Final' AND T1.C_CAGENTE = '6'
      AND T1.SECTOR NOT IN (41,4,412,413,42,34,341)
GROUP BY T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE
"""))


In [ ]:
# CUOTAS TOTALES DE LOS SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #cf_total"))
work_conn.execute(text("""
SELECT T1.AÑO, T1.TRIM,
       T1.SECTOR,
       SUM(T1.DATO) AS DATO
INTO #cf_total
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('AF.521','AF.522') AND T1.C_CUENTA = 'Bce Final'
      AND T1.SECTOR NOT IN (41,4,412,413,42,6,34,341)
GROUP BY T1.AÑO, T1.TRIM, T1.SECTOR
"""))


In [ ]:
# PORCENTAJE PARA CREAR D443 CON EL RM
work_conn.execute(text("DROP TABLE IF EXISTS #porc_ext"))
work_conn.execute(text("""
SELECT t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.DATO / T2.DATO AS DATO
INTO #porc_ext
FROM #cf_exterior T1, #cf_total T2
WHERE T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
"""))


In [ ]:
# D443 INGRESO CONTRAGENTE RM
work_conn.execute(text("DROP TABLE IF EXISTS #d443_rm"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM, T1.SECTOR, '6' AS C_CAGENTE, 'YG' AS C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA, 'PS' AS FUENTE, '0702c' AS PROC,
       SUM(T1.DATO * T2.DATO) AS DATO
INTO #d443_rm
FROM TABLAS.dbo.BD_CTSI t1, #porc_ext T2
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('D.443') AND T1.C_CUENTA = 'YG' AND T1.SECTOR NOT IN (41,4,412,413,42,6,34,341)
      AND T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
GROUP BY t1.AÑO, T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA
"""))


In [ ]:
# D443 AJUSTE EN CONTRAGENTE 33901
work_conn.execute(text("DROP TABLE IF EXISTS #d443_fmnm"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, '33901' AS C_CAGENTE, 'YG' AS C_CUENTA, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA, T1.FUENTE, T1.PROC,
       SUM(T1.DATO) * -1 AS DATO
INTO #d443_fmnm
FROM #d443_rm T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN, T1.C_ENTRADA, T1.FUENTE, T1.PROC
"""))


In [ ]:
# D443 IMPUTA EN GASTO DEL RM CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d443_6"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, 6 AS SECTOR, '53' AS C_CAGENTE, 'YG' AS C_CUENTA, T1.C_SCN, T1.N_SCN, 'D' AS C_ENTRADA, T1.FUENTE, T1.PROC,
       SUM(T1.DATO) AS DATO
INTO #d443_6
FROM #d443_rm T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""))


In [ ]:
# AJUSTA EN GASTO D42 DEL RM CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_6"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM, 6 AS SECTOR, '53' AS C_CAGENTE, 'YG' AS C_CUENTA, 'D.42' AS C_SCN, 'Renta distribuida de las sociedades' AS N_SCN, 'D' AS C_ENTRADA, T1.FUENTE, T1.PROC,
       SUM(T1.DATO) * -1 AS DATO
INTO #d42_6
FROM #d443_rm T1
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.FUENTE, T1.PROC
"""))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_rm"))
_log("APPEND TABLAS.BD_CTSI desde D443_RM", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_fmnm"))
_log("APPEND TABLAS.BD_CTSI desde D443_FMNM", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_6"))
_log("APPEND TABLAS.BD_CTSI desde D443_6", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_6"))
_log("APPEND TABLAS.BD_CTSI desde D42_6", res.rowcount)
for t in ["#d443_fmnm", "#d443_rm", "#porc_ext", "#cf_exterior", "#cf_total", "#d443_6", "#d42_6"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CONCILIA TOTAL DEBE Y HABER DEL D443. MANDA EL TOTAL PAGADO (DEBE), DIF LA IMPUTA AL HABER (RECIBIDO) DEL SECTOR 51022 CON CA CORRESPONDIENTE
work_conn.execute(text("DROP TABLE IF EXISTS #d443_cierre"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       51022 AS SECTOR,
       (CASE WHEN T1.SECTOR = 3390101 AND T1.C_ENTRADA = 'D' THEN '3390101'
             WHEN T1.SECTOR IN (3390102,339011) AND T1.C_ENTRADA = 'D' THEN '33901'
             WHEN T1.C_CAGENTE NOT IN ('3390101') THEN '33901'
             ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta de la inversión atribuida a los accionistas de los fondos de inversión colectiva' AS N_SCN,
       'PS' AS FUENTE,
       '0703' AS PROC
INTO #d443_cierre
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.443' AND T1.SECTOR IN (3390102,339011,3390101,6)) OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.443')
GROUP BY T1.AÑO, T1.TRIM,
         (CASE WHEN T1.SECTOR = 3390101 AND T1.C_ENTRADA = 'D' THEN '3390101'
               WHEN T1.SECTOR IN (3390102,339011) AND T1.C_ENTRADA = 'D' THEN '33901'
               WHEN T1.C_CAGENTE NOT IN ('3390101') THEN '33901'
               ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, T1.C_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d443_cierre"))
_log("APPEND TABLAS.BD_CTSI desde D443_CIERRE", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d443_cierre"))


In [ ]:
# ELIMINA RENTAS DISTRIBUIDAS RECIBIDOS Y PAGADOS EN EL SECTOR 5111 Y 51022
work_conn.execute(text("DROP TABLE IF EXISTS #d42_5111"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0701' AS PROC
INTO #d42_5111
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.SECTOR = 5111 AND T1.C_SCN = 'D.42') OR (T1.SECTOR = 51022 AND T1.C_SCN = 'D.42' AND T1.MONEDA = 'P')
GROUP BY T1.MONEDA, T1.AÑO, T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_5111"))
_log("APPEND TABLAS.BD_CTSI desde D42_5111", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d42_5111"))


In [ ]:
# CIERRE 2019. CAMBIA CONTRAGENTE EN BANCOS DESDE AFP A FP EN DIVIDENDOS PAGADOS POR LOS BANCOS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '34', PROC = '0800'
WHERE SECTOR = 321 AND C_SCN = 'D.42' AND C_ENTRADA = 'D' AND C_CAGENTE = '361'
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0800 (321)", res.rowcount)


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR LOS SEGUROS CAMBIA CONTRAGENTE GOBIERNO A RESTO DE EMPRESAS
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0800'
WHERE SECTOR IN (351,352,353,35) AND C_SCN = 'D.42' AND C_ENTRADA = 'D' AND C_CAGENTE = '41'
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0800 (seguros)", res.rowcount)


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR LOS EMPRESAS PRIVADAS CAMBIA CONTRAGENTE HOGARES A RESTO DE EMPRESAS, PARA LUEGO RECALCULAR EL PAGADO A HOGARES
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0800'
WHERE SECTOR = 51021 AND C_SCN = 'D.42' AND C_ENTRADA = 'D' AND C_CAGENTE = '511'
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0800 (51021)", res.rowcount)


In [ ]:
# CIERRE 2019. IMPUTA DIVIDENDOS PAGADOS POR EMPRESAS SUPERVISADAS PRIVADAS A HOGARES UTILIZANDO MISMO CRITERIO QUE LA IMPUTACIÓN DEL AF5 DE EMPRESAS EN HOGARES
# O SEA, UN PORCENTAJE SOBRE EL TOTAL PAGADO POR EL SECTOR Y RESTADO DE LO PAGADO AL RESTO DE LA ECONOMÍA. IMPUTACIÓN CON CA HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #d42_51021"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       '511' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * 0.036281 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0801' AS PROC
INTO #d42_51021
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR = 51021 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'D'
GROUP BY T1.AÑO, T1.SECTOR, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))


In [ ]:
# IMPUTACIÓN CON CA HOGARES (contrapartida)
work_conn.execute(text("DROP TABLE IF EXISTS #d42_resto"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0801b' AS PROC
INTO #d42_resto
FROM #d42_51021 t1
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_51021"))
_log("APPEND TABLAS.BD_CTSI desde D42_51021", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_resto"))
_log("APPEND TABLAS.BD_CTSI desde D42_RESTO", res.rowcount)
for t in ["#d42_51021", "#d42_resto"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. CAMBIA CONTRAGENTE vacíos en todos los sectores a 53 en los dividendos
res = work_conn.execute(text("""
UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE = '53', PROC = '0802'
WHERE C_SCN = 'D.42' AND (C_CAGENTE IS NULL OR C_CAGENTE IN ('','512'))
"""))
_log("UPDATE TABLAS.BD_CTSI PROC=0802", res.rowcount)


In [ ]:
# CIERRE 2019. EN DIVIDENDOS RECIBIDOS POR EL RESTO DEL MUNDO REALIZA APERTURA POR CONTRAPARTIDA UTILIZANDO LA ESTRUCTURA DEL AF5 ACTIVO DEL SECTOR GENERADO EN LA SÍNTESIS
# BALANCE FINAL AF5 ACTIVO RESTO DEL MUNDO CON CA
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_a_ca"))
work_conn.execute(text("""
SELECT T1.AÑO, T1.TRIM,
       T1.SECTOR,
       (CASE WHEN T1.C_CAGENTE = '351/352' THEN '351' ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO
INTO #af5_rm_a_ca
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'AF.5' AND T1.C_ENTRADA = 'D' AND T1.C_CUENTA = 'Bce Final'
GROUP BY T1.AÑO, T1.SECTOR, T1.TRIM,
         (CASE WHEN T1.C_CAGENTE = '351/352' THEN '351' ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, T1.C_ENTRADA
"""))
af5_rm_a_ca = pd.read_sql(text("SELECT * FROM #af5_rm_a_ca"), work_conn)
_log("af5_rm_a_ca", af5_rm_a_ca)


In [ ]:
# BALANCE FINAL AF5 ACTIVO RESTO DEL MUNDO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_a"))
work_conn.execute(text("""
SELECT T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO
INTO #af5_rm_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'AF.5' AND T1.C_ENTRADA = 'D' AND T1.C_CUENTA = 'Bce Final'
GROUP BY T1.AÑO, T1.SECTOR, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA
"""))
af5_rm_a = pd.read_sql(text("SELECT * FROM #af5_rm_a"), work_conn)
_log("af5_rm_a", af5_rm_a)


In [ ]:
# ESTRUCTURA
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_est"))
work_conn.execute(text("""
SELECT T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.DATO / T2.DATO AS DATO
INTO #af5_rm_est
FROM #af5_rm_a_ca T1, #af5_rm_a T2
WHERE T1.SECTOR = T2.SECTOR AND T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM
"""))
af5_rm_est = pd.read_sql(text("SELECT * FROM #af5_rm_est"), work_conn)
_log("af5_rm_est", af5_rm_est)


In [ ]:
# IMPUTACIÓN APERTURA DIVIDENDOS RECIBIDOS
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_rec"))
work_conn.execute(text("""
SELECT 'P' AS MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T2.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * T2.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0803' AS PROC
INTO #d42_rm_rec
FROM TABLAS.dbo.BD_CTSI t1, #af5_rm_est T2
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'H' AND T1.SECTOR = T2.SECTOR AND T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM
GROUP BY T1.AÑO, T1.SECTOR, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T2.C_CAGENTE
"""))


In [ ]:
# AJUSTA IMPUTACION ANTERIOR EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_aj"))
work_conn.execute(text("""
SELECT T1.MONEDA, T1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0803b' AS PROC
INTO #d42_rm_aj
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'H'
GROUP BY T1.MONEDA, T1.AÑO, T1.SECTOR, T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_rm_rec"))
_log("APPEND TABLAS.BD_CTSI desde D42_RM_REC", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi}) SELECT {cols_bd_ctsi} FROM #d42_rm_aj"))
_log("APPEND TABLAS.BD_CTSI desde D42_RM_AJ", res.rowcount)
for t in ["#d42_rm_rec", "#d42_rm_aj", "#af5_rm_a_ca", "#af5_rm_a", "#af5_rm_est"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR EL RESTO DEL MUNDO REALIZA APERTURA POR CONTRAPARTIDA UTILIZANDO LA ESTRUCTURA DEL AF5 PASIVO DEL SECTOR GENERADO EN LA SÍNTESIS
# CONSIDERA LO QUE TIENEN EL ACTIVO DE LOS SECTORES CON CA 6..ESO ES LO DEFINITIVO
# BALANCE FINAL AF5 PASIVO RESTO DEL MUNDO CON CA
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_p_ca"))
work_conn.execute(text("""
SELECT T1.AÑO, T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO
INTO #af5_rm_p_ca
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_CAGENTE = '6' AND T1.C_SCN = 'AF.5' AND T1.C_ENTRADA = 'D' AND T1.C_CUENTA = 'Bce Final'
GROUP BY T1.AÑO, TRY_CAST(T1.C_CAGENTE AS float), T1.TRIM,
         LTRIM(CAST(T1.SECTOR AS varchar(11))), T1.C_CUENTA, T1.C_ENTRADA
"""))
af5_rm_p_ca = pd.read_sql(text("SELECT * FROM #af5_rm_p_ca"), work_conn)
_log("af5_rm_p_ca", af5_rm_p_ca)


In [ ]:
# BALANCE FINAL AF5 PASIVO RESTO DEL MUNDO
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_p"))
work_conn.execute(text("""
SELECT AÑO, TRIM,
       SECTOR,
       C_CUENTA,
       C_ENTRADA,
       SUM(DATO) AS DATO
INTO #af5_rm_p
FROM #af5_rm_p_ca t1
GROUP BY AÑO, SECTOR, TRIM, C_CUENTA, C_ENTRADA
"""))
af5_rm_p = pd.read_sql(text("SELECT * FROM #af5_rm_p"), work_conn)
_log("af5_rm_p", af5_rm_p)
# NOTA: este tramo (1/8) termina aquí, en línea con el bloque PROC SQL que crea WORK.AF5_RM_P.
# #af5_rm_p_ca y #af5_rm_p quedan vivas en la sesión (work_conn) para ser consumidas por tramos posteriores.


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_rm_est_p"))
# ESTRUCTURA
sql_af5_rm_est_p = """
SELECT t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.DATO / T2.DATO AS DATO
INTO #af5_rm_est_p
FROM #af5_rm_p_ca t1, #af5_rm_p T2
WHERE T1.SECTOR = T2.SECTOR AND T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM
"""
work_conn.execute(text(sql_af5_rm_est_p))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_pag"))
# IMPUTACIÓN APERTURA DIVIDENDOS PAGADOS
sql_d42_rm_pag = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T2.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * T2.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0804' AS PROC
INTO #d42_rm_pag
FROM TABLAS.dbo.BD_CTSI t1, #af5_rm_est_p T2
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'D' AND (T1.SECTOR = T2.SECTOR AND T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM)
GROUP BY t1.AÑO, T1.SECTOR, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T2.C_CAGENTE
"""
work_conn.execute(text(sql_d42_rm_pag))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_p_aj"))
# AJUSTA IMPUTACION ANTERIOR EN CA 53
sql_d42_rm_p_aj = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0804b' AS PROC
INTO #d42_rm_p_aj
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR = 6 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'D'
GROUP BY T1.MONEDA, t1.AÑO, T1.SECTOR, T1.TRIM, t1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d42_rm_p_aj))


In [ ]:
# APPEND server-side de D42_RM_PAG y D42_RM_P_AJ a TABLAS.BD_CTSI (FORCE alinea por nombre de columna)
cols_bd_ctsi_1 = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_rm_pag"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_RM_PAG)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_rm_p_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_RM_P_AJ)", res.rowcount)


In [ ]:
# Drop de temporales: el original nombra D42_RM__PAJ (typo, no existe como tal) — se dropean las 5 tablas realmente creadas en este tramo
for t in ["#d42_rm_pag", "#d42_rm_p_aj", "#af5_rm_p_ca", "#af5_rm_p", "#af5_rm_est_p"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS RECIBIDOS POR BANCOS REALIZA APERTURA POR CONTRAPARTIDA UTILIZANDO LA ESTRUCTURA DEL AF5 ACTIVO DEL SECTOR GENERADO EN LA SÍNTESIS
work_conn.execute(text("DROP TABLE IF EXISTS #af5_bcos_a_ca"))
# BALANCE FINAL AF5 ACTIVO CON CA
sql_af5_bcos_a_ca = """
SELECT t1.AÑO, T1.TRIM,
       321 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO
INTO #af5_bcos_a_ca
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR IN (321, 32) AND T1.C_SCN = 'AF.5' AND T1.C_ENTRADA = 'D' AND T1.C_CUENTA = 'Bce Final'
GROUP BY t1.AÑO, T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA
"""
work_conn.execute(text(sql_af5_bcos_a_ca))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_bcos_a"))
# BALANCE FINAL AF5 ACTIVO
sql_af5_bcos_a = """
SELECT t1.AÑO, T1.TRIM,
       321 AS SECTOR,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO
INTO #af5_bcos_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR IN (321, 32) AND T1.C_SCN = 'AF.5' AND T1.C_ENTRADA = 'D' AND T1.C_CUENTA = 'Bce Final'
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA
"""
work_conn.execute(text(sql_af5_bcos_a))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #af5_bcos_est"))
# ESTRUCTURA
sql_af5_bcos_est = """
SELECT t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.DATO / T2.DATO AS DATO
INTO #af5_bcos_est
FROM #af5_bcos_a_ca t1, #af5_bcos_a T2
WHERE T1.SECTOR = T2.SECTOR AND T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM
"""
work_conn.execute(text(sql_af5_bcos_est))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_rec"))
# IMPUTACIÓN APERTURA DIVIDENDOS RECIBIDOS
sql_d42_bcos_rec = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       T2.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * T2.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0804c' AS PROC
INTO #d42_bcos_rec
FROM TABLAS.dbo.BD_CTSI t1, #af5_bcos_est T2
WHERE T1.SECTOR = 321 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'H' AND (T1.SECTOR = T2.SECTOR AND T1.AÑO = T2.AÑO AND T1.TRIM = T2.TRIM)
GROUP BY t1.AÑO, T1.SECTOR, T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN, T2.C_CAGENTE
"""
work_conn.execute(text(sql_d42_bcos_rec))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_aj"))
# AJUSTA IMPUTACION ANTERIOR EN CA 53
sql_d42_bcos_aj = """
SELECT T1.MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       t1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0804d' AS PROC
INTO #d42_bcos_aj
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.SECTOR = 321 AND T1.C_SCN = 'D.42' AND T1.C_ENTRADA = 'H'
GROUP BY T1.MONEDA, t1.AÑO, T1.SECTOR, T1.TRIM, t1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d42_bcos_aj))


In [ ]:
# APPEND server-side de D42_BCOS_REC y D42_BCOS_AJ a TABLAS.BD_CTSI
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_bcos_rec"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_REC)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_bcos_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_AJ)", res.rowcount)


In [ ]:
for t in ["#d42_bcos_rec", "#d42_bcos_aj", "#af5_bcos_est", "#af5_bcos_a", "#af5_bcos_a_ca"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR EL RESTO DEL MUNDO IMPUTA LO DE BCENTRAL, BCOS, OFIS, SEGUROS y FP. AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_rm_p"))
sql_d42_rm_p = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       T1.TRIM,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END) AS SECTOR,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0805' AS PROC
INTO #d42_rm_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.42' AND T1.SECTOR = 6 AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('31', '32', '33', '35', '34')) OR
      (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.42' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('31', '32', '33', '35', '34') AND T1.C_CAGENTE = '6')
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END),
         (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_rm_p))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_6_53"))
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR RM
sql_d42_6_53 = """
SELECT T1.MONEDA,
       t1.AÑO,
       T1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0805b' AS PROC
INTO #d42_6_53
FROM #d42_rm_p T1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d42_6_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_rm_p"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_RM_P)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_6_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_6_53)", res.rowcount)


In [ ]:
for t in ["#d42_6_53", "#d42_rm_p"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR EL BANCOS IMPUTA LO DE BANCOS,SEGUROS y CORREDORES DE BOLSA. AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_p"))
sql_d42_bcos_p = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       T1.TRIM,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END) AS SECTOR,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0806' AS PROC
INTO #d42_bcos_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.42' AND T1.SECTOR IN (321, 322, 32) AND T1.C_CAGENTE IN ('321', '35', '351', '352', '353', '36904')) OR
      (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.42' AND T1.SECTOR IN (321, 35, 351, 352, 353, 36904) AND T1.C_CAGENTE IN ('321', '322', '32', '3'))
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END),
         (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_bcos_p))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_bcos_53"))
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR BANCOS
sql_d42_bcos_53 = """
SELECT T1.MONEDA,
       t1.AÑO,
       T1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0806b' AS PROC
INTO #d42_bcos_53
FROM #d42_bcos_p T1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d42_bcos_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_bcos_p"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_P)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_bcos_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_BCOS_53)", res.rowcount)


In [ ]:
for t in ["#d42_bcos_p", "#d42_bcos_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR OFIS IMPUTA LO DE BANCOS. AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_ofis_p"))
sql_d42_ofis_p = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       T1.TRIM,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END) AS SECTOR,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0807' AS PROC
INTO #d42_ofis_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.42' AND (SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('33') OR T1.SECTOR = 411) AND T1.C_CAGENTE = '321') OR
      (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.42' AND T1.SECTOR IN (321) AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('33'))
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END),
         (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_ofis_p))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_ofis_53"))
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR BANCOS (OFIS)
sql_d42_ofis_53 = """
SELECT T1.MONEDA,
       t1.AÑO,
       T1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0807b' AS PROC
INTO #d42_ofis_53
FROM #d42_ofis_p T1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d42_ofis_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_ofis_p"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_OFIS_P)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_ofis_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_OFIS_53)", res.rowcount)


In [ ]:
for t in ["#d42_ofis_p", "#d42_ofis_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR SEGUROS IMPUTA LO DE BANCOS. AJUSTA EN CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d42_seg_p"))
sql_d42_seg_p = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       T1.TRIM,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END) AS SECTOR,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0808' AS PROC
INTO #d42_seg_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.42' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('35') AND T1.C_CAGENTE = '321') OR
      (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.42' AND T1.SECTOR IN (321) AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('35'))
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END),
         (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_seg_p))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_seg_53"))
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL SECTOR SEGUROS
sql_d42_seg_53 = """
SELECT T1.MONEDA,
       t1.AÑO,
       T1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0808b' AS PROC
INTO #d42_seg_53
FROM #d42_seg_p T1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d42_seg_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_seg_p"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_SEG_P)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_seg_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_SEG_53)", res.rowcount)


In [ ]:
for t in ["#d42_seg_p", "#d42_seg_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. EN DIVIDENDOS PAGADOS POR AUXILIARES IMPUTA LO DE BANCOS Y LO DEL RESTO DEL MUNDO (SOLO CA 36).
# AJUSTA EN CA 53 SOLO LO DE BANCOS; POR LO DEL RM CON CA 36 SE DEBE ADICIONAR AL PAGADO POR AUXILIARES YA QUE ES EL SECTOR NO MEDIDO
work_conn.execute(text("DROP TABLE IF EXISTS #d42_aux_p"))
sql_d42_aux_p = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       T1.TRIM,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END) AS SECTOR,
       (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0809' AS PROC
INTO #d42_aux_p
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.42' AND SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('36', '37') AND T1.C_CAGENTE IN ('321')) OR
      (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.42' AND T1.SECTOR IN (321) AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36', '37'))
GROUP BY t1.AÑO, T1.TRIM,
         (CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END),
         (CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END),
         T1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d42_aux_p))


In [ ]:
# el DELETE comentado en el SAS original (C_CAGENTE='6' AND SECTOR NOT IN (36)) permanece anulado — no se traduce
work_conn.execute(text("DROP TABLE IF EXISTS #d42_aux_53"))
# AJUSTA IMPUTACIÓN ANTERIOR EN D42 CA 53 DEL AUXILIARES..SOLO LA PARTE DE BANCOS
sql_d42_aux_53 = """
SELECT T1.MONEDA,
       t1.AÑO,
       T1.TRIM,
       t1.SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0809b' AS PROC
INTO #d42_aux_53
FROM #d42_aux_p T1
WHERE T1.C_CAGENTE = '321'
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, t1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d42_aux_53))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_aux_p"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_AUX_P)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_aux_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_AUX_53)", res.rowcount)


In [ ]:
for t in ["#d42_aux_p", "#d42_aux_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# El bloque D42_AUX_R (imputa dividendos recibidos por sector 36) está comentado por completo en el SAS original: queda anulado, no se traduce.

# CONCILIA TOTAL DEBE Y HABER DEL D42. MANDA EL HABER, DIF LA IMPUTA AL DEBE DEL SECTOR 51022 CON CA RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #d42_recibido"))
# CALCULA TOTAL RECIBIDO POR CADA SECTOR
sql_d42_recibido = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       T1.TRIM,
       51022 AS SECTOR,
       (CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LTRIM(CAST(T2.C_SI_publ AS varchar(4))) END) AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       'Renta distribuida de las sociedades' AS N_SCN,
       'PS' AS FUENTE,
       '0704' AS PROC
INTO #d42_recibido
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('D.42') AND T1.SECTOR = T2.C_SI
GROUP BY t1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, t1.C_SCN,
         (CASE WHEN T1.SECTOR = 51022 THEN '53' ELSE LTRIM(CAST(T2.C_SI_publ AS varchar(4))) END)
"""
work_conn.execute(text(sql_d42_recibido))


In [ ]:
work_conn.execute(text("UPDATE #d42_recibido SET C_CAGENTE = '511' WHERE C_CAGENTE = '8'"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_pagado"))
# CALCULA TOTAL PAGADO POR CADA SECTOR
sql_d42_pagado = """
SELECT 'P' AS MONEDA,
       t1.AÑO,
       T1.TRIM,
       51022 AS SECTOR,
       LTRIM(CAST(T2.C_SI_publ AS varchar(4))) AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0704' AS PROC
INTO #d42_pagado
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_CONTRAPARTIDAS T2
WHERE T1.C_ENTRADA = 'D' AND T1.C_SCN IN ('D.42') AND T1.C_CAGENTE = T2.C_CAGENTE
GROUP BY t1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, t1.C_SCN, T1.N_SCN,
         LTRIM(CAST(T2.C_SI_publ AS varchar(4)))
"""
work_conn.execute(text(sql_d42_pagado))


In [ ]:
work_conn.execute(text("UPDATE #d42_pagado SET C_CAGENTE = '511' WHERE C_CAGENTE = '8'"))
# DATA D42_RECIBIDO; SET D42_RECIBIDO D42_PAGADO: concatenación server-side vía INSERT sobre la misma #tmp
work_conn.execute(text("INSERT INTO #d42_recibido SELECT * FROM #d42_pagado"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d42_delta"))
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
sql_d42_delta = """
SELECT T1.MONEDA,
       t1.AÑO,
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #d42_delta
FROM #d42_recibido T1
GROUP BY T1.MONEDA, t1.AÑO, T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, t1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_d42_delta))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d42_delta"))
_log("APPEND TABLAS.dbo.BD_CTSI (D42_DELTA)", res.rowcount)


In [ ]:
for t in ["#d42_delta", "#d42_pagado", "#d42_recibido"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA UTILIDADES REINVERTIDAS RECIBIDAS POR SECTOR 51021 CON CA 53, USANDO INFO DE LO PAGADO POR RM
work_conn.execute(text("DROP TABLE IF EXISTS #d43_h_snf"))
sql_d43_h_snf = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51021 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0705' AS PROC
INTO #d43_h_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.43' AND T1.SECTOR = 6)
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, T1.N_SCN, T1.DATO
"""
work_conn.execute(text(sql_d43_h_snf))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d43_h_snf"))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_H_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d43_h_snf"))


In [ ]:
# IMPUTA UTILIDADES REINVERTIDAS PAGADAS POR SECTOR 51021 CON CA 53, USANDO INFO DE LO RECIBIDO POR RM
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_snf"))
sql_d43_d_snf = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51021 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0706' AS PROC
INTO #d43_d_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.43' AND T1.SECTOR = 6)
GROUP BY t1.AÑO, T1.TRIM, T1.C_CAGENTE, T1.C_CUENTA, t1.C_SCN, T1.N_SCN, T1.DATO
"""
work_conn.execute(text(sql_d43_d_snf))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d43_d_snf"))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_D_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_snf"))


In [ ]:
# IMPUTA UTILIDADES REINVERTIDAS PAGADAS POR SECTOR 32 CON CA 6. AJUSTA EN SECTOR 51 CON CA 6. CIERRE 2021: incorpora apertura entre bancos y seguros
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_sf"))
sql_d43_d_sf = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       T1.SECTOR,
       '6' AS C_CAGENTE,
       'YG' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'D.43' AS C_SCN,
       'Utilidades reinvertidas de la inversión extranjera directa' AS N_SCN,
       'PS' AS FUENTE,
       '0706b' AS PROC
INTO #d43_d_sf
FROM TABLAS.dbo.UR_SF_CR18 t1
"""
work_conn.execute(text(sql_d43_d_sf))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d43_d_aj"))
sql_d43_d_aj = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       'YG' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       'D.43' AS C_SCN,
       'Utilidades reinvertidas de la inversión extranjera directa' AS N_SCN,
       'PS' AS FUENTE,
       '0706c' AS PROC
INTO #d43_d_aj
FROM TABLAS.dbo.UR_SF_CR18 t1
GROUP BY t1.AÑO, T1.TRIM
"""
work_conn.execute(text(sql_d43_d_aj))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d43_d_sf"))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_D_SF)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d43_d_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (D43_D_AJ)", res.rowcount)
for t in ["#d43_d_sf", "#d43_d_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA D45 PAGADO EN SECTOR 51022, USANDO INFO DE LO RECIBIDO POR SECTOR 41
work_conn.execute(text("DROP TABLE IF EXISTS #d45_d_snf"))
sql_d45_d_snf = """
SELECT 'P' AS MONEDA, t1.AÑO, T1.TRIM,
       51022 AS SECTOR,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0707' AS PROC
INTO #d45_d_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.45' AND T1.SECTOR = 41)
GROUP BY t1.AÑO, T1.TRIM, T1.C_CUENTA, t1.C_SCN, T1.N_SCN, T1.DATO
"""
work_conn.execute(text(sql_d45_d_snf))


In [ ]:
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_1}) SELECT {cols_bd_ctsi_1} FROM #d45_d_snf"))
_log("APPEND TABLAS.dbo.BD_CTSI (D45_D_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d45_d_snf"))
# AJUSTA D5: el comentario de cierre del tramo no tiene código asociado en este bloque; el siguiente tramo continúa con D5


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d5_d_snf"))
sql_d5_d_snf = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '41' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0711' AS PROC
INTO #d5_d_snf
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_ENTRADA='H' AND T1.C_SCN='D.5') OR (T1.C_ENTRADA='D' AND T1.C_SCN='D.5' AND T1.SECTOR NOT IN (412))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d5_d_snf))
# APPEND server-side hacia TABLAS.BD_CTSI
cols_bd_ctsi = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d5 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d5_d_snf
"""
res = work_conn.execute(text(sql_append_d5))
_log("APPEND TABLAS.BD_CTSI (D5_D_SNF)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d5_d_snf"))


In [ ]:
# AJUSTA D61
work_conn.execute(text("DROP TABLE IF EXISTS #d61_d"))
sql_d61_d = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '34' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0715' AS PROC
INTO #d61_d
FROM TABLAS.dbo.BD_CTSI t1
-- SECTOR NOT IN (/*411,*/412): el 411 quedó comentado en el original, se respeta tal cual
WHERE (T1.C_SCN='D.61' AND SECTOR NOT IN (412))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d61_d))
sql_append_d61 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d61_d
"""
res = work_conn.execute(text(sql_append_d61))
_log("APPEND TABLAS.BD_CTSI (D61_D)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d61_d"))


In [ ]:
# AJUSTA D62
work_conn.execute(text("DROP TABLE IF EXISTS #d62_d"))
sql_d62_d = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '34' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0719' AS PROC
INTO #d62_d
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.62' AND SECTOR NOT IN (412))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d62_d))
sql_append_d62 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d62_d
"""
res = work_conn.execute(text(sql_append_d62))
_log("APPEND TABLAS.BD_CTSI (D62_D)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d62_d"))


In [ ]:
# AJUSTA D8
work_conn.execute(text("DROP TABLE IF EXISTS #d8"))
sql_d8 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       34 AS SECTOR,
       '511' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       'Ajustes por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE,
       '0722' AS PROC
INTO #d8
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.8')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d8))
sql_append_d8 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d8
"""
res = work_conn.execute(text(sql_append_d8))
_log("APPEND TABLAS.BD_CTSI (D8)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d8"))


In [ ]:
# CALCULO DEL D8 EN BASE A D61 Y D62 DE SECTORES 34 Y 341. AJUSTA DIFERENCIAS EN D8
work_conn.execute(text("DROP TABLE IF EXISTS #d8_calculo"))
sql_d8_calculo = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       34 AS SECTOR,
       '511' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       'D.8' AS C_SCN,
       'Ajustes por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE,
       '0729' AS PROC
INTO #d8_calculo
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.61' AND T1.C_ENTRADA='H' AND T1.SECTOR IN (34,341)) OR (T1.C_SCN='D.62' AND T1.C_ENTRADA='D' AND T1.SECTOR IN (34,341))
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA
"""
work_conn.execute(text(sql_d8_calculo))

work_conn.execute(text("DROP TABLE IF EXISTS #d8_inicial"))
sql_d8_inicial = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       34 AS SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       'Ajustes por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
       'PS' AS FUENTE,
       '0729' AS PROC
INTO #d8_inicial
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.8' AND T1.C_ENTRADA='D' AND T1.SECTOR IN (34,341))
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CAGENTE, t1.C_CUENTA, t1.C_SCN
"""
work_conn.execute(text(sql_d8_inicial))
# APPEND server-side de #d8_inicial hacia #d8_calculo (PROC APPEND BASE=WORK.D8_CALCULO)
cols_d8_calculo = "MONEDA, AÑO, TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d8calc = f"""
INSERT INTO #d8_calculo ({cols_d8_calculo})
SELECT {cols_d8_calculo}
FROM #d8_inicial
"""
res = work_conn.execute(text(sql_append_d8calc))
_log("APPEND #d8_calculo <- #d8_inicial", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d8_ajuste"))
sql_d8_ajuste = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #d8_ajuste
FROM #d8_calculo t1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, t1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_d8_ajuste))

work_conn.execute(text("DROP TABLE IF EXISTS #d8_511"))
sql_d8_511 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '511' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       '0731' AS PROC
INTO #d8_511
FROM #d8_ajuste t1
"""
work_conn.execute(text(sql_d8_511))

sql_append_d8_ajuste = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d8_ajuste
"""
res = work_conn.execute(text(sql_append_d8_ajuste))
_log("APPEND TABLAS.BD_CTSI (D8_AJUSTE)", res.rowcount)

sql_append_d8_511 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d8_511
"""
res = work_conn.execute(text(sql_append_d8_511))
_log("APPEND TABLAS.BD_CTSI (D8_511)", res.rowcount)

for t in ["#d8_ajuste", "#d8_511", "#d8_inicial", "#d8_calculo"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
raise NotImplementedError(
    "WORK.D71_53 y WORK.D71_6 leen desde una ruta de archivo SAS7BDAT "
    "('/sasdata/BCCH/GEM_DCNI/02_CNSI/02_PRE_SINTESIS/bd_ci_full_cierre.sas7bdat'), "
    "no de una tabla de BD del catalogo de conexiones ni de un dataset WORK previo. "
    "M-001 pide eliminar rutas hardcodeadas pero no define el reemplazo (que archivo/tabla "
    "del proyecto reemplaza a bd_ci_full_cierre.sas7bdat): falta esa fuente para poder "
    "leer con pyreadstat o localizar la tabla BD equivalente."
)


In [ ]:
# IMPUTA D71 Y D72 EN SECTOR RM CON CA 351 Y 352 USANDO INFO DE CONTRAPARTIDA. AJUSTA IMPUTACIONES CONTRA D75 DEL SECTOR RM CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d71_rm"))
sql_d71_rm = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0739' AS PROC
INTO #d71_rm
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.71' AND T1.C_ENTRADA='D' AND T1.SECTOR IN (351,352) AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.SECTOR, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d71_rm))

work_conn.execute(text("DROP TABLE IF EXISTS #d75_rm_h"))
sql_d75_rm_h = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       'D.75' AS C_SCN,
       'Transferencias corrientes diversas' AS N_SCN,
       T1.FUENTE,
       '0748' AS PROC
INTO #d75_rm_h
FROM #d71_rm t1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA, T1.FUENTE
"""
work_conn.execute(text(sql_d75_rm_h))

work_conn.execute(text("DROP TABLE IF EXISTS #d72_rm"))
sql_d72_rm = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       TRY_CAST(T1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(T1.SECTOR AS varchar(11))) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0740' AS PROC
INTO #d72_rm
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.72' AND T1.C_ENTRADA='H' AND T1.SECTOR IN (351,352) AND T1.C_CAGENTE='6')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CAGENTE, T1.SECTOR, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d72_rm))

work_conn.execute(text("DROP TABLE IF EXISTS #d75_rm_d"))
sql_d75_rm_d = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       'D.75' AS C_SCN,
       'Transferencias corrientes diversas' AS N_SCN,
       T1.FUENTE,
       '0749' AS PROC
INTO #d75_rm_d
FROM #d72_rm t1
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA, T1.FUENTE
"""
work_conn.execute(text(sql_d75_rm_d))

sql_append_d71_rm = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d71_rm
"""
res = work_conn.execute(text(sql_append_d71_rm))
_log("APPEND TABLAS.BD_CTSI (D71_RM)", res.rowcount)

# el SAS original hace APPEND BASE=TABLAS.BD_CTSI DATA=D75_RM_H (sin libref WORK): mismo dataset WORK.D75_RM_H
sql_append_d75_rm_h = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d75_rm_h
"""
res = work_conn.execute(text(sql_append_d75_rm_h))
_log("APPEND TABLAS.BD_CTSI (D75_RM_H)", res.rowcount)

sql_append_d72_rm = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d72_rm
"""
res = work_conn.execute(text(sql_append_d72_rm))
_log("APPEND TABLAS.BD_CTSI (D72_RM)", res.rowcount)

sql_append_d75_rm_d = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d75_rm_d
"""
res = work_conn.execute(text(sql_append_d75_rm_d))
_log("APPEND TABLAS.BD_CTSI (D75_RM_d)", res.rowcount)

for t in ["#d71_rm", "#d72_rm", "#d75_rm_h", "#d75_rm_d"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE D71. RESPETA EL TOTAL RECIBIDO, DIFERENCIA LA IMPUTA EN PAGADO DEL SECTOR 51022 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #d71_cierre"))
sql_d71_cierre = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '352' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0744' AS PROC
INTO #d71_cierre
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.71')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d71_cierre))
sql_append_d71_cierre = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d71_cierre
"""
res = work_conn.execute(text(sql_append_d71_cierre))
_log("APPEND TABLAS.BD_CTSI (D71_CIERRE)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d71_cierre"))


In [ ]:
# CIERRE D72. RESPETA EL TOTAL PAGADO, DIFERENCIA LA IMPUTA EN RECIBIDO DEL SECTOR 51022 CON CA 352
work_conn.execute(text("DROP TABLE IF EXISTS #d72_cierre"))
sql_d72_cierre = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '352' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0747' AS PROC
INTO #d72_cierre
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.72')
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d72_cierre))
sql_append_d72_cierre = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d72_cierre
"""
res = work_conn.execute(text(sql_append_d72_cierre))
_log("APPEND TABLAS.BD_CTSI (D72_CIERRE)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d72_cierre"))


In [ ]:
# ELIMINA D75 EN SEGUROS GENERALES (>2009). YA NO APLICA CON MARCO NUEVO DE SEGUROS
work_conn.execute(text("DROP TABLE IF EXISTS #d75_352"))
sql_d75_352 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO*-1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0750' AS PROC
INTO #d75_352
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.75' AND T1.SECTOR=352 AND t1.[AÑO]>2009)
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, t1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_352))
sql_append_d75_352 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d75_352
"""
res = work_conn.execute(text(sql_append_d75_352))
_log("APPEND TABLAS.BD_CTSI (D75_352)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_352"))


In [ ]:
# IMPUTA 14% APROX DE TOTAL DE D75 RECIBIDAS EN LAS RECIBIDAS DEL SECTOR 51022 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d75_51022"))
sql_d75_51022 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO)*0.146847233207581 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0751' AS PROC
INTO #d75_51022
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.75' AND T1.C_ENTRADA='H' AND T1.SECTOR NOT IN (411,412,6))
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_51022))
sql_append_d75_51022 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d75_51022
"""
res = work_conn.execute(text(sql_append_d75_51022))
_log("APPEND TABLAS.BD_CTSI (D75_51022)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_51022"))


In [ ]:
# CIERRE DE D75. COMPARA RECIBIDO Y PAGADO. RESPETA LO RECIBIDO E IMPUTA EL 65% DEL DIFERENCIAL EN LO PAGADO DEL SECTOR 51022 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre"))
sql_d75_cierre = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51022 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='D' THEN T1.DATO*-1 ELSE T1.DATO END)*0.65 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0755' AS PROC
INTO #d75_cierre
FROM TABLAS.dbo.BD_CTSI t1
-- SECTOR NOT IN (/*411,*/412): el 411 quedó comentado en el original, se respeta tal cual
WHERE (T1.C_SCN='D.75' AND T1.SECTOR NOT IN (412))
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_cierre))
sql_append_d75_cierre = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d75_cierre
"""
res = work_conn.execute(text(sql_append_d75_cierre))
_log("APPEND TABLAS.BD_CTSI (D75_CIERRE)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre"))


In [ ]:
# IMPUTA AJUSTE EN TRANSF CORRIENTES PAGADAS DE EMPRESAS PARA QUE NO SEAN NEGATIVAS EN 2018 Y 2019. ESTO AHRÁ QUE SE AJUSTEN LAS RECIBIDAS TMB POR DEFECTO
# APPEND server-side de la tabla permanente TABLAS.AJ_TCORR (no es un WORK, ya vive en la BD)
cols_aj_tcorr = cols_bd_ctsi
sql_append_aj_tcorr = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_aj_tcorr})
SELECT {cols_aj_tcorr}
FROM TABLAS.dbo.AJ_TCORR
"""
res = work_conn.execute(text(sql_append_aj_tcorr))
_log("APPEND TABLAS.BD_CTSI (AJ_TCORR)", res.rowcount)


In [ ]:
# IMPUTA LA DIFERENCIA ENTRE DEBE Y HABER DEL D75 EN EL RECIBIDO DE SECTOR 51 CON CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre_2"))
sql_d75_cierre_2 = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       t1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA='H' THEN T1.DATO*-1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0755b' AS PROC
INTO #d75_cierre_2
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_SCN='D.75' AND T1.SECTOR NOT IN (412))
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d75_cierre_2))
sql_append_d75_cierre_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d75_cierre_2
"""
res = work_conn.execute(text(sql_append_d75_cierre_2))
_log("APPEND TABLAS.BD_CTSI (D75_CIERRE_2)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d75_cierre_2"))


In [ ]:
# CIERRE 2019. ACTUALIZA CONTRAGENTE VACÍOS EN INTERESES A 53
# valor faltante de C_CAGENTE en SAS ('') se traduce como '' O NULL para replicar la semantica de blanco SAS
sql_update_1 = text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:cagente_new, PROC=:proc_new "
    "WHERE (C_CAGENTE IS NULL OR C_CAGENTE='') AND C_SCN=:c_scn AND C_CUENTA=:c_cuenta"
)
res = work_conn.execute(sql_update_1, {"cagente_new": "53", "proc_new": "0755c", "c_scn": "D.41", "c_cuenta": "YG"})
_log("UPDATE TABLAS.BD_CTSI (contragente vacios a 53)", res.rowcount)


In [ ]:
# CIERRE 2019. ACTUALIZA CONTRAGENTE DE INTERESES RECIBIDOS DE GOBIERNO CON CA OFIS (3321) A CA BANCOS
sql_update_2 = text(
    "UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:cagente_new, PROC=:proc_new "
    "WHERE C_CAGENTE=:cagente_old AND C_SCN=:c_scn AND C_CUENTA=:c_cuenta AND C_ENTRADA=:c_entrada AND SECTOR=:sector"
)
res = work_conn.execute(sql_update_2, {"cagente_new": "321", "proc_new": "0755d", "cagente_old": "3321", "c_scn": "D.41", "c_cuenta": "YG", "c_entrada": "H", "sector": 41})
_log("UPDATE TABLAS.BD_CTSI (gobierno ofis->bancos)", res.rowcount)


In [ ]:
# CIERRE 2021. ACTUALIZA CONTRAGENTE DE INTERESES PAGADOS POR OFIS CON CA RESTO Y EMPRESAS A CA BANCOS
sectores_355 = [411, 33211, 33212, 33221, 33222, 33231, 33232, 339011, 3390102]
placeholders = ", ".join(f":sector_{i}" for i in range(len(sectores_355)))
params_update_3 = {"cagente_new": "321", "proc_new": "0755d", "cagente_excl": "321", "c_scn": "D.41", "c_cuenta": "YG", "c_entrada": "D"}
params_update_3.update({f"sector_{i}": s for i, s in enumerate(sectores_355)})
sql_update_3 = text(
    f"UPDATE TABLAS.dbo.BD_CTSI SET C_CAGENTE=:cagente_new, PROC=:proc_new "
    f"WHERE C_CAGENTE<>:cagente_excl AND C_SCN=:c_scn AND C_CUENTA=:c_cuenta AND C_ENTRADA=:c_entrada AND SECTOR IN ({placeholders})"
)
res = work_conn.execute(sql_update_3, params_update_3)
_log("UPDATE TABLAS.BD_CTSI (ofis resto/empresas->bancos)", res.rowcount)


In [ ]:
# CIERRE 2021. AJUSTA INTERESES RECIBIDOS POR BANCO CENTRAL UTILIZANDO ESTRUCTURA DE ACTIVOS QUE DEVENGAN INTERESES EN TODOS SUS CONTRAGENTES EXCEPTO RESTO DEL MUNDO
# INTERESES TOTALES RECIBIDOS POR BCCH DESDE CI
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_h_ci"))
sql_d41_bcch_h_ci = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN
INTO #d41_bcch_h_ci
FROM TABLAS.dbo.BD_CTSI T1
WHERE FUENTE='CI' AND T1.SECTOR=31 AND T1.C_ENTRADA='H' AND T1.C_SCN='D.41'
GROUP BY T1.MONEDA, t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, t1.SECTOR
"""
work_conn.execute(text(sql_d41_bcch_h_ci))
act_bcch = pd.read_sql(text("SELECT * FROM #d41_bcch_h_ci"), work_conn)
_log("d41_bcch_h_ci", act_bcch)


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch"))
sql_act_bcch = """
SELECT t1.[AÑO], T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #act_bcch
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR=31 AND T1.C_ENTRADA='D' AND T1.C_SCN IN ('AF.41','AF.42','AF.32','AF.31','AF.29','AF.22')
  AND T1.C_CUENTA IN ('Bce Final','Bce Inicio') AND T1.C_CAGENTE<>'6'
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_ENTRADA, t1.SECTOR
"""
work_conn.execute(text(sql_act_bcch))
act_bcch = pd.read_sql(text("SELECT * FROM #act_bcch"), work_conn)
_log("act_bcch", act_bcch)


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM, promedio saldo final e inicial
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_2"))
sql_act_bcch_2 = """
SELECT t1.[AÑO], T1.TRIM, t1.SECTOR, T1.C_ENTRADA, AVG(T1.DATO) AS DATO
INTO #act_bcch_2
FROM #act_bcch t1
GROUP BY t1.[AÑO], T1.TRIM, t1.SECTOR, T1.C_ENTRADA
"""
work_conn.execute(text(sql_act_bcch_2))
act_bcch_2 = pd.read_sql(text("SELECT * FROM #act_bcch_2"), work_conn)
_log("act_bcch_2", act_bcch_2)


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM, con detalle contragente
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_c"))
sql_act_bcch_c = """
SELECT t1.[AÑO], T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #act_bcch_c
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR=31 AND T1.C_ENTRADA='D' AND T1.C_SCN IN ('AF.41','AF.42','AF.32','AF.31','AF.29','AF.22')
  AND T1.C_CUENTA IN ('Bce Final','Bce Inicio') AND T1.C_CAGENTE<>'6'
GROUP BY t1.[AÑO], T1.TRIM, t1.C_CUENTA, T1.C_ENTRADA, T1.C_CAGENTE, t1.SECTOR
"""
work_conn.execute(text(sql_act_bcch_c))
act_bcch_c = pd.read_sql(text("SELECT * FROM #act_bcch_c"), work_conn)
_log("act_bcch_c", act_bcch_c)


In [ ]:
# ACTIVOS TOTALES QUE DEVENGAN INTERESES SIN CONTRAGENTE RM, con detalle contragente, promedio saldo final e inicial
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_c2"))
sql_act_bcch_c2 = """
SELECT t1.[AÑO], T1.TRIM, t1.SECTOR, T1.C_CAGENTE, T1.C_ENTRADA, AVG(T1.DATO) AS DATO
INTO #act_bcch_c2
FROM #act_bcch_c t1
GROUP BY t1.[AÑO], T1.TRIM, t1.SECTOR, T1.C_ENTRADA, T1.C_CAGENTE
"""
work_conn.execute(text(sql_act_bcch_c2))
act_bcch_c2 = pd.read_sql(text("SELECT * FROM #act_bcch_c2"), work_conn)
_log("act_bcch_c2", act_bcch_c2)


In [ ]:
# ESTRUCTURA DE ACTIVOS POR CONTRAGENTE
work_conn.execute(text("DROP TABLE IF EXISTS #act_bcch_est"))
sql_act_bcch_est = """
SELECT t1.[AÑO], T1.TRIM, t1.SECTOR, T1.C_CAGENTE, T1.DATO/T2.DATO AS EST
INTO #act_bcch_est
FROM #act_bcch_c2 t1
INNER JOIN #act_bcch_2 T2 ON T1.[AÑO]=T2.[AÑO] AND T1.TRIM=T2.TRIM
"""
work_conn.execute(text(sql_act_bcch_est))
act_bcch_est = pd.read_sql(text("SELECT * FROM #act_bcch_est"), work_conn)
_log("act_bcch_est", act_bcch_est)


In [ ]:
# INTERESES SECTORIZADOS MODIFICADOS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_def"))
sql_d41_bcch_def = """
SELECT T1.MONEDA, t1.[AÑO], T1.TRIM, t1.SECTOR, t1.C_CUENTA, T1.C_ENTRADA, T2.C_CAGENTE,
       T1.DATO*T2.EST AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755d1' AS PROC
INTO #d41_bcch_def
FROM #d41_bcch_h_ci T1
INNER JOIN #act_bcch_est T2 ON T1.[AÑO]=T2.[AÑO] AND T1.TRIM=T2.TRIM
"""
work_conn.execute(text(sql_d41_bcch_def))
d41_bcch_def = pd.read_sql(text("SELECT * FROM #d41_bcch_def"), work_conn)
_log("d41_bcch_def", d41_bcch_def)


In [ ]:
# INTERESES SECTORIZADOS INICIALES A ELIMINAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_drop"))
sql_d41_bcch_drop = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_CAGENTE, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755d1' AS PROC
INTO #d41_bcch_drop
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.FUENTE = 'CI' AND T1.SECTOR = 31 AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE
"""
work_conn.execute(text(sql_d41_bcch_drop))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_BCCH_DROP FORCE (server-side desde #tmp)
cols_d41_bcch_drop = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_bcch_drop = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_bcch_drop})
SELECT {cols_d41_bcch_drop}
FROM #d41_bcch_drop
"""
res = work_conn.execute(text(sql_append_d41_bcch_drop))
_log("APPEND TABLAS.BD_CTSI (D41_BCCH_DROP)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_BCCH_DEF FORCE
# D41_BCCH_DEF fue dejado como #tmp por un tramo anterior del mismo nodo
cols_d41_bcch_def = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_CAGENTE, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_bcch_def = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_bcch_def})
SELECT {cols_d41_bcch_def}
FROM #d41_bcch_def
"""
res = work_conn.execute(text(sql_append_d41_bcch_def))
_log("APPEND TABLAS.BD_CTSI (D41_BCCH_DEF)", res.rowcount)


In [ ]:
# DROP TABLE D41_BCCH_DEF, D41_BCCH_DROP (limpieza de temporales de sesion)
for t in ["#d41_bcch_def", "#d41_bcch_drop"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. AJUSTA INTERESES PAGADOS POR GOBIERNO A BANCOS, UTILIZANDO PTMOS TOTALES CON BANCOS Y TOTAL DE INTERESES PAGADOS EN LA CI
# INTERESES TOTALES PAGADOR POR GOB DESDE CI
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_d_ci"))
sql_d41_gob_d_ci = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, 4 AS SECTOR, T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO, T1.C_SCN, T1.N_SCN
INTO #d41_gob_d_ci
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.FUENTE = 'CI' AND T1.SECTOR = 41 AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_gob_d_ci))


In [ ]:
# PASIVOS TOTALES QUE DEVENGAN INTERESES DESDE CI
work_conn.execute(text("DROP TABLE IF EXISTS #pas_gob_h"))
sql_pas_gob_h = """
SELECT T1.[AÑO], T1.TRIM, 4 AS SECTOR, T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #pas_gob_h
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.FUENTE = 'CI' AND T1.SECTOR IN (41, 412, 413, 42) AND T1.C_ENTRADA = 'H'
  AND T1.C_SCN IN ('AF.41', 'AF.42', 'AF.32', 'AF.31') AND T1.C_CUENTA = 'Bce Final'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA
"""
work_conn.execute(text(sql_pas_gob_h))


In [ ]:
# PASIVOS TOTALES QUE DEVENGAN INTERESES DESDE CI CON BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #pas_gob_bcos"))
sql_pas_gob_bcos = """
SELECT T1.[AÑO], T1.TRIM, 4 AS SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) AS DATO
INTO #pas_gob_bcos
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.FUENTE = 'CI' AND T1.SECTOR IN (41, 412, 413, 42) AND T1.C_CAGENTE IN ('321', '32')
  AND T1.C_ENTRADA = 'H' AND T1.C_SCN IN ('AF.41', 'AF.42', 'AF.32', 'AF.31') AND T1.C_CUENTA = 'Bce Final'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_CAGENTE
"""
work_conn.execute(text(sql_pas_gob_bcos))


In [ ]:
# % QUE REPRESENTA EL PASIVO CON BANCOS DEL TOTAL
work_conn.execute(text("DROP TABLE IF EXISTS #pas_gob_est"))
sql_pas_gob_est = """
SELECT T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.DATO * 1.0 / T2.DATO AS EST
INTO #pas_gob_est
FROM #pas_gob_bcos T1
INNER JOIN #pas_gob_h T2 ON T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM
"""
work_conn.execute(text(sql_pas_gob_est))


In [ ]:
# CALCULA INTERESES PAGADOS A BANCOS CORREGIDOS PARA REEMPLAZAR DATO CI
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_bcos"))
sql_d41_gob_bcos = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T2.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       T1.DATO * T2.EST AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755e' AS PROC
INTO #d41_gob_bcos
FROM #d41_gob_d_ci T1
INNER JOIN #pas_gob_est T2 ON T1.[AÑO] = T2.[AÑO] AND T1.TRIM = T2.TRIM AND T1.SECTOR = T2.SECTOR
"""
work_conn.execute(text(sql_d41_gob_bcos))


In [ ]:
# INTERESES A ELIMINAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_bcos_drop"))
sql_d41_gob_bcos_drop = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       T1.DATO * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755e' AS PROC
INTO #d41_gob_bcos_drop
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.FUENTE = 'CI' AND T1.SECTOR = 41 AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.C_CAGENTE = '321'
"""
work_conn.execute(text(sql_d41_gob_bcos_drop))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_GOB_BCOS FORCE (primera pasada, sin negar)
cols_d41_gob_bcos = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_gob_bcos_1 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_gob_bcos})
SELECT {cols_d41_gob_bcos}
FROM #d41_gob_bcos
"""
res = work_conn.execute(text(sql_append_d41_gob_bcos_1))
_log("APPEND TABLAS.BD_CTSI (D41_GOB_BCOS 1ra pasada)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_GOB_BCOS_DROP FORCE (primera pasada, sin negar)
cols_d41_gob_bcos_drop = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_gob_bcos_drop_1 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_gob_bcos_drop})
SELECT {cols_d41_gob_bcos_drop}
FROM #d41_gob_bcos_drop
"""
res = work_conn.execute(text(sql_append_d41_gob_bcos_drop_1))
_log("APPEND TABLAS.BD_CTSI (D41_GOB_BCOS_DROP 1ra pasada)", res.rowcount)


In [ ]:
# PARA AJUSTARLO CONTRA RESTO DEL MUNDO
work_conn.execute(text("UPDATE #d41_gob_bcos SET DATO = DATO * -1, C_CAGENTE = '6'"))


In [ ]:
# PARA AJUSTARLO CONTRA RESTO DEL MUNDO
work_conn.execute(text("UPDATE #d41_gob_bcos_drop SET DATO = DATO * -1, C_CAGENTE = '6'"))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_GOB_BCOS FORCE (segunda pasada, ya ajustado)
sql_append_d41_gob_bcos_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_gob_bcos})
SELECT {cols_d41_gob_bcos}
FROM #d41_gob_bcos
"""
res = work_conn.execute(text(sql_append_d41_gob_bcos_2))
_log("APPEND TABLAS.BD_CTSI (D41_GOB_BCOS 2da pasada)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_GOB_BCOS_DROP FORCE (segunda pasada, ya ajustado)
sql_append_d41_gob_bcos_drop_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_gob_bcos_drop})
SELECT {cols_d41_gob_bcos_drop}
FROM #d41_gob_bcos_drop
"""
res = work_conn.execute(text(sql_append_d41_gob_bcos_drop_2))
_log("APPEND TABLAS.BD_CTSI (D41_GOB_BCOS_DROP 2da pasada)", res.rowcount)


In [ ]:
# DROP TABLE D41_GOB_D_CI,PAS_GOB_H,PAS_GOB_BCOS,PAS_GOB_EST,D41_GOB_BCOS,D41_GOB_BCOS_DROP
for t in ["#d41_gob_d_ci", "#pas_gob_h", "#pas_gob_bcos", "#pas_gob_est", "#d41_gob_bcos", "#d41_gob_bcos_drop"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. AJUSTA INTERESES PAGADOS POR GOBIERNO A BANCOS Y EMPRESAS POR CONCEPTO DE BONOS USANDO DCV
# DATOS DCV INTERESES PAGADOS POR GOBIERNO A BANCOS Y EMPRESAS
# Fuente es un archivo SAS externo (.sas7bdat) fuera de la BD del proyecto: no hay ruta relativa
# ni dataset de origen declarado para reemplazarlo, y la lectura de .sas7bdat no está en las
# librerías permitidas (no hay pyreadstat.read_sas_dcv equivalente documentado para esta ruta).
raise NotImplementedError(
    "GOB_BR: fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat' "
    "es un archivo SAS externo fuera del catalogo de conexiones de BD y fuera de input_datasets; "
    "se requiere definir su reemplazo (ruta relativa + lector, p.ej. pyreadstat.read_sas7bdat) "
    "antes de traducir este bloque"
)


In [ ]:
# INTERESES PAGADOS POR GOB A EMPRESAS A ELIMINAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_53"))
sql_d41_gob_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755f' AS PROC
INTO #d41_gob_53
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR = 41 AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.C_CAGENTE = '53'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_d41_gob_53))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_GOB_53 FORCE (primera pasada)
cols_d41_gob_53 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_gob_53_1 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_gob_53})
SELECT {cols_d41_gob_53}
FROM #d41_gob_53
"""
res = work_conn.execute(text(sql_append_d41_gob_53_1))
_log("APPEND TABLAS.BD_CTSI (D41_GOB_53 1ra pasada)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=GOB_BR FORCE (primera pasada)
# GOB_BR depende del bloque anterior no resuelto (archivo externo .sas7bdat)
raise NotImplementedError(
    "APPEND de GOB_BR a TABLAS.BD_CTSI (1ra pasada): GOB_BR no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat (ver bloque anterior)"
)


In [ ]:
# AJUSTA INT ELIMINADOS DE EMPRESAS Y LO LLEVA A RESTO DEL MUNDO
work_conn.execute(text("UPDATE #d41_gob_53 SET DATO = DATO * -1, C_CAGENTE = '6'"))


In [ ]:
# AJUSTA INT DCV EN BCOS Y EMPRESAS CONTRA RESTO DEL MUNDO
# GOB_BR no existe (bloque previo no resuelto)
raise NotImplementedError(
    "UPDATE GOB_BR SET DATO=DATO*-1, C_CAGENTE='6': GOB_BR no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_GOB_53 FORCE (segunda pasada, ya ajustado)
sql_append_d41_gob_53_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_gob_53})
SELECT {cols_d41_gob_53}
FROM #d41_gob_53
"""
res = work_conn.execute(text(sql_append_d41_gob_53_2))
_log("APPEND TABLAS.BD_CTSI (D41_GOB_53 2da pasada)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=GOB_BR FORCE (segunda pasada)
raise NotImplementedError(
    "APPEND de GOB_BR a TABLAS.BD_CTSI (2da pasada): GOB_BR no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# DROP TABLE GOB_BR,D41_GOB_53
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_53"))
# #gob_br nunca se creo (fuente externa no resuelta); DROP IF EXISTS es no-op seguro
work_conn.execute(text("DROP TABLE IF EXISTS #gob_br"))


In [ ]:
# CIERRE 2021. IMPUTA INTERESES RECIBIDOS POR FP CON CONTRAGENTE OFIS USANDO DATOS DCV.
# IMPUTA LO QUE ES POR DEBENTURES Y EFECTO DE COMERCIO, PORQUE LA CUENTA YA TIENE LO DE PATRIMONIO SEPARADO. SE AJUSTA CONTRA EMPRESAS
# DATOS DCV INTERESES RECIBIDOS POR FP DESDE OFIS -- misma fuente externa no resuelta
raise NotImplementedError(
    "FP_OFI: fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat' "
    "es un archivo SAS externo fuera del catalogo de conexiones de BD y fuera de input_datasets; "
    "se requiere definir su reemplazo antes de traducir este bloque"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=FP_OFI FORCE (primera pasada)
raise NotImplementedError(
    "APPEND de FP_OFI a TABLAS.BD_CTSI (1ra pasada): FP_OFI no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# AJUSTA CONTRA EMPRESAS
raise NotImplementedError(
    "UPDATE FP_OFI SET DATO=DATO*-1, C_CAGENTE='53': FP_OFI no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=FP_OFI FORCE (segunda pasada)
raise NotImplementedError(
    "APPEND de FP_OFI a TABLAS.BD_CTSI (2da pasada): FP_OFI no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# DROP TABLE FP_OFI (nunca se creo; DROP IF EXISTS es no-op seguro)
work_conn.execute(text("DROP TABLE IF EXISTS #fp_ofi"))


In [ ]:
# CIERRE 2021. AJUSTA INTERESES RECIBIDOS POR SEGUROS CON CONTRAGENTE OFIS USANDO DATOS DCV. SE AJUSTA CONTRA EMPRESAS
# DATOS DCV INTERESES RECIBIDOS POR SEGUROS DESDE OFIS -- misma fuente externa no resuelta
raise NotImplementedError(
    "SEG_OFI: fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat' "
    "es un archivo SAS externo fuera del catalogo de conexiones de BD y fuera de input_datasets; "
    "se requiere definir su reemplazo antes de traducir este bloque"
)


In [ ]:
# INTERESES RECIBIDOS POR SEGUROS DESDE OFIS EN CI A RESTAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_seg_33"))
sql_d41_seg_33 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, 35 AS SECTOR, '33' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755h' AS PROC
INTO #d41_seg_33
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR IN (35, 351, 352, 353) AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41'
  AND SUBSTRING(T1.C_CAGENTE, 1, 2) = '33' AND SUBSTRING(T1.C_CAGENTE, 1, 5) NOT IN ('33901')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_seg_33))


In [ ]:
# data SEG_OFI;set SEG_OFI D41_SEG_33;run; -- SEG_OFI no existe (bloque previo no resuelto)
raise NotImplementedError(
    "concatenacion SEG_OFI + D41_SEG_33: SEG_OFI no se pudo materializar porque su fuente "
    "es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# calcula delta para ajustarse a dato DCV
raise NotImplementedError(
    "SEG_OFI_delta: depende de SEG_OFI, no materializado por fuente externa no resuelta"
)


In [ ]:
# IMPUTACIÓN PARA AJUSTAR EN RESTO DE LA ECONOMIA (CONTRA AJUSTE)
raise NotImplementedError(
    "SEG_RESTO_delta: depende de SEG_OFI_delta, no materializado por fuente externa no resuelta"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=SEG_OFI_delta FORCE
raise NotImplementedError(
    "APPEND de SEG_OFI_delta a TABLAS.BD_CTSI: no materializado por fuente externa no resuelta"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=SEG_RESTO_delta FORCE
raise NotImplementedError(
    "APPEND de SEG_RESTO_delta a TABLAS.BD_CTSI: no materializado por fuente externa no resuelta"
)


In [ ]:
# DROP TABLE SEG_OFI,D41_SEG_33,SEG_OFI_delta,SEG_RESTO_delta
for t in ["#seg_ofi", "#d41_seg_33", "#seg_ofi_delta", "#seg_resto_delta"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. IMPUTA INTERESES PAGADOS EN PATRIMONIO SEPARADO (OFIS) POR BONOS DE PATRIMONIOS SEPARADOS USANDO DATOS DESDE DCV, CON CONTRAGENTE BCOS PORQUE LUEGO SE AJUSTAN EN LA SINTESIS.
# PARA NO ALTERAR LA CUENTA ESTOS MISMOS SE IMPUTAN EN LOS RECIBIDOS CON CONTRAGENTE EMPRESAS, YA Q LOS PTMOS SE VAN A ESE SECTOR
# DATOS DCV INTERESES PAGADOS POR PATRIMONIOS SEPARADOS -- misma fuente externa no resuelta
raise NotImplementedError(
    "PATSEP_D: fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat' "
    "es un archivo SAS externo fuera del catalogo de conexiones de BD y fuera de input_datasets; "
    "se requiere definir su reemplazo antes de traducir este bloque"
)


In [ ]:
# DATOS DCV INTERESES RECIBIDOS POR PATRIMONIOS SEPARADOS
raise NotImplementedError(
    "PATSEP_H: depende de PATSEP_D, no materializado por fuente externa no resuelta"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=PATSEP_D FORCE
raise NotImplementedError(
    "APPEND de PATSEP_D a TABLAS.BD_CTSI: no materializado por fuente externa no resuelta"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=PATSEP_H FORCE
raise NotImplementedError(
    "APPEND de PATSEP_H a TABLAS.BD_CTSI: no materializado por fuente externa no resuelta"
)


In [ ]:
# DROP TABLE PATSEP_H,PATSEP_D
for t in ["#patsep_h", "#patsep_d"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. AJUSTA INTERESES RECIBIDOS POR AUXILIARES CON CONTRAGENTE OFIS,BCENTRAL Y EMPRESAS USANDO DATOS DCV. SE AJUSTA CONTRA BANCOS
# DATOS DCV INTERESES RECIBIDOS POR AUX DESDE OFIS, BCENTRAL Y EMP -- misma fuente externa no resuelta
raise NotImplementedError(
    "AUX_DCV: fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat' "
    "es un archivo SAS externo fuera del catalogo de conexiones de BD y fuera de input_datasets; "
    "se requiere definir su reemplazo antes de traducir este bloque"
)


In [ ]:
# INTERESES RECIBIDOS POR AUXILIARES DESDE BCENTRAL, OFIS Y EMP EN CI A RESTAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_aux_aj"))
sql_d41_aux_aj = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, 36 AS SECTOR, SUBSTRING(T1.C_CAGENTE, 1, 2) AS C_CAGENTE,
       T1.C_CUENTA, T1.C_ENTRADA, SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0755j' AS PROC
INTO #d41_aux_aj
FROM TABLAS.dbo.BD_CTSI T1
WHERE SUBSTRING(CAST(T1.SECTOR AS varchar(8)), 1, 2) = '36' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41'
  AND SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('33', '51', '31', '53')
  AND T1.C_CAGENTE NOT IN ('3390101', '511', '5111')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, SUBSTRING(T1.C_CAGENTE, 1, 2)
"""
work_conn.execute(text(sql_d41_aux_aj))


In [ ]:
# data AUX_DCV;set AUX_DCV D41_AUX_AJ;run; -- AUX_DCV no existe (bloque previo no resuelto)
raise NotImplementedError(
    "concatenacion AUX_DCV + D41_AUX_AJ: AUX_DCV no se pudo materializar porque su fuente "
    "es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# calcula delta para ajustarse a dato DCV
raise NotImplementedError(
    "AUX_DCV_delta: depende de AUX_DCV, no materializado por fuente externa no resuelta"
)


In [ ]:
# IMPUTACIÓN PARA AJUSTAR EN BANCOS (CONTRA AJUSTE)
raise NotImplementedError(
    "AUX_BCO_delta: depende de AUX_DCV_delta, no materializado por fuente externa no resuelta"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=AUX_DCV_delta FORCE
raise NotImplementedError(
    "APPEND de AUX_DCV_delta a TABLAS.BD_CTSI: no materializado por fuente externa no resuelta"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=AUX_BCO_delta FORCE
raise NotImplementedError(
    "APPEND de AUX_BCO_delta a TABLAS.BD_CTSI: no materializado por fuente externa no resuelta"
)


In [ ]:
# DROP TABLE AUX_DCV, D41_AUX_AJ,AUX_DCV_delta,AUX_BCO_delta
for t in ["#aux_dcv", "#d41_aux_aj", "#aux_dcv_delta", "#aux_bco_delta"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INGRESO DE LOS SECTORES OFIS, FP, SEGUROS, AUX FIN Y FM MM EN GASTO DE GOBIERNO.
# REALIZA CONTRA AJUSTE EN CA 6 POR CAMBIOS CIERRE 2021. CIERRE 2021: INCORPORA TMB IMPUTACIÓN DE INTERESES RECIBIDOS POR BCENTRAL CA GOB
work_conn.execute(text("DROP TABLE IF EXISTS #d41_gob_a"))
sql_d41_gob_a = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
       CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0756' AS PROC
INTO #d41_gob_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.SECTOR IN (4, 41, 42, 412, 413)
       AND (SUBSTRING(T1.C_CAGENTE, 1, 2) IN ('36', '33', '37', '35', '39', '34', '31') OR T1.C_CAGENTE = '411'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41'
       AND (SUBSTRING(LTRIM(CAST(T1.SECTOR AS varchar(7))), 1, 2) IN ('36', '33', '37', '35', '39', '34', '31') OR T1.SECTOR = 411)
       AND SUBSTRING(T1.C_CAGENTE, 1, 1) IN ('4') AND T1.C_CAGENTE NOT IN ('411'))
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
         CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END,
         T1.C_CUENTA,
         CASE WHEN T1.C_ENTRADA = 'H' THEN 'D' ELSE 'D' END,
         T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_gob_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D41 CA 6 DEL SECTOR GOBIERNO
work_conn.execute(text("DROP TABLE IF EXISTS #d41_4_53"))
sql_d41_4_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, '6' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, T1.FUENTE, '0756b' AS PROC
INTO #d41_4_53
FROM #d41_gob_a T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d41_4_53))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_GOB_A FORCE
cols_d41_gob_a = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_gob_a = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_gob_a})
SELECT {cols_d41_gob_a}
FROM #d41_gob_a
"""
res = work_conn.execute(text(sql_append_d41_gob_a))
_log("APPEND TABLAS.BD_CTSI (D41_GOB_A)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_4_53 FORCE
cols_d41_4_53 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_4_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_4_53})
SELECT {cols_d41_4_53}
FROM #d41_4_53
"""
res = work_conn.execute(text(sql_append_d41_4_53))
_log("APPEND TABLAS.BD_CTSI (D41_4_53)", res.rowcount)


In [ ]:
# DROP TABLE D41_GOB_A,D41_4_53
for t in ["#d41_gob_a", "#d41_4_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INGRESO DE LOS SECTORES GOB, HOGARES, BCOS, OFIS, FP, SEGUROS, AUX, FM MM EN GASTO DE RM.
# REALIZA CONTRA AJUSTE EN CA 53. CIERRE 2021: ELIMINA CA 334 YA QUE SE CAMBIA A OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_rm_a"))
sql_d41_rm_a = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
       CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0757' AS PROC
INTO #d41_rm_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.SECTOR IN (6) AND T1.C_CAGENTE NOT IN ('31', '53', '5101'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41' AND T1.SECTOR NOT IN (31, 5101, 51021, 51022) AND T1.C_CAGENTE = '6')
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
         CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END,
         T1.C_CUENTA,
         T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_rm_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D41 CA 53 DEL SECTOR RM
work_conn.execute(text("DROP TABLE IF EXISTS #d41_6_53"))
sql_d41_6_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, '53' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, T1.FUENTE, '0757b' AS PROC
INTO #d41_6_53
FROM #d41_rm_a T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d41_6_53))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_RM_A FORCE
cols_d41_rm_a = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_rm_a = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_rm_a})
SELECT {cols_d41_rm_a}
FROM #d41_rm_a
"""
res = work_conn.execute(text(sql_append_d41_rm_a))
_log("APPEND TABLAS.BD_CTSI (D41_RM_A)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_6_53 FORCE
cols_d41_6_53 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_6_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_6_53})
SELECT {cols_d41_6_53}
FROM #d41_6_53
"""
res = work_conn.execute(text(sql_append_d41_6_53))
_log("APPEND TABLAS.BD_CTSI (D41_6_53)", res.rowcount)


In [ ]:
# DROP TABLE D41_RM_A,D41_6_53
for t in ["#d41_rm_a", "#d41_6_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INGRESO DE LOS SECTORES GOB, BCOS (POR CIERRE 2021 NO SE AJUSTA, SE DEJA RESIDUAL), OFIS, FP, SEGUROS, AUX, FM MM EN GASTO DE BANCO CENTRAL.
# REALIZA CONTRA AJUSTE EN CA 53. CIERRE 2021: LOS AJUSTES SE HACEN CONTRA BANCOS COMERCIALES PARA DEJARLO COMO SECTOR RESIDUAL, ADEMÁS ELIMINA CA 334 PORQUE SE DEJA EN AUXILIARES
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bc_a"))
sql_d41_bc_a = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END AS SECTOR,
       CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0758' AS PROC
INTO #d41_bc_a
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41' AND T1.SECTOR IN (31)
       AND T1.C_CAGENTE NOT IN ('6', '511', '8', '31', '53', '9', '321'))
   OR (T1.C_ENTRADA = 'H' AND T1.C_SCN = 'D.41'
       AND T1.SECTOR NOT IN (31, 5101, 51021, 51022, 6, 511, 8, 321) AND T1.C_CAGENTE = '31')
GROUP BY T1.[AÑO], T1.TRIM,
         CASE WHEN T1.C_ENTRADA = 'H' THEN TRY_CAST(T1.C_CAGENTE AS float) ELSE T1.SECTOR END,
         CASE WHEN T1.C_ENTRADA = 'H' THEN LTRIM(CAST(T1.SECTOR AS varchar(7))) ELSE T1.C_CAGENTE END,
         T1.C_CUENTA,
         T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d41_bc_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D41 CA 53 DEL SECTOR BANCO CENTRAL. CIERRE 2021: SE AJUSTA EN BANCOS COMERCIALES NO EN EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_31_53"))
sql_d41_31_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, '321' AS C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, T1.FUENTE, '0758b' AS PROC
INTO #d41_31_53
FROM #d41_bc_a T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE
"""
work_conn.execute(text(sql_d41_31_53))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_BC_A FORCE
cols_d41_bc_a = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_bc_a = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_bc_a})
SELECT {cols_d41_bc_a}
FROM #d41_bc_a
"""
res = work_conn.execute(text(sql_append_d41_bc_a))
_log("APPEND TABLAS.BD_CTSI (D41_BC_A)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_31_53 FORCE
cols_d41_31_53 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_31_53 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_31_53})
SELECT {cols_d41_31_53}
FROM #d41_31_53
"""
res = work_conn.execute(text(sql_append_d41_31_53))
_log("APPEND TABLAS.BD_CTSI (D41_31_53)", res.rowcount)


In [ ]:
# DROP TABLE D41_BC_A,D41_31_53
for t in ["#d41_bc_a", "#d41_31_53"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2021. AJUSTA INTERESES PAGADOS POR BANCO CENTRAL A EMPRESAS Y AUXILIARES POR CONCEPTO DE BONOS USANDO DCV
# DATOS DCV INTERESES PAGADOS POR BCENTRAL A AUXILIARES Y EMPRESAS -- misma fuente externa no resuelta
raise NotImplementedError(
    "BCCH_PAG: fuente '/sasdata/BCCH/GEM_DCNI/02_CNSI/05_DCV/Data/DCVRES/BASE_D41_FINAL.sas7bdat' "
    "es un archivo SAS externo fuera del catalogo de conexiones de BD y fuera de input_datasets; "
    "se requiere definir su reemplazo antes de traducir este bloque"
)


In [ ]:
# INTERESES PAGADOS POR BCCH A EMPRESAS Y AUXILIARES A ELIMINAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_53"))
sql_d41_bcch_53 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO, T1.C_SCN, T1.N_SCN, 'PS' AS FUENTE, '0758b1' AS PROC
INTO #d41_bcch_53
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR = 31 AND T1.C_ENTRADA = 'D' AND T1.C_SCN = 'D.41'
  AND (T1.C_CAGENTE IN ('53', '5101', '51021', '51022', '37', '', '9') OR SUBSTRING(T1.C_CAGENTE, 1, 2) = '36')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.C_CAGENTE
"""
work_conn.execute(text(sql_d41_bcch_53))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_BCCH_53 FORCE (primera pasada)
cols_d41_bcch_53 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_bcch_53_1 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_bcch_53})
SELECT {cols_d41_bcch_53}
FROM #d41_bcch_53
"""
res = work_conn.execute(text(sql_append_d41_bcch_53_1))
_log("APPEND TABLAS.BD_CTSI (D41_BCCH_53 1ra pasada)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=BCCH_PAG FORCE (primera pasada)
raise NotImplementedError(
    "APPEND de BCCH_PAG a TABLAS.BD_CTSI (1ra pasada): BCCH_PAG no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# AJUSTA INT ELIMINADOS DE EMPRESAS Y AUX Y LO LLEVA A SECTOR DE AJUSTE QUE ES BANCOS
work_conn.execute(text("UPDATE #d41_bcch_53 SET DATO = DATO * -1, C_CAGENTE = '321'"))


In [ ]:
# AJUSTA INT DCV IMPUTADDOS EN AUX Y EMPRESAS CONTRA BANCOS
raise NotImplementedError(
    "UPDATE BCCH_PAG SET DATO=DATO*-1, C_CAGENTE='321': BCCH_PAG no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=D41_BCCH_53 FORCE (segunda pasada, ya ajustado)
sql_append_d41_bcch_53_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_d41_bcch_53})
SELECT {cols_d41_bcch_53}
FROM #d41_bcch_53
"""
res = work_conn.execute(text(sql_append_d41_bcch_53_2))
_log("APPEND TABLAS.BD_CTSI (D41_BCCH_53 2da pasada)", res.rowcount)


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=BCCH_PAG FORCE (segunda pasada)
raise NotImplementedError(
    "APPEND de BCCH_PAG a TABLAS.BD_CTSI (2da pasada): BCCH_PAG no se pudo materializar "
    "porque su fuente es el archivo externo BASE_D41_FINAL.sas7bdat"
)


In [ ]:
# DROP TABLE BCCH_PAG,D41_BCCH_53
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcch_53"))
# #bcch_pag nunca se creo (fuente externa no resuelta); DROP IF EXISTS es no-op seguro
work_conn.execute(text("DROP TABLE IF EXISTS #bcch_pag"))


In [ ]:
# Este nodo se procesa en tramos; este es el TRAMO 5 de 8 (offsets ~8594-14051 del SAS original)
# Ajusta imputación de SIFMI de bancos (H) y su contrapartida en CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_bcos_h"))
sql_sifmi_bcos_h = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       t1.TRIM,
       321 AS SECTOR,
       LTRIM(CAST(t1.SECTOR AS varchar(7))) AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(t1.DATO) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       t1.FUENTE,
       '0758c' AS PROC
INTO #sifmi_bcos_h
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.41' AND t1.SECTOR IN (511,41,36,35)
  AND t1.C_CAGENTE IN ('321','322','32') AND t1.FUENTE = 'DI_Aj_SIFMI'
GROUP BY t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CUENTA, t1.C_SCN, t1.N_SCN, t1.FUENTE
"""
work_conn.execute(text(sql_sifmi_bcos_h))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN CA 53 DEL SECTOR BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_321_53"))
sql_sifmi_321_53 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, '0758d' AS PROC
INTO #sifmi_321_53
FROM #sifmi_bcos_h
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_sifmi_321_53))


In [ ]:
# PROC APPEND de ambas tablas a TABLAS.BD_CTSI (server-side, acumulativo — re-ejecutar duplica, igual que el SAS original)
cols_bd_ctsi_sifmi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_sifmi}) SELECT {cols_bd_ctsi_sifmi} FROM #sifmi_bcos_h"))
_log("APPEND TABLAS.dbo.BD_CTSI (sifmi_bcos_h)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_sifmi}) SELECT {cols_bd_ctsi_sifmi} FROM #sifmi_321_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (sifmi_321_53)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_bcos_h"))
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_321_53"))


In [ ]:
# CIERRE 2019. IMPUTA INGRESO DE LOS SECTORES GOB,HH, BCOS, OFIS, FP, SEGUROS, AUX, FM MM EN GASTO DE BCOS.
# REALIZA CONTRA AJUSTE EN CA 53. CIERRE 2021: AGREGA IMPUTACIÒN DE DATO BANCO CENTRAL Y 34 PORQUE AHORA ES AUXILIAR
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcos_a"))
sql_d41_bcos_a = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(CAST(t1.SECTOR AS varchar(7))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0758c' AS PROC
INTO #d41_bcos_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.41' AND t1.SECTOR IN (321,322,32)
       AND t1.C_CAGENTE NOT IN ('6','53','9','51',''))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.41' AND t1.SECTOR NOT IN (5101,51021,51022,6,51)
       AND t1.C_CAGENTE IN ('321','322','32','3'))
GROUP BY t1.[AÑO], t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(CAST(t1.SECTOR AS varchar(7))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_d41_bcos_a))


In [ ]:
# Corrige SECTOR calculado 3 -> 32 (arrastre de INPUT sobre C_CAGENTE='32')
work_conn.execute(text("UPDATE #d41_bcos_a SET SECTOR = 32 WHERE SECTOR = 3"))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D41 CA 53 DEL SECTOR BCOS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_321_53"))
sql_d41_321_53 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, '53' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, '0758d' AS PROC
INTO #d41_321_53
FROM #d41_bcos_a
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_d41_321_53))


In [ ]:
# PROC APPEND server-side (acumulativo — re-ejecutar duplica, igual que el SAS original)
cols_bd_ctsi_d41 = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_d41}) SELECT {cols_bd_ctsi_d41} FROM #d41_bcos_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_bcos_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_d41}) SELECT {cols_bd_ctsi_d41} FROM #d41_321_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_321_53)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_bcos_a"))
work_conn.execute(text("DROP TABLE IF EXISTS #d41_321_53"))


In [ ]:
# CIERRE 2021. AJUSTA INTERESES PAGADOS POR BANCOS A RESTO DEL MUNDO, DADO QUE LA CI NO CONTIENE LOS PAGADOS POR BONOS Y DEPÒSITOS. SE AJUSTA CONTRA EMPRESAS
# ACTIVO DEL RM CON BANCOS POR CONCEPTO DE TÌTULOS Y DEPÒSITOS
work_conn.execute(text("DROP TABLE IF EXISTS #bcos_rm"))
sql_bcos_rm = """
SELECT [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, SUM(DATO) AS DATO
INTO #bcos_rm
FROM TABLAS.dbo.BD_CTSI
WHERE SECTOR = 6 AND C_CAGENTE IN ('321') AND C_SCN IN ('AF.22','AF.29','AF.31','AF.32')
  AND C_ENTRADA = 'D' AND C_CUENTA IN ('Bce Final','Bce Inicio')
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_bcos_rm))


In [ ]:
# SALDOS PROMEDIOS
work_conn.execute(text("DROP TABLE IF EXISTS #bcos_rm_avg"))
sql_bcos_rm_avg = """
SELECT [AÑO], TRIM, SECTOR, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN, AVG(DATO) AS DATO
INTO #bcos_rm_avg
FROM #bcos_rm
GROUP BY [AÑO], TRIM, SECTOR, C_CAGENTE, C_ENTRADA, C_SCN, N_SCN
"""
work_conn.execute(text(sql_bcos_rm_avg))


In [ ]:
# Origen: archivo SAS externo '/sasdata/BCCH/GEM_DCNI/02_CNSI/09_SI_BCOS/321_BCOS/tasas.sas7bdat' (tasas implícitas pagadas desde bancos, usadas en CA RM).
# M-001: ruta hardcodeada reemplazada por ruta relativa al workspace del proyecto.
ruta_tasas = Path("datos_externos") / "321_BCOS" / "tasas.sas7bdat"
tasas_raw, _ = pyreadstat.read_sas7bdat(str(ruta_tasas))
tasas_raw.columns = [c.upper() for c in tasas_raw.columns]
tasas = tasas_raw[
    tasas_raw["C_SCN"].isin(["AF.22", "AF.29", "AF.31", "AF.32"])
    & tasas_raw["C_CAGENTE"].isin(["53"])
    & (tasas_raw["C_ENTRADA"] == "D")
][["AÑO", "TRIM", "C_CAGENTE", "C_SCN", "C_ENTRADA", "TASA"]]
# Sube la tabla de tasas a la sesión SQL para cruzarla server-side con #bcos_rm_avg
tasas.to_sql("#tasas", work_conn, if_exists="replace", index=False)
_log("tasas", tasas)

In [ ]:
# INTERESES A ADICIONAR A LO PAGADO POR BANCO AL RM POR BONOS Y DEP
work_conn.execute(text("DROP TABLE IF EXISTS #int_bcos_rm"))
sql_int_bcos_rm = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM,
       TRY_CAST(t1.C_CAGENTE AS float) AS SECTOR,
       LTRIM(CAST(t1.SECTOR AS varchar(2))) AS C_CAGENTE,
       'D.41' AS C_SCN, 'Intereses' AS N_SCN, t1.C_ENTRADA, 'YG' AS C_CUENTA,
       SUM(t1.DATO * t2.TASA / 100) AS DATO, 'PS' AS FUENTE, '0758d1' AS PROC
INTO #int_bcos_rm
FROM #bcos_rm_avg t1
INNER JOIN #tasas t2
  ON t1.[AÑO] = t2.[AÑO] AND t1.TRIM = t2.TRIM AND t1.C_ENTRADA = t2.C_ENTRADA AND t1.C_SCN = t2.C_SCN
GROUP BY t1.[AÑO], t1.TRIM, t1.C_CAGENTE, t1.SECTOR, t1.C_ENTRADA
"""
work_conn.execute(text(sql_int_bcos_rm))


In [ ]:
# PROC APPEND server-side, primera versión de int_bcos_rm (acumulativo)
cols_bd_ctsi_int = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_SCN, N_SCN, C_ENTRADA, C_CUENTA, DATO, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_int}) SELECT {cols_bd_ctsi_int} FROM #int_bcos_rm"))
_log("APPEND TABLAS.dbo.BD_CTSI (int_bcos_rm)", res.rowcount)


In [ ]:
# AJUSTE CONTRA EMPRESAS: se actualiza la #tmp en la sesión antes del segundo append
work_conn.execute(text("UPDATE #int_bcos_rm SET C_CAGENTE = '53', DATO = DATO * -1"))
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_int}) SELECT {cols_bd_ctsi_int} FROM #int_bcos_rm"))
_log("APPEND TABLAS.dbo.BD_CTSI (int_bcos_rm ajustado)", res.rowcount)
for t in ["#bcos_rm", "#bcos_rm_avg", "#tasas", "#int_bcos_rm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INGRESO DE LOS SECTORES BCENTRAL, FP Y SEGUROS, y OFIS EN GASTO DE OFIS.
# REALIZA CONTRA AJUSTE EN CA 321. CIERRE 2021. SE INCORPORA IMPUTACION DE INGRESOS DE AUXILIARES EN OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_ofis_a"))
sql_d41_ofis_a = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(CAST(t1.SECTOR AS varchar(7))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0758e' AS PROC
INTO #d41_ofis_a
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.41'
       AND SUBSTRING(LTRIM(CAST(t1.SECTOR AS varchar(7))), 1, 2) IN ('33')
       AND SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('33','34','31','35','36'))
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.41'
       AND SUBSTRING(LTRIM(CAST(t1.SECTOR AS varchar(7))), 1, 2) IN ('34','35','31','33','36')
       AND (SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('33') OR t1.C_CAGENTE = '411')
       AND t1.C_CAGENTE NOT IN ('331'))
GROUP BY t1.[AÑO], t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(CAST(t1.SECTOR AS varchar(7))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_d41_ofis_a))


In [ ]:
# AJUSTA IMPUTACIÓN ANTERIOR EN D41 CA 321 DEL SECTOR OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #d41_33_53"))
sql_d41_33_53 = """
SELECT MONEDA, [AÑO], TRIM, SECTOR, '321' AS C_CAGENTE, C_CUENTA, C_ENTRADA,
       SUM(DATO) * -1 AS DATO, C_SCN, N_SCN, FUENTE, '0758f' AS PROC
INTO #d41_33_53
FROM #d41_ofis_a
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, C_SCN, N_SCN, FUENTE
"""
work_conn.execute(text(sql_d41_33_53))


In [ ]:
# PROC APPEND server-side (acumulativo)
cols_bd_ctsi_ofis = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_ofis}) SELECT {cols_bd_ctsi_ofis} FROM #d41_ofis_a"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_ofis_a)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_ofis}) SELECT {cols_bd_ctsi_ofis} FROM #d41_33_53"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_33_53)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_ofis_a"))
work_conn.execute(text("DROP TABLE IF EXISTS #d41_33_53"))


In [ ]:
# CIERRE 2021. IMPUTA INTERESES PAGADOS POR OFIS A EMPRESAS POR TENENCIA DE BONOS EN EMPRESAS DEL SECTOR.
# SE USA TASA IMPLÍCITA DEL DCV APLICADA AL TOTAL DE BONOS EN ACTIVO DE EMPRESAS CA OFIS
# DATOS DCV INTERESES PAGADOS POR OFIS — origen: archivo SAS externo BASE_D41_FINAL.sas7bdat
# M-001: ruta hardcodeada reemplazada por ruta relativa al workspace del proyecto.
ruta_dcv_final = Path("datos_externos") / "05_DCV" / "Data" / "DCVRES" / "BASE_D41_FINAL.sas7bdat"
dcv_final_raw, _ = pyreadstat.read_sas7bdat(str(ruta_dcv_final))
dcv_final_raw.columns = [c.upper() for c in dcv_final_raw.columns]
dcv_final_filtro = dcv_final_raw[
    (
        (dcv_final_raw["C_SI_EMISOR_N"].astype(str).str[:2] == "33")
        | (dcv_final_raw["C_SI_EMISOR_N"] == "411")
        | (dcv_final_raw["C_SI_EMISOR_N"] == "324")
    )
    & (dcv_final_raw["C_SI_EMISOR_N"] != "334")
    & (dcv_final_raw["AÑO"] > 2002)
]
ofis_int = (
    dcv_final_filtro.assign(TRIM=dcv_final_filtro["TRIMESTRE"], SECTOR=33)
    .groupby(["AÑO", "TRIM", "SECTOR"], as_index=False)["DATO"].sum()
)
ofis_int.to_sql("#ofis_int", work_conn, if_exists="replace", index=False)
_log("ofis_int", ofis_int)

In [ ]:
# BONOS EMITIDOS OFIS MERCADO LOCAL
work_conn.execute(text("DROP TABLE IF EXISTS #ofis_bonos"))
sql_ofis_bonos = """
SELECT [AÑO], TRIM, 33 AS SECTOR, SUM(DATO) AS DATO
INTO #ofis_bonos
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'H' AND C_CUENTA = 'Bce Final' AND SUBSTRING(C_SCN, 1, 4) = 'AF.3'
  AND (SUBSTRING(LTRIM(CAST(SECTOR AS varchar(7))), 1, 2) IN ('33') OR SECTOR = 411)
  AND SECTOR NOT IN (334) AND C_CAGENTE NOT IN ('6')
GROUP BY [AÑO], TRIM
"""
work_conn.execute(text(sql_ofis_bonos))


In [ ]:
# TASA IMPLICITA BONOS OFIS
work_conn.execute(text("DROP TABLE IF EXISTS #tasa_b_ofis"))
sql_tasa_b_ofis = """
SELECT t1.[AÑO], t1.TRIM, t1.SECTOR, t2.DATO / t1.DATO AS TASA
INTO #tasa_b_ofis
FROM #ofis_bonos t1
INNER JOIN #ofis_int t2 ON t1.[AÑO] = t2.[AÑO] AND t1.TRIM = t2.TRIM AND t1.SECTOR = t2.SECTOR
"""
work_conn.execute(text(sql_tasa_b_ofis))
tasa_b_ofis = pd.read_sql(text("SELECT * FROM #tasa_b_ofis"), work_conn)
_log("tasa_b_ofis", tasa_b_ofis)


In [ ]:
# BONOS EMITIDOS OFIS EN PODER DE EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #ofis_bonos_emp"))
sql_ofis_bonos_emp = """
SELECT [AÑO], TRIM, 51 AS SECTOR, SUM(DATO) AS DATO
INTO #ofis_bonos_emp
FROM TABLAS.dbo.BD_CTSI
WHERE C_ENTRADA = 'D' AND C_CUENTA = 'Bce Final' AND SUBSTRING(C_SCN, 1, 4) = 'AF.3'
  AND (SUBSTRING(LTRIM(CAST(SECTOR AS varchar(7))), 1, 2) IN ('51') OR SECTOR = 334 OR SECTOR = 53)
  AND SECTOR NOT IN (511)
  AND (SUBSTRING(C_CAGENTE, 1, 2) = '33' OR C_CAGENTE = '411') AND [AÑO] > 2002
GROUP BY [AÑO], TRIM
"""
work_conn.execute(text(sql_ofis_bonos_emp))


In [ ]:
# INTERESES A IMPUTAR CON CA EMPRESAS POR BONOS
work_conn.execute(text("DROP TABLE IF EXISTS #int_ofis_emp"))
sql_int_ofis_emp = """
SELECT t1.[AÑO], t1.TRIM, t1.SECTOR, '51' AS C_CAGENTE, 'D' AS C_ENTRADA, 'YG' AS C_CUENTA,
       'D.41' AS C_SCN, 'Intereses' AS N_SCN, 'PS' AS FUENTE, '0758f' AS PROC,
       t1.TASA * t2.DATO AS DATO
INTO #int_ofis_emp
FROM #tasa_b_ofis t1
INNER JOIN #ofis_bonos_emp t2 ON t1.[AÑO] = t2.[AÑO] AND t1.TRIM = t2.TRIM
"""
work_conn.execute(text(sql_int_ofis_emp))


In [ ]:
# AJUSTE IMPUTACIÓN EN CA BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #int_ofis_bcos"))
sql_int_ofis_bcos = """
SELECT [AÑO], TRIM, SECTOR, '321' AS C_CAGENTE, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, FUENTE,
       '0758g' AS PROC, DATO * -1 AS DATO
INTO #int_ofis_bcos
FROM #int_ofis_emp
"""
work_conn.execute(text(sql_int_ofis_bcos))


In [ ]:
# PROC APPEND server-side (acumulativo)
cols_bd_ctsi_int_ofis = "[AÑO], TRIM, SECTOR, C_CAGENTE, C_ENTRADA, C_CUENTA, C_SCN, N_SCN, FUENTE, PROC, DATO"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_int_ofis}) SELECT {cols_bd_ctsi_int_ofis} FROM #int_ofis_emp"))
_log("APPEND TABLAS.dbo.BD_CTSI (int_ofis_emp)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_int_ofis}) SELECT {cols_bd_ctsi_int_ofis} FROM #int_ofis_bcos"))
_log("APPEND TABLAS.dbo.BD_CTSI (int_ofis_bcos)", res.rowcount)
for t in ["#ofis_int", "#ofis_bonos", "#tasa_b_ofis", "#ofis_bonos_emp", "#int_ofis_emp", "#int_ofis_bcos"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019. IMPUTA INTERESES PAGADOS POR AUXILIARES (SECTOR 37) A BCENTRAL USANDO INFO DEL BANCO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #d41_aux_bc"))
sql_d41_aux_bc = """
SELECT 'P' AS MONEDA,
       t1.[AÑO],
       t1.TRIM,
       CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END AS SECTOR,
       CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(CAST(t1.SECTOR AS varchar(7))) ELSE t1.C_CAGENTE END AS C_CAGENTE,
       t1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN t1.C_ENTRADA = 'D' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
       t1.C_SCN,
       t1.N_SCN,
       'PS' AS FUENTE,
       '0758g' AS PROC
INTO #d41_aux_bc
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.41'
       AND (SUBSTRING(LTRIM(CAST(t1.SECTOR AS varchar(7))), 1, 2) IN ('36') OR t1.SECTOR = 37)
       AND t1.C_CAGENTE = '31')
   OR (t1.C_ENTRADA = 'H' AND t1.C_SCN = 'D.41' AND t1.SECTOR = 31
       AND (SUBSTRING(t1.C_CAGENTE, 1, 2) IN ('36') OR t1.C_CAGENTE = '37'))
GROUP BY t1.[AÑO], t1.TRIM,
         CASE WHEN t1.C_ENTRADA = 'H' THEN TRY_CAST(t1.C_CAGENTE AS float) ELSE t1.SECTOR END,
         CASE WHEN t1.C_ENTRADA = 'H' THEN LTRIM(CAST(t1.SECTOR AS varchar(7))) ELSE t1.C_CAGENTE END,
         t1.C_CUENTA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_d41_aux_bc))


In [ ]:
# intereses pagados sector 37 con contragente (SIN CA 31)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_sc_ca"))
sql_d41_sc_ca = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       SUM(t1.DATO) AS DATO, t1.C_SCN, t1.N_SCN
INTO #d41_sc_ca
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.41' AND t1.SECTOR = 37
  AND t1.C_CAGENTE NOT IN ('31') AND t1.FUENTE NOT IN ('DI_Aj_REAJ')
GROUP BY t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN
"""
work_conn.execute(text(sql_d41_sc_ca))


In [ ]:
# intereses pagados sector 37 sin contragente
work_conn.execute(text("DROP TABLE IF EXISTS #d41_sc"))
sql_d41_sc = """
SELECT 'P' AS MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, SUM(t1.DATO) AS DATO
INTO #d41_sc
FROM TABLAS.dbo.BD_CTSI t1
WHERE t1.C_ENTRADA = 'D' AND t1.C_SCN = 'D.41' AND t1.SECTOR = 37 AND t1.FUENTE NOT IN ('DI_Aj_REAJ')
GROUP BY t1.[AÑO], t1.TRIM, t1.SECTOR
"""
work_conn.execute(text(sql_d41_sc))


In [ ]:
# estructura contragente
work_conn.execute(text("DROP TABLE IF EXISTS #d41_sc_est"))
sql_d41_sc_est = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, t1.C_CAGENTE, t1.DATO / t2.DATO AS EST
INTO #d41_sc_est
FROM #d41_sc_ca t1
INNER JOIN #d41_sc t2 ON t1.[AÑO] = t2.[AÑO] AND t1.SECTOR = t2.SECTOR AND t1.MONEDA = t2.MONEDA AND t1.TRIM = t2.TRIM
"""
work_conn.execute(text(sql_d41_sc_est))


In [ ]:
# AJUSTA IMPUTACIÓN EN D41 CA POR ESTRUCTURA DEL SECTOR AUXILIARES (SECTOR 37)
work_conn.execute(text("DROP TABLE IF EXISTS #d41_aux_aj"))
sql_d41_aux_aj = """
SELECT t1.MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, t2.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA,
       (t1.DATO * t2.EST) * -1 AS DATO, t1.C_SCN, t1.N_SCN, t1.FUENTE, '0758h' AS PROC
INTO #d41_aux_aj
FROM #d41_aux_bc t1
INNER JOIN #d41_sc_est t2
  ON t1.[AÑO] = t2.[AÑO] AND t1.SECTOR = t2.SECTOR AND t1.MONEDA = t2.MONEDA AND t1.TRIM = t2.TRIM
GROUP BY t1.MONEDA, t1.[AÑO], t1.TRIM, t1.SECTOR, t2.C_CAGENTE, t1.C_CUENTA, t1.C_ENTRADA, t1.C_SCN, t1.N_SCN, t1.FUENTE
"""
work_conn.execute(text(sql_d41_aux_aj))


In [ ]:
# PROC APPEND server-side (acumulativo) y limpieza de las #tmp de este tramo
cols_bd_ctsi_aux = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_aux}) SELECT {cols_bd_ctsi_aux} FROM #d41_aux_bc"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_aux_bc)", res.rowcount)
res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi_aux}) SELECT {cols_bd_ctsi_aux} FROM #d41_aux_aj"))
_log("APPEND TABLAS.dbo.BD_CTSI (d41_aux_aj)", res.rowcount)
for t in ["#d41_aux_bc", "#d41_aux_aj", "#d41_sc_ca", "#d41_sc", "#d41_sc_est"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CALCULA IMP A SECTOR 51022 CON CA RESPECTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #d41_delta"))
sql_d41_delta = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #d41_delta
FROM #d41_recibido T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_ENTRADA, T1.C_CUENTA, T1.C_SCN, T1.N_SCN, T1.SECTOR, T1.C_CAGENTE, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_d41_delta))


In [ ]:
# APPEND server-side de D41_DELTA a TABLAS.BD_CTSI (FORCE alinea por nombre)
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_d41_delta = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d41_delta
"""
res = work_conn.execute(text(sql_append_d41_delta))
_log("APPEND TABLAS.BD_CTSI (D41_DELTA)", res.rowcount)


In [ ]:
# DROP TABLE D41_DELTA, D41_PAGADO, D41_RECIBIDO (temporales de sesión)
for t in ["#d41_delta", "#d41_pagado", "#d41_recibido"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA TRANSFERENCIAS DE CAPITAL EN SECTOR 5101 CON CA 41 USANDO INFO DE GOBIERNO
work_conn.execute(text("DROP TABLE IF EXISTS #d9_5101"))
sql_d9_5101 = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.C_SI AS SECTOR,
       '41' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO,
       T1.C_INSTRUMENTO_SCN AS C_SCN,
       'Transferencias de capital' AS N_SCN,
       'PS' AS FUENTE,
       '0760' AS PROC
INTO #d9_5101
FROM TABLAS.dbo.T_TK_GG_EPU T1
"""
work_conn.execute(text(sql_d9_5101))


In [ ]:
sql_append_d9_5101 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d9_5101
"""
res = work_conn.execute(text(sql_append_d9_5101))
_log("APPEND TABLAS.BD_CTSI (D9_5101)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d9_5101"))


In [ ]:
# CAMBIA D9 RECIBIDAS POR SECTOR 41 A PAGADAS MULTIPLICANDO POR -1
sql_update_d9 = """
UPDATE TABLAS.dbo.BD_CTSI
SET DATO = DATO * -1, C_ENTRADA = 'D', PROC = '0761'
WHERE SECTOR = 41 AND C_CUENTA = 'Capital' AND C_SCN = 'D.9' AND C_ENTRADA = 'H'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_update_d9))
    _log("UPDATE TABLAS.BD_CTSI (D9 41->pagadas)", res.rowcount)


In [ ]:
# CIERRE DE TRANSFERENCIAS DE CAPITAL: el diferencial entre lo pagado y lo recibido se imputa a sector 51 (85%)
# IMPUTA 51
work_conn.execute(text("DROP TABLE IF EXISTS #d9_51"))
sql_d9_51 = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) * 1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0764a' AS PROC
INTO #d9_51
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN = 'D.9'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_d9_51))


In [ ]:
sql_append_d9_51 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #d9_51
"""
res = work_conn.execute(text(sql_append_d9_51))
_log("APPEND TABLAS.BD_CTSI (D9_51)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #d9_51"))


In [ ]:
# El bloque D9_511 (segunda vuelta, residual) quedó comentado/anulado en el SAS original: no se traduce (código muerto)


In [ ]:
# IMPUTA K21 EN SECTOR 51022 CON CA 41, USANDO INFO DE SECTOR 41
work_conn.execute(text("DROP TABLE IF EXISTS #k21"))
sql_k21 = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51022 AS SECTOR,
       '41' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0765' AS PROC
INTO #k21
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.C_SCN = 'K.21' AND T1.SECTOR = 41
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_k21))


In [ ]:
sql_append_k21 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #k21
"""
res = work_conn.execute(text(sql_append_k21))
_log("APPEND TABLAS.BD_CTSI (K21)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #k21"))


In [ ]:
# ELIMINA TRANSACCIONES DE SECTOR 5111
work_conn.execute(text("DROP TABLE IF EXISTS #elimina5111"))
sql_elimina5111 = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0766' AS PROC
INTO #elimina5111
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.SECTOR = 5111 AND T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('P.11','P.2','D.1','B.2','K.1'))
   OR (T1.SECTOR = 5111 AND T1.C_CUENTA = 'YG' AND T1.C_SCN IN ('B.2','B.8'))
   OR (T1.SECTOR = 5111 AND T1.C_CUENTA = 'Capital' AND T1.C_SCN IN ('K.1','B.8','B.9','P.51','P.52'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina5111))


In [ ]:
sql_append_elimina5111 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimina5111
"""
res = work_conn.execute(text(sql_append_elimina5111))
_log("APPEND TABLAS.BD_CTSI (ELIMINA5111)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimina5111"))


In [ ]:
# ELIMINA TRANSACCIONES DE CUENTAS DE PRODUCCIÓN DE TODOS LOS SECTORES. OK
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod"))
sql_elimina_prod = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0769' AS PROC
INTO #elimina_prod
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('P.11','P.13') AND T1.C_ENTRADA = 'H')
   OR (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('P.2') AND T1.C_ENTRADA = 'D')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina_prod))


In [ ]:
sql_append_elimina_prod = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimina_prod
"""
res = work_conn.execute(text(sql_append_elimina_prod))
_log("APPEND TABLAS.BD_CTSI (ELIMINA_PROD)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod"))


In [ ]:
# ELIMINA TRANSACCIONES DE CUENTAS DE PRODUCCIÓN DE TODOS LOS SECTORES EXCEPTO SECTOR GOBIERNO E IMPUTACIONES BANCARIAS
# CIERRE 2021: EXCEPTO EN SECTOR FINANCIERO DONDE SE IMPUTARÁN LAS DIFERENCIAS
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod2"))
sql_elimina_prod2 = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       (T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0770' AS PROC
INTO #elimina_prod2
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('B.2','D.1','D.21','D.29','K.1') AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 NOT IN ('S.13','S.9','S.12'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CAGENTE, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina_prod2))


In [ ]:
sql_append_elimina_prod2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimina_prod2
"""
res = work_conn.execute(text(sql_append_elimina_prod2))
_log("APPEND TABLAS.BD_CTSI (ELIMINA_PROD2)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_prod2"))


In [ ]:
# IMPUTA CCAS. CIERRE 2021: EXCEPTO EN SECTOR FINANCIERO DONDE SE IMPUTARÁN LAS DIFERENCIAS
work_conn.execute(text("DROP TABLE IF EXISTS #ccas"))
sql_ccas = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.C_SI AS SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0771' AS PROC
INTO #ccas
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE (T1.C_SCN = T2.C_SCN) AND ((T1.C_SI IN (511) AND T1.C_SCN <> 'B.1') OR (T1.C_SI = 4 AND T1.C_SCN NOT IN ('B.1','K.1')))
"""
work_conn.execute(text(sql_ccas))


In [ ]:
sql_append_ccas = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #ccas
"""
res = work_conn.execute(text(sql_append_ccas))
_log("APPEND TABLAS.BD_CTSI (CCAS)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #ccas"))


In [ ]:
# CIERRE 2021: CALCULA DIFERENCIA A IMPUTAR EN VARIABLES DE CTA DE PRODUCCION DEL SECTOR FINANCIERO
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_sf"))
sql_elimina_sf = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       3 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0771' AS PROC
INTO #elimina_sf
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_SCN IN ('B.2','D.1','D.21','D.29','K.1') AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 IN ('S.12'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimina_sf))


In [ ]:
# El bloque comentado que renombraba D.29 -> D.21 en ELIMINA_SF quedó anulado en el SAS original: no se traduce
work_conn.execute(text("UPDATE #elimina_sf SET C_SCN = 'B.2/B.3', N_SCN = 'Excedente de explotación/ Ingreso Mixto' WHERE C_SCN = 'B.2'"))


In [ ]:
# OBTIENE DATA DEL SF DESDE CCAS
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_sf"))
sql_ccas_sf = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.C_SI AS SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0771' AS PROC
INTO #ccas_sf
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE T1.C_SCN = T2.C_SCN AND T1.C_SI = 3 AND T1.C_SCN <> 'B.1'
"""
work_conn.execute(text(sql_ccas_sf))


In [ ]:
# DATA ELIMINA_SF; SET ELIMINA_SF CCAS_SF; -> concatenación server-side: reconstruye #elimina_sf con la unión
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_sf_concat"))
sql_concat_elimina_sf = """
SELECT * INTO #elimina_sf_concat FROM (
    SELECT * FROM #elimina_sf
    UNION ALL
    SELECT * FROM #ccas_sf
) u
"""
work_conn.execute(text(sql_concat_elimina_sf))
work_conn.execute(text("DROP TABLE IF EXISTS #elimina_sf"))
work_conn.execute(text("EXEC sp_rename '#elimina_sf_concat', '#elimina_sf'"))


In [ ]:
# CALCULA DIFERENCIAS A IMPUTAR EN SF
work_conn.execute(text("DROP TABLE IF EXISTS #dif_prod_sf"))
sql_dif_prod_sf = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #dif_prod_sf
FROM #elimina_sf T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_dif_prod_sf))


In [ ]:
sql_append_dif_prod_sf = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #dif_prod_sf
"""
res = work_conn.execute(text(sql_append_dif_prod_sf))
_log("APPEND TABLAS.BD_CTSI (DIF_PROD_SF)", res.rowcount)
for t in ["#dif_prod_sf", "#elimina_sf", "#ccas_sf"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR AGREGADO
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_2"))
sql_ccas_2 = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.C_SI AS SECTOR,
       'Producción' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0772' AS PROC
INTO #ccas_2
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE (T1.C_SCN = T2.C_SCN) AND (T1.C_SI IN (3,511) AND T1.C_SCN = 'B.1')
"""
work_conn.execute(text(sql_ccas_2))


In [ ]:
sql_append_ccas_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #ccas_2
"""
res = work_conn.execute(text(sql_append_ccas_2))
_log("APPEND TABLAS.BD_CTSI (CCAS_2)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_2"))


In [ ]:
# CAMBIA EN SECTOR GOBIERNO EL EXCEDENTE DE EXPLOTACIÓN A EXCEDENTE DE EXPLOTACIÓN E INGRESO MIXTO
sql_update_gob_exc = """
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'B.2/B.3', N_SCN = 'Excedente de explotación/ Ingreso Mixto'
WHERE C_ENTRADA = 'D' AND SECTOR = 41 AND C_SCN = 'B.2'
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_update_gob_exc))
    _log("UPDATE TABLAS.BD_CTSI (gobierno B.2->B.2/B.3)", res.rowcount)


In [ ]:
# CALCULA DIFERENCIAL ENTRE ECONOMIA NACIONAL Y LOS SECTORES PARA LA CUENTA DE PRODUCCIÓN. DIFERENCIA LO IMPUTA EN SECTOR 51
# SUMA DE SECTORES
work_conn.execute(text("DROP TABLE IF EXISTS #cierre_prod"))
sql_cierre_prod = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T3.N_SCN,
       'PS' AS FUENTE,
       '0777' AS PROC
INTO #cierre_prod
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2, TABLAS.dbo.T_INSTRUMENTOS T3
WHERE (T1.SECTOR = T2.C_SI AND T1.C_SCN = T3.C_SCN) AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T1.SECTOR NOT IN (6,412) AND T2.C_SI_SCN_N1 IN ('S.12','S.13','S.14') AND T1.C_SCN NOT IN ('P.2'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T3.N_SCN
"""
work_conn.execute(text(sql_cierre_prod))


In [ ]:
# DATA ECONOMIA NACIONAL
work_conn.execute(text("DROP TABLE IF EXISTS #ccas_en"))
sql_ccas_en = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51 AS SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       T1.C_SCN,
       T2.N_SCN,
       'PS' AS FUENTE,
       '0777' AS PROC
INTO #ccas_en
FROM TABLAS.dbo.T_CCAST T1, TABLAS.dbo.T_INSTRUMENTOS T2
WHERE (T1.C_SCN = T2.C_SCN) AND (T1.C_SI = 1 AND T1.C_SCN <> 'B.1')
"""
work_conn.execute(text(sql_ccas_en))


In [ ]:
# APPEND BASE=WORK.CIERRE_PROD DATA=WORK.CCAS_EN (destino WORK, no BD): append server-side sobre la #tmp
sql_append_cierre_prod = """
INSERT INTO #cierre_prod (MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC)
SELECT MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC
FROM #ccas_en
"""
res = work_conn.execute(text(sql_append_cierre_prod))
_log("APPEND #cierre_prod (CCAS_EN)", res.rowcount)


In [ ]:
# IMPUTACION
work_conn.execute(text("DROP TABLE IF EXISTS #prod_dif"))
sql_prod_dif = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #prod_dif
FROM #cierre_prod T1 WHERE T1.DATO <> 0
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.PROC, T1.FUENTE
"""
work_conn.execute(text(sql_prod_dif))


In [ ]:
sql_append_prod_dif = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #prod_dif
"""
res = work_conn.execute(text(sql_append_prod_dif))
_log("APPEND TABLAS.BD_CTSI (PROD_DIF)", res.rowcount)
for t in ["#prod_dif", "#cierre_prod", "#ccas_en"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA VALOR AGREGADO EN SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #va_51"))
sql_va_51 = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51 AS SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       'B.1' AS C_SCN,
       'Valor agregado bruto' AS N_SCN,
       'PS' AS FUENTE,
       '0778' AS PROC
INTO #va_51
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 = 'S.11')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA
"""
work_conn.execute(text(sql_va_51))


In [ ]:
sql_append_va_51 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #va_51
"""
res = work_conn.execute(text(sql_append_va_51))
_log("APPEND TABLAS.BD_CTSI (VA_51)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #va_51"))


In [ ]:
# IMPUTA VALOR AGREGADO EN SECTOR 4
work_conn.execute(text("DROP TABLE IF EXISTS #va_4"))
sql_va_4 = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       4 AS SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       'B.1' AS C_SCN,
       'Valor agregado bruto' AS N_SCN,
       'PS' AS FUENTE,
       '0778b' AS PROC
INTO #va_4
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 = 'S.13' AND T1.C_SCN NOT IN ('B.2','B.2/B.3'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA
"""
work_conn.execute(text(sql_va_4))


In [ ]:
sql_append_va_4 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #va_4
"""
res = work_conn.execute(text(sql_append_va_4))
_log("APPEND TABLAS.BD_CTSI (VA_4)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #va_4"))


In [ ]:
# IMPUTA EXCEDENTE DE EXPLOTACION EN LA CTA DE PRODUCCION DEL RM, USANDO LA INFO DEL EXCEDENTE DE LA CTA YG
work_conn.execute(text("DROP TABLE IF EXISTS #exc_rm"))
sql_exc_rm = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0779' AS PROC
INTO #exc_rm
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'H' AND T1.C_SCN = 'B.2' AND T1.SECTOR = 6)
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_exc_rm))


In [ ]:
sql_append_exc_rm = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #exc_rm
"""
res = work_conn.execute(text(sql_append_exc_rm))
_log("APPEND TABLAS.BD_CTSI (EXC_RM)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #exc_rm"))


In [ ]:
# ACTUALIZA CODIGO DE EXCEDENTE DE EXPLOTACION EN TODOS LOS SECTORES EXCEPTO RM
sql_update_exc = """
UPDATE TABLAS.dbo.BD_CTSI
SET C_SCN = 'B.2/B.3', N_SCN = 'Excedente de explotación/ Ingreso mixto', PROC = '0780'
WHERE C_SCN = 'B.2' AND C_ENTRADA = 'H' AND C_CUENTA = 'YG' AND SECTOR <> 6
"""
with engine.begin() as conn:
    res = conn.execute(text(sql_update_exc))
    _log("UPDATE TABLAS.BD_CTSI (excedente todos sectores excepto RM)", res.rowcount)


In [ ]:
# CIERRE 2021: IMPUTA EFECTO NETO DE SIFMI DE LOS SECTORES SEGUROS, AUXILIARES Y BANCOS EN EXCEDENTE, DADO QUE EN SEGUROS Y AUXILIARES IMPLICA MAYOR CI Y EN BANCOS MAYOR VBP
work_conn.execute(text("DROP TABLE IF EXISTS #sifmi_sf"))
sql_sifmi_sf = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.2/B.3' AS C_SCN,
       'Excedente de explotación/ Ingreso mixto' AS N_SCN,
       'PS' AS FUENTE,
       '0780b' AS PROC
INTO #sifmi_sf
FROM TABLAS.dbo.BD_CTSI T1
WHERE T1.SECTOR IN (35,36,321) AND T1.C_CUENTA = 'YG' AND T1.C_SCN = 'D.41' AND T1.FUENTE = 'DI_Aj_SIFMI'
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR
"""
work_conn.execute(text(sql_sifmi_sf))


In [ ]:
# AJUSTA EN EXCEDENTE TOTAL DEL SECTOR
work_conn.execute(text("DROP TABLE IF EXISTS #aj_exc"))
sql_aj_exc = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       3 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0780c' AS PROC
INTO #aj_exc
FROM #sifmi_sf T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_exc))


In [ ]:
sql_append_sifmi_sf = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #sifmi_sf
"""
res = work_conn.execute(text(sql_append_sifmi_sf))
_log("APPEND TABLAS.BD_CTSI (SIFMI_SF)", res.rowcount)

sql_append_aj_exc = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #aj_exc
"""
res = work_conn.execute(text(sql_append_aj_exc))
_log("APPEND TABLAS.BD_CTSI (AJ_EXC)", res.rowcount)
for t in ["#sifmi_sf", "#aj_exc"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ELIMINA EXCEDENTE/ING MIXTO DE LA CUENTA DE YG EN SECTORES SNF Y SF
work_conn.execute(text("DROP TABLE IF EXISTS #elimin_exc"))
sql_elimin_exc = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0781' AS PROC
INTO #elimin_exc
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'H' AND T2.C_SI_SCN_N1 IN ('S.11','S.12') AND T1.C_SCN IN ('B.2/B.3','B.2'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_elimin_exc))


In [ ]:
sql_append_elimin_exc = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #elimin_exc
"""
res = work_conn.execute(text(sql_append_elimin_exc))
_log("APPEND TABLAS.BD_CTSI (ELIMIN_EXC)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #elimin_exc"))


In [ ]:
# IMPUTA EXCEDENTE EN LA CTA YG DE LOS SECTORES SNF Y SF, USANDO INFO DE LA CTA DE PRODUCCION
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_exc"))
sql_imputa_exc = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       'YG' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0782' AS PROC
INTO #imputa_exc
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_ENTRADA = 'D' AND T2.C_SI_SCN_N1 IN ('S.11','S.12') AND T1.C_SCN IN ('B.2/B.3','B.2'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_imputa_exc))


In [ ]:
sql_append_imputa_exc = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #imputa_exc
"""
res = work_conn.execute(text(sql_append_imputa_exc))
_log("APPEND TABLAS.BD_CTSI (IMPUTA_EXC)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_exc"))


In [ ]:
# IMPUTA DIFERENCIAL DE FBCF Y VAR DE EXISTENCIAS EN SECTOR 51022
work_conn.execute(text("DROP TABLE IF EXISTS #fbc_sectores"))
sql_fbc_sectores = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51022 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0768' AS PROC
INTO #fbc_sectores
FROM TABLAS.dbo.BD_CTSI T1
WHERE (T1.C_SCN IN ('P.51','P.52') AND T1.C_ENTRADA = 'D')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_fbc_sectores))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #fbc_total"))
sql_fbc_total = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       51022 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0768' AS PROC
INTO #fbc_total
FROM TABLAS.dbo.CNT T1
WHERE (T1.C_SCN IN ('P.51','P.52') AND T1.C_ENTRADA = 'D')
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_fbc_total))


In [ ]:
# APPEND BASE=WORK.FBC_SECTORES DATA=WORK.FBC_TOTAL FORCE (destino WORK): append server-side sobre la #tmp
cols_fbc = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_fbc_sectores = f"""
INSERT INTO #fbc_sectores ({cols_fbc})
SELECT {cols_fbc}
FROM #fbc_total
"""
res = work_conn.execute(text(sql_append_fbc_sectores))
_log("APPEND #fbc_sectores (FBC_TOTAL)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #fbc_dif"))
sql_fbc_dif = """
SELECT T1.MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #fbc_dif
FROM #fbc_sectores T1
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN, T1.FUENTE, T1.PROC
"""
work_conn.execute(text(sql_fbc_dif))


In [ ]:
sql_append_fbc_dif = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #fbc_dif
"""
res = work_conn.execute(text(sql_append_fbc_dif))
_log("APPEND TABLAS.BD_CTSI (FBC_DIF)", res.rowcount)
for t in ["#fbc_dif", "#fbc_sectores", "#fbc_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# RECALCULA EL AHORRO PARA TODOS LOS SECTORES
# CIERRE 2021: CAMBIA NIVEL DEL SECTOR AL CUAL SE HACE EL CÁLCULO DEL AHORRO
work_conn.execute(text("DROP TABLE IF EXISTS #ahorro"))
sql_ahorro = """
SELECT 'P' AS MONEDA,
       T1.[AÑO],
       T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       'PS' AS FUENTE,
       '0786' AS PROC
INTO #ahorro
FROM TABLAS.dbo.BD_CTSI T1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND ((T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'D' AND T1.C_SCN <> 'P.31') OR (T1.C_CUENTA = 'YG' AND T1.C_ENTRADA = 'H'))
GROUP BY T1.MONEDA, T1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_CUENTA
"""
work_conn.execute(text(sql_ahorro))
_log("AHORRO materializado en #ahorro (server-side)", None)


In [ ]:
# El bloque AHORRO_2 (recálculo con T_SECTORIZACION_N1) quedó comentado/anulado en el SAS original: no se traduce (código muerto)


In [ ]:
# APPEND WORK.AHORRO a TABLAS.BD_CTSI y limpieza de la #tmp
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #ahorro"))
_log("APPEND TABLAS.BD_CTSI (ahorro)", res.rowcount)
# el DROP TABLE de WORK.AHORRO_2 estaba comentado en el SAS original (bloque anulado, no se traduce)
work_conn.execute(text("DROP TABLE IF EXISTS #ahorro"))


In [ ]:
# ELIMINA AHORRO DE LA CTA DE CAPITAL E IMPUTA EL AHORRO DE LA CUENTA YG
work_conn.execute(text("DROP TABLE IF EXISTS #drop_b8_ck"))
sql_drop_b8_ck = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0787' AS PROC
INTO #drop_b8_ck
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA = 'Capital' AND T1.C_SCN = 'B.8')
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_drop_b8_ck))
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #drop_b8_ck"))
_log("APPEND TABLAS.BD_CTSI (drop_b8_ck)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #drop_b8_ck"))


In [ ]:
# IMPUTA el ahorro de la cuenta YG en Capital
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_b8_ck"))
sql_imputa_b8_ck = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0788' AS PROC
INTO #imputa_b8_ck
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA = 'YG' AND T1.C_SCN = 'B.8' AND T1.C_ENTRADA = 'D')
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_imputa_b8_ck))
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #imputa_b8_ck"))
_log("APPEND TABLAS.BD_CTSI (imputa_b8_ck)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_b8_ck"))


In [ ]:
# ELIMINA CCF DE LA CTA DE CAPITAL E IMPUTA EL CCF DE LA CUENTA DE PRODUCCION
work_conn.execute(text("DROP TABLE IF EXISTS #drop_ccf_ck"))
sql_drop_ccf_ck = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0789' AS PROC
INTO #drop_ccf_ck
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.C_CUENTA = 'Capital' AND T1.C_SCN = 'K.1')
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_drop_ccf_ck))
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #drop_ccf_ck"))
_log("APPEND TABLAS.BD_CTSI (drop_ccf_ck)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #drop_ccf_ck"))


In [ ]:
# IMPUTA el CCF en la cuenta de Produccion, cruzando con la tabla lookup de sectorizacion
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_ccf_ck"))
sql_imputa_ccf_ck = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0790' AS PROC
INTO #imputa_ccf_ck
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Producción' AND T1.C_SCN = 'K.1' AND T1.C_ENTRADA = 'D' AND T1.DATO <> 0)
GROUP BY t1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_imputa_ccf_ck))
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #imputa_ccf_ck"))
_log("APPEND TABLAS.BD_CTSI (imputa_ccf_ck)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #imputa_ccf_ck"))


In [ ]:
# CIERRE 2021: recalcula PTMO NETO en la cuenta de Capital, a nivel de codigo de sector de publicacion
# (el bloque anterior CAP_NEC_FMNM quedo comentado/anulado en el SAS original: no se traduce)
work_conn.execute(text("DROP TABLE IF EXISTS #cap_nec"))
sql_cap_nec = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T2.C_SI_publ AS SECTOR,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'D' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       'PS' AS FUENTE, '0793' AS PROC
INTO #cap_nec
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_CUENTA = 'Capital')
GROUP BY t1.[AÑO], T1.TRIM, T2.C_SI_publ, T1.C_CUENTA
"""
work_conn.execute(text(sql_cap_nec))
# el bloque WORK.CAP_NEC_2 quedo comentado/anulado en el SAS original: no se traduce
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #cap_nec"))
_log("APPEND TABLAS.BD_CTSI (cap_nec)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #cap_nec"))


In [ ]:
# AJUSTA PTMO NETO EN SECTOR GOBIERNO AL DE LA CTA DE CAPITAL. AJUSTA LA CF CONTRA AJUSTES DE CONCILIACION
work_conn.execute(text("DROP TABLE IF EXISTS #gob_ck"))
sql_gob_ck = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       41 AS SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_CUENTA = 'Financiera' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE, '0902' AS PROC
INTO #gob_ck
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_SCN = 'B.9' AND T2.C_SI_SCN_N1 = 'S.13')
GROUP BY t1.[AÑO], T1.TRIM
"""
work_conn.execute(text(sql_gob_ck))
gob_ck = pd.read_sql(text("SELECT * FROM #gob_ck"), work_conn)
_log("gob_ck", gob_ck)


In [ ]:
# IMPUTA AF71 en Rec Volumen
work_conn.execute(text("DROP TABLE IF EXISTS #gob_rv"))
sql_gob_rv = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       C_ENTRADA,
       DATO * -1 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE, '0903' AS PROC
INTO #gob_rv
FROM #gob_ck
"""
work_conn.execute(text(sql_gob_rv))


In [ ]:
# IMPUTA AF71 en cta financiera del sector 51
work_conn.execute(text("DROP TABLE IF EXISTS #cf_51"))
sql_cf_51 = """
SELECT MONEDA, [AÑO], TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA, C_ENTRADA,
       DATO * -1 AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE, '0904' AS PROC
INTO #cf_51
FROM #gob_ck
"""
work_conn.execute(text(sql_cf_51))


In [ ]:
# IMPUTA AF71 en Rec Volumen del sector 51
work_conn.execute(text("DROP TABLE IF EXISTS #rv_51"))
sql_rv_51 = """
SELECT MONEDA, [AÑO], TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       C_ENTRADA,
       DATO AS DATO,
       C_SCN, N_SCN,
       'PS' AS FUENTE, '0905' AS PROC
INTO #rv_51
FROM #gob_ck
"""
work_conn.execute(text(sql_rv_51))


In [ ]:
# APPEND de los 4 datasets GOB_CK/GOB_RV/CF_51/RV_51 a TABLAS.BD_CTSI y limpieza
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #gob_ck"))
_log("APPEND TABLAS.BD_CTSI (gob_ck)", res.rowcount)
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #gob_rv"))
_log("APPEND TABLAS.BD_CTSI (gob_rv)", res.rowcount)
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #cf_51"))
_log("APPEND TABLAS.BD_CTSI (cf_51)", res.rowcount)
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #rv_51"))
_log("APPEND TABLAS.BD_CTSI (rv_51)", res.rowcount)
for t in ["#gob_ck", "#gob_rv", "#cf_51", "#rv_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA CUENTA DEL RESTO DEL MUNDO CONSIDERANDO VARIABLES MACRO DADAS POR LAS CNT
# RM desde Sintesis (BD_CTSI)
work_conn.execute(text("DROP TABLE IF EXISTS #rm_cnf"))
sql_rm_cnf = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
       SUM(T1.DATO * -1) AS DATO,
       CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'D.4'
            WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'D.7' ELSE T1.C_SCN END AS C_SCN,
       CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'Renta distribuida de las sociedades'
            WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'Transferencias corrientes diversas' ELSE T1.N_SCN END AS N_SCN,
       'PS' AS FUENTE, '0906' AS PROC
INTO #rm_cnf
FROM TABLAS.dbo.BD_CTSI t1
WHERE (T1.SECTOR = 6 AND T1.C_CUENTA IN ('Producción','YG','Capital')
       AND T1.C_SCN IN ('P.7','P.6','D.1','D.41','D.42','D.43','D.443','D.72','D.71','D.75','B.8'))
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
         CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'D.4'
              WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'D.7' ELSE T1.C_SCN END,
         CASE WHEN T1.C_SCN IN ('D.1','D.41','D.42','D.43','D.443') THEN 'Renta distribuida de las sociedades'
              WHEN T1.C_SCN IN ('D.71','D.72','D.75') THEN 'Transferencias corrientes diversas' ELSE T1.N_SCN END
"""
work_conn.execute(text(sql_rm_cnf))


In [ ]:
# RM desde CNT (tabla lookup de cuentas nacionales trimestrales)
work_conn.execute(text("DROP TABLE IF EXISTS #rm_cnt"))
sql_rm_cnt = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA,
       T1.DATO AS DATO,
       T1.C_SCN,
       CASE WHEN T1.C_SCN = 'D.4' THEN 'Renta distribuida de las sociedades'
            WHEN T1.C_SCN = 'D.7' THEN 'Transferencias corrientes diversas'
            WHEN T1.C_SCN = 'B.8' THEN 'Ahorro' ELSE T1.N_SCN END AS N_SCN,
       'PS' AS FUENTE, '0906' AS PROC
INTO #rm_cnt
FROM TABLAS.dbo.CNT t1
WHERE T1.SECTOR = 6
GROUP BY t1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA, T1.C_ENTRADA, T1.C_SCN,
         CASE WHEN T1.C_SCN = 'D.4' THEN 'Renta distribuida de las sociedades'
              WHEN T1.C_SCN = 'D.7' THEN 'Transferencias corrientes diversas'
              WHEN T1.C_SCN = 'B.8' THEN 'Ahorro' ELSE T1.N_SCN END
"""
work_conn.execute(text(sql_rm_cnt))
# PROC APPEND BASE=WORK.RM_CNF DATA=WORK.RM_CNT: acumula sobre la #tmp de sesión
res = work_conn.execute(text("INSERT INTO #rm_cnf SELECT * FROM #rm_cnt"))
_log("APPEND #rm_cnf (rm_cnt)", res.rowcount)


In [ ]:
# calcula diferencias entre RM_CNF (con lo acumulado de RM_CNT) excluyendo Capital
work_conn.execute(text("DROP TABLE IF EXISTS #rm_dif"))
sql_rm_dif = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '53' AS C_CAGENTE,
       C_CUENTA, C_ENTRADA,
       SUM(DATO) AS DATO,
       CASE WHEN C_SCN = 'D.4' THEN 'D.42'
            WHEN C_SCN = 'D.7' THEN 'D.75' ELSE C_SCN END AS C_SCN,
       N_SCN, FUENTE, PROC
INTO #rm_dif
FROM #rm_cnf
WHERE (C_CUENTA <> 'Capital')
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA,
         CASE WHEN C_SCN = 'D.4' THEN 'D.42' WHEN C_SCN = 'D.7' THEN 'D.75' ELSE C_SCN END,
         N_SCN, FUENTE, PROC
"""
work_conn.execute(text(sql_rm_dif))


In [ ]:
# IMPUTA ahorro en cta de Capital al haber
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b8_ck"))
sql_rm_b8_ck = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       DATO, C_SCN, N_SCN, FUENTE, PROC
INTO #rm_b8_ck
FROM #rm_dif
WHERE (C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_b8_ck))


In [ ]:
# IMPUTA dif en Ptmo Neto cta de Capital al debe
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b9_ck"))
sql_rm_b9_ck = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE,
       'Capital' AS C_CUENTA,
       'D' AS C_ENTRADA,
       DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       FUENTE, PROC
INTO #rm_b9_ck
FROM #rm_dif
WHERE (C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_b9_ck))


In [ ]:
# IMPUTA dif en cta financiera inst AF71 con CA 51 al haber
work_conn.execute(text("DROP TABLE IF EXISTS #rm_af71_cf"))
sql_rm_af71_cf = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       FUENTE, PROC
INTO #rm_af71_cf
FROM #rm_dif
WHERE (C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_af71_cf))


In [ ]:
# IMPUTA dif en Rec Volumen inst AF71 con CA 51 al haber
work_conn.execute(text("DROP TABLE IF EXISTS #rm_af71_rv"))
sql_rm_af71_rv = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR,
       '51' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'H' AS C_ENTRADA,
       DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       FUENTE, PROC
INTO #rm_af71_rv
FROM #rm_dif
WHERE (C_SCN = 'B.8')
"""
work_conn.execute(text(sql_rm_af71_rv))


In [ ]:
# IMPUTA dif en cta financiera sector 51 inst AF71 con CA 6 al haber
work_conn.execute(text("DROP TABLE IF EXISTS #af71_cf_51"))
sql_af71_cf_51 = """
SELECT MONEDA, [AÑO], TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       FUENTE, PROC
INTO #af71_cf_51
FROM #rm_dif
WHERE (C_SCN = 'B.8')
"""
work_conn.execute(text(sql_af71_cf_51))
af71_cf_51 = pd.read_sql(text("SELECT * FROM #af71_cf_51"), work_conn)
_log("af71_cf_51", af71_cf_51)


In [ ]:
# IMPUTA dif en RV sector 51 inst AF71 con CA 6 al haber
work_conn.execute(text("DROP TABLE IF EXISTS #af71_rv_51"))
sql_af71_rv_51 = """
SELECT MONEDA, [AÑO], TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'H' AS C_ENTRADA,
       DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       FUENTE, PROC
INTO #af71_rv_51
FROM #rm_dif
WHERE (C_SCN = 'B.8')
"""
work_conn.execute(text(sql_af71_rv_51))
af71_rv_51 = pd.read_sql(text("SELECT * FROM #af71_rv_51"), work_conn)
_log("af71_rv_51", af71_rv_51)


In [ ]:
# CALCULA ajuste en B2 a cta de Produccion
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b2"))
sql_rm_b2 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE,
       'Producción' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA = 'D' THEN DATO * -1 ELSE DATO END) AS DATO,
       'B.2' AS C_SCN,
       'Excedente de explotación' AS N_SCN,
       FUENTE, PROC
INTO #rm_b2
FROM #rm_dif
WHERE (C_CUENTA = 'Producción')
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, PROC, FUENTE
"""
work_conn.execute(text(sql_rm_b2))


In [ ]:
# CALCULA ajuste en B2 a cta de YG
work_conn.execute(text("DROP TABLE IF EXISTS #rm_b2_yg"))
sql_rm_b2_yg = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE,
       'YG' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN C_ENTRADA = 'D' THEN DATO * -1 ELSE DATO END) AS DATO,
       'B.2' AS C_SCN,
       'Excedente de explotación' AS N_SCN,
       FUENTE, PROC
INTO #rm_b2_yg
FROM #rm_dif
WHERE (C_CUENTA = 'Producción')
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, PROC, FUENTE
"""
work_conn.execute(text(sql_rm_b2_yg))


In [ ]:
# APPEND de los 8 datasets del bloque RM a TABLAS.BD_CTSI y limpieza de todas las #tmp
for tmp_name, label in [
    ("#rm_dif", "rm_dif"), ("#rm_b8_ck", "rm_b8_ck"), ("#rm_b9_ck", "rm_b9_ck"),
    ("#rm_af71_cf", "rm_af71_cf"), ("#rm_af71_rv", "rm_af71_rv"),
    ("#af71_cf_51", "af71_cf_51"), ("#af71_rv_51", "af71_rv_51"),
    ("#rm_b2", "rm_b2"), ("#rm_b2_yg", "rm_b2_yg"),
]:
    res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM {tmp_name}"))
    _log(f"APPEND TABLAS.BD_CTSI ({label})", res.rowcount)
for t in ["#rm_dif", "#rm_cnf", "#rm_cnt", "#rm_b8_ck", "#rm_b9_ck", "#rm_af71_cf", "#rm_af71_rv", "#af71_cf_51", "#af71_rv_51", "#rm_b2", "#rm_b2_yg"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA TOTAL D75 Y D42: calcula diferencia contra el sector 51 con CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d4d7"))
sql_aj_d4d7 = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0909' AS PROC
INTO #aj_d4d7
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_SCN IN ('D.42','D.75')
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d4d7))


In [ ]:
# AJUSTA ahorro cta YG
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8"))
sql_aj_b8 = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE, C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       FUENTE, '0910' AS PROC
INTO #aj_b8
FROM #aj_d4d7
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, FUENTE
"""
work_conn.execute(text(sql_aj_b8))


In [ ]:
# AJUSTA ahorro cta CK
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8_ck"))
sql_aj_b8_ck = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       FUENTE, '0911' AS PROC
INTO #aj_b8_ck
FROM #aj_d4d7
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, FUENTE
"""
work_conn.execute(text(sql_aj_b8_ck))


In [ ]:
# AJUSTA Ptmo Neto cta CK
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b9_ck"))
sql_aj_b9_ck = """
SELECT MONEDA, [AÑO], TRIM,
       SECTOR, C_CAGENTE,
       'Capital' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(DATO) AS DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       FUENTE, '0912' AS PROC
INTO #aj_b9_ck
FROM #aj_d4d7
GROUP BY MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, FUENTE
"""
work_conn.execute(text(sql_aj_b9_ck))
# APPEND de los 4 datasets del bloque AJ_D4D7 y limpieza
for tmp_name, label in [("#aj_d4d7", "aj_d4d7"), ("#aj_b8", "aj_b8"), ("#aj_b8_ck", "aj_b8_ck"), ("#aj_b9_ck", "aj_b9_ck")]:
    res = work_conn.execute(text(f"INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM {tmp_name}"))
    _log(f"APPEND TABLAS.BD_CTSI ({label})", res.rowcount)
for t in ["#aj_d4d7", "#aj_b8", "#aj_b8_ck", "#aj_b9_ck"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CIERRE 2019: los recibidos del RM imputados en PROC 0906 se imputan como pagados por el sector 51 al RM
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d42_rm"))
sql_aj_d42_rm = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '6' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0919' AS PROC
INTO #aj_d42_rm
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_SCN IN ('D.42') AND T1.SECTOR = 6 AND T1.C_ENTRADA = 'H' AND T1.PROC = '0906'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d42_rm))


In [ ]:
# y este mismo valor se rebaja del sector 51 con CA 53
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d42_resto"))
sql_aj_d42_resto = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(T1.DATO) * -1 AS DATO,
       T1.C_SCN, T1.N_SCN,
       'PS' AS FUENTE, '0919' AS PROC
INTO #aj_d42_resto
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_SCN IN ('D.42') AND T1.SECTOR = 6 AND T1.C_ENTRADA = 'H' AND T1.PROC = '0906'
GROUP BY t1.[AÑO], T1.TRIM, T1.C_CUENTA, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d42_resto))
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #aj_d42_rm"))
_log("APPEND TABLAS.BD_CTSI (aj_d42_rm)", res.rowcount)
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM #aj_d42_resto"))
_log("APPEND TABLAS.BD_CTSI (aj_d42_resto)", res.rowcount)
for t in ["#aj_d42_resto", "#aj_d42_rm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IMPUTA AJUSTE EN BONOS para eliminar saldos negativos en AF32: APPEND directo de tabla permanente TABLAS.AJUSTE_BONOS
res = work_conn.execute(text("INSERT INTO TABLAS.dbo.BD_CTSI SELECT * FROM TABLAS.dbo.AJUSTE_BONOS"))
_log("APPEND TABLAS.BD_CTSI (ajuste_bonos)", res.rowcount)


In [ ]:
# CIERRE 2021: AJUSTA PTMO NETO EN SECTOR SEGUROS Y AUXILIARES AL DE LA CTA DE CAPITAL. AJUSTA LA CF CONTRA AJUSTES DE CONCILIACION. TMB EN BCO CENTRAL
work_conn.execute(text("DROP TABLE IF EXISTS #seg_ck"))
sql_seg_ck = """
SELECT 'P' AS MONEDA, t1.[AÑO], T1.TRIM,
       CASE WHEN T2.C_SI_SCN = 'S.125' THEN 35 ELSE 36 END AS SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_CUENTA = 'Financiera' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste conciliación' AS N_SCN,
       'PS' AS FUENTE, '0920' AS PROC
INTO #seg_ck
FROM TABLAS.dbo.BD_CTSI t1, TABLAS.dbo.T_SECTORIZACION T2
WHERE (T1.SECTOR = T2.C_SI) AND (T1.C_SCN = 'B.9' AND T2.C_SI_SCN IN ('S.125','S.124','S.121','S.123'))
GROUP BY t1.[AÑO], T1.TRIM,
         CASE WHEN T2.C_SI_SCN = 'S.125' THEN 35 ELSE 36 END
"""
work_conn.execute(text(sql_seg_ck))
seg_ck = pd.read_sql(text("SELECT * FROM #seg_ck"), work_conn)
_log("seg_ck", seg_ck)
# WORK.SEG_CK sigue abierto: el APPEND/limpieza de esta tabla corresponde al tramo siguiente (8 de 8), no incluido en este corte


In [ ]:
# IMPUTA AF71 EN REC VOLUMEN
work_conn.execute(text("DROP TABLE IF EXISTS #seg_rv"))
sql_seg_rv = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0903' AS PROC
INTO #seg_rv
FROM #seg_ck T1
"""
work_conn.execute(text(sql_seg_rv))


In [ ]:
# IMPUTA AF71 EN CTA FINANCIERA DE SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #cf_51"))
sql_cf_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0904' AS PROC
INTO #cf_51
FROM #seg_ck T1
"""
work_conn.execute(text(sql_cf_51))


In [ ]:
# IMPUTA AF71 EN CTA FINANCIERA DE SECTOR 51
work_conn.execute(text("DROP TABLE IF EXISTS #rv_51"))
sql_rv_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '53' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0905' AS PROC
INTO #rv_51
FROM #seg_ck T1
"""
work_conn.execute(text(sql_rv_51))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.SEG_CK/SEG_RV/CF_51/RV_51 FORCE — server-side, columnas explícitas
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_src in ["#seg_ck", "#seg_rv", "#cf_51", "#rv_51"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_src}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_src}", res.rowcount)


In [ ]:
# limpieza de temporales de este tramo
for t in ["#seg_ck", "#seg_rv", "#cf_51", "#rv_51"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# ELIMINA CTAS DE BALANCE, FINANCIERA, REC PRECIO Y VOLUMEN DE CUENTAS YG
sql_delete_yg = text("""
DELETE FROM TABLAS.dbo.BD_CTSI
WHERE C_CUENTA IN ('Bce Inicio','Financiera','Rec Precio','Rec Precio Reaj','Rec Volumen','Bce Final')
  AND C_SCN IN ('D.41','K.1','P.2','P.51','P.11','D.42','D.5','B.9','B.90','B.10.2','B.10.3','P.52','D.62','D.75','D.1')
""")
res = work_conn.execute(sql_delete_yg)
_log("DELETE TABLAS.dbo.BD_CTSI (ctas balance/financiera/rec precio-volumen YG)", res.rowcount)


In [ ]:
# CALCULA E IMPUTA PTMO NETO, VALOR NETO, Y VARIACIONES DEL VALOR NETO DEBIDO A VOLUMENES Y REC PRECIOS
work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))
sql_saldos = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'B.90'
            WHEN T1.C_CUENTA = 'Financiera' THEN 'B.9'
            WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END AS C_SCN,
       CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'Valor neto'
            WHEN T1.C_CUENTA = 'Financiera' THEN 'Capacidad/Necesidad de financiamiento'
            WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END AS N_SCN,
       'PS' AS FUENTE,
       '0009' AS PROC
INTO #saldos
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_CUENTA IN ('Bce Inicio','Financiera','Rec Precio','Rec Precio Reaj','Rec Volumen','Bce Final')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA,
         CASE WHEN T1.C_ENTRADA = 'H' THEN 'H' ELSE 'H' END,
         CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'B.90'
              WHEN T1.C_CUENTA = 'Financiera' THEN 'B.9'
              WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END,
         CASE WHEN T1.C_CUENTA LIKE '%Bce%' THEN 'Valor neto'
              WHEN T1.C_CUENTA = 'Financiera' THEN 'Capacidad/Necesidad de financiamiento'
              WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END
"""
work_conn.execute(text(sql_saldos))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.SALDOS FORCE
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_saldos = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #saldos
"""
res = work_conn.execute(text(sql_append_saldos))
_log("APPEND TABLAS.dbo.BD_CTSI desde #saldos", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #saldos"))


In [ ]:
# ELIMA CEROS
sql_delete_ceros = text("DELETE FROM TABLAS.dbo.BD_CTSI WHERE DATO = 0")
res = work_conn.execute(sql_delete_ceros)
_log("DELETE TABLAS.dbo.BD_CTSI (ceros)", res.rowcount)


In [ ]:
# AJUSTA REMUNERACIONES EN HOGARES
# AJUSTE REMU RECIBIDAS POR HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_remu"))
sql_aj_remu = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '53' AS C_CAGENTE,
       'H' AS C_ENTRADA,
       'YG' AS C_CUENTA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'DI_Aj_REM' AS FUENTE,
       '0913' AS PROC
INTO #aj_remu
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_SCN = 'D.1'
GROUP BY T1.[AÑO], T1.TRIM, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_remu))


In [ ]:
# AJUSTE AHORRO CTA YG HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8_yg"))
sql_aj_b8_yg = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       T1.C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_b8_yg
FROM #aj_remu T1
"""
work_conn.execute(text(sql_aj_b8_yg))


In [ ]:
# AJUSTE AHORRO CTA K HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b8_k"))
sql_aj_b8_k = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'H' AS C_ENTRADA,
       T1.DATO,
       'B.8' AS C_SCN,
       'Ahorro' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_b8_k
FROM #aj_remu T1
"""
work_conn.execute(text(sql_aj_b8_k))


In [ ]:
# AJUSTE PTMO NETO CTA K HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_b9_k"))
sql_aj_b9_k = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CAGENTE,
       'Capital' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO,
       'B.9' AS C_SCN,
       'Capacidad/Necesidad de financiamiento' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_b9_k
FROM #aj_remu T1
"""
work_conn.execute(text(sql_aj_b9_k))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.AJ_REMU/AJ_B8_YG/AJ_B8_K/AJ_B9_K FORCE
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_ENTRADA, C_CUENTA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_src in ["#aj_remu", "#aj_b8_yg", "#aj_b8_k", "#aj_b9_k"]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
    SELECT {cols_bd_ctsi}
    FROM {tmp_src}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_src}", res.rowcount)
for t in ["#aj_remu", "#aj_b8_yg", "#aj_b8_k", "#aj_b9_k"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO NETO DE LAS SNF. AJUSTA PTMO NETO DE LA CF E IMPUTA NEGATIVO DE ESE AJUSTE EN EL PTMO NETO DE LA CF DE HOGARES
# TMB GENERA AJUSTE Y DISCREPANCIA EN LA CTA DE CADA SECTOR DEBIDO A ESTE AJUSTE
# CALCULA DIFERENCIA
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_snf"))
sql_aj_d9_snf = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0918' AS PROC
INTO #aj_d9_snf
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_SCN = 'B.9' AND T2.C_SI_SCN_N1 = 'S.11'
GROUP BY T1.[AÑO], T1.TRIM, T2.C_SI_SCN_N1, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d9_snf))


In [ ]:
# IMPUTA NEGATIVO EN CTA FIN HH
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_hh"))
sql_aj_d9_hh = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_d9_hh
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_d9_hh))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_51"))
sql_aj_af71_cf_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '511' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_51
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_cf_51))


In [ ]:
# IMPUTA AJUSTE EN CTA REC VOLUMEN INST AF71 EMPRESAS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_51"))
sql_aj_af71_rv_51 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       51 AS SECTOR,
       '511' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_51
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_rv_51))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_511"))
sql_aj_af71_cf_511 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '51' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_511
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_cf_511))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 HOGARES (Rec Volumen)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_511"))
sql_aj_af71_rv_511 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '51' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_511
FROM #aj_d9_snf T1
"""
work_conn.execute(text(sql_aj_af71_rv_511))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.AJ_D9_SNF/AJ_D9_HH/AJ_AF71_CF_51/AJ_AF71_RV_51/AJ_AF71_CF_511/AJ_AF71_RV_511 FORCE
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
cols_bd_ctsi_ag = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_src, cols in [("#aj_d9_snf", cols_bd_ctsi), ("#aj_d9_hh", cols_bd_ctsi),
                      ("#aj_af71_cf_51", cols_bd_ctsi_ag), ("#aj_af71_rv_51", cols_bd_ctsi_ag),
                      ("#aj_af71_cf_511", cols_bd_ctsi_ag), ("#aj_af71_rv_511", cols_bd_ctsi_ag)]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols})
    SELECT {cols}
    FROM {tmp_src}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_src}", res.rowcount)
for t in ["#aj_d9_snf", "#aj_d9_hh", "#aj_af71_cf_51", "#aj_af71_rv_51", "#aj_af71_cf_511", "#aj_af71_rv_511"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# AJUSTA PTMO NETO DE HOGARES. AJUSTA PTMO NETO DE LA CF E IMPUTA NEGATIVO DE ESE AJUSTE EN EL PTMO NETO DE LA CF DE BANCOS
# TMB GENERA AJUSTE Y DISCREPANCIA EN LA CTA DE CADA SECTOR DEBIDO A ESTE AJUSTE
# CALCULA DIFERENCIA
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_hh2"))
sql_aj_d9_hh2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       'Financiera' AS C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       'PS' AS FUENTE,
       '0933' AS PROC
INTO #aj_d9_hh2
FROM TABLAS.dbo.BD_CTSI t1
INNER JOIN TABLAS.dbo.T_SECTORIZACION T2 ON T1.SECTOR = T2.C_SI
WHERE T1.C_SCN = 'B.9' AND T2.C_SI_SCN_N1 = 'S.14'
GROUP BY T1.[AÑO], T1.TRIM, T2.C_SI_SCN_N1, T1.C_SCN, T1.N_SCN
"""
work_conn.execute(text(sql_aj_d9_hh2))


In [ ]:
# IMPUTA NEGATIVO EN CTA FIN DE BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_d9_bcos"))
sql_aj_d9_bcos = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       T1.C_CUENTA,
       T1.C_ENTRADA,
       T1.DATO * -1 AS DATO,
       T1.C_SCN,
       T1.N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_d9_bcos
FROM #aj_d9_hh2 T1
"""
work_conn.execute(text(sql_aj_d9_bcos))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 HOGARES (segunda ronda, contraparte 321)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_511_2"))
sql_aj_af71_cf_511_2 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '321' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_511_2
FROM #aj_d9_hh2 T1
"""
work_conn.execute(text(sql_aj_af71_cf_511_2))


In [ ]:
# IMPUTA AJUSTE EN CTA REC VOLUMEN INST AF71 HOGARES
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_511_2"))
sql_aj_af71_rv_511_2 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       511 AS SECTOR,
       '321' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_511_2
FROM #aj_d9_hh2 T1
"""
work_conn.execute(text(sql_aj_af71_rv_511_2))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 BANCOS
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_cf_321"))
sql_aj_af71_cf_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '511' AS C_CAGENTE,
       'Financiera' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO * -1 AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_cf_321
FROM #aj_d9_hh2 T1
"""
work_conn.execute(text(sql_aj_af71_cf_321))


In [ ]:
# IMPUTA AJUSTE EN CTA FINANCIERA INST AF71 BANCOS (Rec Volumen)
work_conn.execute(text("DROP TABLE IF EXISTS #aj_af71_rv_321"))
sql_aj_af71_rv_321 = """
SELECT T1.MONEDA, T1.[AÑO], T1.TRIM,
       321 AS SECTOR,
       '511' AS C_CAGENTE,
       'Rec Volumen' AS C_CUENTA,
       'D' AS C_ENTRADA,
       T1.DATO AS DATO,
       'AF.71' AS C_SCN,
       'Ajuste de conciliación' AS N_SCN,
       T1.FUENTE,
       T1.PROC
INTO #aj_af71_rv_321
FROM #aj_d9_hh2 T1
"""
work_conn.execute(text(sql_aj_af71_rv_321))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.AJ_D9_HH/AJ_D9_BCOS/AJ_AF71_CF_511/AJ_AF71_RV_511/AJ_AF71_CF_321/AJ_AF71_RV_321 FORCE
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
cols_bd_ctsi_ag = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_src, cols in [("#aj_d9_hh2", cols_bd_ctsi), ("#aj_d9_bcos", cols_bd_ctsi),
                      ("#aj_af71_cf_511_2", cols_bd_ctsi_ag), ("#aj_af71_rv_511_2", cols_bd_ctsi_ag),
                      ("#aj_af71_cf_321", cols_bd_ctsi_ag), ("#aj_af71_rv_321", cols_bd_ctsi_ag)]:
    sql_append = f"""
    INSERT INTO TABLAS.dbo.BD_CTSI ({cols})
    SELECT {cols}
    FROM {tmp_src}
    """
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_src}", res.rowcount)
for t in ["#aj_d9_hh2", "#aj_d9_bcos", "#aj_af71_cf_511_2", "#aj_af71_rv_511_2", "#aj_af71_cf_321", "#aj_af71_rv_321"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# CALCULA E IMPUTA VARIACIONES DEL VALOR NETO DEBIDO A VOLUMENES Y REC PRECIOS
work_conn.execute(text("DROP TABLE IF EXISTS #saldos_2"))
sql_saldos_2 = """
SELECT 'P' AS MONEDA, T1.[AÑO], T1.TRIM,
       T1.SECTOR,
       T1.C_CUENTA,
       'H' AS C_ENTRADA,
       SUM(CASE WHEN T1.C_ENTRADA = 'H' THEN T1.DATO * -1 ELSE T1.DATO END) AS DATO,
       CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END AS C_SCN,
       CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
            WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END AS N_SCN,
       'PS' AS FUENTE,
       '0010' AS PROC
INTO #saldos_2
FROM TABLAS.dbo.BD_CTSI t1
WHERE T1.C_CUENTA IN ('Rec Precio','Rec Precio Reaj','Rec Volumen')
GROUP BY T1.[AÑO], T1.TRIM, T1.SECTOR, T1.C_CUENTA,
         CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'B.10.2'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'B.10.3' END,
         CASE WHEN T1.C_CUENTA = 'Rec Volumen' THEN 'Variaciones del valor neto debidas a otras variaciones de volumen'
              WHEN T1.C_CUENTA LIKE '%Rec Precio%' THEN 'Variaciones del valor neto debidas a ganancias/pérdidas por tenencia nominales' END
"""
work_conn.execute(text(sql_saldos_2))


In [ ]:
# PROC APPEND BASE=TABLAS.BD_CTSI DATA=WORK.SALDOS_2 FORCE
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
sql_append_saldos_2 = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM #saldos_2
"""
res = work_conn.execute(text(sql_append_saldos_2))
_log("APPEND TABLAS.dbo.BD_CTSI desde #saldos_2", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #saldos_2"))


## S2_14_Final

Imputa los reajustes por precio de AF.29/AF.42/AF.522 de fondos de pensiones y cesantía como contribuciones sociales (D.61) pagadas por los hogares y como ajuste patrimonial (D.8) recíproco entre hogares y sector fondos de pensiones, acumulando las 4 tablas resultantes en la base de cuentas nacionales

*confianza: medium · verificador: approve · SAS: PROC SQL con GROUP BY CALCULATED + CASE WHEN agregado, encadenamiento de 4 CREATE TABLE derivadas y PROC APPEND FORCE x4 sobre tabla de BD*

In [ ]:
# ========= S2_14_Final =========
# COMPRIME TABLA PRINCIPAL (COMPRESS=YES es una opción de almacenamiento de SAS sin equivalente en SQL Server; se preserva el contenido de la tabla tal cual)
# Sin cambios de datos que aplicar: el DATA step solo reescribe tablas.BD_CTSI sobre sí misma


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d61_hh"))
# IMPUTA REAJUSTES DE AF29 Y AF42 DE LOS FONDOS DE PENSIONES Y CESANTÍA EN LAS CONTRIBUCIONES SOCIALES PAGADAS POR LOS HOGARES
sql_d61_hh = """
SELECT  'P' AS MONEDA,
        t1.[AÑO],
        t1.TRIM,
        511 AS SECTOR,
        '34' AS C_CAGENTE,
        'YG' AS C_CUENTA,
        'D' AS C_ENTRADA,
        SUM(CASE WHEN t1.C_SCN = 'AF.522' THEN t1.DATO * -1 ELSE t1.DATO END) AS DATO,
        'D.61' AS C_SCN,
        'Contribuciones sociales' AS N_SCN,
        'PS' AS FUENTE,
        '0001d' AS PROC
INTO #d61_hh
FROM TABLAS.dbo.BD_CTSI t1
WHERE (t1.C_ENTRADA = 'D' AND t1.C_SCN IN ('AF.29', 'AF.42') AND t1.FUENTE = 'CI' AND t1.C_CUENTA = 'Rec Precio Reaj' AND t1.SECTOR IN (34, 341, 342))
   OR (t1.C_ENTRADA = 'D' AND t1.C_SCN IN ('AF.522') AND t1.FUENTE = 'PS' AND t1.C_CUENTA = 'Rec Precio Reaj' AND t1.SECTOR IN (34, 341, 342))
GROUP BY t1.[AÑO], t1.TRIM,
         (CASE WHEN t1.C_ENTRADA = 'D' AND t1.C_SCN IN ('AF.29', 'AF.42') AND t1.FUENTE = 'CI' AND t1.C_CUENTA = 'Rec Precio Reaj' AND t1.SECTOR IN (34, 341, 342) THEN 'AF29_42' ELSE 'AF522' END)
"""
# GROUP BY replicado: el SAS agrupa por las 9 columnas CALCULATED, pero MONEDA/SECTOR/C_CAGENTE/C_CUENTA/C_ENTRADA/C_SCN/N_SCN son constantes literales dentro de cada rama del WHERE (no dependen de datos), así que agrupar además por AÑO/TRIM y por la rama de origen (CI/AF29_42 vs PS/AF522) reproduce exactamente los mismos grupos sin colapsar FUENTE/C_SCN
work_conn.execute(text(sql_d61_hh))
d61_hh = pd.read_sql(text("SELECT * FROM #d61_hh"), work_conn)
_log("d61_hh", d61_hh)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d8_hh"))
# IMPUTA REAJUSTES DE AF29 Y AF42 DE LOS FONDOS DE PENSIONES Y CESANTÍA EN D8 DE LOS HOGARES
sql_d8_hh = """
SELECT  t1.MONEDA,
        t1.[AÑO],
        t1.TRIM,
        t1.SECTOR,
        t1.C_CAGENTE,
        t1.C_CUENTA,
        'H' AS C_ENTRADA,
        t1.DATO,
        'D.8' AS C_SCN,
        'Ajuste por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
        t1.FUENTE,
        '0001e' AS PROC
INTO #d8_hh
FROM #d61_hh t1
"""
work_conn.execute(text(sql_d8_hh))
d8_hh = pd.read_sql(text("SELECT * FROM #d8_hh"), work_conn)
_log("d8_hh", d8_hh)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d61_fp"))
# IMPUTA REAJUSTES DE AF29 Y AF42 DE LOS FONDOS DE PENSIONES Y CESANTÍA EN LAS CONTRIBUCIONES SOCIALES EN SECTOR FONDOS DE PENSIONES
sql_d61_fp = """
SELECT  t1.MONEDA,
        t1.[AÑO],
        t1.TRIM,
        34 AS SECTOR,
        '511' AS C_CAGENTE,
        t1.C_CUENTA,
        'H' AS C_ENTRADA,
        t1.DATO,
        t1.C_SCN,
        t1.N_SCN,
        t1.FUENTE,
        '0001f' AS PROC
INTO #d61_fp
FROM #d61_hh t1
"""
work_conn.execute(text(sql_d61_fp))
d61_fp = pd.read_sql(text("SELECT * FROM #d61_fp"), work_conn)
_log("d61_fp", d61_fp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #d8_fp"))
# IMPUTA REAJUSTES DE AF29 Y AF42 DE LOS FONDOS DE PENSIONES Y CESANTÍA EN D8 DE LOS HOGARES
sql_d8_fp = """
SELECT  t1.MONEDA,
        t1.[AÑO],
        t1.TRIM,
        t1.SECTOR,
        t1.C_CAGENTE,
        t1.C_CUENTA,
        'D' AS C_ENTRADA,
        t1.DATO,
        'D.8' AS C_SCN,
        'Ajuste por la variación en la participación neta de los hogares en los fondos de pensiones' AS N_SCN,
        t1.FUENTE,
        '0001g' AS PROC
INTO #d8_fp
FROM #d61_fp t1
"""
work_conn.execute(text(sql_d8_fp))
d8_fp = pd.read_sql(text("SELECT * FROM #d8_fp"), work_conn)
_log("d8_fp", d8_fp)


In [ ]:
# APPEND server-side de las 4 tablas temporales a TABLAS.BD_CTSI (PROC APPEND ... FORCE), en el mismo orden del SAS
cols_bd_ctsi = "MONEDA, [AÑO], TRIM, SECTOR, C_CAGENTE, C_CUENTA, C_ENTRADA, DATO, C_SCN, N_SCN, FUENTE, PROC"
for tmp_name in ["#d61_hh", "#d8_hh", "#d61_fp", "#d8_fp"]:
    sql_append = f"""
INSERT INTO TABLAS.dbo.BD_CTSI ({cols_bd_ctsi})
SELECT {cols_bd_ctsi}
FROM {tmp_name}
"""
    res = work_conn.execute(text(sql_append))
    _log(f"APPEND TABLAS.dbo.BD_CTSI desde {tmp_name}", res.rowcount)


In [ ]:
# DROP TABLE D61_HH, D8_HH, D61_FP, D8_FP (limpieza de temporales de sesión)
for tmp_name in ["#d61_hh", "#d8_hh", "#d61_fp", "#d8_fp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {tmp_name}"))
